# Why does CCMP change path attribution?

This notebook tests whether suppressing messages outside selected paths is sufficient to increase those paths' attribution. It runs inference and gradients only.

All implementation is visible below as normal Python and `%%writefile` cells. There are no encoded source blobs or runtime source decoding.

Merged-graph checkpoints are preferred together with their matching `_test_hyb` graph and field scorer. If no merged checkpoint exists under `outputs/sir4_hyb/`, the notebook prints an explicit fallback and uses the previous frame checkpoint.

The main contrast is `outside_suppress − off`. A positive value means that suppressing outside messages was sufficient to raise attribution for that fixed path in that example. Path weights are mean edge gradients, not probabilities or products of gates. Retrieval scores and ranks are reported separately.


## 1. Configuration

In [ ]:
# Edit only this block for the usual run.
DATASETS = ["sir4_biology", "sir4_cs"]
PREFER_MERGED = True
PRECISIONS = ['bfloat16', 'float32']
PROTECTION = "nodes_all_layers"
RESUME = True
ATOL, RTOL = 1e-5, 1e-4
TOP_PATHS = 5
BEAM_SIZE = 10

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import zipfile

os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = "/content/gfm-rag" + os.pathsep + os.environ.get("PYTHONPATH", "")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/cargo-gfmrag"
SCIGRAPHIR_ROOT = "/content/scigraphir"
DATA_ROOT = f"{SCIGRAPHIR_ROOT}/retriever/data"
S4 = f"{SCIGRAPHIR_ROOT}/experiments"
KGDIR = f"{SCIGRAPHIR_ROOT}/retriever"
RUNS = "/content/runs"
OP_MODEL = "/content/qwen3"
OP_SLUG = "_content-qwen3"
NEED_ENGINE = True
os.environ["SCIGRAPHIR_ROOT"] = SCIGRAPHIR_ROOT
sys.path.insert(0, SCIGRAPHIR_ROOT)
os.makedirs(RUNS, exist_ok=True)

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ.setdefault("HF_TOKEN", "")

env = dict(os.environ, SCIGRAPHIR_ROOT=SCIGRAPHIR_ROOT, PYTHONUNBUFFERED="1")


def sh(command, cwd, extra=None, log=None, check=True):
    """Run a command and stream its output."""
    started = time.time()
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=dict(env, **(extra or {})),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    log_stream = open(log, "w") if log else None
    for raw_line in iter(process.stdout.readline, b""):
        line = raw_line.decode("utf-8", "replace")
        print(line, end="")
        if log_stream:
            log_stream.write(line)
            log_stream.flush()
    process.wait()
    if log_stream:
        log_stream.close()
    print(f"[{time.time() - started:.0f}s, exit {process.returncode}]")
    if check and process.returncode:
        raise RuntimeError("Command failed: " + " ".join(map(str, command)))
    return process.returncode


def sir4_spec(field):
    pair = "physics+biology" if field in ("physics", "biology") else "cs+matsci"
    warm = "biology" if field in ("physics", "biology") else "cs"
    dataset = f"sir4_{field}"
    return {
        "frame": f"{dataset}_test_v16sc",
        "sem": {
            "field": (f"outputs/{dataset}/semantic", dataset),
            "frame": (f"outputs/sir4_zeroshot/semantic_sir4_{warm}", f"sir4_{warm}"),
        },
        "arms": [(
            "frame_ccmp",
            f"outputs/sir4_zeroshot/scigraphir_{pair}_qwenmlp_ccmp_e10_b2",
            "frame", "frame", True,
        )],
    }


SPEC = {f"sir4_{field}": sir4_spec(field)
        for field in ("cs", "biology", "physics", "matsci")}
assert DATASETS and all(dataset in SPEC for dataset in DATASETS)
assert all(precision in ("bfloat16", "float32") for precision in PRECISIONS)
assert PROTECTION in ("nodes_all_layers", "sender_layer")
assert BEAM_SIZE >= TOP_PATHS >= 1
print("Datasets:", DATASETS)
print("Precisions:", PRECISIONS)
print("Prefer merged checkpoints:", PREFER_MERGED)


## 2. Restore data bundles

In [ ]:
# Restore the selected corpora under /content/scigraphir.
FORCE_UNPACK = False
marker = Path(SCIGRAPHIR_ROOT) / ".unpacked.json"
available = set(json.loads(marker.read_text())) if marker.exists() and not FORCE_UNPACK else set()
missing = [dataset for dataset in DATASETS if dataset not in available]

if missing:
    cache_directory = Path(SCIGRAPHIR_ROOT) / "outputs/caches"
    parked_cache = Path("/content/_cargo_caches_keep")
    if available:
        for dataset in missing:
            archive = Path(DRIVE) / f"{dataset}_bundle.zip"
            assert archive.is_file() and zipfile.is_zipfile(archive), f"missing or corrupt {archive}"
            with zipfile.ZipFile(archive) as bundle:
                bundle.extractall(SCIGRAPHIR_ROOT)
            print("unpacked", archive.name)
    else:
        if cache_directory.exists():
            if parked_cache.exists():
                shutil.rmtree(parked_cache)
            shutil.move(cache_directory, parked_cache)
        cargo_directory = Path(SCIGRAPHIR_ROOT)
        if cargo_directory.exists():
            shutil.rmtree(cargo_directory)
        cargo_directory.mkdir(parents=True)
        for dataset in DATASETS:
            archive = Path(DRIVE) / f"{dataset}_bundle.zip"
            assert archive.is_file() and zipfile.is_zipfile(archive), f"missing or corrupt {archive}"
            with zipfile.ZipFile(archive) as bundle:
                bundle.extractall(SCIGRAPHIR_ROOT)
            print("unpacked", archive.name)
        if parked_cache.exists():
            cache_directory.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(parked_cache, cache_directory)
    marker.write_text(json.dumps(sorted(available | set(DATASETS))))
else:
    print("bundles already unpacked:", sorted(available))


## 3. Install the runtime

In [ ]:
# Install the adapted GFM-RAG checkout and runtime dependencies.
archive = Path(DRIVE) / "gfm-rag-adapted.zip"
assert archive.is_file() and zipfile.is_zipfile(archive), f"missing or corrupt {archive}"
engine_directory = Path("/content/gfm-rag")
if engine_directory.exists():
    shutil.rmtree(engine_directory)
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall("/content")
assert (engine_directory / "gfmrag").is_dir(), "archive did not create /content/gfm-rag"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
                str(engine_directory)], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch-geometric", "sentence-transformers", "transformers", "hydra-core",
    "omegaconf", "easydict", "ninja", "faiss-cpu", "pymetis", "wandb",
    "tqdm", "numpy", "pandas", "python-dotenv", "langchain-community",
], check=True)

import torch
assert torch.cuda.is_available(), "Select an A100 or L4 GPU runtime"
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name())


## 4. Actual source code used by this notebook

Each following cell writes one visible source file. You can search, edit and review it directly.

In [ ]:
%%writefile /content/scigraphir/scigraphir_paths.py
"""
scigraphir_paths.py -- one place that decides WHICH corpus a pipeline script is
working on, and where that corpus's caches live.

THE PROBLEM THIS SOLVES. Five scripts in the construction pipeline had the
TOMATO corpus baked in as the literal string "tomato", and their caches were
keyed by SPLIT ALONE:

    sciafford/cache/frames_doc.jsonl          <- which corpus?
    retriever/probes/cache/probes_test.jsonl
    outputs/caches/op_emb/test_doc.npy

Point those scripts at a second corpus and a leftover TOMATO `test_doc.npy`
loads silently: the shape check passes whenever the two corpora happen to have a
similar document count, and the run produces plausible, meaningless numbers. No
exception, no warning. That is the failure this module exists to prevent.

THE CONTRACT. With the default dataset "tomato" every path returned here is
byte-identical to the hardcoded string it replaced, so existing caches still
resolve and published TOMATO numbers stay reproducible. Any other dataset name
additionally scopes every cache into a subdirectory named after it, so two
corpora can never share a cache file.

    DATASET=tomato   ->  .../cache/frames_doc.jsonl          (unchanged)
    DATASET=sir4_cs  ->  .../cache/sir4_cs/frames_doc_test.jsonl

Scripts call `set_dataset()` once after parsing args, then use the helpers.

    import sys, os
    sys.path.insert(0, os.path.expanduser("$SCIGRAPHIR_ROOT"))
    from scigraphir_paths import set_dataset, corpus_dir, frames_path

    set_dataset(args.dataset)
    raw = corpus_dir(args.split)              # .../data/sir4_cs_test/raw
"""
from __future__ import annotations

import os

# Repo root. SCIGRAPHIR_ROOT lets the same scripts run somewhere that is not this
# laptop -- Colab unzips the bundle to /content/scigraphir, where $SCIGRAPHIR_ROOT
# does not exist. Unset, the default is byte-identical to the hardcoded string
# it replaced, so nothing about a local run changes.
CARGO = os.environ.get("SCIGRAPHIR_ROOT") or os.path.expanduser("$SCIGRAPHIR_ROOT")
KG = f"{CARGO}/retriever"
V16 = f"{CARGO}/sciafford"

#: Corpus currently being processed. "tomato" reproduces every legacy path.
DATASET = os.environ.get("SCIGRAPHIR_DATASET", "tomato")


def set_dataset(name: str | None) -> str:
    """Set the active corpus. `None` or "" leaves the current value alone."""
    global DATASET
    if name:
        DATASET = name
    return DATASET


def is_legacy() -> bool:
    """True when paths must stay byte-identical to the pre-refactor strings."""
    return DATASET == "tomato"


def _scoped(root: str) -> str:
    """Cache root, with a per-dataset subdirectory for anything but TOMATO."""
    d = root if is_legacy() else f"{root}/{DATASET}"
    os.makedirs(d, exist_ok=True)
    return d


# --------------------------------------------------------------------------
# corpora and graphs
# --------------------------------------------------------------------------
def corpus_name(split: str) -> str:
    """Directory name of a raw corpus, e.g. `tomato_test` / `sir4_cs_test`."""
    return f"{DATASET}_{split}"


def corpus_dir(split: str) -> str:
    """Absolute path to a raw corpus directory (holds raw/documents.json)."""
    return f"{KG}/data/{corpus_name(split)}"


def graph_name(split: str, suffix: str = "v16sc") -> str:
    """Directory name of a built graph, e.g. `sir4_cs_train_v16sc`."""
    return f"{DATASET}_{split}_{suffix}"


def graph_dir(split: str, suffix: str = "v16sc") -> str:
    return f"{KG}/data/{graph_name(split, suffix)}"


# --------------------------------------------------------------------------
# caches
# --------------------------------------------------------------------------
def frames_path(side: str, split: str) -> str:
    """Extracted-frame cache. `side` is "doc" or "query".

    TOMATO's test frames are stored WITHOUT a split suffix and its train frames
    WITH one -- an asymmetry from when only a test split existed. That is
    preserved exactly for TOMATO and dropped for every other dataset, where the
    split is always in the name.
    """
    root = _scoped(f"{V16}/cache")
    if is_legacy() and split == "test":
        return f"{root}/frames_{side}.jsonl"
    return f"{root}/frames_{side}_{split}.jsonl"


def graph_cache_dir() -> str:
    """Scratch for graph-construction caches, e.g. `ds_{split}_emb_{type}.npy`.

    Those were keyed by split alone too. A SIR-4 build silently loaded TOMATO's
    concept embeddings from `ds_test_emb_method.npy`, which is part of why a
    contaminated build produced TOMATO-shaped output without complaining.
    """
    return _scoped(f"{V16}/cache")


def probes_path(split: str) -> str:
    """LLM probe cache used by the operator's S and M terms."""
    return f"{_scoped(f'{KG}/probes/cache')}/probes_{split}.jsonl"


def emb_dir() -> str:
    """BGE embedding cache (`{split}_doc.npy`, `{split}_query_{hash}.npy`, ...).

    The most dangerous of the three: filenames carry only the split, and a
    document matrix from the wrong corpus fails no assertion that the callers
    make.
    """
    return _scoped(f"{CARGO}/outputs/caches/op_emb")


def add_dataset_arg(ap) -> None:
    """Attach the standard `--dataset` flag to an argparse parser."""
    ap.add_argument(
        "--dataset", default=os.environ.get("SCIGRAPHIR_DATASET", "tomato"),
        help="corpus to operate on (default tomato; e.g. sir4_cs). "
             "Anything but 'tomato' also scopes every cache under this name.")


def banner() -> str:
    return (f"[scigraphir_paths] dataset={DATASET} "
            f"{'(legacy TOMATO paths)' if is_legacy() else '(scoped caches)'}")


In [ ]:
%%writefile /content/scigraphir/retriever/eval/operator_scorer.py
"""
eval/operator.py — the CARGO dense OPERATOR on TOMATO (faithful port of
archive/make_operator_scorer_notebook.py, sections 5-7).

    dense = q . d                          (selected encoder; query instruction on queries)
    S     = sum_probes [cos(probe, d)]+    (probe-sum)
    M     = max_probes [cos(probe, d)]+    (probe-max)
    dem   = leave-one-out probe-sum popularity   (label-free anti-hub denominator)
    S_op  = w0 . z(dense) + w1 . z(S / dem^b) + w2 . z(M / dem^b)

(w0, w1, w2, b) are fit on a TRAIN slice by InfoNCE and selected on a held-out TRAIN dev slice,
then applied to TEST. We report dense retrieval and S_op side by side, so the delta isolates the
probe + anti-hub contribution over the selected dense encoder. NO graph, NO API key.

Embeddings are cached to outputs/caches/op_emb/ (encode once; re-fits are instant).
Reads probes from probes/cache/probes_{train,test}.jsonl (run gen_probes.py first).

Run (Mac solo, or Colab GPU — auto-detects device):
    python eval/operator.py --train_fit 2500 --dev 600
"""
import argparse, hashlib, json, os, sys
import numpy as np
from tqdm import tqdm

# SCIGRAPHIR_ROOT so this runs off-laptop (Colab unzips to /content/scigraphir). Unset,
# it resolves to exactly the path this replaced.
_ROOT = os.environ.get("SCIGRAPHIR_ROOT") or os.path.expanduser("$SCIGRAPHIR_ROOT")
BASE = f"{_ROOT}/retriever"
BGE_QI = "Represent this sentence for searching relevant passages: "
# The Qwen3-Embedding model card's own example task, verbatim. It replaced a
# hand-written task description that named cross-domain transfer explicitly.
# Reason: BGE is run with its stock model-card instruction, so a tuned Qwen3
# instruction made the two dense baselines non-comparable, and any margin over
# Qwen3 partly measured prompt engineering rather than method.
QWEN_QI = ("Instruct: Given a web search query, retrieve relevant passages that "
           "answer the query\nQuery:")
DEFAULT_MODEL = "BAAI/bge-large-en-v1.5"
KS = [1, 5, 10, 25, 50, 100]

# Corpus + cache resolution lives in one module so the pipeline scripts cannot
# drift. Default dataset "tomato" reproduces every path this replaced. The
# embedding cache is the dangerous one: `{split}_doc.npy` carries no corpus
# name, so a stale TOMATO matrix would load into a SIR-4 run and pass the only
# check `cached_encode` makes (row count).
sys.path.insert(0, _ROOT)
from scigraphir_paths import (add_dataset_arg, banner, corpus_dir, emb_dir,  # noqa: E402
                         probes_path, set_dataset)


def model_slug(name):
    """Return a cache suffix so embeddings from different encoders cannot mix."""
    if name == DEFAULT_MODEL:
        return ""
    import re
    return "_" + re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def query_instruction(name):
    """Use the retrieval instruction intended for the selected encoder."""
    return QWEN_QI if "qwen" in name.lower() else BGE_QI


def qi_tag(instruct):
    """Instruction fingerprint, for the QUERY cache key only.

    WITHOUT THIS, CHANGING AN INSTRUCTION DOES NOTHING. The query cache is keyed
    on which query ids are in the split plus the model slug, and cached_encode
    validates row count and nothing else -- so a new instruction reloads the
    previous instruction's embeddings and reports them as the new result. Silent,
    and indistinguishable from "the instruction did not matter".

    Documents and probes are encoded with no instruction at all, so only the
    query cache needs this.
    """
    return "_i" + hashlib.md5(instruct.encode()).hexdigest()[:6]


def load_split(split):
    """Return doc_ids, doc_texts, queries(list of dict id/question/gold/stratum), probes{id:[...]}."""
    root = os.path.join(corpus_dir(split), "raw")
    corpus = json.load(open(os.path.join(root, "documents.json")))           # {doc_id: text}
    queries = json.load(open(os.path.join(root, f"{split}.json")))
    probes = {}
    for line in open(probes_path(split)):
        r = json.loads(line); probes[r["id"]] = r["probes"]
    return list(corpus), corpus, queries, probes


def cached_encode(model, texts, tag, instruct="", batch=64, chunk=1500):
    """Encode in chunks, releasing MPS unified-memory buffers between chunks so the encoder
    can't snowball into swap (the Apple-Silicon MPS accumulation bug). Cached to .npy."""
    import torch
    p = os.path.join(emb_dir(), f"{tag}.npy")   # emb_dir() makes the dir
    if os.path.exists(p):
        e = np.load(p)
        if e.shape[0] == len(texts):
            print(f"  [emb] loaded {tag}: {e.shape}")
            return e
        print(f"  [emb] {tag} stale ({e.shape[0]} != {len(texts)}), re-encoding")
    print(f"  [emb] encoding {tag}: {len(texts)} texts (chunk={chunk}) ...")
    parts = []
    for i in tqdm(range(0, len(texts), chunk), desc=f"  enc {tag}"):
        sub = [instruct + t for t in texts[i:i + chunk]]
        v = model.encode(sub, normalize_embeddings=True, batch_size=batch, show_progress_bar=False)
        parts.append(np.asarray(v, dtype=np.float32))
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()             # release GPU buffers so memory stays bounded
    e = np.concatenate(parts, 0)
    np.save(p, e)
    print(f"  [emb] saved {tag}: {e.shape}")
    return e


def signals(qe, de, pe, own, NQ):
    """dense/S/M/dem exactly as the operator notebook (section 5)."""
    dense = qe @ de.T
    ND = de.shape[0]
    S = np.zeros((NQ, ND), np.float32); M = np.zeros((NQ, ND), np.float32)
    for i in tqdm(range(NQ), desc="  signals", leave=False):
        idx = np.where(own == i)[0]
        if len(idx):
            H = np.clip(pe[idx] @ de.T, 0, None)          # [n_probes_i, ND], ReLU
            S[i] = H.sum(0); M[i] = H.max(0)
    dem = np.clip(S.sum(0, keepdims=True) - S, 1e-6, None)  # leave-one-out popularity
    return dense.astype(np.float32), S, M, dem.astype(np.float32)


def build_signals(model, model_name, split, q_subset=None):
    """Encode (cached) and assemble signals for a split. q_subset = list of query dicts to use."""
    import hashlib
    doc_ids, corpus, queries, probes = load_split(split)
    if q_subset is not None:
        queries = q_subset
    qkey = hashlib.md5("|".join(q["id"] for q in queries).encode()).hexdigest()[:8]  # cache by WHICH queries
    slug = model_slug(model_name)
    docpos = {d: i for i, d in enumerate(doc_ids)}
    de = cached_encode(model, [corpus[d] for d in doc_ids], f"{split}_doc{slug}")
    qi = query_instruction(model_name)
    qe = cached_encode(model, [q["question"] for q in queries],
                       f"{split}_query_{qkey}{slug}{qi_tag(qi)}", instruct=qi)
    flat, own = [], []
    for i, q in enumerate(queries):
        for pr in probes.get(q["id"], []):
            flat.append(pr); own.append(i)
    pe = cached_encode(model, flat, f"{split}_probe_{qkey}{slug}")
    own = np.array(own)
    gold_idx = [[docpos[g] for g in (q.get("supporting_documents") or []) if g in docpos] for q in queries]
    strat = [q.get("stratum") for q in queries]
    sig = signals(qe, de, pe, own, len(queries))
    return sig, gold_idx, strat


def ndcg10(order, gold):
    g = set(gold)
    if not g:
        return 0.0
    dcg = sum(1 / np.log2(r + 2) for r, d in enumerate(order[:10]) if d in g)
    idcg = sum(1 / np.log2(r + 2) for r in range(min(len(g), 10)))
    return dcg / idcg if idcg else 0.0


def evaluate(score, gold_idx, strat, name):
    order = np.argsort(-score, 1)
    buck = {"all": [], "same": [], "cross": []}
    for i, gs in enumerate(gold_idx):
        if not gs:
            continue
        o = list(order[i][:200]); g = set(gs)
        row = ({k: len(set(o[:k]) & g) / len(g) for k in KS}, ndcg10(o, gs))
        buck["all"].append(row)
        if strat[i] in ("same", "cross"):
            buck[strat[i]].append(row)
    print(f"\n=== {name} ===")
    print(f"{'stratum':6} " + " ".join(f'R@{k:<4}' for k in KS) + " nDCG  n")
    out = {}
    for s in ("all", "cross", "same"):
        r = buck[s]
        if not r:
            continue
        mr = {k: 100 * np.mean([x[0][k] for x in r]) for k in KS}
        nd = 100 * np.mean([x[1] for x in r])
        out[s] = mr
        print(f"{s:6} " + " ".join(f'{mr[k]:5.1f}' for k in KS) + f" {nd:4.1f} {len(r)}")
    return out


def main():
    import torch, torch.nn as nn
    from sentence_transformers import SentenceTransformer

    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default=DEFAULT_MODEL)
    ap.add_argument("--train_fit", type=int, default=2500, help="train queries used to fit the 4 params")
    ap.add_argument("--dev", type=int, default=600, help="held-out train queries for model selection")
    ap.add_argument("--epochs", type=int, default=401)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--cpu", action="store_true", help="force CPU encoding (steady, no MPS swap blowup)")
    add_dataset_arg(ap)
    a = ap.parse_args()
    set_dataset(a.dataset)
    print(banner())

    dev_t = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    enc_dev = "cpu" if a.cpu else dev_t
    print(f"compute={dev_t} | encode={enc_dev} | model={a.model}")
    model = SentenceTransformer(a.model, device=enc_dev)
    model.max_seq_length = 512

    # ---- TRAIN: sample fit + dev queries (shared train corpus), build signals ----
    _, _, train_q, _ = load_split("train")
    rng = np.random.default_rng(a.seed); rng.shuffle(train_q)
    dev_q = train_q[:a.dev]
    fit_q = train_q[a.dev:a.dev + a.train_fit]
    print(f"\n[train] fit={len(fit_q)} dev={len(dev_q)} queries (corpus shared)")
    (fD, fS, fM, fDe), fgold, _ = build_signals(model, a.model, "train", q_subset=fit_q)
    (dD, dS, dM, dDe), dgold, _ = build_signals(model, a.model, "train", q_subset=dev_q)

    # ---- operator (4 params), InfoNCE on fit, select on dev nDCG@10 ----
    DEVt = dev_t
    T = lambda x: torch.tensor(x, device=DEVt)
    def zr(X): return (X - X.mean(1, keepdim=True)) / (X.std(1, keepdim=True) + 1e-6)
    class Operator(nn.Module):
        def __init__(s):
            super().__init__(); s.logbeta = nn.Parameter(torch.zeros(1))
            s.w = nn.Parameter(torch.tensor([1., 1., 0.2]))
        def forward(s, dense, ssum, smax, dem):
            degb = dem.clamp_min(1e-6) ** torch.exp(s.logbeta)
            return s.w[0] * zr(dense) + s.w[1] * zr(ssum / degb) + s.w[2] * zr(smax / degb)

    fDt, fSt, fMt, fDet = T(fD), T(fS), T(fM), T(fDe)
    dDt, dSt, dMt, dDet = T(dD), T(dS), T(dM), T(dDe)
    qi, gi = [], []
    for i, gs in enumerate(fgold):
        for g in gs:
            qi.append(i); gi.append(g)
    qi, gi = torch.tensor(qi, device=DEVt), torch.tensor(gi, device=DEVt)

    m = Operator().to(DEVt); opt = torch.optim.Adam(m.parameters(), lr=0.05)
    best, bsd, stale = -1, None, 0
    # Stop once the dev score stops improving. The 401 gradient steps are cheap
    # (four parameters), but each evaluation scores 600 dev queries against the
    # whole corpus and argsorts each one in Python -- that is the real cost, and
    # it runs 21 times. The fit typically plateaus by ~ep 40. Set OP_PATIENCE=0
    # to disable and run the full schedule.
    PATIENCE = int(os.environ.get("OP_PATIENCE", "3"))
    for ep in range(a.epochs):
        m.train(); opt.zero_grad()
        loss = -torch.log_softmax(m(fDt, fSt, fMt, fDet), 1)[qi, gi].mean()
        loss.backward(); opt.step()
        if ep % 20 == 0:
            with torch.no_grad():
                nd = np.mean([ndcg10(list((-m(dDt, dSt, dMt, dDet).cpu().numpy()[i]).argsort()), dgold[i])
                              for i in range(len(dgold)) if dgold[i]])
            tag = ""
            if nd > best:
                best = nd; bsd = {k: v.clone() for k, v in m.state_dict().items()}; tag = "*"
                stale = 0
            else:
                stale += 1
            print(f"ep{ep:3d} loss {loss.item():.3f} dev nDCG@10 {nd:.3f} "
                  f"beta {torch.exp(m.logbeta).item():.2f} w {m.w.detach().cpu().numpy().round(2)} {tag}")
            if PATIENCE and stale >= PATIENCE:
                print(f"[early stop] ep{ep}: no dev gain for {stale} evals "
                      f"({stale * 20} epochs); keeping best {best:.3f}")
                break
    m.load_state_dict(bsd)
    print(f"\nLEARNED beta={torch.exp(m.logbeta).item():.3f} w={m.w.detach().cpu().numpy().round(3)} best dev {best:.3f}")

    # SAVE the fitted parameters. They used to be printed and nothing else, while
    # FusionGraphReasoner warm-starts w and beta from hardcoded W_INIT/BETA_INIT
    # fitted on TOMATO. So a SIR-4 fusion run began from TOMATO's calibration and
    # this fit was thrown away. Written per-dataset so one corpus can never read
    # another's, same contract as scigraphir_paths.
    # ALSO SCOPED BY ENCODER. w and beta are fitted against one encoder's score
    # distributions, so a Qwen3 fit and a BGE fit are different calibrations of the
    # same four parameters. Sharing a filename would let a refit destroy the values
    # that already-trained checkpoints were warm-started from, with nothing to show
    # it happened. Empty slug for the default encoder keeps existing paths intact.
    sfx = ("" if a.dataset == "tomato" else f"_{a.dataset}") + model_slug(a.model)
    params = {"dataset": a.dataset, "encoder": a.model,
              "w": [round(float(x), 6) for x in m.w.detach().cpu().numpy()],
              "beta": round(float(torch.exp(m.logbeta).item()), 6),
              "dev_ndcg10": round(float(best), 6),
              "train_fit": a.train_fit, "dev": a.dev}
    pp = os.path.join(BASE, "eval", f"operator_params{sfx}.json")
    json.dump(params, open(pp, "w"), indent=1)
    print(f"[params] wrote {pp}")
    print(f"[params] fusion warm start -> W_INIT={tuple(params['w'])} BETA_INIT={params['beta']}")

    # ---- TEST: signals on test corpus, apply operator, score by stratum ----
    print("\n[test] building signals ...")
    (teD, teS, teM, teDe), tegold, testrat = build_signals(model, a.model, "test")
    with torch.no_grad():
        S_op = m(T(teD), T(teS), T(teM), T(teDe)).cpu().numpy()
    summary = {}
    dense_name = f"{a.model} (dense)"
    operator_name = f"Operator ({a.model}; probes + anti-hub)"
    summary[dense_name] = evaluate(teD, tegold, testrat, dense_name)
    summary[operator_name] = evaluate(S_op, tegold, testrat, operator_name)
    out = os.path.join(BASE, "eval", f"operator_results{sfx}.json")
    json.dump(summary, open(out, "w"), indent=2)
    # save per-query operator rankings (for downstream operator-seeded PPR)
    te_doc_ids, _, te_q, _ = load_split("test")
    preds = []
    for i, q in enumerate(te_q):
        order = np.argsort(-S_op[i])[:100]
        preds.append({"id": q["id"], "stratum": q.get("stratum"),
                      "supporting_documents": q.get("supporting_documents", []),
                      "predictions": {"document": [[te_doc_ids[j], float(S_op[i, j])] for j in order]}})
    json.dump(preds, open(os.path.join(BASE, "eval", f"predictions_operator{sfx}.json"), "w"))
    print(f"wrote eval/predictions_operator{sfx}.json  encoder={a.model}")
    bo = summary[operator_name]; bg = summary[dense_name]
    print(f"\nHEADLINE cross R@10:  operator {bo['cross'][10]:.1f}  vs  dense {bg['cross'][10]:.1f}  "
          f"(HyDE 25.3) | wrote {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/scigraphir/experiments/eval/semantic_scorer.py
"""
semantic_scorer.py -- controlled comparison of semantic scorer architectures on
SIR-4. No graph, no reasoner, no fusion.

    arm "current"  s = w0 z(r_dir) + w1 z(S/p^beta) + w2 z(M/p^beta)
                   S = sum_j H_qdj, M = max_j H_qdj      (the handcrafted summaries)

    arm "attention"  s = sum_v a_v x_v,  a = softmax_v(MLP([x_v, t_v]))
                     over the direct view and every valid hypothetical answer, so
                     the weights sum to 1. The pooled score is a weighted average,
                     which keeps it on the same scale as the direct view no matter
                     how many hypothetical answers a query has.

    arm "gated"      s = g_dir x_dir + sum_j g_qdj x_qdj,  g = sigmoid(MLP([x, t]))
                     Independent gates that do not sum to 1, so several strong
                     views accumulate. Kept selectable via --arms; note the summed
                     views carry several times the spread of the direct view, so
                     this arm has to learn the channel balance the other two get
                     from normalisation.

    arm "dualsetmlp" h_j   = [x_j, MLP(x_j)]
                       u_all = sum_j h_j
                       u_sel = sum_j softmax_j(tau x_j) h_j
                       s     = linear([x_dir,u_all,u_sel]) + MLP([x_dir,u_all,u_sel])

                     This is the automatic raw-view scorer: additive pooling
                     preserves evidence accumulation, a learned positive
                     temperature provides monotonic selective pooling, and the
                     final residual MLP is not constrained to a weighted average.

All trained arms use the SAME selected loss, frozen encoder outputs, fit/dev
split, and development metric. In a comparison run, the only intended change is
the scorer architecture.

THE OBJECTIVE. `--loss operator` (DEFAULT) is operator_scorer.py's exactly, so the
`current` arm reproduces the scorer as it is fitted everywhere else in the project
and the only thing varying between arms is the architecture.

`--loss fixed` is the multi-gold correction. The operator objective's denominator
runs over the whole corpus INCLUDING the query's other golds; TOMATO has one gold
per query so that is harmless there, but SIR-4 matsci has 3.95, so every step
pushes gold 1 up by pushing golds 2..4 down. Both are available and every output
filename carries which one produced it, so the two can be compared rather than
argued about.

WHY THE HYPOTHETICAL VIEWS SHARE ONE SCALE. Standardising each view separately
sets every view's spread to exactly 1, which is precisely the quantity that says
whether a hypothetical answer discriminates between papers at all. A view that
gives every paper the same score would arrive at the gate looking as confident as
one that separates them. So the views are CENTRED individually and divided by a
single shared scale, which preserves their relative spreads.

Run (after the operator cell has produced the embedding caches):
    python3 eval/semantic_scorer.py --dataset sir4_matsci --model /content/qwen3
"""
from __future__ import annotations

import argparse
import hashlib
import importlib.util
import json
import math
import os
import sys
import time

import numpy as np
# Torch at module level, unlike the rest of this file's function-local imports:
# DeepSetsScorer must inherit nn.Module at class-definition time so that
# train()/eval() actually gate its dropout. Every entry point here needs torch
# within seconds anyway.
import torch as _torch
import torch.nn as _nn

_ROOT = os.environ.get("SCIGRAPHIR_ROOT") or os.path.expanduser("$SCIGRAPHIR_ROOT")
sys.path.insert(0, _ROOT)
from scigraphir_paths import (add_dataset_arg, banner, corpus_dir, emb_dir,  # noqa: E402
                         probes_path, set_dataset)

EPS = 1e-6            # popularity floor and normalisation floor, as specified
KS_REPORT = (1, 3, 5, 10, 25, 100)


def _op_module():
    """Load operator_scorer as a module so the embedding cache keys are IDENTICAL.

    The encoder, the query instruction, the instruction fingerprint and the
    filename layout all live there. Re-implementing any of them here would
    produce a cache MISS at best, and at worst a hit on a file built under a
    different instruction -- which is the same silent-wrong-baseline failure the
    rest of this pipeline is armoured against.
    """
    p = f"{_ROOT}/retriever/eval/operator_scorer.py"
    spec = importlib.util.spec_from_file_location("operator_scorer", p)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m


# --------------------------------------------------------------------------
# stage 1: raw semantic measurements, cached on disk
# --------------------------------------------------------------------------
def build_inputs(op, model_name, split, cache_root, force=False):
    """Materialise dense, H, mask and total_S for one split.

    H is [Q, Jmax, D] float16 in a MEMMAP, never a resident tensor: at CS scale it
    is 5,445 x 8 x 20,203 = 1.8 GB, which is fine on disk and fine to slice a
    minibatch out of, and not fine to hold on a GPU alongside activations.
    """
    doc_ids, corpus, queries, probes = op.load_split(split)
    D, Q = len(doc_ids), len(queries)
    qids = [q["id"] for q in queries]
    qkey = hashlib.md5("|".join(qids).encode()).hexdigest()[:8]
    dockey = hashlib.md5("|".join(doc_ids).encode()).hexdigest()[:8]
    slug = op.model_slug(model_name)
    qi = op.query_instruction(model_name)

    # THE CACHE KEY CARRIES EVERY INPUT THAT CHANGES THE NUMBERS. Dataset and
    # split alone are what the rest of this repo used to key on, and a matrix
    # from the wrong corpus passes a row-count check. Document ORDER is in here
    # too: the arrays are column-indexed by it, so a reordered corpus would
    # misalign every score with nothing to show for it.
    meta = {"dataset": os.path.basename(corpus_dir(split)).rsplit("_", 1)[0],
            "split": split, "encoder": model_name, "query_instruct": qi,
            "qkey": qkey, "dockey": dockey, "Q": Q, "D": D}
    os.makedirs(cache_root, exist_ok=True)
    tag = f"{split}_{qkey}_{dockey}{slug}"
    mpath = f"{cache_root}/semantic_inputs_{tag}.json"
    hpath = f"{cache_root}/semantic_H_{tag}.f16"

    if os.path.exists(mpath) and not force:
        got = json.load(open(mpath))
        stale = {k: (got.get(k), v) for k, v in meta.items() if got.get(k) != v}
        assert not stale, f"stale semantic cache {mpath}: {stale}"
        assert os.path.exists(hpath), f"manifest without matrix: {hpath}"
        H = np.memmap(hpath, np.float16, "r", shape=(Q, got["Jmax"], D))
        z = np.load(f"{cache_root}/semantic_side_{tag}.npz", allow_pickle=True)
        print(f"[sem] loaded cached inputs {tag}: H{H.shape} f16")
        return dict(H=H, dense=z["dense"], mask=z["mask"], total_S=z["total_S"],
                    doc_ids=doc_ids, queries=queries, meta=got)

    # Encode only on a genuine cache miss.  Importing sentence-transformers can
    # initialise a multi-gigabyte model stack, so a cache-only experiment must
    # not require it (and must not fail merely because that optional stack is
    # unavailable on the evaluation machine).
    dpath = f"{emb_dir()}/{split}_doc{slug}.npy"
    qpath = f"{emb_dir()}/{split}_query_{qkey}{slug}{op.qi_tag(qi)}.npy"
    ppath = f"{emb_dir()}/{split}_probe_{qkey}{slug}.npy"
    flat, own = [], []
    for i, q in enumerate(queries):
        for pr in probes.get(q["id"], []):
            flat.append(pr)
            own.append(i)
    if not all(os.path.exists(p) for p in (dpath, qpath, ppath)):
        from sentence_transformers import SentenceTransformer
        print(f"[sem] embeddings absent for {split}; encoding with {model_name}")
        enc = SentenceTransformer(model_name)
        enc.max_seq_length = 512
        op.cached_encode(enc, [corpus[d] for d in doc_ids], f"{split}_doc{slug}")
        op.cached_encode(enc, [q["question"] for q in queries],
                         f"{split}_query_{qkey}{slug}{op.qi_tag(qi)}", instruct=qi)
        op.cached_encode(enc, flat, f"{split}_probe_{qkey}{slug}")
    de = np.load(dpath).astype(np.float32)
    qe = np.load(qpath).astype(np.float32)
    pe = np.load(ppath).astype(np.float32)
    assert de.shape[0] == D and qe.shape[0] == Q and pe.shape[0] == len(flat), (
        f"embedding rows {de.shape[0]}/{qe.shape[0]}/{pe.shape[0]} != {D}/{Q}/{len(flat)}")

    own = np.asarray(own)
    counts = np.bincount(own, minlength=Q) if len(own) else np.zeros(Q, int)
    Jmax = int(counts.max()) if len(own) else 1
    meta["Jmax"] = Jmax
    print(f"[sem] {split}: Q={Q} D={D} Jmax={Jmax} "
          f"(views/query min {counts.min()} mean {counts.mean():.1f})")
    assert counts.min() > 0, "a query has no hypothetical answers; regenerate probes"

    # dense in row chunks -- one Q x D float32 is 440 MB at CS scale
    dense = np.empty((Q, D), np.float16)
    for i in range(0, Q, 512):
        dense[i:i + 512] = (qe[i:i + 512] @ de.T).astype(np.float16)

    H = np.memmap(hpath, np.float16, "w+", shape=(Q, Jmax, D))
    mask = np.zeros((Q, Jmax), bool)
    total_S = np.zeros(D, np.float64)
    t0 = time.time()
    for i in range(Q):
        idx = np.where(own == i)[0]
        h = np.clip(pe[idx] @ de.T, 0, None).astype(np.float32)      # [j_i, D]
        H[i, :len(idx)] = h.astype(np.float16)
        mask[i, :len(idx)] = True
        total_S += h.sum(0)                       # popularity over the WHOLE split
        if i % 500 == 0:
            print(f"  [sem] {split} {i}/{Q}  ({time.time() - t0:.0f}s)", flush=True)
    H.flush()

    np.savez_compressed(f"{cache_root}/semantic_side_{tag}.npz",
                        dense=dense, mask=mask, total_S=total_S.astype(np.float32))
    json.dump(meta, open(mpath, "w"), indent=1)
    print(f"[sem] wrote {hpath} ({os.path.getsize(hpath)/1e6:.0f} MB)")
    return dict(H=np.memmap(hpath, np.float16, "r", shape=(Q, Jmax, D)),
                dense=dense, mask=mask, total_S=total_S.astype(np.float32),
                doc_ids=doc_ids, queries=queries, meta=meta)


def gold_matrix(queries, doc_ids):
    """Padded gold indices + validity mask, and a per-query python list."""
    pos = {d: i for i, d in enumerate(doc_ids)}
    lists = []
    for q in queries:
        gs = [pos[g] for g in (q.get("supporting_documents") or []) if g in pos]
        lists.append(gs)
    G = max(1, max(len(g) for g in lists))
    idx = np.zeros((len(lists), G), np.int64)
    val = np.zeros((len(lists), G), np.float32)
    for i, gs in enumerate(lists):
        idx[i, :len(gs)] = gs
        val[i, :len(gs)] = 1.0
    return idx, val, lists


# --------------------------------------------------------------------------
# stage 2: the two architectures
# --------------------------------------------------------------------------
class Pop:
    """Per-document popularity for the anti-hub division, in one of three modes.

    loo        (total_S - S) per query: the historical estimate. TRANSDUCTIVE --
               a test paper's popularity is computed from the OTHER test queries'
               hypothetical answers, so it cannot run on one query alone.
    bank       mean ReLU cosine against the TRAIN-split answer bank. Query-
               independent: one [D] vector per corpus, computable for unseen
               papers from their embedding plus a frozen train artifact.
    predicted  a small MLP distilled from the bank targets. Fully local: needs
               only the paper's own embedding at inference.

    The object rides in the argument slot that used to carry total_S, so every
    scorer stays a pure function of (H, mask, dense, pop).
    """

    def __init__(self, mode, total_S=None, vec=None, predictor=None,
                 doc_emb=None, target=None):
        self.mode, self.total_S, self.vec = mode, total_S, vec
        # `joint` keeps the predictor LIVE inside the scoring path, so the
        # retrieval gradient reaches it. `predicted` freezes its output to a
        # vector instead. Same network, different training regime.
        self.predictor, self.doc_emb, self.target = predictor, doc_emb, target

    def parameters(self):
        """Predictor parameters, so the trainer can optimise them jointly."""
        return list(self.predictor.parameters()) if self.mode == "joint" else []

    def state(self):
        """Predictor weights, or None. Part of the checkpoint under joint training.

        The scorer and the predictor are ONE model: restoring the scorer to its
        best epoch while the predictor sits at whatever the last epoch left it
        would evaluate a pair that never existed during training.
        """
        if self.mode != "joint":
            return None
        return {k: v.detach().cpu().clone() for k, v in self.predictor.state_dict().items()}

    def load_state(self, st):
        if st is None or self.mode != "joint":
            return
        dev = next(self.predictor.parameters()).device
        self.predictor.load_state_dict({k: v.to(dev) for k, v in st.items()})

    def aux_loss(self):
        """Anchor p_hat to the bank targets in log space.

        Without it the retrieval loss is free to repurpose the predictor as extra
        scorer capacity -- it would stop meaning "how generally matchable is this
        paper" and start meaning "whatever lowers the ranking loss", which is a
        different model wearing the same name. The log keeps a handful of very
        popular papers from dominating.
        """
        if self.mode != "joint" or self.target is None:
            return 0.0
        p = self.predictor(self.doc_emb)
        return ((_torch.log(EPS + p) - _torch.log(EPS + self.target)) ** 2).mean()

    @staticmethod
    def from_data(data, device, variant=""):
        """`variant` selects an alternative popularity stored on the same split.

        The mlp arm carries its own LEARNED popularity as part of the proposal, so
        one run needs two sources live at once: the baseline's leave-one-out and
        the predictor's. They are stored side by side rather than in two runs.
        """
        vk = f"pop_vec{variant}"
        mk = f"pop_mode{variant}"
        if data.get(mk, "loo") == "loo":
            return Pop("loo", total_S=_torch.as_tensor(data["total_S"], device=device))
        return Pop(data[mk], vec=_torch.as_tensor(data[vk], device=device).float())

    def per_query(self, S):
        """[B, D] popularity, given the query's own answer-sum S [B, D]."""
        if self.mode == "loo":
            return (self.total_S.unsqueeze(0) - S).clamp_min(EPS)
        if self.mode == "joint":
            # Recomputed every step and DIFFERENTIABLE: this is the whole point.
            return self.predictor(self.doc_emb).clamp_min(EPS).unsqueeze(0).expand_as(S)
        # Query-independent: the same vector for every query. Includes the
        # query's own answers when the query came from the train split, which is
        # the definition of the bank -- one contribution among thousands.
        return self.vec.unsqueeze(0).clamp_min(EPS).expand_as(S)


def _views(H, mask, dense, pop, beta):
    """Popularity adjustment + the two normalisations. Returns x_dir, x_hyp, Ht.

    H     [B, J, D] float32   raw ReLU'd hypothetical-answer matches
    mask  [B, J]    float32   1 for a real answer, 0 for padding
    dense [B, D]    float32   direct question-paper cosine
    """
    import torch
    m3 = mask.unsqueeze(-1)                                   # [B, J, 1]
    S = (H * m3).sum(1)                                       # [B, D]
    p = pop.per_query(S)
    Ht = H / p.unsqueeze(1).pow(beta)
    Ht = Ht * m3                                              # padding contributes 0

    D = H.shape[-1]
    nj = mask.sum(1).clamp_min(1.0)                           # [B] real answers
    mu = Ht.sum(-1, keepdim=True) / D                         # [B, J, 1]
    c = (Ht - mu) * m3                                        # centre each answer
    # ONE shared scale across this query's valid answers, so a view that barely
    # separates papers stays narrow instead of being inflated to spread 1.
    sig = torch.sqrt((c * c).sum((1, 2)) / (nj * D) + EPS)    # [B]
    x_hyp = c / (sig.view(-1, 1, 1) + EPS)
    x_dir = (dense - dense.mean(1, keepdim=True)) / (dense.std(1, keepdim=True) + EPS)
    return x_dir, x_hyp * m3, Ht


def _set_views(H, mask, dense, pop, beta):
    """Return direct and hypothetical-answer inputs for learned set pooling.

    Unlike ``_views``, this uses one mean and one scale for the complete
    hypothetical-answer set of a query.  Consequently an answer that barely
    separates papers stays weak, and relative offsets between answers are not
    erased before the set network sees them.
    """
    import torch
    m3 = mask.unsqueeze(-1)
    S = (H * m3).sum(1)
    p = pop.per_query(S)
    adjusted = H / p.unsqueeze(1).pow(beta)
    adjusted = adjusted * m3

    D = H.shape[-1]
    n = (mask.sum(1) * D).clamp_min(1.0)
    mu = adjusted.sum((1, 2)) / n
    centred = (adjusted - mu.view(-1, 1, 1)) * m3
    scale = torch.sqrt((centred * centred).sum((1, 2)) / n + EPS)
    x_hyp = centred / (scale.view(-1, 1, 1) + EPS)
    x_dir = (dense - dense.mean(1, keepdim=True)) / (dense.std(1, keepdim=True) + EPS)
    return x_dir, x_hyp * m3


class MatchabilityPredictor(_nn.Module):
    """Query-Independent Document Matchability Estimator.

        p_hat(d) = softplus( g_eta(E(d)) ),   g_eta: dim -> hidden -> 1

    Distils the train-answer bank into a function of the paper embedding alone,
    so popularity at inference needs nothing but the paper itself: no other test
    queries, no bank matmul, no lookup table. Fitted on log targets so a few
    extremely popular papers cannot dominate the loss.
    """

    def __init__(self, dim, hidden=64):
        super().__init__()
        self.net = _nn.Sequential(_nn.Linear(dim, hidden), _nn.GELU(),
                                  _nn.Linear(hidden, 1))

    def forward(self, E):
        return _nn.functional.softplus(self.net(E)).squeeze(-1)


def bank_popularity(bank_emb, doc_emb, chunk=4096):
    """p_bank[d] = mean over bank answers of ReLU(cos(h, d)). Pure numpy, chunked."""
    out = np.zeros(doc_emb.shape[0], np.float64)
    for i in range(0, bank_emb.shape[0], chunk):
        out += np.clip(bank_emb[i:i + chunk] @ doc_emb.T, 0, None).sum(0)
    return (out / max(bank_emb.shape[0], 1)).astype(np.float32)


def fit_matchability(doc_emb, targets, device, seed=0, hidden=64, lr=1e-3,
                     weight_decay=1e-2, epochs=200, patience=10):
    """Fit p_hat to the bank targets with log-MSE, early-stopped on held-out docs."""
    N = doc_emb.shape[0]
    rng = np.random.default_rng(seed)
    perm = rng.permutation(N)
    n_val = max(1, N // 10)
    va, fi = perm[:n_val], perm[n_val:]
    X = _torch.as_tensor(doc_emb, device=device)
    y = _torch.log(EPS + _torch.as_tensor(targets, device=device))
    m = MatchabilityPredictor(doc_emb.shape[1], hidden).to(device)
    groups = [{"params": [p for p in m.parameters() if p.ndim >= 2],
               "weight_decay": weight_decay},
              {"params": [p for p in m.parameters() if p.ndim < 2],
               "weight_decay": 0.0}]
    opt = _torch.optim.AdamW(groups, lr=lr)
    best, best_sd, stale = float("inf"), None, 0
    for ep in range(epochs):
        m.train()
        for bi in range(0, len(fi), 1024):
            sel = _torch.as_tensor(fi[bi:bi + 1024], device=device)
            loss = ((_torch.log(EPS + m(X[sel])) - y[sel]) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        m.eval()
        with _torch.no_grad():
            v = float(((_torch.log(EPS + m(X[va])) - y[va]) ** 2).mean())
        if v < best - 1e-6:
            best, best_sd, stale = v, {k: t.clone() for k, t in m.state_dict().items()}, 0
        else:
            stale += 1
            if stale >= patience:
                break
    m.load_state_dict(best_sd)
    m.eval()
    with _torch.no_grad():
        pred = m(X).cpu().numpy()
        corr = float(np.corrcoef(np.log(EPS + pred), np.log(EPS + targets))[0, 1])
    print(f"[matchability] fitted on {len(fi)} docs, held-out {n_val}: "
          f"log-MSE {best:.4f}, log-corr {corr:.3f} (ep {ep + 1})")
    return m, {"heldout_logmse": round(best, 6), "log_corr": round(corr, 4),
               "epochs_run": ep + 1, "hidden": hidden}


class CurrentScorer:
    """w0 z(dense) + w1 z(S/p^b) + w2 z(M/p^b) -- the handcrafted sum and max."""

    name = "current"

    def __init__(self, device):
        import torch
        import torch.nn as nn
        self.w = nn.Parameter(torch.tensor([1.0, 1.0, 0.2], device=device))
        self.logbeta = nn.Parameter(torch.zeros((), device=device))

    def parameters(self):
        return [self.w, self.logbeta]

    def state(self):
        return {"w": self.w.detach().cpu().tolist(),
                "beta": float(self.logbeta.detach().exp().cpu())}

    def load(self, st):
        import torch
        with torch.no_grad():
            self.w.copy_(torch.tensor(st["w"], device=self.w.device))
            self.logbeta.copy_(torch.tensor(math.log(st["beta"]), device=self.w.device))

    def __call__(self, H, mask, dense, pop):
        import torch
        beta = self.logbeta.exp()
        m3 = mask.unsqueeze(-1)
        S_raw = (H * m3).sum(1)
        degb = pop.per_query(S_raw).pow(beta)
        # max over REAL answers only: padding is 0 and every H is >= 0, so a
        # padded slot would silently act as a floor of zero on an all-zero row.
        M_raw = H.masked_fill(m3 == 0, -1.0).max(1).values.clamp_min(0.0)

        def z(x):
            return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + EPS)

        return (self.w[0] * z(dense) + self.w[1] * z(S_raw / degb)
                + self.w[2] * z(M_raw / degb))


class DenseScorer:
    """The dense baseline: rank by the query-document cosine alone.

    No hypothetical answers, no popularity term, no parameters, no training. It is
    here so the table shows what the hypothetical-answer machinery buys over plain
    query-document similarity in the SAME encoder space -- without it, a reader
    cannot tell whether `current` and `attention` are close to each other because
    both are good or because neither adds anything over the encoder.
    """

    name = "dense"

    def __init__(self, device):
        pass

    def parameters(self):
        return []

    def state(self):
        return {"note": "no parameters; ranks by cos(E(q), E(d))"}

    def load(self, st):
        pass

    def __call__(self, H, mask, dense, pop):
        return (dense - dense.mean(1, keepdim=True)) / (dense.std(1, keepdim=True) + EPS)

class DeepSetsScorer(_nn.Module):
    """Compact Deep Sets, ~62 parameters. No sum, no max, no handcrafted summaries.

        z_qdj = phi([x_hyp_qdj, x_dir_qd])        phi: 2 -> 4 -> 4
        u_qd  = masked mean_j z_qdj               over VALID (and kept) answers
        s     = rho([x_dir_qd, u_qd])             rho: 5 -> 4 -> 1

    Pairing each answer with the direct score inside phi lets the network encode
    interactions ("a strong answer AND a weak direct match") before pooling, and
    the masked MEAN keeps u on one scale regardless of how many answers a query
    has -- the 610-parameter version summed, which tied its output scale to J.

    REGULARISATION IS THE POINT of this version; the large one overfit (best
    train loss of all arms, worse test).
      * 15% whole-answer dropout, training only: an entire hypothetical answer is
        dropped for a query, with the SAME mask for every paper of that query, so
        the model cannot rely on any single answer existing. The masked mean
        renormalises over the kept answers, so no 1/(1-p) scaling is needed.
      * weight decay handled by train_arm (matrices only), dev early stopping,
        and three seeds handled by the --seeds loop in main.

    THIS CLASS INHERITS nn.Module AND THE TRAINERS CALL train()/eval(). The other
    scorer classes are plain objects, so dropout inserted there would stay active
    during evaluation; hasattr guards in train_arm/dev_ndcg/score_rows make the
    mode switch a no-op for them and real for this one.

    If dropout removes every answer of a query (possible at J=1), u is zero and
    the score falls back to a function of x_dir alone, which is the right
    degradation. Permutation invariant: shared phi, symmetric mean.
    """

    name = "deepsets"

    def __init__(self, device, hidden=4, p_drop=0.15):
        super().__init__()
        self.p_drop = float(p_drop)
        self.phi = _nn.Sequential(_nn.Linear(2, hidden), _nn.GELU(),
                                  _nn.Linear(hidden, hidden))
        self.rho = _nn.Sequential(_nn.Linear(hidden + 1, hidden), _nn.GELU(),
                                  _nn.Linear(hidden, 1))
        self.logbeta = _nn.Parameter(_torch.zeros(()))
        self.to(device)

    def state(self):
        return {"sd": {k: v.detach().cpu().tolist() for k, v in self.state_dict().items()},
                "beta": float(self.logbeta.detach().exp().cpu()),
                "p_drop": self.p_drop}

    def load(self, st):
        import torch
        dev = self.logbeta.device
        self.load_state_dict({k: torch.tensor(v, device=dev) for k, v in st["sd"].items()})

    def forward(self, H, mask, dense, pop):
        import torch
        x_dir, x_hyp, _ = _views(H, mask, dense, pop, self.logbeta.exp())
        B, J, D = x_hyp.shape
        m = mask
        if self.training and self.p_drop > 0:
            # One mask per (query, answer), shared across all D papers: the unit
            # being dropped is the ANSWER, not a (paper, answer) cell.
            keep = (torch.rand(B, J, device=m.device) >= self.p_drop).float()
            m = m * keep
        inp = torch.stack([x_hyp, x_dir.unsqueeze(1).expand(B, J, D)], -1)  # [B,J,D,2]
        z = self.phi(inp) * m.view(B, J, 1, 1)                              # [B,J,D,h]
        u = z.sum(1) / m.sum(1).clamp(min=1.0).view(B, 1, 1)                # masked mean
        feat = torch.cat([x_dir.unsqueeze(-1), u], -1)                      # [B,D,h+1]
        return self.rho(feat).squeeze(-1)


class SetMLPScorer(_nn.Module):
    """A small, automatic, permutation-invariant semantic scorer.

    For every adjusted hypothetical-answer match x_j, one shared MLP produces a
    learned representation.  Summing these representations is the standard
    Deep Sets invariant; it is not a precomputed semantic feature.  A final
    residual MLP maps the pooled set and direct query match to one paper score.

    The first per-view channel is an identity path.  It gives gradients a stable
    route and makes the initial model a strong direct-plus-pooled retriever.  All
    output weights, the nonlinear set features, and beta remain trainable.  No
    maximum, top-k statistic, rank position, or manually weighted operator input
    is computed.
    """

    name = "setmlp"

    def __init__(self, device, hidden=8, p_drop=0.0):
        super().__init__()
        hidden = int(hidden)
        assert hidden >= 2
        self.hidden = hidden
        self.p_drop = float(p_drop)
        # h-1 nonlinear channels plus one unmodified identity channel.
        self.phi = _nn.Sequential(
            _nn.Linear(1, hidden), _nn.GELU(), _nn.Linear(hidden, hidden - 1))
        self.linear = _nn.Linear(hidden + 1, 1, bias=False)
        self.residual = _nn.Sequential(
            _nn.Linear(hidden + 1, hidden), _nn.GELU(),
            _nn.Dropout(0.10), _nn.Linear(hidden, 1))
        self.logbeta = _nn.Parameter(_torch.zeros(()))

        # Stable residual learning: epoch zero is direct + the identity set
        # channel.  This is an initialisation only; every coefficient is learned.
        with _torch.no_grad():
            self.linear.weight.zero_()
            self.linear.weight[0, 0] = 1.0       # direct query view
            self.linear.weight[0, 1] = 1.0       # pooled identity view
            self.residual[-1].weight.zero_()
            self.residual[-1].bias.zero_()
        self.to(device)

    def state(self):
        return {"sd": {k: v.detach().cpu().tolist() for k, v in self.state_dict().items()},
                "beta": float(self.logbeta.detach().exp().cpu()),
                "hidden": self.hidden, "p_drop": self.p_drop}

    def load(self, st):
        dev = self.logbeta.device
        self.load_state_dict({k: _torch.tensor(v, device=dev) for k, v in st["sd"].items()})

    def forward(self, H, mask, dense, pop):
        import torch
        x_dir, x_hyp = _set_views(H, mask, dense, pop, self.logbeta.exp())
        B, J, D = x_hyp.shape
        m = mask
        if self.training and self.p_drop > 0:
            keep = (torch.rand(B, J, device=m.device) >= self.p_drop).float()
            # Never erase the entire set: keep the first valid answer if a rare
            # all-dropped row occurs.  The rule is independent of paper scores.
            empty = (keep * m).sum(1) == 0
            if empty.any():
                first = m.float().argmax(1)
                keep[empty, first[empty]] = 1.0
            m = m * keep

        nonlinear = self.phi(x_hyp.unsqueeze(-1))
        per_view = torch.cat([x_hyp.unsqueeze(-1), nonlinear], -1)
        pooled = (per_view * m.view(B, J, 1, 1)).sum(1)              # [B,D,h]

        # Calibrate every learned pooled channel across this query's candidate
        # corpus.  This prevents a high-variance channel from winning merely by
        # scale and gives the final MLP comparable inputs.
        pooled = ((pooled - pooled.mean(1, keepdim=True)) /
                  (pooled.std(1, keepdim=True) + EPS))
        feat = torch.cat([x_dir.unsqueeze(-1), pooled], -1)
        return (self.linear(feat) + self.residual(feat)).squeeze(-1)


class DualSetMLPScorer(_nn.Module):
    """Set MLP with complementary additive and learned-selective pooling.

    ``u_all`` accumulates evidence from every answer. ``u_sel`` uses a learned
    positive temperature and a softmax *inside the set representation* to focus
    on informative answers.  The final score is unconstrained: unlike view-attention as the
    scorer itself, it is not a convex average and therefore does not cap
    evidence accumulation.  Both pools are permutation invariant and neither
    computes a handcrafted maximum.
    """

    name = "dualsetmlp"

    def __init__(self, device, hidden=8, p_drop=0.0):
        super().__init__()
        hidden = int(hidden)
        assert hidden >= 2
        self.hidden = hidden
        self.p_drop = float(p_drop)
        self.phi = _nn.Sequential(
            _nn.Linear(1, hidden), _nn.GELU(), _nn.Linear(hidden, hidden - 1))
        # A single positive temperature is enough to learn the continuum from
        # broad averaging (small tau) to strongest-view selection (large tau).
        # Monotonic selection generalises better than another free MLP here.
        self.logtau = _nn.Parameter(_torch.tensor(math.log(5.0)))
        width = 1 + 2 * hidden
        self.linear = _nn.Linear(width, 1, bias=False)
        self.residual = _nn.Sequential(
            _nn.Linear(width, hidden), _nn.GELU(), _nn.Dropout(0.10),
            _nn.Linear(hidden, 1))
        self.logbeta = _nn.Parameter(_torch.zeros(()))
        with _torch.no_grad():
            self.linear.weight.zero_()
            self.linear.weight[0, 0] = 1.0
            self.linear.weight[0, 1] = 1.0
            self.residual[-1].weight.zero_()
            self.residual[-1].bias.zero_()
        self.to(device)

    def state(self):
        return {"sd": {k: v.detach().cpu().tolist() for k, v in self.state_dict().items()},
                "beta": float(self.logbeta.detach().exp().cpu()),
                "hidden": self.hidden, "p_drop": self.p_drop}

    def load(self, st):
        dev = self.logbeta.device
        self.load_state_dict({k: _torch.tensor(v, device=dev) for k, v in st["sd"].items()})

    def forward(self, H, mask, dense, pop):
        import torch
        x_dir, x_hyp = _set_views(H, mask, dense, pop, self.logbeta.exp())
        B, J, D = x_hyp.shape
        m = mask
        if self.training and self.p_drop > 0:
            keep = (torch.rand(B, J, device=m.device) >= self.p_drop).float()
            empty = (keep * m).sum(1) == 0
            if empty.any():
                first = m.float().argmax(1)
                keep[empty, first[empty]] = 1.0
            m = m * keep

        nonlinear = self.phi(x_hyp.unsqueeze(-1))
        per_view = torch.cat([x_hyp.unsqueeze(-1), nonlinear], -1)
        valid = m.view(B, J, 1)
        u_all = (per_view * valid.unsqueeze(-1)).sum(1)

        logits = self.logtau.exp() * x_hyp
        logits = logits.masked_fill(valid == 0, -float("inf"))
        attn = torch.softmax(logits, 1)
        u_sel = (attn.unsqueeze(-1) * per_view).sum(1)

        def corpus_z(u):
            return (u - u.mean(1, keepdim=True)) / (u.std(1, keepdim=True) + EPS)

        feat = torch.cat([x_dir.unsqueeze(-1), corpus_z(u_all), corpus_z(u_sel)], -1)
        return (self.linear(feat) + self.residual(feat)).squeeze(-1)


class AttentionScorer:
    """The thesis method (Ch.6 Eq 6.10-6.12), exactly as written.

        l_v = f_phi([x_v, t_v])         one shared MLP, t=0 direct, t=1 hypothetical
        a_v = softmax over ALL J+1 views (padded answers masked to -inf)
        s   = sum_v a_v x_v

    No temperature, no warm start, no separate channels: this arm exists to test
    the written method as written. Views are normalised by _views(), i.e. the
    shared-scale refinement (centre each answer, one scale per query) that the
    method adopted in place of per-view standardisation.

    Known structural properties, stated so the result is readable:
      * the score is a weighted average, so it cannot exceed the paper's own
        largest view -- evidence cannot accumulate across views;
      * averaging J partly-independent views shrinks the hypothetical mass's
        spread well below x_dir's, so the direct view tends to dominate.

    Permutation invariant: one MLP scores every view, softmax and the weighted
    sum are symmetric. The direct view is always valid, so no row is fully masked.
    """

    name = "attention"

    def __init__(self, device, hidden=8):
        import torch
        import torch.nn as nn
        self.mlp = nn.Sequential(nn.Linear(2, hidden), nn.GELU(),
                                 nn.Linear(hidden, 1)).to(device)
        self.logbeta = nn.Parameter(torch.zeros((), device=device))

    def parameters(self):
        return list(self.mlp.parameters()) + [self.logbeta]

    def state(self):
        return {"mlp": {k: v.detach().cpu().tolist() for k, v in self.mlp.state_dict().items()},
                "beta": float(self.logbeta.detach().exp().cpu())}

    def load(self, st):
        import torch
        dev = self.logbeta.device
        self.mlp.load_state_dict({k: torch.tensor(v, device=dev)
                                  for k, v in st["mlp"].items()})
        with torch.no_grad():
            self.logbeta.copy_(torch.tensor(math.log(st["beta"]), device=dev))

    def __call__(self, H, mask, dense, pop):
        import torch
        x_dir, x_hyp, _ = _views(H, mask, dense, pop, self.logbeta.exp())
        B, J, D = x_hyp.shape
        dev = mask.device
        views = torch.cat([x_dir.unsqueeze(1), x_hyp], 1)            # [B, 1+J, D]
        vmask = torch.cat([torch.ones(B, 1, device=dev), mask], 1)
        t = torch.cat([torch.zeros(1, device=dev),
                       torch.ones(J, device=dev)]).view(1, 1 + J, 1)
        inp = torch.stack([views, t.expand(B, 1 + J, D)], -1)        # [B, 1+J, D, 2]
        logit = self.mlp(inp).squeeze(-1)                            # [B, 1+J, D]
        logit = logit.masked_fill(vmask.unsqueeze(-1) == 0, -float("inf"))
        a = torch.softmax(logit, 1)
        return (a * views).sum(1)


class SortedMLPScorer:
    """One MLP on the SORTED vector of view scores. No sum, no max, no attention.

        input  = [ x_dir , sort_desc(x_hyp_1..J) padded to jmax , n_valid/jmax ]
        s      = MLP(input)

    NOTHING IS HANDCRAFTED. Sorting is not a summary statistic, it is a
    canonical ordering: it makes the input permutation invariant by construction
    while throwing away nothing. The MLP then learns whatever function of the
    score distribution it wants -- how much the best answer counts, whether the
    second one matters, whether the gap between them matters, how many answers
    need to fire. `sum` and `max` are not computed anywhere.

    IT GENERALISES SUMMARY-BASED SCORING, WHICH IS NOT THE SAME AS CONTAINING IT.
    `current` is w0 x_dir + w1 (sum_j x_hyp) + w2 (max_j x_hyp), and on this input
    the sum is the sum of the sorted coordinates, the max is the first sorted
    coordinate, and x_dir is coordinate 0 -- so a linear readout gets close. But
    `_views` centres each hypothetical answer SEPARATELY before the shared scale,
    so the max channel here is the largest CENTRED value, which is not the same
    quantity `current` computes from raw matches. The honest claim is that this
    arm exposes the complete ordered match profile instead of two fixed summaries
    of it, not that it reproduces the baseline exactly.

    NO SPREAD PROBLEM. Every coordinate gets its own free weight, so the model can
    set any balance between the direct match and the answers. Both previous arms
    failed on exactly this: gated summed the answers and they overwhelmed x_dir,
    attention averaged them and x_dir overwhelmed them.

    Padded slots sort to the end and are zeroed, and the valid count is supplied as
    a feature, so a query with three answers and one with eight are distinguishable
    without the padding masquerading as evidence.
    """

    name = "mlp"

    def __init__(self, device, jmax, hidden=16):
        import torch
        import torch.nn as nn
        self.jmax = int(jmax)
        self.net = nn.Sequential(nn.Linear(self.jmax + 2, hidden), nn.GELU(),
                                 nn.Linear(hidden, 1)).to(device)
        self.logbeta = nn.Parameter(torch.zeros((), device=device))

    def parameters(self):
        return list(self.net.parameters()) + [self.logbeta]

    def state(self):
        return {"net": {k: v.detach().cpu().tolist() for k, v in self.net.state_dict().items()},
                "jmax": self.jmax,
                "beta": float(self.logbeta.detach().exp().cpu())}

    def load(self, st):
        import torch
        dev = self.logbeta.device
        self.net.load_state_dict({k: torch.tensor(v, device=dev)
                                  for k, v in st["net"].items()})
        with torch.no_grad():
            self.logbeta.copy_(torch.tensor(math.log(st["beta"]), device=dev))

    def __call__(self, H, mask, dense, pop):
        import torch
        x_dir, x_hyp, _ = _views(H, mask, dense, pop, self.logbeta.exp())
        B, J, D = x_hyp.shape
        dev = x_hyp.device

        # Invalid answers sort to the end, then get zeroed by position. Comparing
        # against the fill value to find them would be fragile; the count is exact.
        fill = torch.finfo(x_hyp.dtype).min / 4
        xs, _ = torch.sort(x_hyp.masked_fill(mask.unsqueeze(-1) == 0, fill),
                           dim=1, descending=True)
        nv = mask.sum(1)                                            # [B]
        xs = xs * (torch.arange(J, device=dev).view(1, J) < nv.view(B, 1)).float().unsqueeze(-1)

        # One fixed input width, so a split whose Jmax differs cannot change the
        # layer shape. Truncating keeps the HIGHEST scores, which is the useful end.
        if J < self.jmax:
            xs = torch.cat([xs, torch.zeros(B, self.jmax - J, D, device=dev)], 1)
        elif J > self.jmax:
            xs = xs[:, :self.jmax]

        feat = torch.cat([x_dir.unsqueeze(-1),                       # [B, D, 1]
                          xs.permute(0, 2, 1),                       # [B, D, jmax]
                          (nv.view(B, 1, 1) / self.jmax).expand(B, D, 1)], -1)
        return self.net(feat).squeeze(-1)                            # [B, D]


class GatedScorer:
    """Adaptive gated view pooling. Independent sigmoid gates, no softmax."""

    name = "gated"

    def __init__(self, device, hidden=8):
        import torch.nn as nn
        self.mlp = nn.Sequential(nn.Linear(2, hidden), nn.GELU(),
                                 nn.Linear(hidden, 1)).to(device)
        import torch
        self.logbeta = nn.Parameter(torch.zeros((), device=device))

    def parameters(self):
        return list(self.mlp.parameters()) + [self.logbeta]

    def state(self):
        return {"mlp": {k: v.detach().cpu().tolist() for k, v in self.mlp.state_dict().items()},
                "beta": float(self.logbeta.detach().exp().cpu())}

    def load(self, st):
        import torch
        self.mlp.load_state_dict({k: torch.tensor(v, device=self.logbeta.device)
                                  for k, v in st["mlp"].items()})
        with torch.no_grad():
            self.logbeta.copy_(torch.tensor(math.log(st["beta"]),
                                            device=self.logbeta.device))

    def __call__(self, H, mask, dense, pop):
        import torch
        x_dir, x_hyp, _ = _views(H, mask, dense, pop, self.logbeta.exp())
        B, J, D = x_hyp.shape
        views = torch.cat([x_dir.unsqueeze(1), x_hyp], 1)            # [B, 1+J, D]
        vmask = torch.cat([torch.ones(B, 1, device=mask.device), mask], 1)
        t = torch.cat([torch.zeros(1, device=mask.device),
                       torch.ones(J, device=mask.device)]).view(1, 1 + J, 1)
        inp = torch.stack([views, t.expand(B, 1 + J, D)], -1)        # [B, 1+J, D, 2]
        g = torch.sigmoid(self.mlp(inp).squeeze(-1))                 # [B, 1+J, D]
        # SUM, not a weighted average: gates do not sum to 1, so a paper that
        # matches several views well accumulates past one that spikes on a single
        # view. A softmax here would cap every score at its own largest view.
        return (g * views * vmask.unsqueeze(-1)).sum(1)


# --------------------------------------------------------------------------
# stage 3: the corrected multi-gold loss, and dev nDCG@10
# --------------------------------------------------------------------------
def operator_loss(scores, gold_idx, gold_val):
    """operator_scorer.py's objective, reproduced exactly.

        -torch.log_softmax(S_op, 1)[qi, gi].mean()

    i.e. the mean over all (query, gold) PAIRS of -log_softmax(s)[gold], with the
    denominator running over the whole corpus. THIS IS THE DEFAULT, so the
    `current` arm here reproduces the operator as it is actually fitted everywhere
    else in the project and the comparison changes only the architecture.

    It differs from `multigold_loss` in two ways at once, which is worth knowing
    when reading a --loss ablation:
      1. a query's other golds stay inside each gold's denominator;
      2. weighting is per (query, gold) pair, so a 10-gold query counts ten times
         as much as a 1-gold one.
    """
    import torch
    lp = torch.log_softmax(scores, 1).gather(1, gold_idx)
    return -(lp * gold_val).sum() / gold_val.sum().clamp_min(1.0)


def multigold_loss(scores, gold_idx, gold_val):
    """-log( exp(s_g) / (exp(s_g) + sum_{d not gold} exp(s_d)) ), averaged.

    Golds are removed from EACH OTHER's denominator, then averaged within a query
    and equally across queries so a 10-gold query does not outweigh a 1-gold one.
    """
    import torch
    # scatter_ADD, not scatter_. Padded gold slots carry index 0, so a query whose
    # first real gold IS document 0 would have two writes racing at the same cell
    # and the -inf could be overwritten by the padding's value, silently leaving a
    # gold in its own denominator. Accumulating avoids the collision entirely.
    acc = torch.zeros_like(scores)
    acc.scatter_add_(1, gold_idx, gold_val)
    neg = scores.masked_fill(acc > 0, -float("inf"))
    neg_lse = torch.logsumexp(neg, 1)                            # [B]
    sg = scores.gather(1, gold_idx)                              # [B, G]
    per_gold = torch.logaddexp(sg, neg_lse.unsqueeze(1)) - sg
    per_query = (per_gold * gold_val).sum(1) / gold_val.sum(1).clamp_min(1.0)
    return per_query.mean()


def ndcg_at(order, golds, k):
    g = set(golds)
    if not g:
        return 0.0
    dcg = sum(1 / math.log2(r + 2) for r, d in enumerate(order[:k]) if d in g)
    idcg = sum(1 / math.log2(r + 2) for r in range(min(len(g), k)))
    return dcg / idcg if idcg else 0.0


def batches(n, bs, shuffle=False, rng=None):
    idx = np.arange(n)
    if shuffle:
        rng.shuffle(idx)
    for i in range(0, n, bs):
        yield idx[i:i + bs]


def score_rows(model, data, rows, device, qbatch, torch, pop=None):
    """Score a set of queries against the whole corpus, in query minibatches."""
    if hasattr(model, "eval"):
        model.eval()          # dropout must NOT fire at evaluation time
    out = np.empty((len(rows), data["H"].shape[-1]), np.float32)
    pop = Pop.from_data(data, device) if pop is None else pop
    for s in range(0, len(rows), qbatch):
        sel = rows[s:s + qbatch]
        H = torch.as_tensor(np.asarray(data["H"][sel], np.float32), device=device)
        mk = torch.as_tensor(data["mask"][sel].astype(np.float32), device=device)
        dn = torch.as_tensor(np.asarray(data["dense"][sel], np.float32), device=device)
        out[s:s + qbatch] = model(H, mk, dn, pop).detach().float().cpu().numpy()
    return out


def dev_ndcg_multi(model, data, rows, gold_lists, device, qbatch, torch, ks=(10, 100),
                   pop=None):
    """nDCG at several cutoffs from ONE scoring pass over the dev queries.

    Scoring the dev set is the expensive part of an epoch (every query against the
    whole corpus), so computing @10 and @100 with two calls would double it for a
    number that comes free from the same ranking.
    """
    with torch.no_grad():
        sc = score_rows(model, data, rows, device, qbatch, torch, pop)
    acc = {k: [] for k in ks}
    kmax = max(ks)
    for i, r in enumerate(rows):
        if not gold_lists[r]:
            continue
        order = list(np.argsort(-sc[i])[:kmax])
        for k in ks:
            acc[k].append(ndcg_at(order[:k], gold_lists[r], k))
    return tuple(float(np.mean(acc[k])) if acc[k] else 0.0 for k in ks)


def dev_ndcg(model, data, rows, gold_lists, device, qbatch, torch, k=10, pop=None):
    with torch.no_grad():
        sc = score_rows(model, data, rows, device, qbatch, torch, pop)
    vals = []
    for i, r in enumerate(rows):
        if gold_lists[r]:
            vals.append(ndcg_at(list(np.argsort(-sc[i])[:k]), gold_lists[r], k))
    return float(np.mean(vals)) if vals else 0.0


def train_fixed(model, tr, fit_rows, gold_idx, gold_val, device, a, torch,
                n_epochs, lr=None, seed=None):
    """Train for exactly n_epochs on fit_rows. No dev split, no early stopping.

    Used for the FINAL fit of a k-fold run, where the epoch count has already been
    chosen by the folds and there is no held-out data left to select on -- which is
    the point: every train query is in this fit.
    """
    pop = Pop.from_data(tr, device) if pop is None else pop
    # Under joint training the popularity predictor is part of the optimised
    # model, so its matrices belong in the decayed group like any other.
    trainable = list(model.parameters()) + pop.parameters()
    groups = [g for g in (
        {"params": [p for p in trainable if p.ndim >= 2],
         "weight_decay": a.weight_decay},
        {"params": [p for p in trainable if p.ndim < 2],
         "weight_decay": 0.0}) if g["params"]]
    opt = torch.optim.AdamW(groups, lr=a.lr if lr is None else lr)
    gi = torch.as_tensor(gold_idx, device=device)
    gv = torch.as_tensor(gold_val, device=device)
    rng = np.random.default_rng(a.seed if seed is None else seed)
    lossfn = operator_loss if a.loss == "operator" else multigold_loss
    hist = []
    for ep in range(n_epochs):
        if hasattr(model, "train"):
            model.train()
        losses = []
        for bi in batches(len(fit_rows), a.qbatch, True, rng):
            sel = fit_rows[bi]
            H = torch.as_tensor(np.asarray(tr["H"][sel], np.float32), device=device)
            mk = torch.as_tensor(tr["mask"][sel].astype(np.float32), device=device)
            dn = torch.as_tensor(np.asarray(tr["dense"][sel], np.float32), device=device)
            loss = lossfn(model(H, mk, dn, pop), gi[sel], gv[sel])
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(float(loss.detach()))
        hist.append({"epoch": ep + 1, "train_loss": float(np.mean(losses))})
        print(f"  [{model.name}] final ep{ep + 1:3d}/{n_epochs} "
              f"train {np.mean(losses):.4f}", flush=True)
    if hasattr(model, "eval"):
        model.eval()
    return hist


def kfold_epochs(build, tr, all_rows, gold_idx, gold_val, gold_lists, device, a,
                 torch, lr=None, seed=None):
    """Choose the epoch count by K-fold CV over EVERY train query.

    Each query serves in dev exactly once, so the selection signal spans the whole
    train split rather than one 300-query slice, and every query still contributes
    to fitting in K-1 of the K folds. Returns the median selected epoch, which is
    then used for a final fit on all of the data.
    """
    K = a.kfold
    rng = np.random.default_rng(a.seed if seed is None else seed)
    order = all_rows.copy()
    rng.shuffle(order)
    folds = np.array_split(order, K)
    picked = []
    for k in range(K):
        dev_k = folds[k]
        fit_k = np.concatenate([folds[j] for j in range(K) if j != k])
        m = build()
        _, _, hist, _, _ = train_arm(m, tr, fit_k, dev_k, gold_idx, gold_val,
                                     gold_lists, device, a, torch, lr=lr, seed=seed)
        key = "dev_loss" if a.select_on == "loss" else "dev_ndcg@10"
        best_ep = (min(hist, key=lambda r: r[key]) if a.select_on == "loss"
                   else max(hist, key=lambda r: r[key]))["epoch"]
        picked.append(best_ep)
        print(f"  [fold {k + 1}/{K}] fit {len(fit_k)} dev {len(dev_k)} "
              f"-> epoch {best_ep}")
    E = int(np.median(picked))
    print(f"  [kfold] selected epochs per fold {picked} -> median {E}")
    return max(E, 1), picked


def dev_loss(model, data, rows, gold_idx, gold_val, device, qbatch, torch, lossfn,
             pop=None):
    """The training objective evaluated on held-out dev queries.

    WHY SELECT ON THIS RATHER THAN dev nDCG@10. nDCG@10 with binary relevance is a
    STEP function: a query's score moves only when a gold crosses a rank boundary
    inside the top 10, so most parameter updates change it by exactly zero and the
    rest change it in jumps. Taking the best over ~30 epochs x 3 seeds of a chunky
    signal on the same 300 queries selects the checkpoint that got luckiest on
    those queries, not the one that generalises -- which is why the physics run
    inverted: the arm with the WORST dev nDCG won on test, and the arm with the
    best dev nDCG lost. The loss is continuous, moves every step, and is the
    quantity actually being optimised. nDCG is still computed and reported.
    """
    if hasattr(model, "eval"):
        model.eval()
    pop = Pop.from_data(data, device) if pop is None else pop
    gi = torch.as_tensor(gold_idx, device=device)
    gv = torch.as_tensor(gold_val, device=device)
    tot, n = 0.0, 0
    with torch.no_grad():
        for s in range(0, len(rows), qbatch):
            sel = rows[s:s + qbatch]
            H = torch.as_tensor(np.asarray(data["H"][sel], np.float32), device=device)
            mk = torch.as_tensor(data["mask"][sel].astype(np.float32), device=device)
            dn = torch.as_tensor(np.asarray(data["dense"][sel], np.float32), device=device)
            # The loss fns average within a call, so weight by rows to recover the
            # overall mean when the last minibatch is short.
            tot += float(lossfn(model(H, mk, dn, pop), gi[sel], gv[sel])) * len(sel)
            n += len(sel)
    return tot / max(n, 1)


def train_arm(model, tr, fit_rows, dev_rows, gold_idx, gold_val, gold_lists,
              device, a, torch, lr=None, seed=None, pop=None):
    # WEIGHT DECAY ON WEIGHT MATRICES ONLY (ndim >= 2). Biases and the scalar
    # calibration parameters are excluded deliberately: decaying them is not
    # regularisation, it is a prior. log_tau -> 0 means tau -> 1, logbeta -> 0
    # means beta -> 1, and shrinking `current`'s w toward 0 flattens the corpus
    # softmax and inflates the loss the optimiser is trying to reduce. The stated
    # purpose is to discourage sharp MLP functions, and this is the scoping that
    # actually does that and nothing else.
    pop = Pop.from_data(tr, device) if pop is None else pop
    # Under joint training the popularity predictor is part of the optimised
    # model, so its matrices belong in the decayed group like any other.
    trainable = list(model.parameters()) + pop.parameters()
    groups = [g for g in (
        {"params": [p for p in trainable if p.ndim >= 2],
         "weight_decay": a.weight_decay},
        {"params": [p for p in trainable if p.ndim < 2],
         "weight_decay": 0.0}) if g["params"]]
    opt = torch.optim.AdamW(groups, lr=a.lr if lr is None else lr)
    gi = torch.as_tensor(gold_idx, device=device)
    gv = torch.as_tensor(gold_val, device=device)
    rng = np.random.default_rng(a.seed if seed is None else seed)
    lossfn = operator_loss if a.loss == "operator" else multigold_loss
    on_loss = a.select_on == "loss"
    sel_k = 100 if a.select_on == "ndcg100" else 10
    # Lower is better for loss, higher for nDCG, so track the score to BEAT in the
    # sign the criterion wants and compare one way.
    best_sel, best_state, stale, hist = float("inf"), None, 0, []
    best_nd, best_pop = 0.0, None
    for ep in range(a.epochs):
        # Real for nn.Module arms (dropout on/off), no-op for the plain classes.
        if hasattr(model, "train"):
            model.train()
        losses = []
        for bi in batches(len(fit_rows), a.qbatch, True, rng):
            sel = fit_rows[bi]
            H = torch.as_tensor(np.asarray(tr["H"][sel], np.float32), device=device)
            mk = torch.as_tensor(tr["mask"][sel].astype(np.float32), device=device)
            dn = torch.as_tensor(np.asarray(tr["dense"][sel], np.float32), device=device)
            loss = lossfn(model(H, mk, dn, pop), gi[sel], gv[sel])
            aux = pop.aux_loss()
            if not isinstance(aux, float):
                loss = loss + a.pop_lambda * aux
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(float(loss.detach()))
        if hasattr(model, "eval"):
            model.eval()
        dl = dev_loss(model, tr, dev_rows, gold_idx, gold_val, device, a.qbatch,
                      torch, lossfn, pop)
        nd, nd100 = dev_ndcg_multi(model, tr, dev_rows, gold_lists, device,
                                   a.qbatch, torch, (10, 100), pop)
        hist.append({"epoch": ep + 1, "train_loss": float(np.mean(losses)),
                     "dev_loss": dl, "dev_ndcg@10": nd, "dev_ndcg@100": nd100})
        # all three minimised after the sign flip
        sel_now = dl if on_loss else -(nd100 if sel_k == 100 else nd)
        flag = ""
        if sel_now < best_sel - 1e-9:
            best_sel, best_nd, stale, flag = sel_now, nd, 0, " *"
            best_state = json.loads(json.dumps(model.state()))
            best_pop = pop.state()          # None unless the predictor is joint
        else:
            stale += 1
        print(f"  [{model.name}] ep{ep + 1:3d} train {np.mean(losses):.4f} "
              f"dev_loss {dl:.4f} nDCG@10 {nd:.4f} nDCG@100 {nd100:.4f} "
              f"beta {math.exp(float(model.logbeta.detach())):.3f}{flag}", flush=True)
        if stale >= a.patience:
            print(f"  [{model.name}] early stop: {stale} evals without a "
                  f"dev {a.select_on} gain")
            break
    model.load(best_state)
    # pop_tr and pop_te SHARE one predictor object, so restoring it here also
    # restores the popularity the test split will be scored with.
    pop.load_state(best_pop)
    # Return the nDCG AT THE SELECTED CHECKPOINT, not the best nDCG seen. Those
    # differ under loss selection, and reporting the max would reintroduce exactly
    # the optimistic bias this change removes.
    return best_nd, best_state, hist, best_sel, best_pop


def distill_operator(model, teacher, tr, fit_rows, device, a, torch, seed=None):
    """Initialise a raw-view neural scorer from the fitted operator ranking.

    This is representation distillation, not an inference-time ensemble: the
    teacher is used only on source-training queries.  The student never receives
    the operator's sum or maximum as input and the teacher is absent at test.
    """
    groups = [g for g in (
        {"params": [p for p in model.parameters() if p.ndim >= 2],
         "weight_decay": a.weight_decay},
        {"params": [p for p in model.parameters() if p.ndim < 2],
         "weight_decay": 0.0}) if g["params"]]
    opt = torch.optim.AdamW(groups, lr=a.distill_lr)
    pop = Pop.from_data(tr, device)
    rng = np.random.default_rng(a.seed if seed is None else seed)
    teacher_was_training = getattr(teacher, "training", False)
    if hasattr(teacher, "eval"):
        teacher.eval()
    if hasattr(model, "eval"):
        model.eval()  # deterministic teacher matching; gradients remain enabled
    for ep in range(a.distill_epochs):
        losses = []
        for bi in batches(len(fit_rows), a.qbatch, True, rng):
            sel = fit_rows[bi]
            H = torch.as_tensor(np.asarray(tr["H"][sel], np.float32), device=device)
            mk = torch.as_tensor(tr["mask"][sel].astype(np.float32), device=device)
            dn = torch.as_tensor(np.asarray(tr["dense"][sel], np.float32), device=device)
            with torch.no_grad():
                target = teacher(H, mk, dn, pop)
                target = ((target - target.mean(1, keepdim=True)) /
                          (target.std(1, keepdim=True) + EPS))
            pred = model(H, mk, dn, pop)
            pred = ((pred - pred.mean(1, keepdim=True)) /
                    (pred.std(1, keepdim=True) + EPS))
            loss = torch.mean((pred - target) ** 2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(float(loss.detach()))
        print(f"  [distill] ep{ep + 1:3d} mse {np.mean(losses):.6f}", flush=True)
    if hasattr(teacher, "train") and teacher_was_training:
        teacher.train()


def write_predictions(scores, data, path, topk=100):
    recs = []
    for i, q in enumerate(data["queries"]):
        order = np.argsort(-scores[i])[:topk]
        recs.append({"id": q["id"], "stratum": q.get("stratum"),
                     "supporting_documents": q.get("supporting_documents", []),
                     "predictions": {"document": [[data["doc_ids"][j], float(scores[i, j])]
                                                  for j in order]}})
    json.dump(recs, open(path, "w"))
    return path


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="BAAI/bge-large-en-v1.5")
    # 0 = every train query left after the dev slice. The old default of 2,500
    # came from operator_scorer.py, where it fitted FOUR parameters and more data
    # bought nothing. The learned arms here have 34-200 parameters and their
    # measured failure mode is overfitting, so capping the fit set is backwards:
    # it discarded 2,645 CS queries and 6,869 TOMATO queries for no reason.
    ap.add_argument("--train_fit", type=int, default=0,
                    help="fit queries after the dev slice; 0 = all remaining")
    # 300, not operator_scorer.py's 600. Dev is sliced FIRST, so on a small domain
    # it decides how much fit data is left: matsci has 1,304 train queries, where
    # 600 leaves 704 and 300 leaves 1,004. 300 still gives a stable nDCG@10.
    ap.add_argument("--dev", type=int, default=300)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--patience", type=int, default=5)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--weight_decay", type=float, default=1e-2,
                    help="applied to MLP weight MATRICES only; see train_arm")
    # Per-arm override, unset by default so every arm trains at --lr. Kept
    # because attention peaked at epoch 1 under 1e-3 while the others peaked at
    # 4-6, so a separate rate may be wanted again.
    ap.add_argument("--lr_attention", type=float, default=None)
    ap.add_argument("--hidden", type=int, default=8)
    ap.add_argument("--qbatch", type=int, default=8, help="queries per minibatch")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default=None)
    ap.add_argument("--arms", default="dense,current,attention,deepsets,mlp",
                    help="any of dense, current, attention, deepsets, setmlp, "
                         "dualsetmlp, mlp, gated")
    ap.add_argument("--mlp_hidden", type=int, default=16)
    ap.add_argument("--ds_hidden", type=int, default=4,
                    help="compact deepsets phi/rho width (spec: 4)")
    ap.add_argument("--ds_dropout", type=float, default=0.15,
                    help="whole-answer dropout in the deepsets arm, training only")
    ap.add_argument("--set_hidden", type=int, default=8,
                    help="identity-preserving set-MLP width")
    ap.add_argument("--set_dropout", type=float, default=0.0,
                    help="whole-answer dropout in the setmlp arm, training only")
    ap.add_argument("--distill_operator", default=None,
                    help="fitted current-scorer JSON used only to initialise a set MLP")
    ap.add_argument("--distill_epochs", type=int, default=0)
    ap.add_argument("--distill_lr", type=float, default=1e-3)
    ap.add_argument("--seeds", default="0",
                    help="comma-separated training seeds; each trained arm runs once "
                         "per seed and keeps the best dev checkpoint. The fit/dev "
                         "SPLIT stays fixed by --seed, so seeds vary only init, "
                         "batch order and dropout.")
    # DEFAULT IS THE OPERATOR'S OWN OBJECTIVE, so the `current` arm reproduces the
    # scorer as it is fitted elsewhere in the project and the only thing varying
    # between arms is the architecture. `fixed` is the multi-gold correction:
    # SIR-4 averages ~4 golds per query and the operator objective keeps them in
    # each other's denominator, so it pushes a query's own labels apart. Run both
    # to measure that rather than assume it.
    ap.add_argument("--loss", default="fixed", choices=["operator", "fixed"])
    # DEPLOYABILITY OF THE POPULARITY TERM. `loo` needs the OTHER test queries'
    # hypothetical answers, so it cannot score one query alone. `bank` replaces it
    # with the train-answer bank (query-independent, one matmul per corpus at
    # indexing time). `predicted` distils that bank into an MLP of the paper
    # embedding, so inference is fully local. Default stays loo so every prior
    # number reproduces.
    ap.add_argument("--popularity", default="loo", choices=["loo", "bank", "predicted"])
    # The mlp arm's popularity is PART OF THE ARM, not a run-level axis: the whole
    # proposal is a scorer that needs nothing but the paper embedding at inference.
    # Set 0 to make it share --popularity with the other arms instead.
    ap.add_argument("--mlp_popularity", type=int, default=1,
                    help="1 = the mlp arm uses its own learned popularity predictor")
    # TWO-STAGE vs JOINT. Two-stage fits the predictor to the bank targets, freezes
    # it, then trains the scorer -- so a win is attributable to the scorer and the
    # predictor keeps meaning "general matchability". Joint lets the retrieval
    # gradient reach the predictor as well, anchored by --pop_lambda on the same
    # log-MSE target; more expressive, less interpretable, and the predictor can
    # drift into being extra scorer capacity if lambda is too small.
    ap.add_argument("--mlp_pop_joint", type=int, default=0,
                    help="1 = train the popularity predictor jointly with the scorer")
    ap.add_argument("--pop_lambda", type=float, default=1.0,
                    help="weight on the matchability anchor under joint training")
    # Checkpoint/early-stopping criterion. `loss` is the default because dev
    # nDCG@10 is a step function and selecting its max over ~90 evaluations on one
    # 300-query slice picks luck, not generalisation. `ndcg` reproduces the old
    # behaviour.
    # SELECT ON THE METRIC, NOT THE LOSS. Measured on physics: dev loss and dev
    # nDCG@10 move in OPPOSITE directions on the same held-out queries (attention
    # seed 2, ep18 -> ep29: loss 5.8507 -> 5.6360 while nDCG 0.4329 -> 0.3976).
    # The loss is InfoNCE over the whole corpus, so it is rewarded for separating
    # the gold from negatives at rank 2000 that no metric sees; beta collapses
    # (0.89 -> 0.37), switching off the popularity discount that suppresses
    # generically attractive papers at the TOP. Global separation improves, head
    # precision degrades. nDCG@10 is a chunky step function, which is the real
    # problem, but the cure is a SMOOTHER VERSION OF THE METRIC:
    #   ndcg    nDCG@10, the reported metric, chunky but aligned
    #   ndcg100 nDCG@100, responds to a gold moving anywhere in the top 100, so
    #           many more events per epoch, still head-weighted by the log discount
    #   loss    kept only to reproduce the runs that exposed this
    ap.add_argument("--select_on", default="loss",
                    choices=["ndcg", "ndcg100", "loss"],
                    help="ndcg = nDCG@10 (the reported metric); ndcg100 = nDCG@100 "
                         "(same metric, more resolution per epoch); loss = dev loss")
    # 0 = the historical single dev slice. K >= 2 runs K-fold CV over EVERY train
    # query to choose the epoch count, then refits on ALL of them -- so no query is
    # permanently held out of training, and the selection signal is the whole train
    # split instead of a 300-query sample.
    ap.add_argument("--kfold", type=int, default=0)
    ap.add_argument("--force_build", action="store_true")
    ap.add_argument("--cache_only", action="store_true",
                    help="fail rather than load an embedding model if an array is missing")
    ap.add_argument("--selection_only", action="store_true",
                    help="train and select on development data without scoring test")
    add_dataset_arg(ap)
    a = ap.parse_args()
    set_dataset(a.dataset)
    print(banner())

    import torch
    device = ("cuda" if torch.cuda.is_available()
              else "mps" if torch.backends.mps.is_available() else "cpu")
    torch.manual_seed(a.seed)
    op = _op_module()
    out = a.out or f"{_ROOT}/experiments/results/semantic_{a.dataset}"
    os.makedirs(out, exist_ok=True)
    cache = f"{_ROOT}/outputs/caches/semantic/{a.dataset}"
    print(f"device={device} encoder={a.model}\nout={out}")

    if a.cache_only:
        # build_inputs reaches the encoder only when one of these arrays is
        # absent.  Check here so a cache-only run fails before importing any
        # embedding library or constructing a model.
        for split in ("train", "test"):
            _, _, queries, _ = op.load_split(split)
            qkey = hashlib.md5("|".join(q["id"] for q in queries).encode()).hexdigest()[:8]
            slug = op.model_slug(a.model)
            qi = op.query_instruction(a.model)
            required = [f"{emb_dir()}/{split}_doc{slug}.npy",
                        f"{emb_dir()}/{split}_query_{qkey}{slug}{op.qi_tag(qi)}.npy",
                        f"{emb_dir()}/{split}_probe_{qkey}{slug}.npy"]
            missing = [p for p in required if not os.path.exists(p)]
            assert not missing, "cache-only run is missing:\n  " + "\n  ".join(missing)
    tr = build_inputs(op, a.model, "train", cache, a.force_build)
    te = build_inputs(op, a.model, "test", cache, a.force_build)

    arms_req = [x.strip() for x in a.arms.split(",") if x.strip()]
    # THE MLP ARM CARRIES ITS OWN LEARNED POPULARITY. It is not a separate axis:
    # the proposal is one deployable scorer -- learned pooling AND a popularity
    # discount predicted from the paper embedding alone, with no dependence on the
    # other test queries. `current` keeps the leave-one-out term it has always
    # used, so the comparison is proposal-vs-baseline as each is actually meant to
    # be deployed. Both sources are computed once and stored side by side.
    MLP_ARMS = {"mlp": "joint", "mlp2s": "_pred", "mlpbank": "_bank"}
    if any(x in arms_req for x in MLP_ARMS) or a.popularity != "loo":
        # TRAIN targets are already on disk: total_S is the sum over every train
        # answer, so dividing by the answer count IS the bank mean for the train
        # corpus. Only the TEST corpus needs a fresh matmul, against the SAME
        # train bank -- no test query ever contributes to any popularity value.
        slug = op.model_slug(a.model)
        n_bank = int(tr["mask"].sum())
        p_tr = (tr["total_S"] / max(n_bank, 1)).astype(np.float32)
        bank_tr = p_tr.copy()          # the anchor target, before any prediction
        bank_emb = np.load(f"{emb_dir()}/train_probe_{tr['meta']['qkey']}{slug}.npy"
                           ).astype(np.float32)
        assert bank_emb.shape[0] == n_bank, (
            f"train answer bank {bank_emb.shape[0]} rows != {n_bank} valid answers")
        de_te = np.load(f"{emb_dir()}/test_doc{slug}.npy").astype(np.float32)
        assert de_te.shape[0] == len(te["doc_ids"]), "test doc embedding misaligned"
        p_te = bank_popularity(bank_emb, de_te)
        print(f"[pop] bank of {n_bank} train answers; train p in "
              f"[{p_tr.min():.4f}, {p_tr.max():.4f}], test p in "
              f"[{p_te.min():.4f}, {p_te.max():.4f}]")
        # KEEP THE BANK VECTORS. They are the anchor targets AND the `mlpbank`
        # arm's popularity, so overwriting them with predictions below would lose
        # the one configuration measured to beat the baseline.
        bank_te = p_te.copy()
        tr["pop_mode_bank"] = te["pop_mode_bank"] = "bank"
        tr["pop_vec_bank"], te["pop_vec_bank"] = bank_tr, bank_te
        pop_info = {"mode": a.popularity, "bank_answers": n_bank}
        _fitted, pop_joint = False, None
        if a.popularity == "predicted" or a.mlp_popularity:
            _fitted = True
            de_tr = np.load(f"{emb_dir()}/train_doc{slug}.npy").astype(np.float32)
            assert de_tr.shape[0] == len(tr["doc_ids"]), "train doc embedding misaligned"
            gm, fit_info = fit_matchability(de_tr, p_tr, device, seed=a.seed)
            pop_info["fit"] = fit_info
            with _torch.no_grad():
                # The scorer sees PREDICTED popularity on BOTH splits, so its
                # training-time input distribution matches inference exactly.
                p_tr = gm(_torch.as_tensor(de_tr, device=device)).cpu().numpy()
                p_te = gm(_torch.as_tensor(de_te, device=device)).cpu().numpy()
            tr["pop_mode_pred"] = te["pop_mode_pred"] = "predicted"
            tr["pop_vec_pred"], te["pop_vec_pred"] = p_tr, p_te
            _torch.save(gm.state_dict(),
                        f"{out}/matchability_{a.dataset}{slug}.pt")
            # Joint mode rebuilds the predictor per seed, warm-started from this
            # two-stage fit, so it begins at "general matchability" rather than at
            # noise and any drift is attributable to the retrieval gradient.
            pop_joint = {"init": {k: v.detach().cpu() for k, v in gm.state_dict().items()},
                         "dim": de_tr.shape[1], "hidden": fit_info["hidden"],
                         "de_tr": de_tr, "de_te": de_te,
                         "target_tr": bank_tr}
        if a.popularity != "loo":
            tr["pop_mode"] = te["pop_mode"] = a.popularity
            tr["pop_vec"], te["pop_vec"] = p_tr, p_te
        # pop_joint only exists if a predictor was fitted above; the joint arm is
        # unavailable otherwise and falls back to the frozen vectors.
        pop_joint = pop_joint if (a.mlp_pop_joint and _fitted) else None
    else:
        pop_info, pop_joint = {"mode": "loo"}, None
    gidx, gval, glists = gold_matrix(tr["queries"], tr["doc_ids"])
    te_glists = None if a.selection_only else gold_matrix(te["queries"], te["doc_ids"])[2]

    # SAME SLICING RULE AS operator_scorer.py: dev first, then fit from what is
    # left. matsci train holds 1,304 queries, so a 2,500-query fit set does not
    # exist and the request is silently truncated -- print what actually happened.
    Q = len(tr["queries"])
    rng = np.random.default_rng(a.seed)
    perm = rng.permutation(Q)
    dev_rows = perm[:a.dev]
    fit_rows = perm[a.dev:] if a.train_fit <= 0 else perm[a.dev:a.dev + a.train_fit]
    all_rows = perm                      # every train query, for --kfold
    # Under --kfold the single fit/dev split is unused: the folds provide both, and
    # the final model refits on all_rows. So an empty fit_rows is only a problem in
    # the non-kfold path.
    assert a.kfold >= 2 or len(fit_rows) > 0, (
        f"no fit queries left: {Q} train queries, --dev {a.dev}")
    if a.kfold < 2 and ((a.train_fit > 0 and len(fit_rows) < a.train_fit)
                        or len(dev_rows) < a.dev):
        print(f"[split] REQUESTED fit={a.train_fit} dev={a.dev} but the train split "
              f"holds {Q} queries -> using fit={len(fit_rows)} dev={len(dev_rows)}")
    # ONE input width across splits: the two splits can have different Jmax and a
    # layer shaped for one would fail on the other, after training.
    JFIX = max(tr["H"].shape[1], te["H"].shape[1])
    print(f"[split] jmax(train)={tr['H'].shape[1]} jmax(test)={te['H'].shape[1]} -> JFIX={JFIX}")
    if a.kfold >= 2:
        print(f"[split] {a.kfold}-fold CV over ALL {Q} train queries: each fold fits "
              f"on ~{Q - Q // a.kfold} and devs on ~{Q // a.kfold}; the final model "
              f"refits on all {Q}. test={len(te['queries'])}")
    else:
        print(f"[split] fit={len(fit_rows)} dev={len(dev_rows)} of {Q} train queries "
              f"(seed {a.seed}); test={len(te['queries'])}")

    # Filenames are per-arm AND per-loss and never reused, so a rerun adds files
    # rather than silently replacing another configuration's predictions. Without
    # the loss in the name a legacy run would overwrite the corrected one and the
    # two would be indistinguishable afterwards.
    LSUF = "_operatorloss" if a.loss == "operator" else "_fixedloss"
    # bank/predicted results must never overwrite the loo ones they are read against.
    LSUF += "" if a.popularity == "loo" else f"_{a.popularity}pop"
    TAG = {k: f"{k}{LSUF}" for k in
           ("current", "attention", "deepsets", "setmlp", "dualsetmlp", "gated")}
    TAG["dense"] = "dense"          # untrained, so no loss belongs in its name
    for k in ("mlp", "mlp2s", "mlpbank"):
        TAG[k] = f"{k}{LSUF}"
    seeds = [int(x) for x in str(a.seeds).split(",") if x.strip() != ""]
    results, summary = {}, {}
    for arm in [x.strip() for x in a.arms.split(",") if x.strip()]:
        assert arm in TAG, f"unknown arm {arm!r}; choose from {sorted(TAG)}"
        print(f"\n=== arm: {arm} ===")

        def build():
            return (CurrentScorer(device) if arm == "current"
                    else AttentionScorer(device, a.hidden) if arm == "attention"
                    else DeepSetsScorer(device, a.ds_hidden, a.ds_dropout) if arm == "deepsets"
                    else SetMLPScorer(device, a.set_hidden, a.set_dropout) if arm == "setmlp"
                    else DualSetMLPScorer(device, a.set_hidden, a.set_dropout) if arm == "dualsetmlp"
                    else DenseScorer(device) if arm == "dense"
                    else SortedMLPScorer(device, JFIX, a.mlp_hidden) if arm in MLP_ARMS
                    else GatedScorer(device, a.hidden))

        # THE MLP ARM READS ITS OWN POPULARITY. Its learned predictor is part of
        # the proposal, not a run-level axis, so it is selected per arm here while
        # every other arm keeps whatever --popularity chose.
        # EACH MLP VARIANT DIFFERS ONLY IN ITS POPULARITY SOURCE, so one run
        # measures all three against the same baseline on the same split:
        #   mlp      joint  -- predictor trained with the scorer (the proposal)
        #   mlp2s    _pred  -- predictor fitted to the bank, then frozen
        #   mlpbank  _bank  -- the measured bank mean, nothing learned
        want = MLP_ARMS.get(arm, "")
        joint_here = want == "joint" and pop_joint is not None
        vk = "" if joint_here else (want if f"pop_vec{want}" in tr else "")

        def make_pops():
            """Fresh Pop pair. Under joint mode the predictor is rebuilt per seed."""
            if not joint_here:
                return Pop.from_data(tr, device, vk), Pop.from_data(te, device, vk)
            gj = MatchabilityPredictor(pop_joint["dim"], pop_joint["hidden"]).to(device)
            gj.load_state_dict({k: v.to(device) for k, v in pop_joint["init"].items()})
            dtr = _torch.as_tensor(pop_joint["de_tr"], device=device)
            dte = _torch.as_tensor(pop_joint["de_te"], device=device)
            tgt = _torch.as_tensor(pop_joint["target_tr"], device=device)
            # ONE predictor shared by both splits: the test popularity must come
            # from the network that training produced, not a second copy.
            return (Pop("joint", predictor=gj, doc_emb=dtr, target=tgt),
                    Pop("joint", predictor=gj, doc_emb=dte))

        pop_tr, pop_te = make_pops()

        model = build()
        npar = sum(p.numel() for p in model.parameters())
        print(f"  parameters: {npar}"
              + ("  popularity: predictor, JOINT with the scorer" if joint_here
                 else "  popularity: predictor, frozen (two-stage)" if vk == "_pred"
                 else "  popularity: train-answer bank (measured, not learned)" if vk == "_bank"
                 else f"  popularity: {tr.get('pop_mode', 'loo')}"))
        t0 = time.time()
        per_seed = {}
        # BEFORE the branch. An untrained arm never enters the seed loop, so an
        # initialisation inside it leaves this name unbound for `dense` and the
        # predictor-saving step below crashes on the very first arm.
        best_pop_seed = None
        if npar == 0:
            # Nothing to fit. Its dev score is still recorded so the trained arms
            # can be read against a fixed reference on the same queries.
            print("  no parameters: scoring directly, no training")
            best = dev_ndcg(model, tr, dev_rows, glists, device, a.qbatch, torch,
                            pop=pop_tr)
            state, hist = model.state(), []
            print(f"  [dense] dev nDCG@10 {best:.4f}")
        else:
            lr = a.lr_attention if (arm == "attention" and a.lr_attention) else a.lr
            ndec = sum(p.numel() for p in model.parameters() if p.ndim >= 2)
            print(f"  lr: {lr}  weight_decay: {a.weight_decay} on {ndec} of {npar} params"
                  + (f"  seeds: {seeds}" if len(seeds) > 1 else ""))
            # SEEDS VARY TRAINING ONLY (init, batch order, dropout). The fit/dev
            # split is pinned by --seed, so every seed and every arm sees the same
            # queries and the best-dev selection is a fair comparison.
            # SEEDS ARE SELECTED ON THE SAME CRITERION AS EPOCHS. Picking the
            # best-of-3 by dev nDCG while epochs are picked by dev loss would put
            # the discarded step-function signal straight back in, one level up.
            best_sel_seed, best, state, hist = float("inf"), -1.0, None, []
            for sd in seeds:
                torch.manual_seed(sd)
                # Rebuild for EVERY seed, including a one-seed invocation such
                # as --seeds 2.  Otherwise that invocation silently retains the
                # model constructed under --seed (normally zero).
                model = build()
                # REBUILD THE PREDICTOR TOO. Left outside the loop it would carry
                # seed 0's trained weights into seed 1, so the seeds would not be
                # independent and later ones would start pre-trained.
                pop_tr, pop_te = make_pops()
                if len(seeds) > 1:
                    print(f"  -- seed {sd} --")
                if arm in ("setmlp", "dualsetmlp") and a.distill_epochs > 0:
                    assert a.distill_operator and os.path.exists(a.distill_operator), (
                        "--distill_epochs requires an existing --distill_operator JSON")
                    teacher = CurrentScorer(device)
                    teacher.load(json.load(open(a.distill_operator)))
                    print(f"  distilling fitted operator for {a.distill_epochs} epochs; "
                          "teacher is not used at inference")
                    distill_operator(model, teacher, tr, fit_rows, device, a, torch, seed=sd)
                if a.kfold >= 2:
                    # Folds choose the epoch count; the final fit then uses EVERY
                    # train query. There is no held-out data left to select on,
                    # which is the intended trade: all the data trains the model.
                    E, picked = kfold_epochs(build, tr, all_rows, gidx, gval, glists,
                                             device, a, torch, lr=lr, seed=sd)
                    model = build()
                    h = train_fixed(model, tr, all_rows, gidx, gval, device, a,
                                    torch, E, lr=lr, seed=sd)
                    st = json.loads(json.dumps(model.state()))
                    b = dev_ndcg(model, tr, all_rows, glists, device, a.qbatch, torch)
                    bsel, bpop = float(h[-1]["train_loss"]), pop_tr.state()
                    per_seed[sd] = {"kfold_epochs": picked, "epochs_used": E,
                                    "train_ndcg@10_insample": round(b, 4)}
                else:
                    b, st, h, bsel, bpop = train_arm(model, tr, fit_rows, dev_rows,
                                                     gidx, gval, glists, device, a,
                                                     torch, lr=lr, seed=sd, pop=pop_tr)
                    # Name the field after what was actually selected on: bsel is
                    # the loss under --select_on loss and MINUS the nDCG otherwise,
                    # so a fixed "dev_loss" label printed the nDCG twice.
                    per_seed[sd] = {"dev_ndcg@10": round(b, 4),
                                    f"selected_on_{a.select_on}":
                                        round(bsel if a.select_on == "loss" else -bsel, 4)}
                if bsel < best_sel_seed - 1e-9:
                    best_sel_seed, best, state, hist = bsel, b, st, h
                    best_pop_seed, best_pop_te = bpop, pop_te
            if len(seeds) > 1:
                print(f"  per-seed: {per_seed}")
                if a.kfold >= 2:
                    # NO HELD-OUT DATA REMAINS, so a seed cannot be selected on
                    # merit. Keeping the lowest FINAL TRAINING loss is a tie-break,
                    # not model selection, and it is reported as such.
                    print("  kfold: every train query is in the final fit, so no "
                          "held-out selection is possible; seeds differ only by "
                          "init/order and the lowest final train loss is kept")
                else:
                    print(f"  kept the seed with the best dev "
                          f"{'loss' if a.select_on == 'loss' else 'nDCG@10'}")
            model.load(state)
            if best_pop_seed is not None:
                # Restore the WINNING seed's predictor into the Pop that test
                # scoring uses, so both halves come from one checkpoint.
                best_pop_te.load_state(best_pop_seed)
                pop_te = best_pop_te
        tag = TAG[arm]
        json.dump(state, open(f"{out}/params_semantic_{tag}_{a.dataset}.json", "w"), indent=1)
        # THE JOINT PREDICTOR IS HALF THE MODEL, SO IT HAS TO BE HALF THE CHECKPOINT.
        # Only the pre-joint two-stage fit was ever written to disk, so once the
        # process exited the trained popularity was gone and the params json on disk
        # described a scorer paired with a predictor that no longer existed. Anything
        # reloading this arm -- the graph fusion warm start, a rerun, a transfer --
        # would silently get the wrong half. Written next to the scorer under a
        # matching name so the pair cannot be separated by accident.
        if best_pop_seed is not None:
            _torch.save(best_pop_seed, f"{out}/popnet_semantic_{tag}_{a.dataset}.pt")
            print(f"[{arm}] saved joint popularity predictor -> "
                  f"popnet_semantic_{tag}_{a.dataset}.pt")
        pred, test_ndcg = None, None
        if not a.selection_only:
            # pop_te, NOT the run-level default. Without it a joint run scored
            # test with leave-one-out popularity and the trained predictor was
            # never used at inference at all.
            sc = score_rows(model, te, np.arange(len(te["queries"])),
                            device, a.qbatch, torch, pop=pop_te)
            pred = write_predictions(
                sc, te, f"{out}/predictions_semantic_{tag}_{a.dataset}_test.json")
            test_ndcg = float(np.mean(
                [ndcg_at(list(np.argsort(-sc[i])[:10]), te_glists[i], 10)
                 for i in range(len(te_glists)) if te_glists[i]]))
        results[arm] = {"best_dev_ndcg@10": best, "params": state, "history": hist,
                        "per_seed_dev_ndcg@10": per_seed,
                        "predictions": pred, "minutes": round((time.time() - t0) / 60, 2)}
        summary[arm] = {"dev_ndcg@10": round(best, 4),
                        "test_ndcg@10_quick": (None if test_ndcg is None
                                                else round(test_ndcg, 4))}
        destination = "test deliberately not scored" if pred is None else os.path.basename(pred)
        print(f"[{arm}] best dev nDCG@10 {best:.4f} -> {destination}")

    meta = {"dataset": a.dataset, "encoder": a.model, "device": device,
            "hyperparameters": {k: getattr(a, k) for k in
                                ("train_fit", "dev", "epochs", "patience", "lr",
                                 "hidden", "ds_hidden", "ds_dropout", "mlp_hidden",
                                 "set_hidden", "set_dropout",
                                 "distill_operator", "distill_epochs", "distill_lr",
                                 "qbatch", "seed", "seeds", "loss", "popularity",
                                 "select_on", "kfold", "mlp_popularity",
                                 "mlp_pop_joint", "pop_lambda",
                                 "lr_attention", "weight_decay")},
            "popularity": pop_info,
            "actual_split": {"fit": int(len(fit_rows)), "dev": int(len(dev_rows)),
                             "train_queries": int(Q), "test_queries": len(te["queries"])},
            "loss": ("operator objective (operator_scorer.py): full-corpus denominator "
                     "including the query's other golds, weighted per (query, gold) pair"
                     if a.loss == "operator" else
                     "multi-gold corrected: other golds excluded from each denominator, "
                     "averaged within query then equally across queries"),
            "arms": results, "quick_summary": summary}
    mp = f"{out}/semantic_comparison_{a.dataset}{LSUF}.json"
    json.dump(meta, open(mp, "w"), indent=1)
    print(f"\nwrote {mp}")
    print("\nquick development selection" +
          (" (test not scored):" if a.selection_only else "/test nDCG@10:"))
    for k, v in summary.items():
        test_text = "not scored" if v["test_ndcg@10_quick"] is None else f"{v['test_ndcg@10_quick']:.4f}"
        print(f"  {k:10} dev {v['dev_ndcg@10']:.4f}  test {test_text}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/scigraphir/retriever/precompute/precompute_operator_components.py
"""
precompute_operator_components.py — cache the operator's RAW ingredients (not the final score) so the
operator's weights w=[w0,w1,w2] and exponent beta can be LEARNED jointly inside FusionGraphReasoner.

The operator score is   S_op = w0 z(dense) + w1 z(S/dem^beta) + w2 z(M/dem^beta),  where
  dense = q . d            (query-doc similarity in the selected encoder space)
  S, M  = probe-sum / probe-max similarity   (HyDE bridge probes vs docs)
  dem   = total_S - S      (leave-one-out anti-hub popularity; total_S = S.sum over queries)
Caching dense, S, M and total_S lets the wrapper recompute S_op live with LEARNABLE w, beta (dem is
derived as total_S - S). All arrays are aligned to the graph's nodes.csv document order.

Saves data/<graph>/operator_components.npz {dense,S,M: float16 [Q x n_doc], total_S: float32 [n_doc],
query_ids:[...]}. Local, no API (reuses the op_emb caches). Mirrors precompute_operator_scores.py.
  python precompute/precompute_operator_components.py --graph tomato_train_v16sc --split train
  python precompute/precompute_operator_components.py --graph tomato_test_v16sc  --split test
"""
import argparse, csv, hashlib, json, os, sys
import numpy as np
from tqdm import tqdm

# SCIGRAPHIR_ROOT so this runs off-laptop (Colab unzips to /content/scigraphir). Unset,
# it resolves to exactly the path this replaced.
_ROOT = os.environ.get("SCIGRAPHIR_ROOT") or os.path.expanduser("$SCIGRAPHIR_ROOT")
BASE = f"{_ROOT}/retriever"
sys.path.insert(0, BASE)
# One resolver for corpus + caches; default "tomato" reproduces every legacy path.
sys.path.insert(0, _ROOT)
from scigraphir_paths import add_dataset_arg, banner, corpus_dir, emb_dir, probes_path, set_dataset  # noqa: E402
csv.field_size_limit(10 ** 7)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--graph", required=True, help="graph dir (for doc-node order), e.g. tomato_train_v16sc")
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--model", default="BAAI/bge-large-en-v1.5",
                    help="sentence-transformer used for dense and probe similarities")
    add_dataset_arg(ap)
    a = ap.parse_args()
    set_dataset(a.dataset)
    print(banner())

    s1 = f"{BASE}/data/{a.graph}/processed/stage1"
    docnodes = [n["name"] for n in csv.DictReader(open(f"{s1}/nodes.csv")) if n["type"] == "document"]
    corpus = json.load(open(f"{corpus_dir(a.split)}/raw/documents.json"))
    doc_ids = list(corpus)
    d2i = {d: i for i, d in enumerate(doc_ids)}
    assert all(n in d2i for n in docnodes), "graph has document nodes not in documents.json"
    col = [d2i[name] for name in docnodes]              # corpus order -> nodes.csv doc order
    raw = json.load(open(f"{corpus_dir(a.split)}/raw/{a.split}.json"))
    qids = [q["id"] for q in raw]
    qkey = hashlib.md5("|".join(qids).encode()).hexdigest()[:8]

    probes = {json.loads(l)["id"]: json.loads(l)["probes"]
              for l in open(probes_path(a.split))}

    # ENCODE ON MISS, rather than assuming a previous run happened to use the
    # same query subset. The embedding cache is keyed by an md5 of the query
    # ids, and operator_scorer.py only ever encodes its FIT and DEV samples
    # (e.g. 30 + 10 of 50, or 2,500 + 600 of 5,445). This script needs ALL of
    # them, so its hash never matched and the load died with a bare
    # FileNotFoundError naming a hash that nothing had produced.
    import importlib.util
    spec = importlib.util.spec_from_file_location("op", f"{BASE}/eval/operator_scorer.py")
    op = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(op)                          # same encoder, instruction and cache key
    slug = op.model_slug(a.model)
    qi    = op.query_instruction(a.model)
    qslug = f"{slug}{op.qi_tag(qi)}"        # the instruction is part of what queries ARE
    dpath = f"{emb_dir()}/{a.split}_doc{slug}.npy"
    qpath = f"{emb_dir()}/{a.split}_query_{qkey}{qslug}.npy"
    ppath = f"{emb_dir()}/{a.split}_probe_{qkey}{slug}.npy"
    if not all(os.path.exists(p) for p in (dpath, qpath, ppath)):
        from sentence_transformers import SentenceTransformer
        print(f"[op-comp] {a.model} full-split embeddings absent for {a.split} "
              f"({len(qids)} queries); encoding once")
        model = SentenceTransformer(a.model)
        model.max_seq_length = 512
        op.cached_encode(model, [corpus[d] for d in doc_ids], f"{a.split}_doc{slug}")
        op.cached_encode(model, [q["question"] for q in raw],
                         f"{a.split}_query_{qkey}{qslug}", instruct=qi)
        flat = [pr for q in raw for pr in probes.get(q["id"], [])]
        op.cached_encode(model, flat, f"{a.split}_probe_{qkey}{slug}")

    de = np.load(dpath).astype(np.float32)
    qe = np.load(qpath).astype(np.float32)
    pe = np.load(ppath).astype(np.float32)
    own = []
    for q in raw:
        own += [q["id"]] * len(probes.get(q["id"], []))
    own = np.array(own)

    Q, D = len(raw), len(doc_ids)
    print(f"[op-comp] {Q} queries x {D} docs; computing raw components ...", flush=True)
    print(f"[op-comp] dense: one {Q}x{len(qe[0])} @ {len(de[0])}x{D} matmul ...", flush=True)
    dense = qe @ de.T                                    # [Q, D]
    S = np.zeros((Q, D), np.float32); M = np.zeros((Q, D), np.float32)
    # The probe loop is the long pole: one matmul per query against the whole
    # corpus. It ran completely silently, so a 5,445-query train split looked
    # indistinguishable from a hang for several minutes.
    for i, q in enumerate(tqdm(raw, desc="[op-comp] probes", unit="q")):
        idx = np.where(own == q["id"])[0]
        if len(idx):
            H = np.clip(pe[idx] @ de.T, 0, None); S[i] = H.sum(0); M[i] = H.max(0)

    # reorder columns to nodes.csv document order (so they align with graph.nodes_by_type['document'])
    dense = dense[:, col]; S = S[:, col]; M = M[:, col]
    total_S = S.sum(0)                                   # [D] popularity (for dem = total_S - S)

    # MODEL-SCOPED, because the components ARE the encoder's output. Writing every
    # encoder to one filename would let a Qwen3 run silently clobber the BGE
    # components that already-trained checkpoints were calibrated against -- and the
    # ResearchBench transfer is consuming exactly those right now. The slug is empty
    # for the default encoder, so existing paths are unchanged.
    out = f"{BASE}/data/{a.graph}/operator_components{slug}.npz"
    # Record the instruction, not just the encoder. `dense` is the only array the
    # instruction touches (probes and documents are encoded without one), so a
    # components file built under a different instruction is a different baseline
    # wearing the same filename. Consumers can now check instead of assuming.
    np.savez_compressed(out,
                        dense=dense.astype(np.float16), S=S.astype(np.float16), M=M.astype(np.float16),
                        total_S=total_S.astype(np.float32), query_ids=np.array(qids),
                        encoder=np.array(a.model), query_instruct=np.array(qi))
    sz = os.path.getsize(out) / 1e6
    print(f"[op-comp] saved {out}  ({sz:.0f} MB)  dense/S/M={dense.shape} f16, total_S={total_S.shape} f32")
    print(f"[op-comp] env:  OPERATOR_COMPONENTS{'_TEST' if a.split=='test' else ''}={out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/scigraphir/retriever/precompute/precompute_semantic_components.py
"""
precompute_semantic_components.py — cache what the LEARNED semantic scorer needs inside the
graph fusion, aligned to the graph's nodes.csv document order.

The operator version of this script (precompute_operator_components.py) caches S and M, the two
handcrafted summaries of the hypothetical-answer matches. The sorted-MLP does not use summaries:
it reads the whole ordered match profile, so it needs the FULL per-answer matrix

    H  [Q, Jmax, n_doc]   float16   ReLU(cos(hypothetical answer j, document d))

plus the direct query-document similarity, the answer-validity mask, and the document embeddings
that the popularity predictor p_hat(d) = softplus(g(E(d))) reads.

H IS NOT COPIED. semantic_scorer.py already materialises exactly this matrix as a memmap under
outputs/caches/semantic/<dataset>/, and at CS scale it is 1.76 GB — duplicating it per graph would
cost more disk than every other artefact in the project combined, and would introduce a second copy
that can go stale. This script instead records the memmap's PATH and the column permutation `col`
that maps corpus document order to nodes.csv document order, and the fusion applies `col` to the
few rows it slices per batch. Everything small (dense, mask, doc_emb, total_S) is reordered here
and stored outright, because those are cheap and reordering them per batch would not be.

Run it AFTER the semantic scorer (notebook section 5d), which is what builds the memmap and trains
the checkpoint the fusion warm-starts from. If the memmap is absent this rebuilds it from the
cached embeddings; if the embeddings are absent too it will encode, which is the slow path.

  python precompute/precompute_semantic_components.py \
      --dataset sir4_physics --model /content/qwen3 --graph sir4_physics_train_v16sc --split train
"""
import argparse
import csv
import hashlib
import importlib.util
import json
import os
import sys

import numpy as np

# SCIGRAPHIR_ROOT so this runs off-laptop (Colab unzips to /content/scigraphir). Unset, it
# resolves to exactly the path this replaced.
_ROOT = os.environ.get("SCIGRAPHIR_ROOT") or os.path.expanduser("$SCIGRAPHIR_ROOT")
BASE = f"{_ROOT}/retriever"
sys.path.insert(0, BASE)
sys.path.insert(0, _ROOT)
from scigraphir_paths import add_dataset_arg, banner, corpus_dir, emb_dir, set_dataset  # noqa: E402

csv.field_size_limit(10 ** 7)


def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--graph", required=True, help="graph dir supplying nodes.csv document order")
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--model", default="BAAI/bge-large-en-v1.5")
    ap.add_argument("--force", action="store_true", help="rebuild H even if cached")
    add_dataset_arg(ap)
    a = ap.parse_args()
    set_dataset(a.dataset)
    print(banner())

    op = _load("op", f"{BASE}/eval/operator_scorer.py")
    sem = _load("sem", f"{_ROOT}/experiments/eval/semantic_scorer.py")
    slug = op.model_slug(a.model)

    # ---- the graph's document order -------------------------------------------------
    s1 = f"{BASE}/data/{a.graph}/processed/stage1"
    docnodes = [n["name"] for n in csv.DictReader(open(f"{s1}/nodes.csv")) if n["type"] == "document"]
    corpus = json.load(open(f"{corpus_dir(a.split)}/raw/documents.json"))
    doc_ids = list(corpus)
    d2i = {d: i for i, d in enumerate(doc_ids)}
    assert all(n in d2i for n in docnodes), "graph has document nodes not in documents.json"
    col = np.asarray([d2i[name] for name in docnodes], dtype=np.int64)   # corpus -> nodes.csv

    # ---- the semantic scorer's own inputs, in CORPUS order ---------------------------
    cache = f"{_ROOT}/outputs/caches/semantic/{a.dataset}"
    data = sem.build_inputs(op, a.model, a.split, cache, force=a.force)
    meta = data["meta"]
    Q, Jmax, D = data["H"].shape
    assert D == len(doc_ids), f"H has {D} document columns, corpus has {len(doc_ids)}"
    assert len(data["queries"]) == Q

    # The memmap path is reconstructed from the SAME key build_inputs used, so a cache
    # written under a different encoder or a reordered corpus cannot be picked up here.
    tag = f"{a.split}_{meta['qkey']}_{meta['dockey']}{slug}"
    h_path = f"{cache}/semantic_H_{tag}.f16"
    assert os.path.exists(h_path), f"H memmap missing: {h_path}"

    # ---- document embeddings, for the popularity predictor ---------------------------
    de = np.load(f"{emb_dir()}/{a.split}_doc{slug}.npy").astype(np.float32)
    assert de.shape[0] == D, f"doc embeddings {de.shape[0]} rows != {D} documents"

    qids = [q["id"] for q in data["queries"]]
    raw_qids = [q["id"] for q in json.load(open(f"{corpus_dir(a.split)}/raw/{a.split}.json"))]
    assert qids == raw_qids, "build_inputs query order differs from the raw split order"
    assert meta["qkey"] == hashlib.md5("|".join(qids).encode()).hexdigest()[:8]

    out = f"{BASE}/data/{a.graph}/semantic_components{slug}.npz"
    np.savez_compressed(
        out,
        # reordered to nodes.csv document order, exactly like the operator components
        dense=np.asarray(data["dense"])[:, col].astype(np.float16),
        doc_emb=de[col].astype(np.float16),
        total_S=np.asarray(data["total_S"])[col].astype(np.float32),
        mask=np.asarray(data["mask"]),                       # [Q, Jmax] bool, order-independent
        # H stays where semantic_scorer.py put it; the fusion applies `col` per batch
        col=col, h_path=np.array(h_path), h_shape=np.array([Q, Jmax, D]),
        query_ids=np.array(qids), Jmax=np.array(Jmax),
        encoder=np.array(a.model), query_instruct=np.array(meta["query_instruct"]),
        dockey=np.array(meta["dockey"]),
    )
    sz = os.path.getsize(out) / 1e6
    print(f"[sem-comp] saved {out}  ({sz:.0f} MB)")
    print(f"[sem-comp]   dense {(Q, len(col))} f16 | doc_emb {(len(col), de.shape[1])} f16 "
          f"| mask {(Q, Jmax)} bool")
    print(f"[sem-comp]   H referenced at {h_path} ({os.path.getsize(h_path)/1e6:.0f} MB, not copied)")
    print(f"[sem-comp] env:  SEMANTIC_COMPONENTS{'_TEST' if a.split == 'test' else ''}={out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/gfm-rag/gfmrag/models/cqig.py
"""
cqig.py — Cross-Query Informativeness Gating.

A node earns the right to send a message by responding to THIS query differently from how
the nodes of this graph TYPICALLY respond to unrelated scientific problems. A node that
answers every query the same way is broadcasting background graph activity, and under sum
aggregation that broadcast reaches every one of its neighbours identically, so it shifts a
whole domain rather than discriminating within it.

    mu_v^l    = (1/M) sum_m h_{a_m v}^l                       reference mean,     PER NODE
    V_v^l     = (1/(M-1)) sum_m || h_{a_m v}^l - mu_v ||^2    reference variance, PER NODE
    s^l       = median_{v : V_v^l > delta} V_v^l              reference scale,    PER LAYER
    I_qv^l    = || h_qv^l - mu_v^l ||^2 / (s^l + eps)
    tau_l(G)  = tau_ref_l(G) + dtau_l                         graph term + LEARNED term
    g_qv^l    = 1 - lam_l [ 1 - sigmoid( alpha_l ( log(1+I) - tau_l(G) ) ) ]
    m~        = g_qv^l * m_{v->u}^l                           op="gate", the default

WHY THE DENOMINATOR IS A LAYER-WIDE SCALE AND NOT THE NODE'S OWN VARIANCE. Two dead ends
came first and both are kept as ablations, because the difference between them IS the
hypothesis:

  norm="energy"  divide by E_v = mean ||h||^2. Dead on arrival under
                 `use_ent_emb: early-late-fusion`: h_qv is dominated by the node's static
                 entity embedding and the query-dependent part is ~1% of it in norm, so
                 I ~ 1e-4 for EVERY node and the sigmoid never leaves its midpoint.
  norm="node"    divide by the node's OWN variance V_v. Numerically alive (I ~ 1) but it
                 cancels the very comparison the method is built on: a generic node that
                 always moves by 0.01 and a discriminative node that always moves by 1.0
                 both come out at 0.01/0.01 = 1.0/1.0 = 1. Every node looks equally
                 informative and the gate is a near-constant rescaling.
  norm="layer"   divide by ONE robust scale for the whole layer. A weak generic response is
                 now small, a strong query-specific response is large, and a node that
                 never responds stays at zero. This is the only form in which I is
                 comparable BETWEEN nodes, which is what "this node is informative" has to
                 mean. Default.

WHY CALIBRATION IS TWO PASSES AND NOT ONE. The one-pass identity E||h-mu||^2 = E||h||^2 -
||mu||^2 is a catastrophic cancellation here. The deviation is ~1e-4 of the energy under
early-late fusion, float32 carries ~1e-7 relative, and the accumulation itself rounds. At
the real proportions (D=1024, deviation 1% of the state norm) that leaves a node which
never responds with a variance up to 4e-3 of a genuine one -- ragged, node-dependent, and
close enough to a real response to survive any sane cut. The second pass computes
||h - mu||^2 directly: the residue drops to ~1e-10 of the working scale and, more useful
still, becomes UNIFORM across the unreachable part of the graph, so one threshold removes
all of it. It costs one more forward over M references.

WHY THE THRESHOLD IS MEASURED, NOT DERIVED. The same second pass records the deviations
themselves, so tau_ref is the median of log(1+I) over the (node, reference query) pairs
that respond at all. Deriving it from V instead would compare a per-query deviation against
a mean-over-queries deviation, which sit at systematically different points of the same
distribution. Measuring puts the operating point exactly at "a typical reference query's
typical responding node", so dtau is a learned offset in interpretable units. Both the
variance and the recorded deviations are corrected to leave-one-out, since mu is built from
the same M references: V uses the M-1 denominator, and a stored deviation is scaled by
(M/(M-1))^2, which is exactly ||h_m - mu_{-m}||^2. A query the bank has never seen then
lands in the distribution tau_ref was measured on.

WHY I IS DETACHED FROM THE AUTOGRAD GRAPH BY DEFAULT. With a gradient path through I the
GNN can open every gate at once by drifting h away from mu -- mu is only recomputed every
CQIG_RECAL steps, so that direction is free, it raises I for every query equally, and it
destroys the mechanism while lowering the loss. Detached, the gate is a modulation whose
shape (lam, alpha, dtau) is still learned, and h still receives gradient through the
product h*g. Set grad_through_I=True to restore the differentiable version as an ablation.
Detaching also lets I be computed in float32 for free, which is worth having but is not a
cure. h is bfloat16 under autocast and the deviation is a small fraction of it, so most of
each component's significant digits are gone before the gate sees them. Measured against
float64 at D=1024: with the deviation at 1% of the state norm, doing the subtraction and
the sum in float32 gives ~2.6% median error on I against ~5.2% for the all-bfloat16 path
the earlier version used -- a factor of about two across M and across deviation ratios from
0.3% to 3%. The remainder is h's OWN quantisation and cannot be recovered here. What that
error scales with is the deviation ratio, not the arithmetic: at 0.3% of norm even the
float32 path carries ~29%, at 3% it carries ~0.3%. If a layer's states are nearly identical
across queries, I is noise there whatever the dtype, which is what `responding` in the
calibration report is for.

WHY A FORWARD PRE-HOOK AND NOT A FORK OF THE MESSAGE PASSING. The layer computes
`message = input_j * relation_j` for DistMult, which is LINEAR in the source state, and the
boundary condition arrives as a separate argument while the residual is added outside the
layer. Scaling `input` before the call is therefore exactly equivalent to scaling every
message leaving that node, and touches neither the boundary nor the shortcut.

THREE OPERATORS, ONE COEFFICIENT (`op`). The same pre-hook and the same statistic support a
second family, selected by `op`. Write d_qv = h_qv - mu_v for the query-specific residual.
Because a DistMult message is LINEAR in the source state and the relation embedding is a
function of the graph's relation text alone -- `rel_mlp(graph.rel_attr)` is expanded across
the batch and never sees the query -- E_a[ m(h_av, r) ] = m(mu_v, r) holds EXACTLY, so
"propagate the residual and retain a kappa-fraction of the background" is a subtraction on
the input and the hook implements it exactly rather than approximately:

    m(d_qv, r) + kappa m(mu_v, r) = m( h_qv - (1 - kappa) mu_v , r )

    op="gate"          h_qv * g_qv                 scale the whole state, the original
    op="centre"        h_qv - (1 - g_qv) mu_v      subtract background in proportion to
                                                   how UNinformative the node is
    op="centre-fixed"  h_qv - lam_l mu_v           the same subtraction at a constant rate

with kappa_qv = g_qv, so one coefficient covers all of it and lam_l = 0 recovers the ungated
reasoner exactly under every op. The ladder is nested: fixed centring is adaptive centring
with the coefficient frozen, and `op="centre-fixed"` at lam=1 is full centring, residual only.

WHAT ACTUALLY DIFFERS BETWEEN GATING AND CENTRING, given both are driven by the same g:

  * DIRECTION. Gating is a scalar multiple, so the message keeps its direction and only its
    magnitude moves. Centring removes a component, so the direction moves too. At full
    damping a gated node sends nothing while a centred node sends its residual d_qv. Whether
    silence or the residual is better is an empirical question about what a generic node's
    small d_qv contains -- encoder noise, or a weak but real cross-domain cue -- and it is
    the one thing the pair of arms genuinely tests.
  * WHICH NODES ARE TOUCHED. A node no reference query reaches has mu_v = 0 EXACTLY, so
    centring leaves it alone, while gating sees I = 0 and damps it to the floor. When seed
    coverage is partial these are not small print: they are opposite treatments of whatever
    fraction of the graph the references never entered, which `mu_zero_frac` reports.
  * QUERY-CONDITIONING. mu_v does not depend on the query, so `op="centre-fixed"` subtracts a
    query-INDEPENDENT background from every node. It still changes rankings -- a node fed by
    many broadly-active neighbours loses more -- but it is a hub correction, not
    query-conditioned reasoning, and alpha and dtau receive no gradient in that arm. Only
    the adaptive coefficient makes the operator a function of the query.
  * SIZE OF THE EDIT. Under `use_ent_emb: early-late-fusion` mu is dominated by the node's
    static entity embedding, so at a given lam the subtraction can be a far larger
    perturbation than the multiplication. `edit_rel` in the live report measures it directly
    as ||h~ - h|| / ||h||, which is also the number to read against `layer_norm: yes`: the
    norm renormalises much of a magnitude change away and leaves the direction change, so a
    centring arm should be expected to act through the direction.

GATING MORE THAN ONE LAYER. Calibration of layer l assumes the state entering it was
produced by the same computation at inference. With layers upstream of l gated, that holds
only if their gates were also active while l was calibrated. `rounds=1` calibrates every
gated layer with no gate active anywhere, which is exact for a single gated layer -- a
layer's own gate cannot change its own input -- and leaves the upstream mismatch in place
for a multi-layer arm. `rounds=2` repeats both passes with the round-1 gates live, which
removes it. Anything above 2 is a fixed-point iteration and has not been needed.

STATISTICS ARE PER GRAPH AND MUST BE SELECTED BEFORE A FORWARD. mu has one row per node, so
the train graph's mu is not even shape-compatible with the test graph's states. `_stats` is
keyed by a caller-supplied graph key and `use_graph(key)` must be called before any gated
forward. Reading statistics for the wrong graph raises rather than broadcasting.

ZERO-SHOT, AND WHAT THAT ACTUALLY REQUIRES. The reference bank is stored in a
GRAPH-INDEPENDENT form: a frozen question embedding plus the entity-linked start nodes as
NAMES with their attachment weights. `materialise(node2id, num_nodes)` re-links those same
source problems against any target graph, exactly as the indexer does
(`start_mask[node2id[name]] = weight`), and returns both the rebuilt batches and the
coverage of the source seeds in the target vocabulary. Every graph is therefore calibrated
with the SAME M individual reference problems, and nothing is ever read from the target
corpus's own query set. A reloaded checkpoint carries the bank in its state dict, so an
unseen graph needs one unlabelled calibration pass at indexing time and nothing else.

EXACT NAME LOOKUP IS NOT ENOUGH, AND THE FAILURE IS SILENT. Re-linking by string equality
assumes the target graph spells its concepts the way the source graph does. It does not:
on sir4_physics the source seeds resolved 501/501 against the train graph and 25/501
against the TEST graph, same domain, different papers. mu was then estimated from
references that barely entered the graph, and nothing failed loudly -- coverage was a
printed number, not an error. `link="semantic"` fixes the linker rather than the bank:

    exact match if the name exists; otherwise take the K nearest target nodes to the
    seed's FROZEN source embedding, reject any below a source-selected cosine floor,
    and split the seed's attachment weight across the survivors by softmax(cos / T).

The seed's total starting weight is preserved, so a semantically linked reference injects
exactly as much probability mass as an exactly linked one and the two are comparable. This
is still zero-shot: the reference questions are fixed on source data, the encoder is frozen,
the floor is chosen from source-side statistics, the target's labels and queries are never
touched, and nothing is fine-tuned. What changes is only that a reference problem can now
enter a target graph that does not happen to use the source's exact phrasing.

lam_l = 0 recovers the ungated reasoner exactly, so the ablation is one scalar.
"""
import math
import os

import torch
import torch.nn as nn

# A normalised response below this counts as "this node did not respond to this query".
# I is ~1 for a typical responding node by construction, so 1e-6 is six orders below the
# working range. With the deviation computed directly it only ever catches exact zeros.
RESP = 1e-6

# Bound on the transient float32 buffer used to compute ||h - mu||^2 without ever
# subtracting in bfloat16. ~256 MB.
_CHUNK_BYTES = 1 << 28

# Bound on the reference deviations kept for the threshold and the gate report. Whole
# rows are kept, never a flattened subsample, so a per-node scale stays aligned.
_SAMPLE_ELEMS = 8_000_000

# Target nodes per chunk when matching seeds by cosine. A seed set is a few hundred rows
# and a graph is a few hundred thousand, so the full product is ~0.5 GB in float32 and is
# built one slice at a time instead.
_LINK_CHUNK = 65_536


def topk_cosine(q, node_emb, k, chunk=_LINK_CHUNK):
    """Top-k cosine of each row of `q` against every row of `node_emb`.

    Both are L2-normalised here rather than by the caller, so a caller cannot pass one
    normalised and one not and get a silently rescaled similarity. Chunked over target
    nodes: the full [seeds, nodes] product is what makes this expensive, and it is never
    needed all at once.
    """
    qn = q.float()
    qn = qn / qn.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    n = int(node_emb.shape[0])
    k = max(1, min(int(k), n))
    best_c = torch.full((qn.shape[0], k), -2.0, device=qn.device)
    best_i = torch.zeros((qn.shape[0], k), dtype=torch.long, device=qn.device)
    for a in range(0, n, chunk):
        blk = node_emb[a:a + chunk].to(qn.device).float()
        blk = blk / blk.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        c = qn @ blk.T                                       # [S, chunk]
        kk = min(k, c.shape[1])
        cv, ci = torch.topk(c, kk, dim=1)
        best_c = torch.cat([best_c, cv], 1)
        best_i = torch.cat([best_i, ci + a], 1)
        best_c, order = torch.topk(best_c, k, dim=1)
        best_i = torch.gather(best_i, 1, order)
    return best_c, best_i


def _inv_softplus(y: float) -> float:
    return math.log(math.expm1(y))


def _logit(p: float) -> float:
    return math.log(p / (1.0 - p))


def _describe(t, qs=(0.1, 0.5, 0.9)):
    """min / quantiles / max / mean / std of a tensor, as plain floats.

    min, max, mean and std are exact over the whole tensor; only the quantiles fall back
    to a random subsample, because torch.quantile has a hard size ceiling.
    """
    keys = [f"p{int(round(q * 100))}" for q in qs]
    if t is None or t.numel() == 0:
        return dict({"n": 0, "min": 0.0, "max": 0.0, "mean": 0.0, "std": 0.0},
                    **{k: 0.0 for k in keys})
    t = t.detach().float().flatten()
    out = {"n": int(t.numel()), "min": float(t.min()), "max": float(t.max()),
           "mean": float(t.mean()),
           "std": float(t.std(unbiased=False)) if t.numel() > 1 else 0.0}
    s = t
    if s.numel() > 1_000_000:
        s = s[torch.randperm(s.numel(), device=s.device)[:1_000_000]]
    qv = torch.quantile(s, torch.tensor(list(qs), device=s.device, dtype=s.dtype))
    for k, v in zip(keys, qv):
        out[k] = float(v)
    return out


def parse_layers(spec, n_layers):
    """Resolve a gated-layer specification to 0-indexed layer positions.

    NUMBERS ARE 1-INDEXED, matching how the layers are talked about: with six layers,
    "6" is the last one and "3-6" is the second half. Accepted forms:

        None / "" / "last"     the final layer only (default, exact at rounds=1)
        "all"                  every layer
        "6" / "3-6" / "3,5,6"  1-indexed layer numbers, ranges inclusive
        [3, 4, 5, 6]           the same, as a list

    Returns 0-indexed positions, which is what the hook and the checkpoint use.
    """
    if spec is None:
        return {n_layers - 1}
    if isinstance(spec, str):
        s = spec.strip().strip("'\"").lower()
        if s in ("", "none", "null", "last"):
            return {n_layers - 1}
        if s == "all":
            return set(range(n_layers))
        nums = set()
        for part in s.replace(" ", "").split(","):
            if not part:
                continue
            if "-" in part.lstrip("-"):
                a, b = part.split("-", 1)
                lo, hi = int(a), int(b)
                assert lo <= hi, f"cqig layer range {part!r} runs backwards"
                nums.update(range(lo, hi + 1))
            else:
                nums.add(int(part))
    else:
        nums = {int(x) for x in spec}
    assert nums, f"cqig layer spec {spec!r} selects no layers"
    bad = sorted(x for x in nums if not 1 <= x <= n_layers)
    assert not bad, (f"cqig layer spec {spec!r} names layer(s) {bad}, but the model has "
                     f"{n_layers} layers, numbered 1..{n_layers} (1-indexed)")
    return {x - 1 for x in nums}


class CQIGate(nn.Module):
    """Per-layer informativeness gate, attached to an entity model's conv layers.

    `mode` is WHAT THE HOOK IS DOING RIGHT NOW and `op` is WHICH OPERATOR the arm runs.
    They are independent and the names are close, so they are spelled out here:

    mode: "off"        pass through untouched; the model is bit-identical to the ungated one
          "calibrate"  accumulate reference statistics for the active graph (two phases)
          "gate"       compute I against the active graph's statistics and edit the state

    op:   "gate"          h * g                 the original, and the default
          "centre"        h - (1 - g) * mu      adaptive centring
          "centre-fixed"  h - lam * mu          query-independent centring

    mu:   "ref"           mu from the reference bank; the method
          "zero"          mu forced to 0; THE ABLATION, see mu_zero below
    """

    OPS = ("gate", "centre", "centre-fixed")
    MUS = ("ref", "zero")

    def __init__(self, layers, lam_init=0.1, alpha_init=1.0, rho=1e-3, eps=1e-12,
                 gate_layers=None, norm="layer", grad_through_I=False, op="gate",
                 mu="ref"):
        super().__init__()
        self.n_layers = len(layers)
        self.gate_layers = parse_layers(gate_layers, self.n_layers)
        self.rho, self.eps = rho, eps
        assert norm in ("layer", "node", "energy"), (
            f"unknown cqig norm {norm!r}; one of 'layer' (default), 'node', 'energy'")
        self.norm = norm
        self.grad_through_I = bool(grad_through_I)
        op = str(op or "gate").strip().lower()
        assert op in self.OPS, (
            f"unknown cqig op {op!r}; one of {', '.join(map(repr, self.OPS))}. This is the "
            f"OPERATOR (what is done to the state), not `mode` (what the hook is doing).")
        self.op = op
        self.centring = op != "gate"
        mu = str(mu or "ref").strip().lower()
        assert mu in self.MUS, f"unknown cqig mu {mu!r}; one of {', '.join(map(repr, self.MUS))}"
        self.mu_source = mu
        self.mu_zero = mu == "zero"

        # lam in [0,1] via sigmoid, alpha > 0 via softplus.
        # lam_init is deliberately NOT 0: at exactly 0 the sigmoid derivative is negligible
        # and the gate could never learn to open, which would present as "it did nothing".
        self.lam_raw = nn.Parameter(torch.full((self.n_layers,), _logit(lam_init)))
        self.alpha_raw = nn.Parameter(torch.full((self.n_layers,), _inv_softplus(alpha_init)))
        # LEARNED OFFSET ONLY. The absolute threshold is graph-specific and recomputed at
        # every calibration; a single learned tau was silently overwritten by each recal,
        # so it never actually learned anything.
        self.dtau = nn.Parameter(torch.zeros(self.n_layers))

        # The reference bank travels with the model, in a form that is not tied to any
        # graph, so a reloaded checkpoint can calibrate a corpus it has never seen.
        self.ref_bank = None                  # list[dict]: {"qemb", "seeds", "id"}
        self.ref_meta = {}

        self.mode = "off"
        self._stats: dict = {}                # graph_key -> {layer: {mu, scale, ...}}
        self._active = None                   # graph_key currently selected
        self._acc: dict = {}                  # layer -> running sums for the current phase
        self._mu: dict = {}                   # layer -> mu, between the two phases
        self._energy: dict = {}               # layer -> mean ||h||^2, for norm="energy"
        self._acc_key, self._n_ref, self._m_ref = None, 0, 0
        self._phase = "mean"
        self._calib_gate = False              # apply existing gates while calibrating
        self._live = None                     # layer -> running gate histogram, when on
        self._live_bins = 256
        self._align: dict = {}                # layer -> gold-vs-hard-negative tally
        self._edit: dict = {}                 # layer -> ||h~ - h|| / ||h|| tally
        self.last_I: dict = {}                # layer -> [B, N] I of the last forward
        self._handles = [
            layer.register_forward_pre_hook(self._make_hook(i))
            for i, layer in enumerate(layers)
        ]
        self.last_gate = {}
        self._lowp_note = False

    # ------------------------------------------------------------------ precision
    def _apply(self, *args, **kwargs):
        """Keep lam_raw, alpha_raw and dtau in float32 through `model.to(dtype=...)`.

        THIS IS A CORRECTNESS FIX, NOT AN OPTIMISATION. The trainer casts the whole model
        with `model.to(dtype=torch.bfloat16)` (utils/setup_training.py), which catches these
        three scalars. A bfloat16 value has a 2^-8 relative ULP, so half an ULP is
        |w| * 2^-9 .. |w| * 2^-8, while an AdamW step is at most `lr`. At lr 5e-4 every
        parameter above |w| ~= 0.128 has its update rounded straight back to the old value
        on EVERY step, and because round-to-nearest keeps no remainder it never accumulates.

        lam_raw = logit(0.9) = 2.197 and alpha_raw = inv_softplus(1) = 0.541 are both above
        that line, so neither ever learned. The logs are their own proof: float32 init would
        report lam exactly 0.9000000 and alpha exactly 1.0000000, and what every run to date
        actually reported was lam = 0.9023438 = sigmoid(bf16(2.197)) and alpha = 1.00104 =
        softplus(bf16(0.541)). Those digits ARE the rounding error. dtau starts at 0 where
        bfloat16 resolves finely so it did move, but it stalls hard at |dtau| = 0.25, where
        half an ULP overtakes lr again.

        `.to()`, `.cuda()`, `.float()` and DDP all funnel through `_apply`, so refusing the
        cast here covers every call site and travels with this file alone. Device moves are
        unaffected: the saved copy is restored onto the parameter's NEW device. The values
        kept are the pre-cast float32 ones, so lam_init is exactly logit(0.9) again rather
        than its bfloat16 neighbour.

        Nothing downstream needs a matching change. I is already float32 out of `_sqdist`,
        and both operator branches cast the coefficient back themselves -- `g.to(h.dtype)`
        in `_apply_op` and the per-chunk `.to(h.dtype)` in `_centre`. So the gate is
        computed in float32 and only the coefficient crosses into bfloat16, which is what
        mixed precision was supposed to be doing. Set CQIG_ALLOW_LOWP=1 to reproduce the old
        frozen-scalar runs.
        """
        saved = {n: p.detach().clone().float()
                 for n, p in self.named_parameters(recurse=False)}
        out = super()._apply(*args, **kwargs)
        if os.environ.get("CQIG_ALLOW_LOWP", "0") == "1":
            return out
        pinned = []
        for n, p in self.named_parameters(recurse=False):
            if n in saved and p.is_floating_point() and p.dtype != torch.float32:
                was = p.dtype
                p.data = saved[n].to(device=p.device)
                pinned.append((n, was))
        if pinned and not self._lowp_note:
            self._lowp_note = True
            print(f"[cqig] gate scalars pinned to float32 against a "
                  f"{pinned[0][1]} model cast: {', '.join(n for n, _ in pinned)}. "
                  f"These are learnable; in {pinned[0][1]} an lr-5e-4 step is below half "
                  f"an ULP for lam_raw and alpha_raw, so they would never move.")
        return out

    # ------------------------------------------------------------------ parameters
    def lam(self, i):
        return torch.sigmoid(self.lam_raw[i])

    def alpha(self, i):
        return nn.functional.softplus(self.alpha_raw[i])

    def tau(self, i, key=None):
        st = self._stats.get(key if key is not None else self._active) or {}
        return st.get(i, {}).get("tau_ref", 0.0) + float(self.dtau[i].detach())

    def _g_of(self, i, I, tau):
        return 1.0 - self.lam(i) * (
            1.0 - torch.sigmoid(self.alpha(i) * (torch.log1p(I) - tau))
        )

    @staticmethod
    def _g_scalars(I, lam, alpha, tau):
        """The same gate from plain floats, for reporting off-device."""
        return 1.0 - lam * (1.0 - torch.sigmoid(alpha * (torch.log1p(I) - tau)))

    # ------------------------------------------------------------------ graph selection
    def use_graph(self, key):
        """Select which graph's statistics the gate reads. Required before a gated forward."""
        self._active = key

    def calibrated(self, key):
        return key in self._stats

    def known_graphs(self):
        return sorted(self._stats)

    # ------------------------------------------------------------------ reference bank
    def set_reference_bank(self, refs, meta=None):
        """Store M source reference problems in a GRAPH-INDEPENDENT form.

        Each ref is {"qemb": [1, D] float tensor, "seeds": [(name, weight), ...], "id": str}
        and optionally "seed_emb": [len(seeds), D], the FROZEN source-graph embedding of each
        seed name. Names alone re-link only where the target spells a concept identically;
        the embeddings are what let `link="semantic"` find "thin-film growth" from "thin film
        deposition". They are source-side artefacts of a frozen encoder, so carrying them
        costs one float32 row per seed (~1.8 MB at M=16) and concedes nothing about zero-shot.
        """
        bank = []
        for r in refs:
            e = r["qemb"].detach().float().cpu()
            assert e.dim() == 2 and e.shape[0] == 1, (
                f"a reference must be ONE query, got question_embeddings of shape "
                f"{tuple(e.shape)}; batches of queries are not references")
            seeds = [(str(n), float(w)) for n, w in r["seeds"]]
            se = r.get("seed_emb")
            if se is not None:
                se = se.detach().float().cpu()
                # ROW k OF seed_emb IS SEED k. If that correspondence slips, every semantic
                # link is to the wrong concept and nothing downstream can detect it.
                assert se.dim() == 2 and se.shape[0] == len(seeds), (
                    f"seed_emb has {tuple(se.shape)} rows for {len(seeds)} seeds; they are "
                    f"positionally paired and must be built together")
            bank.append({"qemb": e, "seeds": seeds, "seed_emb": se, "id": r.get("id")})
        assert len(bank) >= 2, (
            f"a reference bank of {len(bank)} cannot give a cross-query variance; "
            f"raise CQIG_M")
        self.ref_bank = bank
        self.ref_meta = dict(meta or {})

    def materialise(self, node2id, num_nodes, device=None, dtype=None, batch_size=1,
                    node_emb=None, link="exact", link_k=3, link_temp=0.05, link_floor=0.0):
        """Re-link the stored source problems against THIS graph's vocabulary.

        `link="exact"` is the operation the indexer performs when it builds start_nodes_mask:
        `start_mask[node2id[name]] = weight`, silently dropping names the graph does not
        contain. That silent drop is the failure mode: 5% coverage and 100% coverage produce
        the same shapes and the same absence of errors.

        `link="semantic"` keeps every exact match and rescues the rest. A seed the target
        does not spell is embedded (its frozen source embedding is in the bank), matched to
        its `link_k` nearest target nodes, filtered at `link_floor`, and its weight split
        across the survivors by softmax(cos / link_temp). The SPLIT IS NORMALISED OVER THE
        SURVIVORS, so the seed contributes exactly its original weight whether it landed on
        one node or three, and an exactly linked seed and a semantically linked one inject
        the same mass. `link_floor` comes from the source side (see the trainer), so no
        target statistic sets it.

        Returns (batches, coverage). Coverage separates exact from semantic hits, because
        an aggregate that mixes them cannot answer the question the arm exists to ask.

        Every graph gets the SAME M problems, so `coverage["queries"]` is M on every graph
        and two graphs' statistics are computed from equal-sized reference sets.
        """
        assert self.ref_bank, "no reference bank; call set_reference_bank first"
        assert link in ("exact", "semantic"), f"unknown cqig link mode {link!r}"
        semantic = link == "semantic"
        if semantic:
            assert node_emb is not None, (
                "link='semantic' needs the TARGET graph's node embeddings (graph.x); it is "
                "None, so this graph was indexed without node features")
            assert int(node_emb.shape[0]) == int(num_nodes), (
                f"node_emb has {int(node_emb.shape[0])} rows for a {num_nodes}-node graph")
            missing = [r["id"] for r in self.ref_bank if r.get("seed_emb") is None]
            assert not missing, (
                f"{len(missing)} reference(s) carry no seed embeddings, so they cannot be "
                f"linked semantically. The bank was frozen before this feature existed; "
                f"retrain, or run this arm with CQIG_LINK=exact")
        masks, embs, total, empty = [], [], 0, 0
        n_exact = n_sem = 0
        sem_cos, sem_deg = [], []
        for r in self.ref_bank:
            mask = torch.zeros(num_nodes, dtype=torch.float32)
            hit, pend = 0, []
            for k, (name, w) in enumerate(r["seeds"]):
                j = node2id.get(name)
                if j is not None and 0 <= int(j) < num_nodes:
                    # ACCUMULATE. Exact hits have distinct ids so this matches the old
                    # assignment, but a semantic link can land on an already-seeded node
                    # and overwriting there would quietly destroy the other seed's weight.
                    mask[int(j)] += w
                    n_exact += 1
                    hit += 1
                elif semantic:
                    pend.append((k, w))
            if pend:
                # Onto the GRAPH's device, not the bank's: the bank is a few hundred CPU
                # rows and the graph is hundreds of thousands, so moving the small side is
                # the only direction that does not stream the whole graph through host RAM.
                q = r["seed_emb"][[k for k, _ in pend]].to(node_emb.device)
                cos, idx = topk_cosine(q, node_emb, link_k)
                cos, idx = cos.cpu(), idx.cpu()
                for t, (_, w) in enumerate(pend):
                    keep = cos[t] >= link_floor
                    if not bool(keep.any()):
                        continue                      # below the floor: a real miss, not a link
                    c, ix = cos[t][keep], idx[t][keep]
                    # Softmax over the SURVIVORS, times w: total attachment weight preserved.
                    wt = torch.softmax(c / max(float(link_temp), 1e-6), dim=0) * w
                    mask.index_add_(0, ix, wt)
                    n_sem += 1
                    hit += 1
                    sem_cos.append(float(c.max()))
                    sem_deg.append(int(ix.numel()))
            total += len(r["seeds"])
            empty += int(hit == 0)
            masks.append(mask)
            embs.append(r["qemb"])
        bs = max(1, int(batch_size))
        batches = []
        for a in range(0, len(masks), bs):
            b = {"question_embeddings": torch.cat(embs[a:a + bs], 0).clone(),
                 "start_nodes_mask": torch.stack(masks[a:a + bs], 0)}
            if device is not None:
                b = {k: v.to(device) for k, v in b.items()}
            if dtype is not None:
                b = {k: (v.to(dtype) if v.is_floating_point() else v) for k, v in b.items()}
            batches.append(b)
        found = n_exact + n_sem
        med = sorted(sem_cos)[len(sem_cos) // 2] if sem_cos else float("nan")
        cov = {"queries": len(masks),
               "batches": len(batches),
               "seeds_found": found,
               "seeds_total": total,
               "seed_coverage": (found / total) if total else 0.0,
               # KEPT SEPARATE ON PURPOSE. The semantic arm's whole claim is that it raises
               # coverage, so an aggregate that hides which half moved cannot test it.
               "seeds_exact": n_exact,
               "seeds_semantic": n_sem,
               "seeds_unmatched": total - found,
               "exact_coverage": (n_exact / total) if total else 0.0,
               "link": link,
               "link_k": int(link_k) if semantic else 0,
               "link_temp": float(link_temp) if semantic else 0.0,
               "link_floor": float(link_floor) if semantic else 0.0,
               "sem_cos_median": med,
               "sem_nodes_per_seed": (sum(sem_deg) / len(sem_deg)) if sem_deg else 0.0,
               "queries_with_no_seed": empty}
        return batches, cov

    # ------------------------------------------------------------------ calibration
    def start_calibration(self, key, phase="mean", apply_existing_gates=False):
        """Begin one reference pass for `key`.

        phase="mean" accumulates sum h and sum ||h||^2; phase="dev" accumulates
        sum ||h - mu||^2 and keeps a sample of the per-query deviations. Both passes must
        see the same M references, which finish_calibration checks.

        apply_existing_gates: run the already-calibrated gates while re-accumulating. This
        is what makes a second round consistent when more than one layer is gated -- see
        the module docstring. It is a no-op on the first round, when nothing is calibrated.
        """
        assert phase in ("mean", "dev"), f"unknown calibration phase {phase!r}"
        if phase == "dev":
            assert self._mu, "phase 'dev' needs phase 'mean' to have finished first"
        else:
            self._mu, self._energy = {}, {}
        self.mode, self._acc, self._acc_key, self._n_ref = "calibrate", {}, key, 0
        self._phase = phase
        self._active = key
        self._calib_gate = bool(apply_existing_gates) and key in self._stats
        if phase == "mean" and not self._calib_gate:
            # DROP THE OUTGOING STATISTICS BEFORE BUILDING THEIR REPLACEMENT. mu is one
            # float32 row per node, which on the CS train graph is 950 MB per gated layer;
            # holding the old set alive through a recalibration doubles the peak for no
            # reason, since nothing reads it until the new one is installed. Kept only when
            # apply_existing_gates asked for it, which is the round-2 path.
            self._stats.pop(key, None)

    def note_reference_batch(self, b):
        self._n_ref += b

    def finish_mean(self):
        """Close phase "mean": turn the running sums into mu (and the energy)."""
        assert self._n_ref > 0, "no reference queries were run"
        assert self._acc, "calibration collected nothing; are any layers in gate_layers?"
        # div_, not div: the accumulator IS the mean once divided, and nothing else holds a
        # reference to it. Allocating a second [N, D] float32 here would be another 950 MB
        # per gated layer on the CS graph, at the moment memory is already at its peak.
        #
        # mu="zero" IS THE ABLATION THAT DECIDES WHETHER THE REFERENCE BANK DOES ANYTHING.
        # With mu = 0 the statistic degenerates to I = ||h||^2 / s, an activation-MAGNITUDE
        # gate, and CQIG's whole claim -- that a node earns its message by responding to
        # THIS query unlike it responds to unrelated ones -- is gone. If this arm reproduces
        # the method's score, the reference bank was never load-bearing and the mechanism is
        # magnitude gating under another name.
        #
        # The mean pass still RUNS and its result is discarded. That wastes M forwards, and
        # it is the point: the two arms then differ in the value of mu and in nothing else,
        # not in code path, not in RNG consumption. Zeroed in place, so no extra memory, and
        # every quantity derived downstream (V, the layer scale s, tau_ref) is recomputed
        # consistently against mu = 0 rather than being inherited from a real one.
        self._mu = {i: (a["sum_h"].zero_() if self.mu_zero else a["sum_h"].div_(self._n_ref))
                    for i, a in self._acc.items()}
        self._energy = {i: a["sum_e"].div_(self._n_ref) for i, a in self._acc.items()}
        self._m_ref, self._acc, self.mode = self._n_ref, {}, "off"
        return len(self._mu)

    def finish_calibration(self):
        """Close phase "dev": variance, layer scale, threshold, and what the gate will do.

        s is the MEDIAN reference variance over responding nodes, found in two steps: a
        median over V > 0 fixes the order of magnitude, then nodes more than six orders
        below it are dropped and the median is retaken. A single median over V > 0 would be
        pulled down by whatever residue the barely-reachable part of the graph leaves.
        """
        M = self._n_ref
        assert M >= 2, f"the variance needs at least 2 reference queries, got {M}"
        assert M == self._m_ref, (
            f"the mean pass saw {self._m_ref} reference queries and the deviation pass "
            f"{M}; the two must be the same set")
        assert self._acc, "the deviation pass collected nothing"
        loo = (M / (M - 1.0)) ** 2         # in-sample deviation -> leave-one-out deviation
        report, per_layer = {}, {}
        for i, a in self._acc.items():
            mu = self._mu[i]
            V = a["sum_sq"] / (M - 1.0)                    # unbiased cross-query variance
            # WHICH NODES COUNT AS RESPONDING, AND WHY THE CUT IS ANCHORED AT THE TOP.
            # mu is a mean of M float32 values, so a node whose state never moves still
            # lands ~1e-10 of the working scale away from it -- uniformly, on every
            # unreachable node. Most of a KG is unreachable from any one query's seeds, so
            # a cut anchored at a MEDIAN over V > 0 sits inside that dust cloud, keeps
            # every node "live", and hands back a scale ten orders too small. The dust is
            # many orders below any real response and the response range spans at most a
            # few, so anchoring six orders under the largest variance separates them with
            # room on both sides.
            vmax = float(V.max()) if V.numel() else 0.0
            n_pos = int((V > 0).sum())
            live = (V > RESP * vmax) if vmax > 0 else (V > 0)
            s = float(V[live].median()) if int(live.sum()) else 0.0
            n_live = int(live.sum())
            vbar = float(V[live].mean()) if n_live else 0.0
            degenerate = not (s > 0)
            if self.norm == "layer":
                # ONE robust scale for the whole layer, so I is comparable BETWEEN nodes.
                # s == 0 means no node responded to any reference query; 1.0 keeps I finite
                # and the report says so rather than dividing by eps.
                scale = torch.full((1, 1), (s if s > 0 else 1.0) + self.eps,
                                   device=V.device, dtype=torch.float32)
            elif self.norm == "node":                      # ablation: self-normalising
                scale = (V + (self.rho * vbar + self.eps)).unsqueeze(0)
            else:                                          # ablation: the original energy
                scale = (self._energy[i] + (self.rho + self.eps)).unsqueeze(0)
            per_layer[i] = {"mu": mu, "scale": scale, "live": live, "V": V, "s": s}

            # --- what a reference query actually scores here, and what the gate does ----
            sample = torch.cat(a["rows"], 0) * loo         # [rows, N] on cpu
            I = sample / scale.detach().to(sample.device)
            resp = I[I > RESP]
            if resp.numel() >= 100:
                tau_ref = float(torch.log1p(resp).median())
                tau_src = "reference queries"
            else:
                # Too few responding pairs for a median to mean anything: fall back to the
                # variance, which is the same quantity averaged over queries.
                iota = (V.unsqueeze(0) / scale).flatten()
                tau_ref = float(torch.log1p(iota[live]).median()) if n_live else 0.0
                tau_src = f"reference variance (only {int(resp.numel())} responding pairs)"
            per_layer[i]["tau_ref"] = tau_ref
            per_layer[i]["tau_src"] = tau_src
            lam, alpha = float(self.lam(i).detach()), float(self.alpha(i).detach())
            tau = tau_ref + float(self.dtau[i].detach())
            gv = self._g_scalars(I, lam, alpha, tau)
            # WHAT THE ACTIVE OP MULTIPLIES BY, not what "gate" would have. Reporting g
            # under a centring arm would describe an operator that arm does not run.
            cv = (gv if self.op == "gate" else
                  torch.full_like(gv, lam) if self.op == "centre-fixed" else 1.0 - gv)
            # mu_v = 0 EXACTLY is "no reference query ever reached this node". A centring op
            # cannot touch those nodes at all while a gating op damps them hardest, so this
            # fraction is how much of the graph the two arms treat oppositely.
            mu_zero = int((mu.norm(dim=-1) == 0).sum())
            report[i] = {
                "nodes": int(V.numel()), "responding": n_pos, "live": n_live,
                "responding_frac": n_live / max(int(V.numel()), 1),
                "scale": s, "var_max": vmax, "var_mean_live": vbar,
                "var_q": _describe(V[live]) if n_live else _describe(None),
                "I": _describe(I), "gate": _describe(gv),
                "op": self.op, "coef": _describe(cv),
                "mu_source": self.mu_source, "mu_zero": mu_zero,
                "mu_zero_frac": mu_zero / max(int(V.numel()), 1),
                "gate_unreached": float(self._g_scalars(
                    torch.zeros(()), lam, alpha, tau)),
                "resp_frac": float(resp.numel()) / max(int(I.numel()), 1),
                "sample_queries": int(sample.shape[0]),
                "degenerate": degenerate, "lam": lam, "alpha": alpha, "tau": tau,
                "tau_ref": tau_ref, "tau_src": tau_src, "M": M,
            }
        self._stats[self._acc_key] = per_layer
        self._active = self._acc_key
        self._acc, self._mu, self._energy = {}, {}, {}
        self._acc_key, self.mode, self._calib_gate = None, "off", False
        return report

    # ------------------------------------------------------------------ live measurement
    def reset_live(self, bins=1024):
        """Start accumulating the gate distribution over REAL forwards, and the
        gold-vs-hard-negative alignment. Both are measurement only: nothing is calibrated
        from them and no target query or label enters the statistics."""
        self._live, self._align, self._edit, self.last_I = {}, {}, {}, {}
        self._live_bins = int(bins)

    def stop_live(self):
        self._live, self._align, self._edit, self.last_I = None, {}, {}, {}

    def recording(self):
        return self._live is not None

    def _note_live(self, i, g):
        a = self._live.get(i)
        if a is None:
            a = self._live[i] = {
                "n": 0,
                "sum": torch.zeros((), device=g.device, dtype=torch.float64),
                "sqs": torch.zeros((), device=g.device, dtype=torch.float64),
                "min": torch.full((), float("inf"), device=g.device),
                "max": torch.full((), float("-inf"), device=g.device),
                "hist": torch.zeros(self._live_bins, device=g.device),
            }
        f = g.flatten().float()
        a["n"] += int(f.numel())
        a["sum"] += f.sum().double()
        a["sqs"] += (f * f).sum().double()
        a["min"] = torch.minimum(a["min"], f.min())
        a["max"] = torch.maximum(a["max"], f.max())
        a["hist"] += torch.histc(f, bins=self._live_bins, min=0.0, max=1.0)

    def _note_edit(self, i, hh, h):
        """Running ||h~ - h|| / ||h|| over nodes, for whichever op is active.

        Measured on the states themselves rather than derived from the coefficient, because
        the two ops move a state by different amounts at the same coefficient and the point
        of the number is to compare them. Nodes with a numerically zero state are dropped
        rather than clamped: most of a KG is unreachable from one query's seeds, and a ratio
        of 0/eps there would dominate the mean with an artefact.
        """
        a = self._edit.get(i)
        if a is None:
            a = self._edit[i] = {
                "n": 0,
                "sum": torch.zeros((), device=h.device, dtype=torch.float64),
                "max": torch.zeros((), device=h.device),
                "moved": 0,
            }
        hn = h.float().norm(dim=-1)                              # [B, N]
        dn = (hh.float() - h.float()).norm(dim=-1)
        live = hn > 0
        if not bool(live.any()):
            return
        rel = dn[live] / hn[live]
        a["n"] += int(rel.numel())
        a["sum"] += rel.sum().double()
        a["max"] = torch.maximum(a["max"], rel.max())
        a["moved"] += int((dn[live] > 0).sum())

    def note_alignment(self, target_mask, doc_idx, doc_scores, k=50):
        """Is a high I actually concentrated on the gold papers?

        The hypothesis behind the gate is that informative nodes lie on the paths that
        matter. This is its most direct falsifiable form: for every query, compare I at the
        GOLD document nodes against I at the `k` documents the semantic scorer ranks
        highest among the non-golds -- the same hard negatives the training objective uses.
        The statistic is the AUC, i.e. the probability that a random gold outscores a random
        hard negative in informativeness, with ties counted as half. 0.5 is no alignment,
        and no amount of tuning lam/alpha/tau can rescue a gate whose statistic sits there.
        """
        if self._live is None or not self.last_I or target_mask is None:
            return
        with torch.no_grad():
            tgt = target_mask.index_select(1, doc_idx.to(target_mask.device)).bool()
            for i, I in self.last_I.items():
                Id = I.index_select(1, doc_idx.to(I.device)).float()
                a = self._align.setdefault(
                    i, {"wins": 0.0, "n": 0, "gold": 0.0, "neg": 0.0})
                for b in range(min(Id.shape[0], tgt.shape[0])):
                    gold = tgt[b].nonzero(as_tuple=False).flatten()
                    if gold.numel() == 0:
                        continue
                    s = doc_scores[b].detach().float().clone()
                    s[gold] = float("-inf")
                    kk = int(min(k, s.numel() - gold.numel()))
                    if kk <= 0:
                        continue
                    neg = s.topk(kk).indices
                    gi = Id[b].index_select(0, gold.to(Id.device)).unsqueeze(1)
                    ni = Id[b].index_select(0, neg.to(Id.device)).unsqueeze(0)
                    a["wins"] += float((gi > ni).float().mean()
                                       + 0.5 * (gi == ni).float().mean())
                    a["n"] += 1
                    a["gold"] += float(gi.mean())
                    a["neg"] += float(ni.mean())

    def live_report(self):
        """Gate distribution over the real forwards, plus the alignment AUC."""
        out = {}
        for i, a in (self._live or {}).items():
            n = max(a["n"], 1)
            mean = float(a["sum"]) / n
            var = max(float(a["sqs"]) / n - mean * mean, 0.0)
            lo, hi = float(a["min"]), float(a["max"])
            h = a["hist"]
            cdf = torch.cumsum(h, 0) / max(float(h.sum()), 1.0)
            edges = (torch.arange(self._live_bins, device=h.device).float() + 0.5) \
                / self._live_bins
            qv = {}
            for q in (0.1, 0.5, 0.9):
                j = int(torch.searchsorted(cdf, torch.tensor(q, device=h.device)).clamp(
                    max=self._live_bins - 1))
                # CLAMPED INTO THE OBSERVED RANGE. min and max are exact while the
                # quantiles are bin centres, so an unclamped p10 can print below the min
                # and read as a bug in the gate rather than in the histogram.
                qv[f"p{int(q * 100)}"] = min(max(float(edges[j]), lo), hi)
            al = self._align.get(i)
            ok = bool(al and al["n"])
            ed = self._edit.get(i)
            out[i] = {"n": a["n"], "min": float(a["min"]), "max": float(a["max"]),
                      "mean": mean, "std": var ** 0.5, **qv,
                      "op": self.op,
                      "auc": (al["wins"] / al["n"]) if ok else float("nan"),
                      "auc_n": (al["n"] if al else 0),
                      "I_gold": (al["gold"] / al["n"]) if ok else float("nan"),
                      "I_neg": (al["neg"] / al["n"]) if ok else float("nan"),
                      "edit_rel_mean": (float(ed["sum"]) / max(ed["n"], 1)) if ed else 0.0,
                      "edit_rel_max": float(ed["max"]) if ed else 0.0,
                      # What fraction of live nodes the operator moved AT ALL. Under a
                      # centring op this is the coverage question in its bluntest form:
                      # mu_v = 0 means the node is untouched however uninformative it is.
                      "edit_frac": (ed["moved"] / max(ed["n"], 1)) if ed else 0.0}
        return out

    # ------------------------------------------------------------------ the hook
    @staticmethod
    def _chunk(n_rows, d):
        return max(1024, int(_CHUNK_BYTES // max(n_rows * d * 4, 1)))

    def _accumulate_mean(self, i, h):
        """Running sum_h and sum_||h||^2, in float32 and in chunks over the node axis.

        FLOAT32 ACCUMULATORS. Under autocast h is bfloat16, whose ~8-bit mantissa would
        lose most of a 64-term running sum. Chunked because the float32 upcast of a
        [B, N, D] state is 2 GB at B=8 on this graph.
        """
        b, n, d = h.shape
        acc = self._acc.setdefault(i, {
            "sum_h": torch.zeros(n, d, device=h.device, dtype=torch.float32),
            "sum_e": torch.zeros(n, device=h.device, dtype=torch.float32),
        })
        step = self._chunk(b, d)
        for a0 in range(0, n, step):
            a1 = min(a0 + step, n)
            hd = h[:, a0:a1].detach().float()
            acc["sum_h"][a0:a1] += hd.sum(0)
            acc["sum_e"][a0:a1] += (hd * hd).sum(-1).sum(0)

    def _accumulate_dev(self, i, h):
        """Running sum ||h - mu||^2, plus a bounded sample of the deviations themselves."""
        mu = self._mu.get(i)
        if mu is None or mu.shape[0] != h.shape[1]:
            return
        sq = self._sqdist(h.detach(), mu)                  # [B, N] float32, exact
        acc = self._acc.setdefault(i, {
            "sum_sq": torch.zeros(h.shape[1], device=h.device, dtype=torch.float32),
            "rows": [], "kept": 0})
        acc["sum_sq"] += sq.sum(0)
        cap = max(1, _SAMPLE_ELEMS // max(h.shape[1], 1))
        if acc["kept"] < cap:
            take = min(sq.shape[0], cap - acc["kept"])
            acc["rows"].append(sq[:take].cpu())
            acc["kept"] += take

    def _sqdist(self, h, mu):
        """||h - mu||^2 per node, in float32, chunked over the node axis.

        The subtraction must not happen in bfloat16: the deviation is ~1% of the state
        under early-late fusion and bfloat16 resolves ~0.4%, so a bfloat16 difference is
        ~40% rounding noise. Chunking keeps the float32 temporary bounded.
        """
        b, n, d = h.shape
        out = torch.empty(b, n, device=h.device, dtype=torch.float32)
        step = self._chunk(b, d)
        for a0 in range(0, n, step):
            a1 = min(a0 + step, n)
            df = h[:, a0:a1].float() - mu[a0:a1].unsqueeze(0)
            out[:, a0:a1] = (df * df).sum(-1)
        return out

    def _centre(self, h, mu, c):
        """h - c * mu per node, chunked over the node axis, never in float32 at full size.

        `c` is the subtraction coefficient: [B, N] for an adaptive op, 0-dim for a fixed one.
        Written naively as `h - c.unsqueeze(-1) * mu` this allocates a float32 [B, N, D]
        because mu is float32 -- 1.9 GB at B=1 on the CS graph, twice the state it is
        editing, and then another copy to cast back. Chunked, with the cast to h's dtype
        INSIDE the chunk, the transient is bounded by _CHUNK_BYTES the same way _sqdist is.
        The slice assignments are autograd-tracked copies into a fresh tensor, so gradient
        still reaches h, lam, alpha and dtau; the loop covers 0..n so nothing is left
        uninitialised.
        """
        b, n, d = h.shape
        out = torch.empty_like(h)
        step = self._chunk(b, d)
        fixed = c.dim() == 0
        for a0 in range(0, n, step):
            a1 = min(a0 + step, n)
            cc = (c if fixed else c[:, a0:a1].unsqueeze(-1)).to(h.dtype)
            out[:, a0:a1] = h[:, a0:a1] - cc * mu[a0:a1].to(h.dtype).unsqueeze(0)
        return out

    def _coef(self, i, g):
        """The coefficient the active op multiplies by, given the gate value.

        Returned as the quantity actually applied, so the report and the live histogram
        describe what happened rather than what would have happened under `op="gate"`.
        """
        if self.op == "gate":
            return g
        if self.op == "centre-fixed":
            return self.lam(i)                 # 0-dim: query-independent by construction
        return 1.0 - g                         # "centre": damp the background, not the state

    def _apply_op(self, i, h, st, I, detach_dtau=False):
        """Edit the state under the active op. Returns (h~, g), g for the live report."""
        dt = float(self.dtau[i].detach()) if detach_dtau else self.dtau[i]
        g = self._g_of(i, I, st["tau_ref"] + dt)
        if self.op == "gate":
            return h * g.to(h.dtype).unsqueeze(-1), g
        # mu is the statistic being subtracted, so a centring op is a NO-OP wherever the
        # references never reached (mu_v = 0 exactly), which is the opposite of what gating
        # does there. See the module docstring; mu_zero_frac reports how much of the graph
        # that is.
        return self._centre(h, st["mu"], self._coef(i, g)), g

    def _stats_for(self, i, h):
        assert self._active is not None, (
            "cqig is in gate mode with no active graph; call use_graph(key) first")
        st = self._stats.get(self._active)
        assert st is not None and i in st, (
            f"cqig has no statistics for graph {self._active!r} at layer {i}; "
            f"calibrate this graph before scoring it (known: {self.known_graphs()})")
        # A wrong-graph lookup would otherwise fail as a bare broadcast error deep in
        # autograd. Say which graph and which shapes.
        assert st[i]["mu"].shape[0] == h.shape[1], (
            f"cqig statistics for graph {self._active!r} have {st[i]['mu'].shape[0]} nodes "
            f"but this forward has {h.shape[1]}; the wrong graph's statistics are active")
        return st[i]

    def _make_hook(self, i):
        def hook(_module, args):
            # args = (input, query, boundary, edge_index, edge_type, size, edge_weight)
            if i not in self.gate_layers or self.mode == "off":
                return None                    # not gated: never calibrated either
            h = args[0]                                    # [B, N, D] = h^(l)
            if self.mode == "calibrate":
                with torch.no_grad():
                    if self._phase == "mean":
                        self._accumulate_mean(i, h)
                    else:
                        self._accumulate_dev(i, h)
                    if not self._calib_gate:
                        return None                        # pass through unchanged
                    st = (self._stats.get(self._active) or {}).get(i)
                    if st is None or st["mu"].shape[0] != h.shape[1]:
                        return None
                    I = self._sqdist(h, st["mu"]) / st["scale"]
                    hh, _ = self._apply_op(i, h, st, I, detach_dtau=True)
                return (hh,) + tuple(args[1:])
            if self.mode != "gate":
                return None
            st = self._stats_for(i, h)
            # DETACHED BY DEFAULT: see the module docstring. A gradient path through I is a
            # free direction that opens every gate at once, and detaching also lets the
            # float32 chunked subtraction cost nothing in stored activations.
            if self.grad_through_I:
                I = self._sqdist(h, st["mu"]) / st["scale"]
            else:
                with torch.no_grad():
                    I = self._sqdist(h, st["mu"]) / st["scale"]
            # Only the messages are edited. The boundary condition is a separate argument
            # and the residual is added outside the layer, so both are left alone. The
            # coefficient is cast back to h's dtype inside the op: a float32 one would
            # silently promote the whole state and double every gated layer's activations.
            hh, g = self._apply_op(i, h, st, I)
            # KEPT AS A TENSOR. float() here is a device sync on every gated layer of every
            # training step, which at four gated layers is four stalls per step for a
            # number only summary() ever reads.
            self.last_gate[i] = g.detach().mean()
            if self._live is not None:
                with torch.no_grad():
                    self._note_live(i, self._coef(i, g).detach().expand_as(g))
                    self.last_I[i] = I.detach()
                    # HOW BIG THE EDIT ACTUALLY IS, relative to the state it edits. Under
                    # early-late fusion h is dominated by the static entity embedding, so
                    # the same lam is a very different perturbation under the two ops, and
                    # `layer_norm: yes` renormalises much of a magnitude change away. This
                    # is the number that says whether the operator did anything at all.
                    self._note_edit(i, hh.detach(), h.detach())
            return (hh,) + tuple(args[1:])
        return hook

    # ------------------------------------------------------------------ checkpointing
    def get_extra_state(self):
        return {"ref_bank": self.ref_bank, "ref_meta": self.ref_meta,
                "gate_layers": sorted(self.gate_layers), "norm": self.norm,
                "op": self.op, "mu": self.mu_source}

    def set_extra_state(self, state):
        if not state:
            return
        # STATISTICS BELONG TO A (WEIGHTS, GRAPH) PAIR, SO NEW WEIGHTS INVALIDATE THEM.
        # `load_best_model_at_end` defaults to true, so the final evaluate() and predict()
        # run on the BEST checkpoint while _stats still held mu from the LAST epoch's
        # weights -- and the recalibration guard, which keys on the training step, saw no
        # reason to refresh them. The reported numbers would have been the best model
        # gated by a different model's reference statistics. Clearing here forces one
        # calibration pass on the next inference, which is the cheap and correct answer.
        if self._stats:
            print(f"[cqig] checkpoint loaded: dropping reference statistics for "
                  f"{self.known_graphs()}; they belong to the previous weights")
        self._stats, self._active = {}, None
        self.ref_bank = state.get("ref_bank")
        self.ref_meta = state.get("ref_meta", {})
        if state.get("gate_layers"):
            want = set(state["gate_layers"])
            if want != self.gate_layers:
                print(f"[cqig] checkpoint gates layers {sorted(want)} (0-indexed), the "
                      f"config asked for {sorted(self.gate_layers)}; using the checkpoint's")
            self.gate_layers = want
        if state.get("norm"):
            if state["norm"] != self.norm:
                print(f"[cqig] checkpoint was normalised by {state['norm']!r}, the config "
                      f"asked for {self.norm!r}; using the checkpoint's")
            self.norm = state["norm"]
        # THE OPERATOR IS PART OF THE TRAINED MODEL, not a scoring-time choice: lam, alpha
        # and dtau were fitted under one of them. Absent on a checkpoint written before this
        # existed, which can only have been "gate".
        op = state.get("op", "gate")
        if op != self.op:
            print(f"[cqig] checkpoint was trained with op={op!r}, the config asked for "
                  f"{self.op!r}; using the checkpoint's")
        self.op, self.centring = op, op != "gate"
        # Same argument for mu: an ablated model recalibrating with a real mu at scoring
        # time would report the method's number under the ablation's name.
        mu = state.get("mu", "ref")
        if mu != self.mu_source:
            print(f"[cqig] checkpoint was trained with mu={mu!r}, the config asked for "
                  f"{self.mu_source!r}; using the checkpoint's")
        self.mu_source, self.mu_zero = mu, mu == "zero"

    # ------------------------------------------------------------------ diagnostics
    def n_reference(self):
        return self._n_ref

    def stats_bytes(self):
        return sum(v["mu"].numel() * v["mu"].element_size()
                   for st in self._stats.values() for v in st.values())

    def summary(self):
        return {
            "gate_layers_1indexed": sorted(i + 1 for i in self.gate_layers),
            "op": self.op,
            "mu": self.mu_source,
            "norm": self.norm,
            "grad_through_I": self.grad_through_I,
            "lam": [round(float(self.lam(i).detach()), 4) for i in sorted(self.gate_layers)],
            "alpha": [round(float(self.alpha(i).detach()), 4) for i in sorted(self.gate_layers)],
            "dtau": [round(float(self.dtau[i].detach()), 4) for i in sorted(self.gate_layers)],
            "active_graph": self._active,
            "calibrated_graphs": self.known_graphs(),
            "reference_queries": len(self.ref_bank or []),
            "stats_mb": round(self.stats_bytes() / 2**20, 1),
            "mean_gate": {k + 1: round(float(v), 4)
                          for k, v in sorted(self.last_gate.items())},
        }

    def remove(self):
        for h in self._handles:
            h.remove()
        self._handles = []


In [ ]:
%%writefile /content/gfm-rag/gfmrag/models/fusion_reasoner.py
"""
fusion_reasoner.py — CARGO fusion: v16sc G-Reasoner + operator, EVERYTHING learned jointly in one run.

Wraps the standard GraphReasoner. The operator is recomputed LIVE from its raw ingredients so its
weights and exponent are trainable (matches the interim report: beta and the fusion weights are learned):

    S_op      = w0 z(dense) + w1 z(S / dem^beta) + w2 z(M / dem^beta)   # dem = total_S - S (anti-hub)
    fused_doc = z(S_op) + gamma_q * relu( z(graph_doc) )
    gamma_q   = softplus( gate(coverage) )                             # per-query gate >= 0

TWO FUSION FORMS, selected by FUSION_FORM (default 'additive' = the arithmetic above,
bit-identical to every run made before the option existed).

    FUSION_FORM=mixture   fused_doc = log( (1 - a_q) p_s + a_q p_g )
                          p_s = softmax( z(S_op) )            # tau_s fixed at 1
                          p_g = softmax( z(graph_doc)/tau_g ) # tau_g learned in [0.25, 4]
                          a_q = a_max * sigmoid( router(phi_q) )

The additive form cannot promote a document, at any gamma it actually learns: a z-score
over this corpus tops out near 5-10 and the converged gamma is 0.036, so the graph's
largest possible contribution to any document is ~0.36 z-units against a semantic top-50
spread of 1-3. It can reorder neighbours, never rescue a gold the semantic scorer missed,
which is exactly the cross-domain case. Mixing calibrated DISTRIBUTIONS instead of scores
makes the response exponential, so the graph's few confident documents can move hundreds
of places, while a flat p_g leaves the ranking exactly unchanged rather than adding noise
to every document. The cost is that the graph's confident MISTAKES are promoted just as
hard, and the graph ranking is substantially weaker overall, so a concentrated p_g puts
large mass on wrong documents on many queries; a_max bounds that. Full detail, including
why tau_s is frozen and why tau_g is bounded rather than free, in __init__.

Trained end-to-end from one ranking loss on fused_doc:
  - the GNN weights (the v16sc graph reasoner, from scratch),
  - the gate (when to trust the graph), and
  - the operator scalars w=[w0,w1,w2] and beta.
The BGE encoder that produced dense/S/M is frozen (we only learn the handful of combination scalars).
w, beta are warm-started at the fitted values and learn at the base LR (op_lr_scale=1.0) — 4 params on
a near-convex objective, so they converge fast. relu => the graph can only promote a doc (aggregate floor).

Ingredients come from env OPERATOR_COMPONENTS (train) / OPERATOR_COMPONENTS_TEST (test): npz with
{dense,S,M: float16 [Q x n_doc] in nodes.csv doc order, total_S: float32 [n_doc], query_ids:[...]}.
Query ids are split-unique, so both tables are merged and looked up by batch['id'].

MULTI-CORPUS TRAINING. Either variable also accepts a COMMA-SEPARATED list of npz
paths, which is what joint Physics+Biology training needs: the trainer walks a list
of graphs and each carries its own corpus, so `dense` has a different document-column
count per graph (physics 10,349 vs biology 15,588) and one merged table cannot serve
both. Each file becomes its own table, and because query ids are unique across SIR-4
datasets the existing id -> (tag, row) lookup already routes a batch to the right one.
GraphDatasetLoader keeps one graph resident at a time, so a batch never spans two
corpora and the single-tag assert in _operator still holds.

THE SEMANTIC CHANNEL IS SELECTABLE (`semantic:` in the config).

  "operator" (default)  the handcrafted scorer above. Unchanged, bit-identical.
  "mlp"                 the LEARNED scorer from semantic_scorer.py: one MLP reading the
                        SORTED vector of per-answer match scores, with a popularity
                        discount predicted from the document embedding alone.

Only the semantic channel changes. The gate, the relu floor, the fusion arithmetic and the
hard-negative mining are identical either way, so a run pair isolates "handcrafted vs learned
semantic scorer" with the graph half held fixed. The learned scorer is warm-started from a
trained 5d checkpoint (SEMANTIC_CKPT + SEMANTIC_POPNET) and keeps training under the fusion's
ranking loss unless semantic_train=False.

WHY THE POPULARITY PREDICTOR AND NOT THE LEAVE-ONE-OUT TERM. The operator's anti-hub
denominator is total_S - S, i.e. a document's popularity measured from the OTHER queries'
hypothetical answers in the same split. Inside the fusion that is the same transductive
dependency it has always been. The learned scorer replaces it with a function of the
document's own embedding, so the fused model can score one query against an unseen corpus.

Ingredients come from env SEMANTIC_COMPONENTS / SEMANTIC_COMPONENTS_TEST (see
precompute_semantic_components.py). The per-answer matrix H is NOT copied into those files:
they carry the memmap's path plus the corpus->nodes.csv column permutation, and the rows for
one batch are sliced and permuted on demand.
"""
import json
import math
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F  # noqa: N812

from gfmrag.models.gfm_reasoner import GraphReasoner

W_INIT = (1.05, 1.05, 0.25)   # fitted operator fusion weights
BETA_INIT = 0.95              # fitted anti-hub exponent


class FusionGraphReasoner(nn.Module):
    def __init__(self, entity_model, feat_dim, gamma_init=0.5, gate_hidden=8,
                 op_lr_scale=1.0, semantic="operator", semantic_train=True,
                 cqig=False, cqig_lam=0.1, cqig_layers=None, cqig_norm=None,
                 cqig_rho=1e-3, cqig_grad_I=None, cqig_op=None, cqig_mu=None, **kwargs):
        # The operator was first proposed as `cqig_mode`. Accepted as an alias rather than
        # left to fall through **kwargs into GraphReasoner, where it would be a silent
        # no-op and the arm would train as the default operator under the other one's name.
        if "cqig_mode" in kwargs:
            cqig_op = kwargs.pop("cqig_mode") if cqig_op is None else cqig_op
        super().__init__()
        self.base = GraphReasoner(entity_model, feat_dim, **kwargs)
        assert semantic in ("operator", "mlp"), f"unknown semantic channel {semantic!r}"
        self.semantic = semantic

        # --- Cross-Query Informativeness Gating (off by default, see cqig.py) ---
        # Attaches a forward pre-hook to each conv layer. With cqig=False nothing is
        # registered at all, so the ungated arm is bit-identical.
        #
        # The three knobs that define an arm of the controlled experiment come from the
        # ENVIRONMENT when the config leaves them unset, for the same reason every other
        # run knob does: a Hydra override of a string like "3-6" has to survive quoting
        # through subprocess, an env var does not.
        # --- CCMP responsibility head (off by default) --------------------------
        # One projection per layer because self.dims may differ layer to layer, then a
        # SHARED trunk plus a layer embedding, so the predictor is one function of
        # (state, remaining depth) rather than L unrelated predictors. Built here, not
        # lazily on first forward, because a module created after the optimiser is
        # constructed never receives a gradient step and the arm would silently run as
        # its own control.
        _em = self.base.entity_model
        # --- Routing baselines for the CCMP comparison (off by default) ----------
        # ROUTE=astar  A*Net-style: the SAME responsibility head CCMP uses, but trained
        #              only through the ranking loss (no continuation targets) and applied
        #              as a hard top-K selection over the reached frontier per layer.
        # ROUTE=attn   RED-GNN-style: query-conditioned attention over each receiver's
        #              incoming edges, trained only through the ranking loss.
        # Either is a REPLACEMENT for CCMP, never an addition, so the three arms differ
        # in exactly one thing: where the routing signal comes from.
        _route = os.environ.get("ROUTE", "").strip().lower()
        assert _route in ("", "none", "astar", "attn"), f"unknown ROUTE={_route!r}"
        _route = "" if _route == "none" else _route
        _ccmp_on = os.environ.get("CCMP", "0") == "1"
        assert not (_ccmp_on and _route), "ROUTE replaces CCMP; unset one of CCMP / ROUTE"
        _em.route_mode = _route
        if _ccmp_on or _route == "astar":
            # RNG STATE SAVED AND RESTORED AROUND THIS BLOCK. Constructing these modules
            # draws from the global generator, so every parameter initialised AFTER this
            # point -- the fusion gate, the router, the operator scalars -- would start
            # from different values in the CCMP arm than in the control. The pair would
            # then differ in the loss AND in the initialisation, and the control's own
            # reruns already move 0.0118 nDCG@5.
            _rng = torch.get_rng_state()
            _dims = list(_em.dims)[:-1]
            _h = int(os.environ.get("CCMP_HID", "64"))
            _em.resp_proj = nn.ModuleList([nn.Linear(_d, _h) for _d in _dims])
            _em.resp_emb = nn.Embedding(len(_dims), _h)
            nn.init.zeros_(_em.resp_emb.weight)
            _em.resp_head = nn.Sequential(nn.ReLU(), nn.Linear(_h, _h),
                                          nn.ReLU(), nn.Linear(_h, 1))
            # Output bias at 0 => yhat starts at 0.5 => a mean-normalised gate starts at
            # exactly 1.0 at every node, so the CCMP arm's epoch 0 is the control's epoch 0
            # and any difference later is the loss, not a different initialisation.
            nn.init.zeros_(_em.resp_head[-1].bias)
            nn.init.zeros_(_em.resp_head[-1].weight)
            _em.resp_gate = os.environ.get("CCMP_GATE", "1") == "1"
            _em.resp_gate_norm = os.environ.get("CCMP_GATE_NORM", "1") == "1"
            _em.resp_eta = float(os.environ.get("CCMP_ETA", "0.5"))
            torch.set_rng_state(_rng)
            if _route == "astar":
                _em.route_k = int(os.environ.get("ROUTE_K", "1024"))
                _em.resp_gate = True
                print(f"[route] mode=astar: same head ({len(_dims)} layers, hid={_h}), "
                      f"hard top-{_em.route_k} of the reached frontier per layer, priority "
                      f"trained by the ranking loss only (no CCMP targets)", flush=True)
            else:
                print(f"[ccmp] responsibility head on {len(_dims)} layers, hid={_h}, "
                      f"gate={_em.resp_gate} mean_norm={_em.resp_gate_norm} "
                      f"eta={_em.resp_eta}", flush=True)
        else:
            _em.resp_proj = None
            _em.resp_gate = False
        if _route == "attn":
            # Same RNG discipline as the CCMP head, same hidden width, and a zero-initialised
            # output layer so attention starts UNIFORM: after degree normalisation every
            # edge weight is exactly 1 and epoch 0 is the control's epoch 0.
            _rng = torch.get_rng_state()
            _dims = list(_em.dims)[:-1]
            _h = int(os.environ.get("ROUTE_HID", os.environ.get("CCMP_HID", "64")))
            _em.attn_node = nn.ModuleList([nn.Linear(_d, _h) for _d in _dims])
            _em.attn_rel = nn.Linear(int(_em.dims[0]), _h)
            _em.attn_query = nn.Linear(int(_em.dims[0]), _h)
            _em.attn_emb = nn.Embedding(len(_dims), _h)
            nn.init.zeros_(_em.attn_emb.weight)
            _em.attn_out = nn.Linear(_h, 1)
            nn.init.zeros_(_em.attn_out.weight)
            nn.init.zeros_(_em.attn_out.bias)
            _em.attn_norm = os.environ.get("ROUTE_ATTN_NORM", "1") == "1"
            torch.set_rng_state(_rng)
            print(f"[route] mode=attn: query-conditioned edge attention on {len(_dims)} layers "
                  f"(hid={_h}, degree-normalised={_em.attn_norm}), trained by the ranking "
                  f"loss only", flush=True)
        else:
            _em.attn_node = None

        self.cqig = None
        if cqig:
            from gfmrag.models.cqig import CQIGate
            spec = cqig_layers if cqig_layers is not None else os.environ.get("CQIG_LAYERS")
            norm = cqig_norm if cqig_norm is not None else os.environ.get("CQIG_NORM", "layer")
            grad = (cqig_grad_I if cqig_grad_I is not None
                    else os.environ.get("CQIG_GRAD_I", "0") == "1")
            op = cqig_op if cqig_op is not None else os.environ.get("CQIG_OP", "gate")
            mu = cqig_mu if cqig_mu is not None else os.environ.get("CQIG_MU", "ref")
            self.cqig = CQIGate(self.base.entity_model.layers, lam_init=cqig_lam,
                                rho=cqig_rho, gate_layers=spec, norm=norm,
                                grad_through_I=grad, op=op, mu=mu)
            print(f"[cqig] attached to {self.cqig.n_layers} layers; gating "
                  f"{sorted(i + 1 for i in self.cqig.gate_layers)} (1-indexed) "
                  f"= positions {sorted(self.cqig.gate_layers)}; norm={norm!r} "
                  f"lam_init={cqig_lam} grad_through_I={grad} op={self.cqig.op!r} "
                  f"mu={self.cqig.mu_source!r}")
            if self.cqig.mu_zero:
                print("[cqig] mu=zero: THIS IS THE ABLATION, NOT THE METHOD. mu is forced to "
                      "0, so I = ||h||^2/s is an activation-MAGNITUDE gate and the reference "
                      "bank contributes nothing. The references are still run and discarded "
                      "so the two arms differ in mu alone. If this reproduces the method's "
                      "score, the bank was never load-bearing.")
            if self.cqig.mu_zero and self.cqig.centring:
                raise AssertionError(
                    "cqig mu='zero' with a centring op is a literal no-op: the operator "
                    "subtracts c*mu and mu is 0, so the model is the ungated reasoner. "
                    "Ablate mu against op='gate', which is the arm whose result is in doubt.")
            if self.cqig.centring:
                print(f"[cqig] op={self.cqig.op!r}: the state is CENTRED, h - c*mu, not "
                      f"scaled. c is "
                      + ("lam (constant, query-independent; alpha and dtau get no "
                         "gradient in this arm)" if self.cqig.op == "centre-fixed"
                         else "1-g, so an uninformative node loses more of its background")
                      + ". Nodes no reference reached have mu=0 and are left ALONE here, "
                        "where a gating arm damps them hardest.")

        # --- per-query gate: gamma_q = softplus(gate(phi_q)) ---
        #
        # ROUTER FEATURES ARE NOW SHARED BY BOTH FUSION FORMS. This used to be read only
        # inside the mixture branch, so on an ADDITIVE arm `FUSION_ROUTER` was accepted,
        # logged, and never used: the gate was nn.Linear(1, ...) on the semantic top-5
        # mean alone. Every additive run recorded before this change was therefore a `cov`
        # run whatever its flag said, and the matsci two-stage 2x2's legacy-vs-phi5 axis
        # compared two identical configurations (measured: -0.0004 / +0.0021 / -0.0019 /
        # -0.0038 against a 0.0118 noise floor, i.e. the null the wiring predicts).
        #
        # `full` is the default because a gate on semantic confidence alone cannot express
        # the one thing worth conditioning on: whether the GRAPH is worth listening to on
        # this query. Set FUSION_ROUTER=cov to reproduce any pre-existing additive run.
        self.router_feats = os.environ.get("FUSION_ROUTER", "full").lower()
        assert self.router_feats in ("full", "cov"), self.router_feats
        n_feat = 5 if self.router_feats == "full" else 1
        self.gate = nn.Sequential(
            nn.Linear(n_feat, gate_hidden), nn.ReLU(), nn.Linear(gate_hidden, 1)
        )
        # INIT IS UNCHANGED BY n_feat. The last layer's weight is zeroed, so gamma_q is
        # exactly softplus(bias) = gamma_init for every query at step 0 no matter how many
        # features feed it. A `full` arm and a `cov` arm therefore start from the identical
        # ranking and diverge only as the gate learns, which is what makes them a pair.
        nn.init.zeros_(self.gate[-1].weight)
        nn.init.constant_(self.gate[-1].bias, math.log(math.expm1(max(gamma_init, 1e-3))))
        self._gamma_lowp_note = False

        # --- FUSION FORM: additive z-scores (default) or a mixture of calibrated
        #     distributions (FUSION_FORM=mixture) ------------------------------------
        #
        # WHY A SECOND FORM AT ALL. The additive form cannot promote a document, at any
        # gamma it actually learns. A z-score over a 15k-60k document corpus tops out
        # near 5-10 even for the graph's single most confident paper, and the converged
        # gamma is 0.036, so the largest contribution the graph can make to ANY document
        # is about 0.36 z-units. The semantic channel's own spread across its top 50 is
        # 1-3 z-units. The graph is therefore structurally unable to lift a paper from
        # outside the top 50 into the top 5 however certain it is; it can only reorder
        # documents the semantic scorer already placed next to each other. That is a
        # property of the functional form, not of the gate, the gating authority lam, or
        # the bf16 freeze, and it caps the measured contribution at +0.017 nDCG@5.
        #
        # THE MIXTURE. Combine calibrated distributions instead of scores:
        #
        #     log p_s = z(s_op)      - logsumexp(z(s_op))        # tau_s FIXED at 1
        #     log p_g = z(g)/tau_g   - logsumexp(z(g)/tau_g)
        #     a_q     = a_max * sigmoid(router(phi_q))           # in [0, a_max]
        #     fused   = log( (1-a_q) p_s + a_q p_g )
        #
        # Four properties, each of which the additive form lacks:
        #
        #  1. EXPONENTIAL RESPONSE. p_g for a document the graph ranks first out of 60k
        #     is O(0.1); p_s in the semantic tail is O(1e-5). Even at a_q = 0.05 the
        #     second term dominates and that document moves hundreds of places. This is
        #     the only mechanism by which a 0.27-nDCG channel can rescue a gold the
        #     semantic scorer missed outright, which is the cross-domain case.
        #  2. AN UNINFORMATIVE GRAPH IS EXACTLY INERT. A flat p_g is a constant, and
        #     logaddexp(const, log(1-a) + log p_s) is strictly increasing in p_s, so the
        #     ranking does not move AT ALL -- not approximately, exactly. The weak-channel
        #     problem is handled by the algebra instead of by keeping gamma small. tau_g
        #     controls how sharp the graph's vote is relative to the semantic's, which is
        #     the one quantity that matters, and it is the only new scalar.
        #  3. MIXTURE, NOT PRODUCT. A product of experts (log p_s + b log p_g) would let
        #     the weak channel VETO a document the semantic scorer got right, driving its
        #     score toward zero. A mixture can only ADD mass, so no document's score ever
        #     falls, a_q -> 0 recovers the semantic ranking and a_max bounds the worst case.
        #     READ THAT AS A STATEMENT ABOUT SCORES, NOT RANKS. Rank is zero-sum: a
        #     confidently-wrong graph promotes other documents OVER a correct one and
        #     demotes it just as effectively as a veto would. Measured: a gold at semantic
        #     rank 1 falls to rank 3 when the graph is certain about six wrong documents,
        #     its own score gain being exactly 0.0 while theirs are 0.13 to 9.75. The
        #     additive form's relu is non-negative too, so this floor is NOT what
        #     distinguishes the two forms; a_max and property 2 are.
        #  4. GRADIENT FROM THE RANKING LOSS EVERYWHERE. relu(z(g)) zeroes the FUSED
        #     loss's gradient to the GNN for every document below the graph's own mean --
        #     precisely the buried golds. Under the mixture the graph softmax couples the
        #     whole corpus, so every document gets a gradient from the final ranking loss,
        #     attenuated rather than deleted. NOTE the precise claim: the graph is not
        #     gradient-starved today, because AUX_W=1.0 runs a separate graph-only
        #     contrastive on _raw_doc_z that never passes through the relu. What changes
        #     is that the RANKING loss itself now reaches every graph logit, instead of
        #     the graph learning about its buried documents only from the auxiliary term.
        #
        # THE RISK, STATED PLAINLY. Property 1 works just as well on the graph's confident
        # MISTAKES. nDCG@5 = 0.27 does NOT establish a top-1 error rate -- with ~1.9 golds
        # per query it is not a Hits@1 measurement and nothing here should be read as one
        # -- but the graph ranking is substantially weaker overall, so a concentrated p_g
        # will put large mass on wrong documents on many queries. The additive form is
        # safe because it is inert; this one is useful because it is not. a_max is the
        # only thing bounding that, which is why it defaults to 0.5 rather than 1.0 and
        # why a_q is learned per query rather than fixed. Whether the rescues outnumber
        # the false promotions is the empirical question the run exists to answer.
        #
        # tau_s IS DEFINED AND FROZEN AT 1, NOT LEARNED. The training loss is already
        # `logsumexp(lineup) - gold`, a softmax cross-entropy, so a learnable tau_s would
        # be softmaxing a softmax: shrinking it sharpens log p_s, which lowers the loss
        # whenever the gold already leads, without improving any ranking. Frozen at 1 it
        # buys something instead: log p_s = z(s_op) - const, cross-entropy is shift
        # invariant per query, so AT INITIALISATION (a_q small) this arm's loss is
        # numerically the same objective the additive arm trains under. No LR retuning,
        # and the two arms stay comparable.
        self.fusion_form = os.environ.get("FUSION_FORM", "additive").lower()
        assert self.fusion_form in ("additive", "mixture"), (
            f"FUSION_FORM={self.fusion_form!r}; expected 'additive' or 'mixture'")
        self._fusion_note = ""
        if self.fusion_form == "mixture":
            self.a_max = float(os.environ.get("FUSION_AMAX", "0.5"))
            a_init = float(os.environ.get("FUSION_AINIT", "0.1"))
            assert 0.0 < a_init < self.a_max <= 1.0, (
                f"need 0 < FUSION_AINIT ({a_init}) < FUSION_AMAX ({self.a_max}) <= 1")
            # a_q must not start AT zero: sigmoid'(-inf) = 0 and the router would get no
            # gradient to tell queries apart with. 0.1 is small enough that epoch 1 is
            # still essentially the semantic ranking.
            # router_feats and n_feat are set once above, for BOTH forms; the mixture just
            # builds a second head on the same input.
            self.router = nn.Sequential(
                nn.Linear(n_feat, gate_hidden), nn.ReLU(), nn.Linear(gate_hidden, 1)
            )
            nn.init.zeros_(self.router[-1].weight)
            nn.init.constant_(self.router[-1].bias,
                              math.log(a_init / (self.a_max - a_init)))
            # tau_g = exp(log_tau_g), so it stays positive without a clamp. log_tau_g = 0
            # is tau_g = 1, i.e. the graph's vote starts exactly as sharp as z(g) makes it.
            #
            # tau_g IS BOUNDED, NOT FREE. Two separate problems, only one of which is
            # already handled:
            #
            #   SCALE IDENTIFIABILITY -- handled, and this is why the graph channel is
            #   z-scored before it gets here. softmax(c*g / (c*tau_g)) = softmax(g/tau_g),
            #   so with RAW graph logits the readout could inflate its own scale while
            #   tau_g grew to match and the pair would not be separately identifiable.
            #   z() is exactly scale-invariant (z(c*g) = z(g)), so that degeneracy cannot
            #   arise. Feeding raw logits here would reintroduce it, and would also expose
            #   tau_g to the GNN's score scale drifting across epochs.
            #
            #   OVERCONFIDENCE -- NOT handled by z(), and this is the reason for the
            #   bounds. Shrinking tau_g sharpens p_g, which lowers the training
            #   cross-entropy on any query whose gold already leads, WITHOUT the graph
            #   ranking any better. That is the same degenerate direction that tau_s is
            #   frozen to avoid, and z-scoring does nothing about it. tau_g is therefore
            #   parameterised as tau_min + (tau_max - tau_min) * sigmoid(tau_hat), so the
            #   sharpest and flattest the graph's vote can get are both fixed in advance.
            #   The default [0.25, 4] is a factor of 4 either side of neutral.
            #
            # The stronger version of this is post-hoc calibration: train the reasoner,
            # freeze its readout, then fit tau_g and the router on held-out SOURCE queries
            # and freeze both for target inference, which makes tau_g a genuine
            # calibration parameter instead of one more jointly-optimised scale. That
            # costs an extra stage; the bounds are the cheap version of the same
            # protection. FUSION_TAUG=<float> freezes tau_g outright if that stage is
            # ever run and its fitted value is known.
            self.tau_min = float(os.environ.get("FUSION_TAU_MIN", "0.25"))
            self.tau_max = float(os.environ.get("FUSION_TAU_MAX", "4.0"))
            assert 0 < self.tau_min < self.tau_max, (self.tau_min, self.tau_max)
            tg = os.environ.get("FUSION_TAUG", "learn").lower()
            if tg == "learn":
                # tau_hat such that tau_g starts at exactly 1: the graph's vote begins
                # neither sharpened nor flattened relative to z(g).
                f = (1.0 - self.tau_min) / (self.tau_max - self.tau_min)
                assert 0.0 < f < 1.0, (
                    f"FUSION_TAU_MIN/MAX = [{self.tau_min}, {self.tau_max}] must bracket "
                    f"tau_g = 1, or the arm cannot start at the neutral sharpness")
                self.tau_g_hat = nn.Parameter(torch.tensor(math.log(f / (1.0 - f))))
            else:  # frozen at a given value, for the ablation or a calibrated fit
                v = float(tg)
                assert self.tau_min <= v <= self.tau_max, (
                    f"FUSION_TAUG={v} is outside [{self.tau_min}, {self.tau_max}]")
                f = (v - self.tau_min) / (self.tau_max - self.tau_min)
                self.register_buffer("tau_g_hat", torch.tensor(math.log(f / (1.0 - f))))
            print(f"[fusion] form=mixture: fused = log((1-a_q) p_s + a_q p_g), "
                  f"a_max={self.a_max} a_init={a_init} router={self.router_feats!r} "
                  f"({n_feat} features) tau_s=1 (frozen) tau_g={tg}. The diagnostic "
                  f"column printed as 'gamma' is now a_q, the mixing weight in "
                  f"[0, {self.a_max}] -- NOT the additive arm's gamma, and the two are "
                  f"not on the same scale.")
        else:
            # SAY WHICH ROUTER RAN. The additive arm used to print nothing about the
            # router, which is how FUSION_ROUTER stayed dead in this path unnoticed
            # across a whole 2x2. A run that does not log the knob it varied cannot be
            # audited from its own console output.
            print(f"[fusion] form=additive: fused = z(s_op) + gamma_q * relu(z(graph)), "
                  f"gamma_q = softplus(gate(phi_q)), router={self.router_feats!r} "
                  f"({n_feat} feature{'s' if n_feat > 1 else ''}), gamma_init={gamma_init}. "
                  + ("phi_q = [semantic confidence, semantic peakedness, graph confidence, "
                     "graph peakedness, channel agreement]." if n_feat > 1 else
                     "phi_q = the semantic top-5 mean alone (the pre-2026-08-16 behaviour)."))

        # --- trainable operator scalars (warm-started at fitted values; base LR via op_lr_scale) ---
        # effective value = init + op_lr_scale * delta, delta starts at 0. With Adam this makes the
        # operator learn at op_lr_scale x the graph/gate LR (1.0 = same rate).
        self.op_lr_scale = float(op_lr_scale)
        self.register_buffer("w_init", torch.tensor(W_INIT, dtype=torch.float32))
        self.w_delta = nn.Parameter(torch.zeros(3))
        self.beta_delta = nn.Parameter(torch.zeros(()))

        # --- operator raw ingredients (CPU float16; rows moved to GPU per batch) ---
        self._dense: dict[str, torch.Tensor] = {}
        self._S: dict[str, torch.Tensor] = {}
        self._M: dict[str, torch.Tensor] = {}
        self._totS: dict[str, torch.Tensor] = {}
        self._row: dict[str, tuple] = {}
        seen_paths: dict[str, str] = {}          # abspath -> tag already holding it
        for split, ev in ((("train", "OPERATOR_COMPONENTS"), ("test", "OPERATOR_COMPONENTS_TEST"))
                          if semantic == "operator" else ()):
            # Not loaded under semantic='mlp'. dense+S+M is 660 MB of resident CPU
            # memory on CS, and nothing in that arm reads it.
            raw = os.environ.get(ev)
            if not raw:
                continue
            paths = [x.strip() for x in raw.split(",") if x.strip()]
            for j, p in enumerate(paths):
                # THE SAME FILE FOR BOTH VARIABLES IS THE ZERO-SHOT IDIOM. Predict runs
                # have one corpus and set OPERATOR_COMPONENTS=OPERATOR_COMPONENTS_TEST=x,
                # which must stay legal: the duplicate check below exists to catch two
                # DIFFERENT tables claiming the same query ids (a real routing bug), not
                # one table registered twice (the same rows either way).
                ap = os.path.abspath(p)
                if ap in seen_paths:
                    print(f"[fusion] {ev}[{j}] is the same file as '{seen_paths[ap]}', reusing it")
                    continue
                # One table per file. The tag stays "train"/"test" for the single-file
                # case so existing runs and checkpoints are byte-identical; only a list
                # introduces the suffixed tags.
                tag = split if len(paths) == 1 else f"{split}#{j}"
                seen_paths[ap] = tag
                d = np.load(p, allow_pickle=True)
                self._dense[tag] = torch.from_numpy(np.asarray(d["dense"], dtype=np.float16))
                self._S[tag] = torch.from_numpy(np.asarray(d["S"], dtype=np.float16))
                self._M[tag] = torch.from_numpy(np.asarray(d["M"], dtype=np.float16))
                self._totS[tag] = torch.from_numpy(np.asarray(d["total_S"], dtype=np.float32))
                # A query id appearing in two tables would make routing order-dependent
                # and silently score half the batch against the wrong corpus.
                dup = [str(q) for q in d["query_ids"] if str(q) in self._row]
                assert not dup, (
                    f"{p}: {len(dup)} query ids already claimed by another components "
                    f"table (e.g. {dup[:3]}); tables must cover disjoint query sets")
                for i, q in enumerate(d["query_ids"]):
                    self._row[str(q)] = (tag, i)
                print(f"[fusion] loaded {ev}[{j}] as '{tag}': dense/S/M "
                      f"{tuple(self._dense[tag].shape)} ({len(d['query_ids'])} queries) "
                      f"{os.path.basename(p)}")
        if semantic == "operator":
            assert self._row, ("no operator components: set OPERATOR_COMPONENTS / "
                               "OPERATOR_COMPONENTS_TEST")
        else:
            self._init_semantic(semantic_train)

        self._raw_doc = None   # graph-alone doc scores, cached each forward for the aux loss
        self._doc_ids = None
        # [B, n_doc] detached SEMANTIC scores, cached for hard-negative mining. Named
        # _s_op for back-compatibility with the trainer and the routed reasoner, but it
        # holds whichever channel `semantic` selected, not necessarily the operator.
        self._s_op = None

    # ------------------------------------------------------------------ precision
    def _apply(self, *args, **kwargs):
        """Keep the gamma gate's weights in float32 through `model.to(dtype=...)`.

        SAME BUG AS CQIGate._apply, DIFFERENT MODULE. The trainer casts the whole model
        with `model.to(dtype=torch.bfloat16)` (utils/setup_training.py). A bfloat16 value
        has a 2^-8 relative ULP, so any parameter above |w| ~= 256*lr = 0.128 at lr 5e-4
        has every AdamW step rounded straight back, permanently, since round-to-nearest
        keeps no remainder.

        `self.gate[-1].bias` is initialised at log(expm1(gamma_init)) = -4.6 for the
        configured gamma_init=0.01. That is 36x the freeze line, so gamma's output bias
        could never move at all, and gate[0]'s weights are drawn from
        U(-1/sqrt(8), 1/sqrt(8)) = U(-0.354, 0.354), so most of those froze as well.

        The measured consequence: gamma stops dead at epoch 4 in EVERY arm and holds to
        four decimals for the next seven epochs while the graph channel's own nDCG still
        moves by 0.017. Whatever gamma reached in those four epochs is the run's final
        answer, because fused = z(s_op) + gamma*relu(z(graph)) and gamma is the only term
        deciding how loud the graph is. matsci: gamma froze at 0.0279 -> 0.5207,
        0.0359 -> 0.5340, 0.0288 -> 0.5218. The score is a function of the freeze point.

        This makes the lam=0.9 result unsafe to attribute to lam: graph-channel nDCG and
        informativeness AUC are the same in all three arms, so what lam changed was where
        gamma happened to stop, not the quality of the gating. CQIG_ALLOW_LOWP=1
        reproduces the old frozen behaviour for both this and the CQIG scalars.

        EVERY FUSION-HEAD PARAMETER IS PINNED, not just the gate. The mixture arm's
        router carries the same -1.39 output bias and its tau_g would drift past the
        freeze line within an epoch, so pinning the gate alone would reintroduce exactly
        this bug under a new name. These are a few dozen scalars in total; float32 for
        all of them costs nothing measurable.
        """
        heads = [("gate", self.gate)]
        # THE CCMP HEAD SITS ON THE SAME CLIFF AS GAMMA. The resp_head trunk initialises
        # at +-0.125 against a 256*lr = 0.128 freeze line, so roughly half its weights are
        # one small drift away from never updating again -- the exact failure that decided
        # every CQIG arm. Empty in the control, where these attributes are None.
        _rm = self.base.entity_model
        # ROUTE=attn's head is on the same cliff, and `_route_attention` runs it with
        # autocast off on float32 inputs, so leaving it in bfloat16 is not a precision
        # loss but a crash: "mat1 and mat2 must have the same dtype" on the first step
        # (tomato attn smoke, 2026-09-07). Empty unless ROUTE=attn.
        for _t in ("resp_proj", "resp_emb", "resp_head",
                   "attn_node", "attn_rel", "attn_query", "attn_emb", "attn_out"):
            if getattr(_rm, _t, None) is not None:
                heads.append((_t, getattr(_rm, _t)))
        if getattr(self, "fusion_form", "additive") == "mixture":
            heads.append(("router", self.router))
            if isinstance(getattr(self, "tau_g_hat", None), nn.Parameter):
                # `self` with recurse=False is tau_g_hat AND the operator scalars
                # w_delta/beta_delta. Pinning those too is deliberate: w_delta starts at
                # 0 but drifts, and W_INIT is 1.05, so they sit on the same cliff. Only
                # reached under form=mixture, so the additive arm is untouched.
                heads.append(("fusion", self))
        saved = {f"{tag}.{n}": p.detach().clone().float()
                 for tag, mod in heads
                 for n, p in (mod.named_parameters(recurse=False)
                              if mod is self else mod.named_parameters())}
        out = super()._apply(*args, **kwargs)
        if os.environ.get("CQIG_ALLOW_LOWP", "0") == "1":
            return out
        pinned = []
        for tag, mod in heads:
            it = (mod.named_parameters(recurse=False) if mod is self
                  else mod.named_parameters())
            for n, p in it:
                k = f"{tag}.{n}"
                if k in saved and p.is_floating_point() and p.dtype != torch.float32:
                    was = p.dtype
                    p.data = saved[k].to(device=p.device)
                    pinned.append((k, was))
        if pinned and not self._gamma_lowp_note:
            self._gamma_lowp_note = True
            print(f"[fusion] fusion head pinned to float32 against a {pinned[0][1]} model "
                  f"cast: {', '.join(n for n, _ in pinned)}. The output bias starts at "
                  f"{float(self.gate[-1].bias.flatten()[0]):.3f}, far above the "
                  f"|w| > 256*lr freeze line, so in {pinned[0][1]} gamma stopped moving "
                  f"at epoch ~4 and the run's score was fixed by whatever it reached.")
        return out

    def _router_phi(self, sz, gz, cov):
        """The per-query router input, shared by the additive gate and the mixture router.

        ONE DEFINITION, TWO CONSUMERS. These features were written for the mixture and
        lived inside it, which is how the additive arm ended up accepting FUSION_ROUTER
        and ignoring it. Both forms now call this, so "router=full" means the same five
        numbers whichever fusion is running and the two forms stay comparable.

        `cov` (the semantic top-5 mean) is feature 1 under `full` and the whole vector
        under `cov`, so the ablation is a strict narrowing rather than a different input.

        Divisors are NOMINAL, not tuned: a top-5 mean of a z-score over a corpus this size
        runs about 3-6 and a top1-to-top5 margin about 1-3, so these put all five features
        on the same O(1) footing and stop the [0,1] overlap feature from starting with a
        hundredth of the gradient of the others. They are constants, not parameters, so
        nothing here can be fitted to the test set.
        """
        if self.router_feats == "cov":
            return cov
        k = min(5, sz.shape[1])
        st, gt = sz.topk(k, dim=-1).values, gz.topk(k, dim=-1).values
        s_cov, g_cov = st.mean(-1, keepdim=True), gt.mean(-1, keepdim=True)
        ko = min(10, sz.shape[1])
        si = sz.topk(ko, dim=-1).indices                        # [B, ko]
        gi = gz.topk(ko, dim=-1).indices
        # agreement: what fraction of the semantic top-10 the graph also ranks top-10.
        # Low overlap with a confident graph is the only configuration in which the
        # graph has something to say that the semantic channel has not already said.
        ov = (si.unsqueeze(2) == gi.unsqueeze(1)).any(-1).float().mean(-1, keepdim=True)
        return torch.cat([s_cov / 5.0,                          # semantic confidence
                          (st[:, :1] - s_cov) / 2.0,            # semantic peakedness
                          g_cov / 5.0,                          # graph confidence
                          (gt[:, :1] - g_cov) / 2.0,            # graph peakedness
                          ov], dim=-1)                          # channel agreement

    def _fuse_mixture(self, sz, gz, cov):
        """log( (1-a_q) p_s + a_q p_g ) from the two standardised channels.

        sz, gz : [B, n_doc] z-scored semantic and graph document scores (float32).
        cov    : [B, 1] the additive arm's router feature, reused as feature 1.
        returns (a_q [B], fused [B, n_doc]).
        """
        # ROUTER FEATURES. The additive gate saw one number, the semantic top-5 mean, so
        # it could tell whether the SEMANTIC scorer was confident but had no way to know
        # whether the graph was worth listening to on this query. Under a mixture the
        # flat-graph case is already handled exactly by the algebra, so these features are
        # not load-bearing for safety -- they are what lets a_q open further on the
        # queries where the graph is peaked AND disagrees, which is where a promote-only
        # channel can actually change the answer. FUSION_ROUTER=cov drops back to the one
        # feature if the richer input ever needs to be ablated out.
        #
        # Divisors are NOMINAL, not tuned: a top-5 mean of a z-score over a corpus this
        # size runs about 3-6 and a top1-to-top5 margin about 1-3, so these put all five
        # features on the same O(1) footing and stop the [0,1] overlap feature from
        # starting with a hundredth of the gradient of the others. They are constants, not
        # parameters, so nothing here can be fitted to the test set.
        phi = self._router_phi(sz, gz, cov)
        # AUTOCAST OFF, same reason as the additive gate: under AMP an nn.Linear emits
        # bfloat16 whatever its parameter dtype, and the output bias sits at -1.39 where
        # a bfloat16 ULP is 0.005, which is noise of the same order as the quantity being
        # learned. Pinning the weights in `_apply` fixes the master copy, not this.
        with torch.autocast(device_type=sz.device.type, enabled=False):
            a = self.a_max * torch.sigmoid(self.router(phi.float())).squeeze(-1).float()
        # FUSION_AFIX=<float> REPLACES THE ROUTER WITH A CONSTANT. The first mixture run
        # (matsci, a_max=0.5) is why this exists: a_q climbed to 0.371 and spanned only
        # [0.309, 0.437] over 331 queries, i.e. the router did not route, it just opened.
        # It opened because it is trained on the TRAINING loss, where the GNN fits its own
        # queries far better than the 0.27 nDCG@5 it scores at test, so it calibrated to a
        # graph quality that does not exist at inference. Fused came out at 0.4265 against
        # a 0.5178 semantic channel: -0.091, exactly the predicted damage at that weight.
        # A constant a removes the train/test mismatch and makes the arm a one-parameter
        # sweep, which is the only honest way to find out whether ANY a > 0 helps here.
        if os.environ.get("FUSION_AFIX", "") != "":
            a = torch.full_like(a, float(os.environ["FUSION_AFIX"]))
        # DELIBERATELY OUTSIDE THE BRANCH ABOVE. tau_g used to be assigned inside it, so
        # the router path -- i.e. every mixture arm run so far -- reached the division
        # below with the name unbound. It has not fired yet only because those runs
        # predate the FUSION_AFIX block; the next router run would raise UnboundLocalError.
        tau_g = self.tau_min + (self.tau_max - self.tau_min) * torch.sigmoid(
            self.tau_g_hat.float())
        # tau_s is 1 and not learned, so log p_s = sz - logsumexp(sz) exactly (see the
        # note in __init__). The per-query logsumexp is a constant within a query, and
        # both the ranking and the cross-entropy loss are invariant to it, which is why
        # this arm starts from the same objective the additive arm trains under.
        lg = gz / tau_g.clamp(min=1e-3)
        log_ps = sz - torch.logsumexp(sz, dim=-1, keepdim=True)
        log_pg = lg - torch.logsumexp(lg, dim=-1, keepdim=True)
        # FUSION_TOPK=<int> CONFINES THE GRAPH TO THE SEMANTIC HEAD. This is the one
        # structural difference between the two forms, and it is why the additive arm wins
        # despite its arithmetic ceiling. gamma*relu(z(g)) can move a document a few places
        # WITHIN the semantic head and can essentially never lift one out of the deep tail;
        # it is a safe local reordering. The mixture is the opposite: every document with
        # a*p_g > p_s is scored log a + log p_g, i.e. purely by graph order, so they arrive
        # as a contiguous block whose depth is set by a. nDCG@5 has five slots, and the
        # graph's own top-5 is worth 0.245 against the semantic channel's 0.517, so once
        # that block reaches rank 5 the trade is losing by construction. Masking p_g outside
        # the semantic top-K keeps the calibrated reweighting (which the additive form can
        # only approximate, capped at 0.36 z-units) and drops the flooding.
        #
        # Out-of-set documents get logaddexp(l1a + log_ps, -inf) = l1a + log_ps exactly:
        # the semantic order shifted by one per-query constant, so their relative order is
        # untouched and in-set documents can only rise past them, never the reverse. The
        # renormalise keeps a's meaning ("fraction of the mixture's mass taken from the
        # graph") rather than silently shrinking it by the mass that was masked away.
        _k = int(os.environ.get("FUSION_TOPK", "0"))
        if 0 < _k < sz.shape[1]:
            keep = torch.zeros_like(sz, dtype=torch.bool)
            keep.scatter_(1, sz.topk(_k, dim=-1).indices, True)
            log_pg = log_pg.masked_fill(~keep, float("-inf"))
            log_pg = log_pg - torch.logsumexp(log_pg, dim=-1, keepdim=True)
        # clamp_min on a, and log1p(-a) rather than log(1-a), so the two mixing logs stay
        # finite if the router saturates at either end. a_max <= 1 keeps 1-a positive.
        la = torch.log(a.clamp(min=1e-8))[:, None]
        l1a = torch.log1p(-a.clamp(max=1.0 - 1e-6))[:, None]
        fused = torch.logaddexp(l1a + log_ps, la + log_pg)
        # ONE FLOAT32 CAVEAT, measured. d/dx log((1-a)e^x + a e^k) > 0, so a constant
        # (uninformative) p_g preserves the semantic order EXACTLY in real arithmetic.
        # In float32 it preserves it only until the a*p_g floor swamps the semantic term
        # and two documents COLLIDE onto the same value -- a tie, not a reorder. Measured
        # on a 15,588-document corpus at a_q=0.1: top-300 bit-identical to the semantic
        # order, first collision at rank ~3,100 between documents whose semantic z-scores
        # differ by 4e-7, ~40 tied of 15,588. Far below anything reported (@5, @10, @100),
        # so the invariance holds where it is read. It would NOT hold at recall@5000, and
        # a metric that deep would need the tie broken by the semantic score.
        with torch.no_grad():
            self._fusion_note = (f"mix a_q {a.mean().item():.4f} "
                                 f"[{a.min().item():.4f},{a.max().item():.4f}] "
                                 f"/{self.a_max:g}  tau_g {tau_g.item():.4f}")
        return a, fused

    @staticmethod
    def _z(x):
        return (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)

    def _operator(self, ids, n_doc, device):
        """Recompute S_op live from cached ingredients with the current (trainable) w, beta."""
        order = [self._row[str(x.item() if hasattr(x, "item") else x)] for x in ids]
        tag = order[0][0]
        # One graph is resident per batch, so one table serves the whole batch. A
        # mixed batch means the loader changed, and index_select below would silently
        # read rows of the wrong corpus rather than fail.
        assert all(t == tag for t, _ in order), (
            f"batch mixes operator tables {sorted({t for t, _ in order})}")
        idx = torch.tensor([r for _, r in order], dtype=torch.long)

        dense = self._dense[tag].index_select(0, idx).to(device, torch.float32)
        S = self._S[tag].index_select(0, idx).to(device, torch.float32)
        M = self._M[tag].index_select(0, idx).to(device, torch.float32)
        totS = self._totS[tag].to(device)                       # [n_doc]
        assert dense.shape[1] == n_doc, f"operator cols {dense.shape[1]} != {n_doc} doc nodes (alignment)"

        w = self.w_init + self.op_lr_scale * self.w_delta       # [3]
        beta = (BETA_INIT + self.op_lr_scale * self.beta_delta).clamp(min=0.05)
        dem = (totS.unsqueeze(0) - S).clamp(min=1e-6)           # [B, n_doc] leave-one-out popularity
        degb = dem.pow(beta)
        return w[0] * self._z(dense) + w[1] * self._z(S / degb) + w[2] * self._z(M / degb)

    # ---------------------------------------------------------------- learned scorer
    @staticmethod
    def _load_semantic_module():
        """Import semantic_scorer.py from the repo rather than vendoring a copy of it.

        The scorer, its normalisation and its popularity object are one design. A second
        copy living here would drift from the one that produced the checkpoint, and that
        drift would surface as a quietly different score rather than an import error.
        """
        import importlib.util
        import sys
        root = os.environ.get("SCIGRAPHIR_ROOT") or os.path.expanduser("$SCIGRAPHIR_ROOT")
        path = f"{root}/experiments/eval/semantic_scorer.py"
        assert os.path.exists(path), (
            f"semantic scorer source not found at {path}; set SCIGRAPHIR_ROOT to the repo root")
        for p in (root, f"{root}/retriever"):
            if p not in sys.path:
                sys.path.insert(0, p)
        spec = importlib.util.spec_from_file_location("cargo_semantic_scorer", path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod

    def _init_semantic(self, trainable):
        sem = self._load_semantic_module()
        self._sem = sem
        self._sem_tab, self._sem_row, self._sem_last_tag = {}, {}, None
        seen: dict[str, str] = {}
        for split, ev in (("train", "SEMANTIC_COMPONENTS"), ("test", "SEMANTIC_COMPONENTS_TEST")):
            raw = os.environ.get(ev)
            if not raw:
                continue
            for j, p in enumerate([x.strip() for x in raw.split(",") if x.strip()]):
                ap = os.path.abspath(p)
                if ap in seen:
                    print(f"[fusion] {ev}[{j}] is the same file as '{seen[ap]}', reusing it")
                    continue
                tag = split if raw.count(",") == 0 else f"{split}#{j}"
                seen[ap] = tag
                d = np.load(p, allow_pickle=True)
                Q, jmax, D = (int(x) for x in d["h_shape"])
                h_path = str(d["h_path"])
                assert os.path.exists(h_path), (
                    f"{p} references the per-answer matrix at {h_path}, which is missing. "
                    "It lives in the semantic scorer's own cache, which does not survive a "
                    "runtime reset; rerun section 5d or the precompute with --force.")
                col = np.asarray(d["col"], dtype=np.int64)
                nb = max(float(np.asarray(d["mask"]).sum()), 1.0)
                self._sem_tab[tag] = {
                    # memmap: only the batch's rows and this graph's columns are ever read
                    "H": np.memmap(h_path, np.float16, "r", shape=(Q, jmax, D)),
                    "col": col,
                    "dense": torch.from_numpy(np.asarray(d["dense"], dtype=np.float16)),
                    "mask": torch.from_numpy(np.asarray(d["mask"]).astype(np.float32)),
                    "doc_emb": torch.from_numpy(np.asarray(d["doc_emb"], dtype=np.float32)),
                    # the anchor target: total_S / (number of answers) IS the bank mean
                    "pop_target": torch.from_numpy(
                        np.asarray(d["total_S"], dtype=np.float32) / nb),
                    "doc_emb_gpu": None, "pop_target_gpu": None,
                }
                dup = [str(q) for q in d["query_ids"] if str(q) in self._sem_row]
                assert not dup, (f"{p}: {len(dup)} query ids already claimed by another "
                                 f"semantic components table (e.g. {dup[:3]})")
                for i, q in enumerate(d["query_ids"]):
                    self._sem_row[str(q)] = (tag, i)
                print(f"[fusion] loaded {ev}[{j}] as '{tag}': H {(Q, jmax, D)} -> {len(col)} "
                      f"doc nodes ({Q} queries) {os.path.basename(p)}")
        assert self._sem_row, ("semantic='mlp' needs SEMANTIC_COMPONENTS / "
                               "SEMANTIC_COMPONENTS_TEST from precompute_semantic_components.py")

        # --- the trained scorer and its predictor, as ONE warm start ---
        # Loading the scorer from a joint run without its predictor would pair a trained
        # readout with an untrained popularity, which is a model that never existed.
        ck, pn = os.environ.get("SEMANTIC_CKPT", ""), os.environ.get("SEMANTIC_POPNET", "")
        assert ck and os.path.exists(ck), (
            "semantic='mlp' needs SEMANTIC_CKPT: a params_semantic_mlp_*.json from section 5d")
        assert pn and os.path.exists(pn), (
            "semantic='mlp' needs SEMANTIC_POPNET: the popnet_semantic_mlp_*.pt written "
            "alongside it by a joint run (--mlp_pop_joint 1)")
        st = json.load(open(ck))
        scorer = sem.SortedMLPScorer("cpu", int(st["jmax"]), hidden=len(st["net"]["0.weight"]))
        scorer.load(st)
        psd = torch.load(pn, map_location="cpu")
        p_hidden, p_dim = psd["net.0.weight"].shape
        popnet = sem.MatchabilityPredictor(int(p_dim), int(p_hidden))
        popnet.load_state_dict(psd)

        # Registered as attributes so the optimiser sees them and .to(device) moves them.
        # These are the SAME objects the scorer holds, so updates propagate both ways.
        self.sem_net, self.sem_logbeta, self.sem_popnet = scorer.net, scorer.logbeta, popnet
        self._sem_scorer = scorer
        if not trainable:
            for prm in self.sem_net.parameters():
                prm.requires_grad_(False)
            self.sem_logbeta.requires_grad_(False)
            for prm in self.sem_popnet.parameters():
                prm.requires_grad_(False)
        print(f"[fusion] semantic='mlp' warm start: jmax={st['jmax']} beta={st['beta']:.4f} "
              f"popnet {p_dim}->{p_hidden}->1, trainable={trainable}")

    def _semantic(self, ids, n_doc, device):
        """Score with the learned sorted-MLP, recomputed live so it keeps training."""
        order = [self._sem_row[str(x.item() if hasattr(x, "item") else x)] for x in ids]
        tag = order[0][0]
        assert all(t == tag for t, _ in order), (
            f"batch mixes semantic tables {sorted({t for t, _ in order})}")
        tab, rows = self._sem_tab[tag], [r for _, r in order]
        self._sem_last_tag = tag

        # Slice rows from disk, then permute to nodes.csv document order. The resident
        # cost is B x Jmax x n_doc, not the whole 0.1-1.8 GB matrix.
        H = torch.from_numpy(
            np.asarray(tab["H"][rows])[:, :, tab["col"]].astype(np.float32)).to(device)
        idx = torch.tensor(rows, dtype=torch.long)
        dense = tab["dense"].index_select(0, idx).to(device, torch.float32)
        mask = tab["mask"].index_select(0, idx).to(device, torch.float32)
        assert H.shape[-1] == n_doc, (
            f"semantic cols {H.shape[-1]} != {n_doc} doc nodes (alignment)")

        if tab["doc_emb_gpu"] is None or tab["doc_emb_gpu"].device != device:
            tab["doc_emb_gpu"] = tab["doc_emb"].to(device, torch.float32)
            tab["pop_target_gpu"] = tab["pop_target"].to(device, torch.float32)
        pop = self._sem.Pop("joint", predictor=self.sem_popnet, doc_emb=tab["doc_emb_gpu"])
        return self._sem_scorer(H, mask, dense, pop)

    # ---------------------------------------------------------------- CQIG calibration
    def _cqig_run_refs(self, graph, batches, dev, count=False):
        for b in batches:
            b = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in b.items()}
            self.base(graph, b)
            if count:
                # COUNT QUERIES, NOT BATCHES. `len(b["id"])` counted whatever the loader
                # happened to pack, so a bank of 16 items meant 16 queries at train batch
                # size 1 and 123 at eval batch size 8, and the two calibrations were not
                # comparable. Re-linked reference batches carry no "id" at all.
                self.cqig.note_reference_batch(int(b["question_embeddings"].shape[0]))

    def cqig_calibrate(self, graph, graph_key, ref_batches=None, device=None, rounds=1):
        """Calibrate THIS graph from the fixed reference problems, in two passes per round.

        Pass "mean" accumulates mu. Pass "dev" measures ||h - mu||^2 directly, which gives
        both the variance -- exactly zero for a node whose state does not move, unlike the
        one-pass identity -- and the deviations tau_ref is the median of.

        `rounds` > 1 repeats both passes with the previous round's gates active. That
        matters only when more than one layer is gated: a layer's own gate cannot change
        its own input, so a single gated layer is already exact at rounds=1.

        Unlabelled and target-query-free: this is the indexing-time pass that gives an
        unseen graph its own reference statistics. Statistics are per graph because mu has
        one row per node, so the training graph's mu is not even shape-compatible with a
        different corpus. `ref_batches` defaults to the bank stored on the gate, which is
        what makes a reloaded checkpoint able to calibrate a corpus it has never seen.
        """
        assert self.cqig is not None, "cqig is not enabled on this model"
        batches = ref_batches if ref_batches is not None else self.cqig.ref_bank
        assert batches, ("no reference bank: the trainer sets one during training and it "
                         "is carried in the checkpoint; cannot calibrate without it")
        dev = device or next(self.parameters()).device
        was_training = self.training
        self.eval()
        report = {}
        with torch.no_grad():
            for r in range(max(1, int(rounds))):
                gated = r > 0
                for phase in ("mean", "dev"):
                    self.cqig.start_calibration(graph_key, phase=phase,
                                                apply_existing_gates=gated)
                    self._cqig_run_refs(graph, batches, dev, count=True)
                    if phase == "mean":
                        self.cqig.finish_mean()
                    else:
                        report = self.cqig.finish_calibration()
                for row in report.values():
                    row["round"] = r + 1
        if was_training:
            self.train()
        self.cqig.use_graph(graph_key)
        self.cqig.mode = "gate"
        return report

    def cqig_use_graph(self, graph_key):
        """Point the gate at an already-calibrated graph, e.g. back to train after eval."""
        assert self.cqig is not None and self.cqig.calibrated(graph_key), (
            f"graph {graph_key!r} has not been calibrated "
            f"(known: {self.cqig.known_graphs() if self.cqig else []})")
        self.cqig.use_graph(graph_key)
        self.cqig.mode = "gate"

    def semantic_aux_loss(self):
        """Log-space anchor holding p_hat near measured popularity. Mirrors Pop.aux_loss.

        Without it the fusion's ranking gradient is free to repurpose the predictor as extra
        scorer capacity, exactly as it would in the standalone run. Uses the tag of the last
        forward, which is the corpus the current batch came from.
        """
        tab = self._sem_tab.get(self._sem_last_tag) if self.semantic == "mlp" else None
        if tab is None or tab["doc_emb_gpu"] is None:
            return 0.0
        eps = 1e-6
        p = self.sem_popnet(tab["doc_emb_gpu"])
        return ((torch.log(eps + p) - torch.log(eps + tab["pop_target_gpu"])) ** 2).mean()

    def forward(self, graph, batch, entities_weight=None):
        # The frontier A^(l) is expanded inside bellmanford, which sees the edge index but
        # not the batch. Handed over here rather than threaded through GraphReasoner's
        # signature, which every other reasoner shares.
        if getattr(self.base.entity_model, "resp_proj", None) is not None:
            self.base.entity_model._ccmp_seeds = batch.get("start_nodes_mask")
        g = self.base(graph, batch, entities_weight)            # [B, N] per-node scores
        doc = graph.nodes_by_type["document"].to(g.device)      # nodes.csv document order
        gdoc = g.index_select(1, doc).float()                   # [B, n_doc] graph-alone doc scores

        # THE ONLY LINE THAT DIFFERS BETWEEN THE TWO ARMS. Everything below -- gate,
        # relu floor, fusion arithmetic, hard-negative mining -- is shared, so a run
        # pair isolates the semantic scorer with the graph half held fixed.
        s_op = (self._operator(batch["id"], doc.numel(), g.device)
                if self.semantic == "operator"
                else self._semantic(batch["id"], doc.numel(), g.device))
        # IS A HIGH INFORMATIVENESS ACTUALLY ON THE GOLD PAPERS? Measurement only, and only
        # while the trainer has recording switched on (around evaluate()). Nothing here
        # feeds back into the statistics, so it is not a transductive path -- but it is the
        # test that decides whether the gate's premise holds at all, so it needs the labels
        # and the hard negatives, which exist only here.
        if self.cqig is not None and self.cqig.recording():
            tgt = batch["target_nodes_mask"] if "target_nodes_mask" in batch else None
            self.cqig.note_alignment(tgt, doc, s_op.detach())
        # THE GATE READS THE STANDARDISED SCORE, like the fusion does. On the raw score
        # the semantic scorer could add a constant or scale everything up and change
        # gamma_q while leaving z(s_op) and its own ranking untouched, so the gate would
        # be keyed to a quantity with no semantic content. That is live for the learned
        # arm in particular: an MLP's output scale is free, unlike the operator's
        # warm-started w. After z(), a peaked top-5 genuinely means a confident scorer.
        sz = self._z(s_op).float()
        cov = sz.topk(min(5, sz.shape[1]), dim=-1).values.mean(-1, keepdim=True)
        # .float() on both operands, explicitly. Under AMP autocast the gate and the
        # learned scorer are nn.Linear and emit bfloat16 regardless of input dtype, and
        # a bfloat16 sum would round away a graph contribution of order gamma=0.01 for
        # the same reason the bfloat16 output write did. This currently lands in float32
        # by accident, via promotion from `gz`; relying on that is one config change away
        # from silently losing the effect.
        # AUTOCAST OFF FOR THE GATE. Pinning the weights to float32 (see `_apply`) is not
        # sufficient on its own: under AMP an nn.Linear is autocast to bfloat16 whatever
        # its parameter dtype, so the pre-activation would be quantised at an ULP of
        # 4.6 * 2^-8 = 0.018 around the -4.6 output bias, roughly 0.6% of gamma itself.
        # The master weights would still update in float32, but gamma would carry
        # avoidable noise. This gate is a 1->8->1 MLP, so float32 here costs nothing.
        gz = self._z(gdoc).float()                              # what the ranking uses
        if self.fusion_form == "mixture":
            gamma, fused = self._fuse_mixture(sz, gz, cov)
        else:
            # phi_q, not cov. Under FUSION_ROUTER=cov this is exactly the old one-feature
            # input; under the default `full` the gate additionally sees graph confidence,
            # graph peakedness and channel agreement, so it can condition gamma on whether
            # the GRAPH is worth listening to rather than only on semantic confidence.
            # Needs `gz`, which is why this sits after the gz line above and not with cov.
            with torch.autocast(device_type=cov.device.type, enabled=False):
                phi = self._router_phi(sz, gz, cov)
                gamma = F.softplus(self.gate(phi.float())).squeeze(-1).float()  # [B] per-query gate
            # FUSION_GAMMAFIX=<float>, the additive twin of FUSION_AFIX, and the only way
            # to find out whether the 0.5340 arm is a result or a coincidence. That number
            # came from a run whose gamma was bf16-FROZEN at 0.0359; the fp32-learnable
            # rerun froze at 0.0288 and scored 0.5218, and 0.012 nDCG@5 on n=331 is inside
            # what two training runs differ by anyway. Pinning gamma at 0.0359 makes the
            # comparison paired instead of accidental. Everything the mixture is being
            # asked to beat rests on that one number reproducing.
            if os.environ.get("FUSION_GAMMAFIX", "") != "":
                gamma = torch.full_like(gamma, float(os.environ["FUSION_GAMMAFIX"]))
            fused = sz + gamma[:, None] * torch.relu(gz)
            self._last_gamma = gamma.detach()   # exposed for interpret() dumps

        # FLOAT32 OUT. `g` is bfloat16, whose epsilon near 1.0 is 2^-8 = 0.0039 -- and
        # the graph's contribution is gamma_q * relu(z(graph)), which at the gate's 0.01
        # init is the same order. Writing the fused score back into a bfloat16 tensor
        # therefore rounds away most of the graph bonus and creates ranking ties, i.e.
        # it destroys precisely the signal being measured. clone() first so a float32
        # `g` is never mutated in place.
        out = g.clone().float()
        out[:, doc] = fused
        self._raw_doc = gdoc          # RAW graph scores: mining and diagnostics (both top-k, scale-free)
        # STANDARDISED graph scores: what the auxiliary loss must train, because it is
        # what the fused ranking consumes. z() is scale-invariant, so a loss on the raw
        # score has a free direction -- multiply every score by a constant and the loss
        # falls while the fused ranking does not move at all. Kept pre-relu on purpose:
        # relu would zero the gradient for exactly the golds sitting below the graph
        # mean, which are the ones that most need lifting.
        #
        # DELIBERATELY z(g) UNDER BOTH FORMS, not lg = z(g)/tau_g. Cross-entropy is
        # shift-invariant, so handing it lg would differ from gz only by the 1/tau_g
        # factor, i.e. it would couple the auxiliary loss's temperature to a fusion
        # parameter and give tau_g a gradient path that has nothing to do with fusion
        # quality. Keeping gz makes the graph-alone objective IDENTICAL in the additive
        # and mixture arms, so a run pair isolates the fusion form and nothing else.
        self._raw_doc_z = gz
        self._doc_ids = doc
        # The SELECTED scorer, so switching `semantic` also switches which documents the
        # contrastive loss makes the graph beat. One variable, every consumer.
        self._s_op = s_op.detach()
        # The gate, cached for diagnostics. Whether gamma_q ever leaves its 0.01 init is
        # the first thing to know about a fusion run: if it does not, the fused score IS
        # the semantic score and no loss change can matter.
        #
        # UNDER form=mixture THIS IS a_q, THE MIXING WEIGHT IN [0, a_max], NOT gamma. The
        # eval hook prints it in the same column, so a mixture run's "gamma mean" is on a
        # different scale from every additive run's and the two must not be read off the
        # same axis. a_q ~ 0.1 is its initialisation, not a stall.
        self._gamma = gamma.detach()
        return out


In [ ]:
%%writefile /content/gfm-rag/gfmrag/models/gfm_reasoner/model.py
import os
from typing import Any, Literal

import torch
from torch import autograd, nn
from torch_geometric.data import Data

from gfmrag.models.base_model import BaseGNNModel
from gfmrag.models.ultra import QueryNBFNet


class QueryGNN(BaseGNNModel):
    """A neural network module for query embedding in graph neural networks.

    This class implements a query embedding model that combines relation embeddings with an entity-based graph neural network
    for knowledge graph completion tasks.

    Args:
        entity_model (EntityNBFNet): The entity-based neural network model for reasoning on graph structure.
        feat_dim (int): Dimension of the entity and relation embeddings.
        *args: Variable length argument list.
        **kwargs: Arbitrary keyword arguments.

    Attributes:
        feat_dim (int): Dimension of entity and relation embeddings.
        entity_model (EntityNBFNet): The entity model instance.
        rel_mlp (nn.Linear): Linear transformation layer for relation embeddings.
        use_ent_emb (Literal[None, "early-fusion", "late-fusion"]): Specifies how to use entity embeddings.
            - None: No entity embeddings used
            - "early-fusion": Entity embeddings are fused early in the model
            - "late-fusion": Entity embeddings are fused late in the model

    Methods:
        forward(data: Data, batch: torch.Tensor) -> torch.Tensor:
            Forward pass of the query GNN model.

            Args:
                data (Data): Graph data object containing the knowledge graph structure and features.
                batch (torch.Tensor): Batch of triples with shape (batch_size, 1+num_negatives, 3),
                                    where each triple contains (head, tail, relation) indices.

            Returns:
                torch.Tensor: Scoring tensor for the input triples.
    """

    def __init__(
        self,
        entity_model: QueryNBFNet,
        feat_dim: int,
        use_ent_emb: Literal[
            None, "early-fusion", "late-fusion", "early-late-fusion"
        ] = None,
        *args: Any,
        **kwargs: Any,
    ) -> None:
        """Initialize the model.

        Args:
            entity_model (QueryNBFNet): The entity model component
            feat_dim (int): Dimension of relation embeddings
            use_ent_emb (Literal[None, "early-fusion", "late-fusion"]): Specifies how to use entity embeddings.
                - None: No entity embeddings used
                - "early-fusion": Entity embeddings are fused early in the model
                - "late-fusion": Entity embeddings are fused late in the model
            *args (Any): Variable length argument list
            **kwargs (Any): Arbitrary keyword arguments

        """

        super().__init__()
        self.feat_dim = feat_dim
        self.use_ent_emb = use_ent_emb
        self.entity_model = entity_model
        self.rel_mlp = nn.Linear(feat_dim, self.entity_model.dims[0])
        self.question_mlp = nn.Linear(self.feat_dim, self.entity_model.dims[0])

        if self.use_ent_emb is not None:
            self.ent_mlp = nn.Linear(feat_dim, self.entity_model.dims[0])

        if (
            self.use_ent_emb == "early-fusion"
            or self.use_ent_emb == "early-late-fusion"
        ):
            self.early_fuse_mlp = nn.Sequential(
                nn.Linear(self.entity_model.dims[0] * 2, self.entity_model.dims[0]),
                nn.ReLU(),
                nn.Linear(self.entity_model.dims[0], self.entity_model.dims[0]),
            )

        if self.use_ent_emb == "late-fusion" or self.use_ent_emb == "early-late-fusion":
            self.predict_mlp = nn.Sequential(
                nn.Linear(self.entity_model.dims[0] * 3, self.entity_model.dims[0]),
                nn.ReLU(),
                nn.Linear(self.entity_model.dims[0], 1),
            )

    def get_input_node_feature(
        self, graph: Data, query_head: torch.Tensor, query_representation: torch.Tensor
    ) -> torch.Tensor:
        """
        Get the input node features for the GNN model.

        Args:
            graph (Data): Graph data object containing entity embeddings and graph structure.
            query_head (torch.Tensor): Tensor of head indices for the query.
            query_representation (torch.Tensor): query relation representations.

        Returns:
            torch.Tensor: The input node features for GNN
        """
        batch_size = len(query_head)
        index = query_head.unsqueeze(-1).expand_as(query_representation)
        # initial (boundary) condition - initialize all node states as zeros
        boundary = torch.zeros(
            batch_size,
            graph.num_nodes,
            self.entity_model.dims[0],
            device=query_head.device,
        )
        # by the scatter operation we put query (relation) embeddings as init features of source (index) nodes
        boundary.scatter_add_(1, index.unsqueeze(1), query_representation.unsqueeze(1))

        if self.use_ent_emb == "early-fusion":
            # if we use early-fusion, we add entity embeddings to the boundary condition
            ent_emb = self.ent_mlp(graph.x)
            boundary += ent_emb.unsqueeze(0).expand_as(boundary)

        return boundary

    def forward(self, graph: Data, batch: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            graph (Data): Graph data object containing entity embeddings and graph structure.
            batch (torch.Tensor): Batch of triple indices with shape (batch_size, 1+num_negatives, 3),
                                where each triple contains (head_idx, tail_idx, relation_idx).

        Returns:
            torch.Tensor: Scores for the triples in the batch.

        Notes:
            - Relations are assumed to be the same across all positive and negative triples
            - Easy edges are removed before processing to encourage learning of non-trivial paths
            - The batch tensor contains both positive and negative samples where the first sample
              is positive and the rest are negative samples
        """
        # batch shape: (bs, 1+num_negs, 3)
        # relations are the same all positive and negative triples, so we can extract only one from the first triple among 1+nug_negs
        batch_size = len(batch)
        relation_representations = (
            self.rel_mlp(graph.rel_attr).unsqueeze(0).expand(batch_size, -1, -1)
        )
        h_index, t_index, r_index = batch.unbind(-1)

        # Obtain entity embeddings
        # turn all triples in a batch into a tail prediction mode
        h_index, t_index, r_index = self.entity_model.negative_sample_to_tail(
            h_index, t_index, r_index, num_direct_rel=graph.num_relations // 2
        )
        assert (h_index[:, [0]] == h_index).all()
        assert (r_index[:, [0]] == r_index).all()
        query_head = h_index[
            :, 0
        ]  # take the first head index for all triples in the batch
        query_relation = r_index[
            :, 0
        ]  # take the first relation index for all triples in the batch

        # Get the input embedding for the query head and relation
        raw_rel_emb = graph.rel_attr.unsqueeze(0).expand(batch_size, -1, -1)
        query_relation_emb = raw_rel_emb[
            torch.arange(batch_size, device=r_index.device), query_relation
        ]
        query_embedding = self.question_mlp(query_relation_emb)  # shape: (bs, emb_dim)
        node_embedding = self.get_input_node_feature(graph, query_head, query_embedding)

        # to make NBFNet iteration learn non-trivial paths
        graph = self.entity_model.remove_easy_edges(graph, h_index, t_index, r_index)
        score = self.entity_model(
            graph, node_embedding, relation_representations, query_embedding
        )

        return score


class GraphReasoner(QueryGNN):
    """A Query-dependent Graph Neural Network that reasons over the graph structure to identify relevant information.

    This class extends QueryGNN to implement a GNN-based reasoner system that processes question embeddings and entity information to find relevant information from a graph.

    Attributes:
        question_mlp (nn.Linear): Linear layer for transforming question embeddings.

    Args:
        entity_model (QueryNBFNet): The underlying query-dependent GNN for reasoning on graph.
        feat_dim (int): Dimension of relation embeddings.
        *args (Any): Variable length argument list.
        **kwargs (Any): Arbitrary keyword arguments.

    Methods:
        forward(graph, batch, entities_weight=None):
            Processes the input graph and question embeddings to generate reasoning scores.

            Args:
                graph (Data): The input graph structure.
                batch (dict[str, torch.Tensor]): Batch of input data containing question embeddings and masks.
                entities_weight (torch.Tensor, optional): Optional weights for entities.

            Returns:
                torch.Tensor: Output scores.

        visualize(graph, sample, entities_weight=None):
            Generates visualization data for the model's reasoning process.

            Args:
                graph (Data): The input graph structure.
                sample (dict[str, torch.Tensor]): Single sample data containing question embeddings and masks.
                entities_weight (torch.Tensor, optional): Optional weights for entities.

            Returns:
                dict[int, torch.Tensor]: Visualization data for each reasoning step.

    Note:
        The visualization method currently only supports batch size of 1.
    """

    """Wrap the GNN model for reasoning."""

    def forward(  # type: ignore[override]
        self,
        graph: Data,
        batch: dict[str, torch.Tensor],
        entities_weight: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Forward pass of the model.

        This method processes a graph and question embeddings to produce entity-level reasoning output.

        Args:
            graph (Data): A PyTorch Geometric Data object containing the graph structure and features.
            batch (dict[str, torch.Tensor]): A dictionary containing:
                - question_embeddings: Tensor of question embeddings
                - start_nodes_mask: Tensor of masks for question entities
            entities_weight (torch.Tensor | None, optional): Optional weight tensor for entities. Defaults to None.

        Returns:
            torch.Tensor: The output tensor representing entity-level reasoning results.

        Notes:
            The forward pass includes:
            1. Processing question embeddings through MLP
            2. Expanding relation representations
            3. Applying optional entity weights
            4. Computing entity-question interaction
            5. Running entity-level reasoning model
        """

        question_emb = batch["question_embeddings"]
        question_entities_mask = batch["start_nodes_mask"]

        # --- optional: down-weight hub seed frames by 1/degree^p of their graph node ---
        # Graph-intrinsic anti-hub prior on the restart distribution. Depends ONLY on the
        # fixed corpus (node degree), so it is single-query deployable — no cross-query info.
        # Redistributes each query's seed mass away from generic (high-degree) frames toward
        # discriminative ones; per-row total mass is preserved so only the *shape* changes.
        # Off by default; enable with env SEED_DEGREE_WEIGHT (the exponent p, e.g. "1.0").
        _sdw = os.environ.get("SEED_DEGREE_WEIGHT")
        if _sdw:
            p = float(_sdw)
            dw = getattr(self, "_seed_deg_weight", None)
            if dw is None or dw.numel() != graph.num_nodes:
                ei = graph.edge_index
                deg = torch.zeros(graph.num_nodes, device=ei.device, dtype=torch.float32)
                ones = torch.ones(ei.size(1), device=ei.device, dtype=torch.float32)
                deg.scatter_add_(0, ei[0], ones)
                deg.scatter_add_(0, ei[1], ones)
                dw = deg.clamp(min=1.0).pow(-p)
                self._seed_deg_weight = dw
            m = question_entities_mask * dw.to(question_entities_mask.dtype).unsqueeze(0)
            orig = question_entities_mask.sum(-1, keepdim=True).clamp(min=1e-6)
            new = m.sum(-1, keepdim=True).clamp(min=1e-6)
            question_entities_mask = m * (orig / new)  # preserve per-row total mass

        question_embedding = self.question_mlp(question_emb)  # shape: (bs, emb_dim)
        batch_size = question_embedding.size(0)
        relation_representations = (
            self.rel_mlp(graph.rel_attr).unsqueeze(0).expand(batch_size, -1, -1)
        )

        # initialize the input with the fuzzy set and question embs
        if entities_weight is not None:
            question_entities_mask = question_entities_mask * entities_weight.unsqueeze(
                0
            )

        node_embedding = torch.einsum(
            "bn, bd -> bnd", question_entities_mask, question_embedding
        )
        if (
            self.use_ent_emb == "early-fusion"
            or self.use_ent_emb == "early-late-fusion"
        ):
            # if we use early-fusion, we add entity embeddings to the input
            ent_emb = self.ent_mlp(graph.x)
            # node_embedding += ent_emb.unsqueeze(0).expand_as(node_embedding)
            node_embedding = self.early_fuse_mlp(
                torch.cat(
                    [node_embedding, ent_emb.unsqueeze(0).expand_as(node_embedding)],
                    dim=-1,
                )
            )

        # GNN model: run the entity-level reasoner to get a scalar distribution over nodes
        output = self.entity_model(
            graph, node_embedding, relation_representations, question_embedding
        )  # shape: (bs, num_nodes, emb_dim)
        if self.use_ent_emb == "late-fusion":
            ent_late_emb = (
                self.ent_mlp(graph.x).unsqueeze(0).expand(batch_size, -1, -1)
            )  # shape: (bs, num_nodes, emb_dim)
            output = self.predict_mlp(
                torch.cat([output, ent_late_emb], dim=-1)
            ).squeeze(-1)  # shape: (bs, num_nodes)
        elif self.use_ent_emb == "early-late-fusion":
            output = self.predict_mlp(
                torch.cat(
                    [output, ent_emb.unsqueeze(0).expand(batch_size, -1, -1)], dim=-1
                )
            ).squeeze(-1)  # shape: (bs, num_nodes)

        return output

    def visualize(
        self,
        graph: Data,
        sample: dict[str, torch.Tensor],
        entities_weight: torch.Tensor | None = None,
    ) -> dict[int, torch.Tensor]:
        """Visualizes attention weights and intermediate states for the model.

        This function generates visualization data for understanding how the model processes
        inputs and generates entity predictions. It is designed for debugging and analysis purposes.

        Args:
            graph (Data): The input knowledge graph structure containing entity and relation information
            sample (dict[str, torch.Tensor]): Dictionary containing:
                - question_embeddings: Tensor of question text embeddings
                - start_nodes_mask: Binary mask tensor indicating question entities
            entities_weight (torch.Tensor | None, optional): Optional tensor of entity weights to apply.
                Defaults to None.

        Returns:
            dict[int, torch.Tensor]: Dictionary mapping layer indices to attention weight tensors,
                allowing visualization of attention patterns at different model depths.

        Note:
            Currently only supports batch size of 1 for visualization purposes.

        Raises:
            AssertionError: If batch size is not 1
        """

        question_emb = sample["question_embeddings"]
        question_entities_mask = sample["start_nodes_mask"]
        question_embedding = self.question_mlp(question_emb)  # shape: (bs, emb_dim)
        batch_size = question_embedding.size(0)

        assert batch_size == 1, "Currently only supports batch size 1 for visualization"

        relation_representations = (
            self.rel_mlp(graph.rel_attr).unsqueeze(0).expand(batch_size, -1, -1)
        )

        # initialize the input with the fuzzy set and question embs
        if entities_weight is not None:
            question_entities_mask = question_entities_mask * entities_weight.unsqueeze(
                0
            )

        node_embedding = torch.einsum(
            "bn, bd -> bnd", question_entities_mask, question_embedding
        )
        if (
            self.use_ent_emb == "early-fusion"
            or self.use_ent_emb == "early-late-fusion"
        ):
            # if we use early-fusion, we add entity embeddings to the input
            ent_emb = self.ent_mlp(graph.x)
            # node_embedding += ent_emb.unsqueeze(0).expand_as(node_embedding)
            node_embedding = self.early_fuse_mlp(
                torch.cat(
                    [node_embedding, ent_emb.unsqueeze(0).expand_as(node_embedding)],
                    dim=-1,
                )
            )

        for layer in self.entity_model.layers:
            layer.relation = relation_representations

        output = self.entity_model.bellmanford(
            graph, node_embedding, question_embedding, separate_grad=True
        )

        node_feature = output["node_feature"]

        if self.use_ent_emb == "late-fusion":
            ent_late_emb = (
                self.ent_mlp(graph.x).unsqueeze(0).expand(batch_size, -1, -1)
            )  # shape: (bs, num_nodes, emb_dim)
            all_score = self.predict_mlp(
                torch.cat([node_feature, ent_late_emb], dim=-1)
            ).squeeze(-1)  # shape: (bs, num_nodes)
        elif self.use_ent_emb == "early-late-fusion":
            all_score = self.predict_mlp(
                torch.cat(
                    [node_feature, ent_emb.unsqueeze(0).expand(batch_size, -1, -1)],
                    dim=-1,
                )
            ).squeeze(-1)  # shape: (bs, num_nodes)

        edge_weights = output["edge_weights"]
        question_entities_mask = sample["start_nodes_mask"]
        target_entities_mask = sample["target_nodes_mask"]
        query_entities_index = question_entities_mask.nonzero(as_tuple=True)[1]
        target_entities_index = target_entities_mask.nonzero(as_tuple=True)[1]

        paths_results = {}
        for t_index in target_entities_index:
            score = all_score[:, t_index].squeeze(0)

            edge_grads = autograd.grad(score, edge_weights, retain_graph=True)
            distances, back_edges = self.entity_model.beam_search_distance(
                graph,
                edge_grads,
                query_entities_index,
                t_index,
                self.entity_model.num_beam,
            )
            paths, weights = self.entity_model.topk_average_length(
                distances, back_edges, t_index, self.entity_model.path_topk
            )
            paths_results[t_index.item()] = (paths, weights)
        return paths_results


In [ ]:
%%writefile /content/gfm-rag/gfmrag/models/ultra/base_nbfnet.py
# mypy: ignore-errors
import copy
from collections.abc import Sequence

import torch
from torch import autograd, nn

from gfmrag.models.ultra import variadic

from . import tasks


class BaseNBFNet(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dims,
        num_relation,
        message_func="distmult",
        aggregate_func="sum",
        short_cut=False,
        layer_norm=False,
        activation="relu",
        concat_hidden=False,
        num_mlp_layer=2,
        dependent=False,
        remove_one_hop=False,
        num_beam=10,
        path_topk=10,
        **kwargs,
    ):
        super().__init__()

        if not isinstance(hidden_dims, Sequence):
            hidden_dims = [hidden_dims]

        self.dims = [input_dim] + list(hidden_dims)
        self.num_relation = num_relation
        self.short_cut = (
            short_cut  # whether to use residual connections between GNN layers
        )
        self.concat_hidden = concat_hidden  # whether to compute final states as a function of all layer outputs or last
        self.remove_one_hop = remove_one_hop  # whether to dynamically remove one-hop edges from edge_index
        self.num_beam = num_beam
        self.path_topk = path_topk

        self.message_func = message_func
        self.aggregate_func = aggregate_func
        self.layer_norm = layer_norm
        self.activation = activation
        self.num_mlp_layers = num_mlp_layer

        # self.layers = nn.ModuleList()
        # for i in range(len(self.dims) - 1):
        #     self.layers.append(layers.GeneralizedRelationalConv(self.dims[i], self.dims[i + 1], num_relation,
        #                                                         self.dims[0], message_func, aggregate_func, layer_norm,
        #                                                         activation, dependent))

        # feature_dim = (sum(hidden_dims) if concat_hidden else hidden_dims[-1]) + input_dim

        # # additional relation embedding which serves as an initial 'query' for the NBFNet forward pass
        # # each layer has its own learnable relations matrix, so we send the total number of relations, too
        # self.query = nn.Embedding(num_relation, input_dim)
        # self.mlp = nn.Sequential()
        # mlp = []
        # for i in range(num_mlp_layer - 1):
        #     mlp.append(nn.Linear(feature_dim, feature_dim))
        #     mlp.append(nn.ReLU())
        # mlp.append(nn.Linear(feature_dim, 1))
        # self.mlp = nn.Sequential(*mlp)

    def remove_easy_edges(self, data, h_index, t_index, r_index=None):
        # we remove training edges (we need to predict them at training time) from the edge index
        # think of it as a dynamic edge dropout
        h_index_ext = torch.cat([h_index, t_index], dim=-1)
        t_index_ext = torch.cat([t_index, h_index], dim=-1)
        r_index_ext = torch.cat([r_index, r_index + data.num_relations // 2], dim=-1)
        if self.remove_one_hop:
            # we remove all existing immediate edges between heads and tails in the batch
            edge_index = data.edge_index
            easy_edge = torch.stack([h_index_ext, t_index_ext]).flatten(1)
            index = tasks.edge_match(edge_index, easy_edge)[0]
            mask = ~index_to_mask(index, data.num_edges)
        else:
            # we remove existing immediate edges between heads and tails in the batch with the given relation
            edge_index = torch.cat([data.edge_index, data.edge_type.unsqueeze(0)])
            # note that here we add relation types r_index_ext to the matching query
            easy_edge = torch.stack([h_index_ext, t_index_ext, r_index_ext]).flatten(1)
            index = tasks.edge_match(edge_index, easy_edge)[0]
            mask = ~index_to_mask(index, data.num_edges)

        data = copy.copy(data)
        data.edge_index = data.edge_index[:, mask]
        data.edge_type = data.edge_type[mask]
        return data

    def negative_sample_to_tail(self, h_index, t_index, r_index, num_direct_rel):
        # convert p(h | t, r) to p(t' | h', r')
        # h' = t, r' = r^{-1}, t' = h
        is_t_neg = (h_index == h_index[:, [0]]).all(dim=-1, keepdim=True)
        new_h_index = torch.where(is_t_neg, h_index, t_index)
        new_t_index = torch.where(is_t_neg, t_index, h_index)
        new_r_index = torch.where(is_t_neg, r_index, r_index + num_direct_rel)
        return new_h_index, new_t_index, new_r_index

    def bellmanford(self, data, h_index, r_index, separate_grad=False):
        batch_size = len(r_index)

        # initialize queries (relation types of the given triples)
        query = self.query(r_index)
        index = h_index.unsqueeze(-1).expand_as(query)

        # initial (boundary) condition - initialize all node states as zeros
        boundary = torch.zeros(
            batch_size, data.num_nodes, self.dims[0], device=h_index.device
        )
        # by the scatter operation we put query (relation) embeddings as init features of source (index) nodes
        boundary.scatter_add_(1, index.unsqueeze(1), query.unsqueeze(1))
        size = (data.num_nodes, data.num_nodes)
        edge_weight = torch.ones(data.num_edges, device=h_index.device)

        hiddens = []
        edge_weights = []
        layer_input = boundary

        for layer in self.layers:
            if separate_grad:
                edge_weight = edge_weight.clone().requires_grad_()
            # Bellman-Ford iteration, we send the original boundary condition in addition to the updated node states
            hidden = layer(
                layer_input,
                query,
                boundary,
                data.edge_index,
                data.edge_type,
                size,
                edge_weight,
            )
            if self.short_cut and hidden.shape == layer_input.shape:
                # residual connection here
                hidden = hidden + layer_input
            hiddens.append(hidden)
            edge_weights.append(edge_weight)
            layer_input = hidden

        # original query (relation type) embeddings
        node_query = query.unsqueeze(1).expand(
            -1, data.num_nodes, -1
        )  # (batch_size, num_nodes, input_dim)
        if self.concat_hidden:
            output = torch.cat(hiddens + [node_query], dim=-1)
        else:
            output = torch.cat([hiddens[-1], node_query], dim=-1)

        return {
            "node_feature": output,
            "edge_weights": edge_weights,
        }

    def forward(self, data, batch):
        h_index, t_index, r_index = batch.unbind(-1)
        if self.training:
            # Edge dropout in the training mode
            # here we want to remove immediate edges (head, relation, tail) from the edge_index and edge_types
            # to make NBFNet iteration learn non-trivial paths
            data = self.remove_easy_edges(
                data, h_index, t_index, r_index, data.num_relations // 2
            )

        shape = h_index.shape
        # turn all triples in a batch into a tail prediction mode
        h_index, t_index, r_index = self.negative_sample_to_tail(
            h_index, t_index, r_index, num_direct_rel=data.num_relations // 2
        )
        assert (h_index[:, [0]] == h_index).all()
        assert (r_index[:, [0]] == r_index).all()

        # message passing and updated node representations
        output = self.bellmanford(
            data, h_index[:, 0], r_index[:, 0]
        )  # (num_nodes, batch_size, feature_dim）
        feature = output["node_feature"]
        index = t_index.unsqueeze(-1).expand(-1, -1, feature.shape[-1])
        # extract representations of tail entities from the updated node states
        feature = feature.gather(
            1, index
        )  # (batch_size, num_negative + 1, feature_dim)

        # probability logit for each tail node in the batch
        # (batch_size, num_negative + 1, dim) -> (batch_size, num_negative + 1)
        score = self.mlp(feature).squeeze(-1)
        return score.view(shape)

    def visualize(self, data, batch):
        assert batch.shape == (1, 3)
        h_index, t_index, r_index = batch.unbind(-1)

        output = self.bellmanford(data, h_index, r_index, separate_grad=True)
        feature = output["node_feature"]
        edge_weights = output["edge_weights"]

        index = t_index.unsqueeze(0).unsqueeze(-1).expand(-1, -1, feature.shape[-1])
        feature = feature.gather(1, index).squeeze(0)
        score = self.mlp(feature).squeeze(-1)

        edge_grads = autograd.grad(score, edge_weights)
        distances, back_edges = self.beam_search_distance(
            data, edge_grads, h_index, t_index, self.num_beam
        )
        paths, weights = self.topk_average_length(
            distances, back_edges, t_index, self.path_topk
        )

        return paths, weights

    @torch.no_grad()
    def beam_search_distance(self, data, edge_grads, h_index, t_index, num_beam=10):
        # beam search the top-k distance from h to t (and to every other node)
        num_nodes = data.num_nodes
        input = torch.full((num_nodes, num_beam), float("-inf"), device=h_index.device)
        input[h_index, 0] = 0
        edge_mask = data.edge_index[0, :] != t_index

        distances = []
        back_edges = []
        for edge_grad in edge_grads:
            # we don't allow any path goes out of t once it arrives at t
            node_in, node_out = data.edge_index[:, edge_mask]
            relation = data.edge_type[edge_mask]
            edge_grad = edge_grad[edge_mask]

            message = input[node_in] + edge_grad.unsqueeze(-1)  # (num_edges, num_beam)
            # (num_edges, num_beam, 3)
            msg_source = (
                torch.stack([node_in, node_out, relation], dim=-1)
                .unsqueeze(1)
                .expand(-1, num_beam, -1)
            )

            # (num_edges, num_beam)
            is_duplicate = torch.isclose(
                message.unsqueeze(-1), message.unsqueeze(-2)
            ) & (msg_source.unsqueeze(-2) == msg_source.unsqueeze(-3)).all(dim=-1)
            # pick the first occurrence as the ranking in the previous node's beam
            # this makes deduplication easier later
            # and store it in msg_source
            is_duplicate = is_duplicate.float() - torch.arange(
                num_beam, dtype=torch.float, device=message.device
            ) / (num_beam + 1)
            prev_rank = is_duplicate.argmax(dim=-1, keepdim=True)
            msg_source = torch.cat(
                [msg_source, prev_rank], dim=-1
            )  # (num_edges, num_beam, 4)

            node_out, order = node_out.sort()
            node_out_set = torch.unique(node_out)
            # sort messages w.r.t. node_out
            message = message[order].flatten()  # (num_edges * num_beam)
            msg_source = msg_source[order].flatten(0, -2)  # (num_edges * num_beam, 4)
            size = node_out.bincount(minlength=num_nodes)
            msg2out = size_to_index(size[node_out_set] * num_beam)
            # deduplicate messages that are from the same source and the same beam
            is_duplicate = (msg_source[1:] == msg_source[:-1]).all(dim=-1)
            is_duplicate = torch.cat(
                [torch.zeros(1, dtype=torch.bool, device=message.device), is_duplicate]
            )
            message = message[~is_duplicate]
            msg_source = msg_source[~is_duplicate]
            msg2out = msg2out[~is_duplicate]
            size = msg2out.bincount(minlength=len(node_out_set))

            if not torch.isinf(message).all():
                # take the topk messages from the neighborhood
                # distance: (len(node_out_set) * num_beam)
                distance, rel_index = scatter_topk(message, size, k=num_beam)
                abs_index = rel_index + (size.cumsum(0) - size).unsqueeze(-1)
                # store msg_source for backtracking
                back_edge = msg_source[abs_index]  # (len(node_out_set) * num_beam, 4)
                distance = distance.view(len(node_out_set), num_beam)
                back_edge = back_edge.view(len(node_out_set), num_beam, 4)

                distance = variadic.native_scatter(
                    distance, node_out_set, dim=0, dim_size=num_nodes, reduce="sum"
                )  # (num_nodes, num_beam)
                back_edge = variadic.native_scatter(
                    back_edge, node_out_set, dim=0, dim_size=num_nodes, reduce="sum"
                )  # (num_nodes, num_beam, 4)
            else:
                distance = torch.full(
                    (num_nodes, num_beam), float("-inf"), device=message.device
                )
                back_edge = torch.zeros(
                    num_nodes, num_beam, 4, dtype=torch.long, device=message.device
                )

            distances.append(distance)
            back_edges.append(back_edge)
            input = distance

        return distances, back_edges

    def topk_average_length(self, distances, back_edges, t_index, k=10):
        # backtrack distances and back_edges to generate the paths
        paths = []
        average_lengths = []

        for i in range(len(distances)):
            distance, order = distances[i][t_index].flatten(0, -1).sort(descending=True)
            back_edge = back_edges[i][t_index].flatten(0, -2)[order]
            for d, (h, t, r, prev_rank) in zip(
                distance[:k].tolist(), back_edge[:k].tolist()
            ):
                if d == float("-inf"):
                    break
                path = [(h, t, r)]
                for j in range(i - 1, -1, -1):
                    h, t, r, prev_rank = back_edges[j][h, prev_rank].tolist()
                    path.append((h, t, r))
                paths.append(path[::-1])
                average_lengths.append(d / len(path))

        if paths:
            average_lengths, paths = zip(
                *sorted(zip(average_lengths, paths), reverse=True)[:k]
            )

        return paths, average_lengths


def index_to_mask(index, size):
    index = index.view(-1)
    size = int(index.max()) + 1 if size is None else size
    mask = index.new_zeros(size, dtype=torch.bool)
    mask[index] = True
    return mask


def size_to_index(size):
    range = torch.arange(len(size), device=size.device)
    index2sample = range.repeat_interleave(size)
    return index2sample


def multi_slice_mask(starts, ends, length):
    values = torch.cat([torch.ones_like(starts), -torch.ones_like(ends)])
    slices = torch.cat([starts, ends])
    mask = variadic.native_scatter(
        values, slices, dim=0, dim_size=length + 1, reduce="sum"
    )[:-1]

    mask = mask.cumsum(0).bool()
    return mask


def scatter_extend(data, size, input, input_size):
    new_size = size + input_size
    new_cum_size = new_size.cumsum(0)
    new_data = torch.zeros(
        new_cum_size[-1], *data.shape[1:], dtype=data.dtype, device=data.device
    )
    starts = new_cum_size - new_size
    ends = starts + size
    index = multi_slice_mask(starts, ends, new_cum_size[-1])
    new_data[index] = data
    new_data[~index] = input
    return new_data, new_size


def scatter_topk(input, size, k, largest=True):
    index2graph = size_to_index(size)
    index2graph = index2graph.view([-1] + [1] * (input.ndim - 1))

    mask = ~torch.isinf(input)
    max = input[mask].max().item()
    min = input[mask].min().item()
    safe_input = input.clamp(2 * min - max, 2 * max - min)
    offset = (max - min) * 4
    if largest:
        offset = -offset
    input_ext = safe_input + offset * index2graph
    index_ext = input_ext.argsort(dim=0, descending=largest)
    num_actual = size.clamp(max=k)
    num_padding = k - num_actual
    starts = size.cumsum(0) - size
    ends = starts + num_actual
    mask = multi_slice_mask(starts, ends, len(index_ext)).nonzero().flatten()

    if (num_padding > 0).any():
        # special case: size < k, pad with the last valid index
        padding = ends - 1
        padding2graph = size_to_index(num_padding)
        mask = scatter_extend(mask, num_actual, padding[padding2graph], num_padding)[0]

    index = index_ext[mask]  # (N * k, ...)
    value = input.gather(0, index)
    if isinstance(k, torch.Tensor) and k.shape == size.shape:
        value = value.view(-1, *input.shape[1:])
        index = index.view(-1, *input.shape[1:])
        index = index - (size.cumsum(0) - size).repeat_interleave(k).view(
            [-1] + [1] * (index.ndim - 1)
        )
    else:
        value = value.view(-1, k, *input.shape[1:])
        index = index.view(-1, k, *input.shape[1:])
        index = index - (size.cumsum(0) - size).view([-1] + [1] * (index.ndim - 1))

    return value, index


In [ ]:
%%writefile /content/gfm-rag/gfmrag/models/ultra/layers.py
# mypy: ignore-errors
import os

import torch
import torch_geometric
from torch import nn
from torch.nn import functional as F  # noqa:N812
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import degree

from gfmrag.models.ultra import variadic


class GeneralizedRelationalConv(MessagePassing):
    eps = 1e-6

    message2mul = {
        "transe": "add",
        "distmult": "mul",
    }
    aggr2reduce = {"sum": "sum", "mean": "mean", "min": "amin", "max": "amax"}

    # TODO for compile() - doesn't work currently
    # propagate_type = {"edge_index": torch.LongTensor, "size": Tuple[int, int]}

    def __init__(
        self,
        input_dim,
        output_dim,
        num_relation,
        query_input_dim,
        message_func="distmult",
        aggregate_func="pna",
        layer_norm=False,
        activation="relu",
        dependent=False,
        project_relations=False,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_relation = num_relation
        self.query_input_dim = query_input_dim
        self.message_func = message_func
        self.aggregate_func = aggregate_func
        self.dependent = dependent
        self.project_relations = project_relations

        if layer_norm:
            self.layer_norm = nn.LayerNorm(output_dim)
        else:
            self.layer_norm = None
        if isinstance(activation, str):
            self.activation = getattr(F, activation)
        else:
            self.activation = activation

        if self.aggregate_func == "pna":
            self.linear = nn.Linear(input_dim * 13, output_dim)
        else:
            self.linear = nn.Linear(input_dim * 2, output_dim)

        if dependent:
            # obtain relation embeddings as a projection of the query relation
            self.relation_linear = nn.Linear(query_input_dim, num_relation * input_dim)
        else:
            if not self.project_relations:
                # relation embeddings as an independent embedding matrix per each layer
                self.relation = nn.Embedding(num_relation, input_dim)
            else:
                # will be initialized after the pass over relation graph
                self.relation = None
                self.relation_projection = nn.Sequential(
                    nn.Linear(input_dim, input_dim),
                    nn.ReLU(),
                    nn.Linear(input_dim, input_dim),
                )

    def forward(
        self, input, query, boundary, edge_index, edge_type, size, edge_weight=None
    ):
        batch_size = len(query)

        if self.dependent:
            # layer-specific relation features as a projection of input "query" (relation) embeddings
            relation = self.relation_linear(query).view(
                batch_size, self.num_relation, self.input_dim
            )
        else:
            if not self.project_relations:
                # layer-specific relation features as a special embedding matrix unique to each layer
                relation = self.relation.weight.expand(batch_size, -1, -1)
            else:
                # NEW and only change:
                # projecting relation features to unique features for this layer, then resizing for the current batch
                relation = self.relation_projection(self.relation)
        if edge_weight is None:
            edge_weight = torch.ones(len(edge_type), device=input.device)

        # note that we send the initial boundary condition (node states at layer0) to the message passing
        # correspond to Eq.6 on p5 in https://arxiv.org/pdf/2106.06935.pdf
        # output = self.propagate(input=input, relation=relation, boundary=boundary, edge_index=edge_index,
        #                         edge_type=edge_type, size=size, edge_weight=edge_weight)
        output = self.propagate(
            edge_index=edge_index,
            size=size,
            edge_type=edge_type,
            input=input,
            relation=relation,
            boundary=boundary,
            edge_weight=edge_weight,
        )

        return output

    def propagate(self, edge_index, size=None, **kwargs):
        # A [B, E] edge_weight (per-query attention, ROUTE=attn) goes through the rspmm kernel
        # ONE QUERY AT A TIME (message_and_aggregate loops the batch): the kernel takes one
        # weight per edge and its backward returns the weight gradient, so per-query weights
        # need neither the O(|E| d) unfused path nor gradient checkpointing. Measured on the
        # TOMATO attn arm (8 Sep): unfused + checkpoint 3.3 s/step vs ~1 s/step fused.
        # Only sum/mean aggregation is wired for the per-query loop; rotate stays unfused.
        # RSPMM_FORCE_UNFUSED=1 takes the old path (used by the equivalence test).
        _w = kwargs["edge_weight"]
        _per_query_fused = (_w.dim() == 2 and self.message_func != "rotate"
                            and self.aggregate_func in ("sum", "mean")
                            and os.environ.get("RSPMM_FORCE_UNFUSED", "0") != "1")
        if not _per_query_fused and (_w.requires_grad or self.message_func == "rotate" or _w.dim() == 2):
            # the rspmm cuda kernel only works for TransE and DistMult message functions
            # otherwise we invoke separate message & aggregate functions
            return super().propagate(edge_index, size, **kwargs)

        for hook in self._propagate_forward_pre_hooks.values():
            res = hook(self, (edge_index, size, kwargs))
            if res is not None:
                edge_index, size, kwargs = res

        # in newer PyG,
        # __check_input__ -> _check_input()
        # __collect__ -> _collect()
        # __fused_user_args__ -> _fuser_user_args
        size = self._check_input(edge_index, size)
        coll_dict = self._collect(self._fused_user_args, edge_index, size, kwargs)

        # TODO: use from packaging.version import parse as parse_version as by default 2.4 > 2.14 which is wrong
        # Let's collectively hope there will be PyG 3.0 after 2.9 and not 2.10
        # PyG reports e.g. "2.8.0.post1" on Colab and locally; keep only the leading numeric
        # fields. (The notebooks used to patch this line in place; now it is fixed at source.)
        pyg_version = [int(i) for i in torch_geometric.__version__.split(".") if i.isdigit()]
        col_fn = (
            self.inspector.distribute
            if pyg_version[1] <= 4
            else self.inspector.collect_param_data
        )

        msg_aggr_kwargs = col_fn("message_and_aggregate", coll_dict)
        for hook in self._message_and_aggregate_forward_pre_hooks.values():
            res = hook(self, (edge_index, msg_aggr_kwargs))
            if res is not None:
                edge_index, msg_aggr_kwargs = res
        out = self.message_and_aggregate(edge_index, **msg_aggr_kwargs)
        for hook in self._message_and_aggregate_forward_hooks.values():
            res = hook(self, (edge_index, msg_aggr_kwargs), out)
            if res is not None:
                out = res

        # PyG 2.5 + distribute -> collect_param_data
        update_kwargs = col_fn("update", coll_dict)
        out = self.update(out, **update_kwargs)

        for hook in self._propagate_forward_hooks.values():
            res = hook(self, (edge_index, size, kwargs), out)
            if res is not None:
                out = res

        return out

    def message(self, input_j, relation, boundary, edge_type):
        relation_j = relation.index_select(self.node_dim, edge_type)

        if self.message_func == "transe":
            message = input_j + relation_j
        elif self.message_func == "distmult":
            message = input_j * relation_j
        elif self.message_func == "rotate":
            x_j_re, x_j_im = input_j.chunk(2, dim=-1)
            r_j_re, r_j_im = relation_j.chunk(2, dim=-1)
            message_re = x_j_re * r_j_re - x_j_im * r_j_im
            message_im = x_j_re * r_j_im + x_j_im * r_j_re
            message = torch.cat([message_re, message_im], dim=-1)
        else:
            raise ValueError(f"Unknown message function `{self.message_func}`")

        # augment messages with the boundary condition
        message = torch.cat(
            [message, boundary], dim=self.node_dim
        )  # (num_edges + num_nodes, batch_size, input_dim)

        return message

    def aggregate(self, input, edge_weight, index, dim_size):
        # augment aggregation index with self-loops for the boundary condition
        index = torch.cat(
            [index, torch.arange(dim_size, device=input.device)]
        )  # (num_edges + num_nodes,)
        if edge_weight.dim() == 2:
            # per-query weights [B, E]; the boundary self-loops get weight 1 for every query
            assert input.ndim == 3, "per-query edge weights expect input [B, E + N, D]"
            edge_weight = torch.cat(
                [edge_weight, torch.ones(edge_weight.shape[0], dim_size,
                                         device=input.device, dtype=edge_weight.dtype)], dim=1
            ).to(input.dtype).unsqueeze(-1)                       # [B, E + N, 1]
        else:
            edge_weight = torch.cat(
                [edge_weight, torch.ones(dim_size, device=input.device)]
            )
            shape = [1] * input.ndim
            shape[self.node_dim] = -1
            edge_weight = edge_weight.view(shape)

        if self.aggregate_func == "pna":
            mean = variadic.native_scatter(
                input * edge_weight,
                index,
                dim=self.node_dim,
                dim_size=dim_size,
                reduce="mean",
            )
            sq_mean = variadic.native_scatter(
                input**2 * edge_weight,
                index,
                dim=self.node_dim,
                dim_size=dim_size,
                reduce="mean",
            )
            max = variadic.native_scatter(
                input * edge_weight,
                index,
                dim=self.node_dim,
                dim_size=dim_size,
                reduce="amax",
            )
            min = variadic.native_scatter(
                input * edge_weight,
                index,
                dim=self.node_dim,
                dim_size=dim_size,
                reduce="amin",
            )
            std = (sq_mean - mean**2).clamp(min=self.eps).sqrt()
            features = torch.cat(
                [
                    mean.unsqueeze(-1),
                    max.unsqueeze(-1),
                    min.unsqueeze(-1),
                    std.unsqueeze(-1),
                ],
                dim=-1,
            )
            features = features.flatten(-2)
            degree_out = degree(index, dim_size).unsqueeze(0).unsqueeze(-1)
            scale = degree_out.log()
            scale = scale / scale.mean()
            scales = torch.cat(
                [torch.ones_like(scale), scale, 1 / scale.clamp(min=1e-2)], dim=-1
            )
            output = (features.unsqueeze(-1) * scales.unsqueeze(-2)).flatten(-2)
        else:
            output = variadic.native_scatter(
                input * edge_weight,
                index,
                dim=self.node_dim,
                dim_size=dim_size,
                reduce=self.aggr2reduce[self.aggregate_func],
            )

        return output

    def message_and_aggregate(
        self,
        edge_index,
        input,
        relation,
        boundary,
        edge_type,
        edge_weight,
        index,
        dim_size,
    ):
        # fused computation of message and aggregate steps with the custom rspmm cuda kernel
        # speed up computation by several times
        # reduce memory complexity from O(|E|d) to O(|V|d), so we can apply it to larger graphs
        from .rspmm import generalized_rspmm

        batch_size, num_node = input.shape[:2]
        input = input.transpose(0, 1).flatten(1)
        relation = relation.transpose(0, 1).flatten(1)
        boundary = boundary.transpose(0, 1).flatten(1)
        degree_out = degree(index, dim_size).unsqueeze(-1) + 1

        if self.message_func in self.message2mul:
            mul = self.message2mul[self.message_func]
        else:
            raise ValueError(f"Unknown message function `{self.message_func}`")

        def _rspmm_add(w):
            """sum='add' rspmm; a [B, E] w runs the kernel once per query on that query's
            [N, D] slice of the flattened [N, B*D] input (same total work as one call)."""
            if w.dim() == 1:
                return generalized_rspmm(edge_index, edge_type, w, relation, input, sum="add", mul=mul)
            assert w.shape[0] == batch_size, f"edge_weight {tuple(w.shape)} vs batch {batch_size}"
            d = input.shape[1] // batch_size
            return torch.cat([
                generalized_rspmm(edge_index, edge_type, w[b].contiguous(),
                                  relation[:, b * d:(b + 1) * d].contiguous(),
                                  input[:, b * d:(b + 1) * d].contiguous(), sum="add", mul=mul)
                for b in range(batch_size)], dim=1)

        if self.aggregate_func == "sum":
            update = _rspmm_add(edge_weight)
            update = update + boundary
        elif self.aggregate_func == "mean":
            update = _rspmm_add(edge_weight)
            update = (update + boundary) / degree_out
        elif self.aggregate_func == "max":
            update = generalized_rspmm(
                edge_index, edge_type, edge_weight, relation, input, sum="max", mul=mul
            )
            update = torch.max(update, boundary)
        elif self.aggregate_func == "pna":
            # we use PNA with 4 aggregators (mean / max / min / std)
            # and 3 scalars (identity / log degree / reciprocal of log degree)
            sum = generalized_rspmm(
                edge_index, edge_type, edge_weight, relation, input, sum="add", mul=mul
            )
            sq_sum = generalized_rspmm(
                edge_index,
                edge_type,
                edge_weight,
                relation**2,
                input**2,
                sum="add",
                mul=mul,
            )
            max = generalized_rspmm(
                edge_index, edge_type, edge_weight, relation, input, sum="max", mul=mul
            )
            min = generalized_rspmm(
                edge_index, edge_type, edge_weight, relation, input, sum="min", mul=mul
            )
            mean = (sum + boundary) / degree_out
            sq_mean = (sq_sum + boundary**2) / degree_out
            max = torch.max(max, boundary)
            min = torch.min(min, boundary)  # (node, batch_size * input_dim)
            std = (sq_mean - mean**2).clamp(min=self.eps).sqrt()
            features = torch.cat(
                [
                    mean.unsqueeze(-1),
                    max.unsqueeze(-1),
                    min.unsqueeze(-1),
                    std.unsqueeze(-1),
                ],
                dim=-1,
            )
            features = features.flatten(-2)  # (node, batch_size * input_dim * 4)
            scale = degree_out.log()
            scale = scale / scale.mean()
            scales = torch.cat(
                [torch.ones_like(scale), scale, 1 / scale.clamp(min=1e-2)], dim=-1
            )  # (node, 3)
            update = (features.unsqueeze(-1) * scales.unsqueeze(-2)).flatten(
                -2
            )  # (node, batch_size * input_dim * 4 * 3)
        else:
            raise ValueError(f"Unknown aggregation function `{self.aggregate_func}`")

        update = update.view(num_node, batch_size, -1).transpose(0, 1)
        return update

    def update(self, update, input):
        # node update as a function of old states (input) and this layer output (update)
        output = self.linear(torch.cat([input, update], dim=-1))
        if self.layer_norm:
            output = self.layer_norm(output)
        if self.activation:
            output = self.activation(output)
        return output


In [ ]:
%%writefile /content/gfm-rag/gfmrag/models/ultra/models.py
# mypy: ignore-errors
import os
import torch
from torch import autograd, nn

from . import layers
from .base_nbfnet import BaseNBFNet


class EntityNBFNet(BaseNBFNet):
    """Neural Bellman-Ford Network for entity prediction."""

    def __init__(self, input_dim, hidden_dims, num_relation=1, **kwargs):
        # dummy num_relation = 1 as we won't use it in the NBFNet layer
        super().__init__(input_dim, hidden_dims, num_relation, **kwargs)
        self.return_hidden = kwargs.get("return_hidden", False)
        self.layers = nn.ModuleList()
        for i in range(len(self.dims) - 1):
            self.layers.append(
                layers.GeneralizedRelationalConv(
                    self.dims[i],
                    self.dims[i + 1],
                    num_relation,
                    self.dims[0],
                    self.message_func,
                    self.aggregate_func,
                    self.layer_norm,
                    self.activation,
                    dependent=False,
                    project_relations=True,
                )
            )

        feature_dim = (
            sum(hidden_dims) if self.concat_hidden else hidden_dims[-1]
        ) + input_dim
        if not self.return_hidden:
            self.mlp = nn.Sequential()
            mlp = []
            for i in range(self.num_mlp_layers - 1):
                mlp.append(nn.Linear(feature_dim, feature_dim))
                mlp.append(nn.ReLU())
            mlp.append(nn.Linear(feature_dim, 1))
            self.mlp = nn.Sequential(*mlp)

    def bellmanford(self, data, h_index, r_index, separate_grad=False):
        batch_size = len(r_index)

        # initialize queries (relation types of the given triples)
        query = self.query[torch.arange(batch_size, device=r_index.device), r_index]
        index = h_index.unsqueeze(-1).expand_as(query)

        # initial (boundary) condition - initialize all node states as zeros
        boundary = torch.zeros(
            batch_size, data.num_nodes, self.dims[0], device=h_index.device
        )
        # by the scatter operation we put query (relation) embeddings as init features of source (index) nodes
        boundary.scatter_add_(1, index.unsqueeze(1), query.unsqueeze(1))

        size = (data.num_nodes, data.num_nodes)
        edge_weight = torch.ones(data.num_edges, device=h_index.device)

        hiddens = []
        edge_weights = []
        layer_input = boundary

        for layer in self.layers:
            # for visualization
            if separate_grad:
                edge_weight = edge_weight.clone().requires_grad_()

            # Bellman-Ford iteration, we send the original boundary condition in addition to the updated node states
            hidden = layer(
                layer_input,
                query,
                boundary,
                data.edge_index,
                data.edge_type,
                size,
                edge_weight,
            )
            if self.short_cut and hidden.shape == layer_input.shape:
                # residual connection here
                hidden = hidden + layer_input
            hiddens.append(hidden)
            edge_weights.append(edge_weight)
            layer_input = hidden

        # original query (relation type) embeddings
        node_query = query.unsqueeze(1).expand(
            -1, data.num_nodes, -1
        )  # (batch_size, num_nodes, input_dim)
        if self.concat_hidden:
            output = torch.cat(hiddens + [node_query], dim=-1)
        else:
            output = torch.cat([hiddens[-1], node_query], dim=-1)

        return {
            "node_feature": output,
            "edge_weights": edge_weights,
        }

    def forward(self, data, relation_representations, batch):
        h_index, t_index, r_index = batch.unbind(-1)

        # initial query representations are those from the relation graph
        self.query = relation_representations

        # initialize relations in each NBFNet layer (with uinque projection internally)
        for layer in self.layers:
            layer.relation = relation_representations

        # if self.training:
        # Edge dropout in the training mode
        # here we want to remove immediate edges (head, relation, tail) from the edge_index and edge_types
        # to make NBFNet iteration learn non-trivial paths
        # data = self.remove_easy_edges(data, h_index, t_index, r_index)

        shape = h_index.shape
        # turn all triples in a batch into a tail prediction mode
        h_index, t_index, r_index = self.negative_sample_to_tail(
            h_index, t_index, r_index, num_direct_rel=data.num_relations // 2
        )
        assert (h_index[:, [0]] == h_index).all()
        assert (r_index[:, [0]] == r_index).all()

        # message passing and updated node representations
        output = self.bellmanford(
            data, h_index[:, 0], r_index[:, 0]
        )  # (num_nodes, batch_size, feature_dim）
        feature = output["node_feature"]
        index = t_index.unsqueeze(-1).expand(-1, -1, feature.shape[-1])
        # extract representations of tail entities from the updated node states
        feature = feature.gather(
            1, index
        )  # (batch_size, num_negative + 1, feature_dim)

        # probability logit for each tail node in the batch
        # (batch_size, num_negative + 1, dim) -> (batch_size, num_negative + 1)
        score = self.mlp(feature).squeeze(-1)
        return score.view(shape)


class QueryNBFNet(EntityNBFNet):
    """
    The entity-level reasoner for UltraQuery-like complex query answering pipelines.

    This class extends EntityNBFNet to handle query-specific reasoning in knowledge graphs.
    Key differences from EntityNBFNet include:

    1. Initial node features are provided during forward pass rather than read from triples batch
    2. Query comes from outer loop
    3. Returns distribution over all nodes (assuming t_index covers all nodes)

    Attributes:
        layers: List of neural network layers for message passing
        short_cut: Boolean flag for using residual connections
        concat_hidden: Boolean flag for concatenating hidden states
        mlp: Multi-layer perceptron for final scoring
        num_beam: Beam size for path search
        path_topk: Number of top paths to return

    Methods:
        bellmanford(data, node_features, query, separate_grad=False):
            Performs Bellman-Ford message passing iterations.
            Args:
                data: Graph data object containing edge information
                node_features: Initial node representations
                query: Query representation
                separate_grad: Whether to track gradients separately for edges
            Returns:
                dict: Contains node features and edge weights

        forward(data, node_features, relation_representations, query):
            Main forward pass of the model.
            Args:
                data: Graph data object
                node_features: Initial node features
                relation_representations: Representations for relations
                query: Query representation
            Returns:
                torch.Tensor: Scores for each node

        visualize(data, sample, node_features, relation_representations, query):
            Visualizes reasoning paths for given entities.
            Args:
                data: Graph data object
                sample: Dictionary containing entity masks
                node_features: Initial node features
                relation_representations: Representations for relations
                query: Query representation
            Returns:
                dict: Contains paths and weights for target entities
    """

    def _route_attention(self, li, h, query, layer, data, edge_weight):
        """RED-GNN-style query-conditioned attention, one weight per (query, edge).

        logit[b, e] = a . tanh(W_n h_b[src(e)] + W_r r_b[type(e)] + W_q q_b + emb[layer])
        alpha       = softmax of logit over every receiver's INCOMING edges, per query
        weight      = alpha * in_degree(receiver)        (ROUTE_ATTN_NORM=1, default)

        The degree factor makes the weights average 1 over each receiver's incoming
        edges, so the aggregation keeps the control's scale and a uniform attention is
        exactly the ungated layer (the output layer is zero-initialised, so that is
        epoch 0). ROUTE_ATTN_NORM=0 gives RED-GNN's raw softmax, which also changes the
        aggregation from a sum to a weighted mean and is therefore a second variable.
        Trained by the ranking loss only. Runs in fp32 like the CCMP head.
        """
        from torch_geometric.utils import softmax as _pyg_softmax
        src, dst = data.edge_index[0], data.edge_index[1]
        B = h.shape[0]
        with torch.autocast(device_type=h.device.type, enabled=False):
            zn = self.attn_node[li](h.float())                          # [B, N, H]
            rel = layer.relation
            if isinstance(rel, nn.Embedding):
                rel = rel.weight
            if rel.dim() == 2:
                rel = rel.unsqueeze(0).expand(B, -1, -1)
            zr = self.attn_rel(rel.float())                             # [B, R, H]
            zq = self.attn_query(query.float()).unsqueeze(1)            # [B, 1, H]
            ze = (zn.index_select(1, src) + zr.index_select(1, data.edge_type) + zq
                  + self.attn_emb.weight[li].float())                   # [B, E, H]
            logit = self.attn_out(torch.tanh(ze)).squeeze(-1)           # [B, E]
            alpha = _pyg_softmax(logit.t().contiguous(), dst, num_nodes=data.num_nodes)  # [E, B]
            if getattr(self, "attn_norm", True):
                deg = torch.bincount(dst, minlength=data.num_nodes).to(alpha.dtype)
                alpha = alpha * deg.index_select(0, dst).unsqueeze(-1)
            w = alpha.t().contiguous()                                  # [B, E]
            w = w * edge_weight.float().unsqueeze(0)
        return w

    def bellmanford(self, data, node_features, query, separate_grad=False):
        import torch.distributed as dist

        dist_context = getattr(data, "dist_context", None)
        is_dist = dist_context is not None and dist.is_initialized()
        boundary_mode = getattr(data, "boundary_mode", False)

        size = (data.num_nodes, data.num_nodes)
        edge_weight = torch.ones(data.num_edges, device=query.device, dtype=query.dtype)

        hiddens = []
        edge_weights = []

        if is_dist and boundary_mode:
            # ------------------------------------------------------------------
            # Boundary-only AllGather (METIS partition)
            # ------------------------------------------------------------------
            # Scalable variant: only communicate boundary source node states
            # instead of the full hidden tensor.  The compact tensor layout is
            # [local_nodes | boundary_nodes] with edges pre-remapped to this
            # space by partition_graph_metis().
            # ------------------------------------------------------------------
            rank, world_size = dist_context
            num_nodes = data.num_nodes
            local_nodes = data.local_nodes
            boundary_nodes = data.boundary_nodes
            compact_size = data.compact_size
            node2part = data.node2part
            local_N = local_nodes.shape[0]
            boundary_N = boundary_nodes.shape[0]

            # Collect local_nodes from all ranks for scatter-back.
            all_local_nodes_list = [None] * world_size
            dist.all_gather_object(all_local_nodes_list, local_nodes.cpu())
            all_local_N = [len(ns) for ns in all_local_nodes_list]
            max_local_N = max(all_local_N)

            def _scatter_allgather(local_t):
                """AllGather local outputs and scatter to global positions."""
                B, loc_n, D = local_t.shape
                if loc_n < max_local_N:
                    pad = local_t.new_zeros(B, max_local_N - loc_n, D)
                    padded = torch.cat([local_t, pad], dim=1).contiguous()
                else:
                    padded = local_t.contiguous()
                chunks = [torch.zeros_like(padded) for _ in range(world_size)]
                dist.all_gather(chunks, padded)
                # Scatter each rank's results to correct global positions.
                output = local_t.new_zeros(B, num_nodes, D)
                for r in range(world_size):
                    r_nodes = all_local_nodes_list[r].to(local_t.device)
                    r_data = chunks[r][:, : all_local_N[r], :]
                    output[:, r_nodes, :] = r_data
                return output

            # Precompute boundary exchange info (once per forward).
            boundary_owners = node2part[boundary_nodes]
            all_boundary_nodes_list = [None] * world_size
            dist.all_gather_object(all_boundary_nodes_list, boundary_nodes.cpu())

            local_nodes_set = set(local_nodes.cpu().tolist())
            send_indices = {}
            for r in range(world_size):
                if r == rank:
                    continue
                their_boundary = set(all_boundary_nodes_list[r].tolist())
                needed_from_us = their_boundary & local_nodes_set
                if needed_from_us:
                    needed_t = torch.tensor(sorted(needed_from_us), device=query.device)
                    idx = torch.searchsorted(local_nodes, needed_t)
                    send_indices[r] = idx

            recv_indices = {}
            for r in range(world_size):
                if r == rank:
                    continue
                mask = boundary_owners == r
                if mask.any():
                    recv_indices[r] = mask.nonzero(as_tuple=True)[0]

            # Initial local hidden states and compact boundary condition.
            local_layer_input = node_features[:, local_nodes, :].clone()
            compact_boundary = torch.cat([
                node_features[:, local_nodes, :],
                node_features[:, boundary_nodes, :],
            ], dim=1)

            compact_edge_size = (compact_size, compact_size)

            for layer in self.layers:
                if separate_grad:
                    edge_weight = edge_weight.clone().requires_grad_()

                # Exchange boundary states.
                B, _, D = local_layer_input.shape
                boundary_hidden = local_layer_input.new_zeros(B, boundary_N, D)

                send_data = {}
                for r, idx in send_indices.items():
                    send_data[r] = local_layer_input[:, idx, :].contiguous()

                all_send_data = [None] * world_size
                dist.all_gather_object(all_send_data, send_data)

                for r in range(world_size):
                    if r == rank:
                        continue
                    if rank in all_send_data[r]:
                        states = all_send_data[r][rank].to(query.device)
                        boundary_hidden[:, recv_indices[r], :] = states

                compact_input = torch.cat(
                    [local_layer_input, boundary_hidden], dim=1
                )

                hidden = layer(
                    compact_input,
                    query,
                    compact_boundary,
                    data.edge_index,
                    data.edge_type,
                    compact_edge_size,
                    edge_weight,
                )

                local_hidden = hidden[:, :local_N, :]
                if self.short_cut and local_hidden.shape == local_layer_input.shape:
                    local_hidden = local_hidden + local_layer_input
                hiddens.append(local_hidden)
                edge_weights.append(edge_weight)
                local_layer_input = local_hidden

            node_query_local = (
                query.unsqueeze(1).expand(-1, local_N, -1).contiguous()
            )
            if self.concat_hidden:
                local_output = torch.cat(hiddens + [node_query_local], dim=-1)
            else:
                local_output = torch.cat([hiddens[-1], node_query_local], dim=-1)

            output = _scatter_allgather(local_output)

        elif is_dist:
            # ------------------------------------------------------------------
            # Distributed split-graph inference
            # ------------------------------------------------------------------
            # Strategy (mathematically exact):
            #   1. Each rank owns nodes [local_start, local_end) and the edges
            #      whose *target* falls in that slice (set by partition_graph_edges).
            #   2. Before each layer: AllGather local hidden states â†’ full (B,N,D).
            #   3. Run the layer with the full source states but local-only edges.
            #      The layer output is correct at local target positions; non-local
            #      positions contain boundary-only values and are discarded.
            #   4. Slice to local portion, apply residual, store.
            #   5. After all layers: AllGather the local concatenated output once
            #      to reconstruct the full result on every rank.
            # ------------------------------------------------------------------
            rank, world_size = dist_context
            num_nodes = data.num_nodes
            base_N = (num_nodes + world_size - 1) // world_size  # ceiling division
            local_start = rank * base_N
            local_end = min((rank + 1) * base_N, num_nodes)
            local_N = local_end - local_start

            # Collect each rank's actual local_N once (for uneven last partition).
            local_N_t = torch.tensor(local_N, device=query.device)
            all_local_N_list = [torch.zeros_like(local_N_t) for _ in range(world_size)]
            dist.all_gather(all_local_N_list, local_N_t)
            all_local_N = [int(x.item()) for x in all_local_N_list]
            max_local_N = max(all_local_N)  # == base_N

            def _allgather(local_t: torch.Tensor) -> torch.Tensor:
                """AllGather (B, local_N, D) across ranks into (B, N, D).

                Handles the case where the last rank may have fewer nodes than
                the others by zero-padding to max_local_N before gathering,
                then slicing each chunk to its actual size before concatenation.
                """
                B, loc_n, D = local_t.shape
                if loc_n < max_local_N:
                    pad = local_t.new_zeros(B, max_local_N - loc_n, D)
                    padded = torch.cat([local_t, pad], dim=1).contiguous()
                else:
                    padded = local_t.contiguous()
                chunks = [torch.zeros_like(padded) for _ in range(world_size)]
                dist.all_gather(chunks, padded)
                return torch.cat(
                    [chunks[r][:, : all_local_N[r], :] for r in range(world_size)],
                    dim=1,
                )  # (B, N, D)

            # Local slice of the initial boundary / layer input.
            local_layer_input = node_features[:, local_start:local_end, :].clone()

            for layer in self.layers:
                if separate_grad:
                    edge_weight = edge_weight.clone().requires_grad_()

                # AllGather â†’ full source-node states on each rank.
                global_input = _allgather(local_layer_input)  # (B, N, D)

                # Layer forward with full input but local-target edges.
                # hidden[v] is correct for v in [local_start, local_end);
                # non-local positions contain boundary-only values (discarded).
                hidden = layer(
                    global_input,
                    query,
                    node_features,  # boundary: full (B, N, D), same on all ranks
                    data.edge_index,
                    data.edge_type,
                    size,
                    edge_weight,
                )

                # Slice to local, apply residual, then store local hidden.
                local_hidden = hidden[:, local_start:local_end, :]  # (B, local_N, D)
                if self.short_cut and local_hidden.shape == local_layer_input.shape:
                    local_hidden = local_hidden + local_layer_input
                hiddens.append(local_hidden)
                edge_weights.append(edge_weight)
                local_layer_input = local_hidden

            # Concatenate local hidden slices (+ local node_query) then AllGather.
            node_query_local = (
                query.unsqueeze(1).expand(-1, local_N, -1).contiguous()
            )  # (B, local_N, input_dim)
            if self.concat_hidden:
                local_output = torch.cat(hiddens + [node_query_local], dim=-1)
            else:
                local_output = torch.cat([hiddens[-1], node_query_local], dim=-1)
            # local_output: (B, local_N, out_dim)

            output = _allgather(local_output)  # (B, N, out_dim)

        else:
            # ------------------------------------------------------------------
            # Standard single-process path (unchanged)
            # ------------------------------------------------------------------
            layer_input = node_features
            # --- CCMP: contrastive continuation message passing -------------------
            # A node's predicted responsibility scales its OUTGOING messages. Scaling
            # `layer_input` does exactly that: every message leaving v is computed from
            # layer_input[v]. `node_features` is left alone on purpose -- that is the
            # query's own seed injection (the boundary condition), not a routing choice.
            #
            # THE GATE IS MEAN-NORMALISED, and that is a deliberate departure from
            # d + (1-d)*yhat as specified. That form lies in [d, 1], so it can only
            # ATTENUATE, and the factors compound over L layers: even a perfect gate
            # emitting 0.9 everywhere leaves a 6-hop route at 0.53 and a 3-hop route at
            # 0.73. That is a short-path prior applied on top of a model whose measured
            # failure is a hop2->hop3 rank cliff, i.e. it pushes the wrong way. Dividing
            # by the mean keeps the relative selectivity, which is the whole mechanism,
            # and drops the depth-dependent global shrink, which is an artefact.
            # CCMP_GATE_NORM=0 restores the attenuating form for the ablation.
            _rp = []
            _rp_stat = []
            _heads = getattr(self, "resp_proj", None)
            # STRUCTURAL REACH, not "the state is nonzero". Early-late entity fusion puts a
            # static text embedding on EVERY node before propagation, so a nonzero-state
            # test is true almost everywhere from layer 1 and the "activated set" was in
            # practice the whole graph. Then the gate's mean was a graph-wide mean and the
            # normalisation said nothing about what the query had reached.
            _reach = None
            _seed = getattr(self, "_ccmp_seeds", None)
            # Path interpretation (trainer.interpret) asks for the per-layer frontier so a
            # node's responsibility can be read against the frontier mean it was normalised
            # by. Off by default; costs one [B, N] clone per layer when on.
            self._reach_layers = []
            # GATE ATTRIBUTION (interpretation only): keep each layer's gate as a leaf so the trainer can
            # take d score / d g per node and layer; the responsibility head is detached on purpose, the
            # gate is treated as the intervention variable.
            self._gate_tensors = []
            if _heads is not None and _seed is not None:
                if getattr(self, "_ccmp_adj_key", None) != id(data):
                    _ei = data.edge_index
                    _r2 = torch.cat([_ei[0], _ei[1]])
                    _c2 = torch.cat([_ei[1], _ei[0]])
                    self._ccmp_adj = torch.sparse_coo_tensor(
                        torch.stack([_r2, _c2]),
                        torch.ones(_r2.numel(), device=_ei.device),
                        (data.num_nodes, data.num_nodes)).coalesce()
                    self._ccmp_adj_key = id(data)
                _reach = (_seed.to(layer_input.device) > 0).float()
            for _li, layer in enumerate(self.layers):
                if getattr(self, "_keep_reach", False):
                    self._reach_layers.append(None if _reach is None else _reach.detach().clone())
                if _heads is not None:
                    # Pinning the weights to fp32 is not sufficient on its own:
                    # under AMP an nn.Linear emits bf16 whatever its parameter dtype.
                    # The projection stays autocast (big matmul into a 64-dim
                    # bottleneck); the trunk and the sigmoid run in fp32, which is where
                    # the gate's precision actually lives -- bf16's ULP near 1.0 is
                    # 0.0039 and the gate only spans about [0.5, 1.5] at eta=0.5.
                    _z = (_heads[_li](layer_input).float()
                          + self.resp_emb.weight[_li].float())
                    with torch.autocast(device_type=_z.device.type, enabled=False):
                        _yh = torch.sigmoid(self.resp_head(_z).squeeze(-1))  # [B, N]
                    _rp.append(_yh)
                    if getattr(self, "route_mode", "") == "astar":
                        # A*Net-STYLE HARD ROUTING. Keep the top-K reached nodes by the
                        # head's priority and drop the rest of the frontier; kept nodes
                        # are weighted by priority / mean(priority over kept) so the head
                        # receives a gradient from the ranking loss (A*Net multiplies
                        # selected messages by the priority for the same reason) and so
                        # the selection is mean-preserving like CCMP's gate. yhat starts
                        # at 0.5 everywhere, so at initialisation this is a pure top-K.
                        # Off the frontier the gate is 1, exactly as for CCMP.
                        _k = int(getattr(self, "route_k", 1024))
                        _act = (_reach if _reach is not None
                                else torch.ones_like(_yh)).to(_yh.dtype)
                        _sc = torch.where(_act.bool(), _yh, torch.full_like(_yh, -1.0))
                        _top = _sc.topk(min(_k, _sc.shape[-1]), dim=-1).indices
                        _keep = torch.zeros_like(_yh).scatter_(-1, _top, 1.0) * _act
                        _num = _yh * _keep
                        _den = (_num.sum(-1, keepdim=True)
                                / _keep.sum(-1, keepdim=True).clamp(min=1.0)).clamp(min=1e-6)
                        _gt = torch.where(_keep.bool(), _num / _den, torch.zeros_like(_yh))
                        if _reach is not None:
                            _gt = torch.where(_reach.bool(), _gt, torch.ones_like(_gt))
                        with torch.no_grad():
                            _sel = _gt[_keep.bool()]
                            if _sel.numel():
                                _rp_stat.append((float(_sel.mean()), float(_sel.max()),
                                                 float(_sel.quantile(0.95))))
                        _msg = layer_input * _gt.unsqueeze(-1).to(layer_input.dtype)
                    elif getattr(self, "resp_gate", False):
                        _eta = getattr(self, "resp_eta", 1.0)
                        _num = 1e-6 + _yh
                        if getattr(self, "resp_gate_norm", True):
                            # NORMALISE FIRST, THEN INTERPOLATE. mean(ybar) = 1 by
                            # construction, so mean(g) = (1-eta) + eta = 1 for EVERY eta:
                            # the strength knob and the mean-preservation are independent,
                            # and eta = 0 recovers the ungated GNN exactly. Doing it the
                            # other way round (floor, then divide by the mean) also gives
                            # mean 1 but makes the floor a range-compression whose meaning
                            # changes with the spread of yhat.
                            #
                            # The mean is over the ACTIVATED set: nodes the query has
                            # actually reached at this layer. Averaging over all ~60k nodes
                            # would divide by a number dominated by unreached nodes, and
                            # the gate would scale with how far propagation has spread
                            # rather than with what it found.
                            _act = (_reach if _reach is not None
                                    else (layer_input.detach().abs().sum(-1) > 0).float()
                                    ).to(_num.dtype)
                            _den = ((_num * _act).sum(-1, keepdim=True)
                                    / _act.sum(-1, keepdim=True).clamp(min=1.0))
                            _num = _num / _den.clamp(min=1e-6)
                        _gt = (1.0 - _eta) + _eta * _num
                        # GATE ONLY WHAT IS SUPERVISED. The loss trains yhat on A^(l)
                        # alone, so off the frontier the prediction is whatever the head
                        # happens to emit -- and under early fusion those nodes still hold
                        # static text embeddings and still send messages. Applying an
                        # unsupervised gate to them lets CCMP perturb the graph in a
                        # direction no gradient ever checked. Gate 1 there leaves the
                        # background computation exactly as the control computes it.
                        if _reach is not None:
                            _gt = torch.where(_reach.bool(), _gt, torch.ones_like(_gt))
                        # GATE DECOMPOSITION (interpretation only, never set in training): restrict
                        # the gate to a set of layers and/or a set of sender nodes, gate 1 elsewhere.
                        # trainer.gate_decomposition() sets these to ask which gates move a route's
                        # weight: the ones on the route's own senders, at the attributed layers, or
                        # the ones on everybody else.
                        _glm = getattr(self, "_gate_layers", None)
                        if _glm is not None and _li not in _glm:
                            _gt = torch.ones_like(_gt)
                        _gnm = getattr(self, "_gate_nodes", None)
                        if _gnm is not None:
                            _gt = torch.where(_gnm.to(_gt.device).unsqueeze(0).expand_as(_gt).bool(),
                                              _gt, torch.ones_like(_gt))
                        if getattr(self, "_gate_grad", False):
                            _gt = _gt.detach().float().requires_grad_(True)
                            self._gate_tensors.append(_gt)
                        # Unbounded above: the normaliser is a MEAN, so if most reached
                        # nodes sit near 0 a few can be amplified hard, and six layers
                        # compound it. Recorded rather than clipped -- a clip would hide
                        # the instability instead of showing it.
                        with torch.no_grad():
                            _sel = _gt[_reach.bool()] if _reach is not None else _gt
                            if _sel.numel():
                                _rp_stat.append((float(_sel.mean()), float(_sel.max()),
                                                 float(_sel.quantile(0.95))))
                        # GATE THE MESSAGES ONLY. `_raw` is what the residual adds back.
                        # Overwriting layer_input here would put the gate on the residual
                        # too, i.e. h^(l+1) = Conv(g h) + g h instead of Conv(g h) + h, so
                        # the scaling would compound through all six layers and the method
                        # would be re-weighting states rather than routing messages --
                        # which is not what CCMP claims to do.
                        _msg = layer_input * _gt.unsqueeze(-1).to(layer_input.dtype)
                    else:
                        _msg = layer_input
                    if _reach is not None:
                        # AUTOCAST OFF. torch.sparse.mm is on the autocast list, so under
                        # bf16 AMP both operands are cast and there is no
                        # addmm_sparse_cuda kernel for BFloat16 -- it raises rather than
                        # falling back. The frontier is pure topology, so nothing here
                        # should be cast or differentiated in the first place.
                        with torch.no_grad(), torch.autocast(
                                device_type=_reach.device.type, enabled=False):
                            _nx = torch.sparse.mm(
                                self._ccmp_adj, _reach.t().contiguous().float()).t()
                            _reach = ((_nx > 0) | (_reach > 0)).float()
                else:
                    _msg = layer_input
                if separate_grad:
                    edge_weight = edge_weight.clone().requires_grad_()

                # RED-GNN-STYLE EDGE ATTENTION (ROUTE=attn): per-query weights on every
                # edge, computed from the sender state, the relation and the query. Passed
                # as a [B, E] edge_weight, which routes the layer through the unfused
                # message/aggregate path (the rspmm kernel takes one weight per edge).
                _ew = edge_weight
                # ROUTE_CKPT defaults OFF since 8 Sep: layers.py now runs per-query weights
                # through the fused rspmm kernel (O(N d) memory, kernel backward gives the
                # weight gradient), so the checkpoint's recompute only costs time. Set
                # ROUTE_CKPT=1 to get the old unfused+checkpoint path back.
                if getattr(self, "attn_node", None) is not None and \
                        os.environ.get("ROUTE_CKPT", "0") == "1" and torch.is_grad_enabled():
                    # GRADIENT CHECKPOINT THE ATTENTION LAYER. The unfused path keeps every
                    # [B, E+N, D] intermediate (gathered senders, gathered relations, the
                    # product, the boundary concat, the weighted copy) alive for backward:
                    # ~1.5 GB each in bf16 on TOMATO's 253k-edge train graph, x6 layers, which
                    # is how the attn smoke reached 75 GB on an 80 GB A100 (8 Sep) while the
                    # fused arms need a fraction of that. Recompute each layer's forward in
                    # backward instead: peak memory becomes one layer's transient, the
                    # numbers are identical, and the forward is paid twice. The attention
                    # logits are inside the checkpoint so their [B, E, H] tensors are not kept
                    # either. ROUTE_CKPT=0 restores the plain path.
                    from torch.utils.checkpoint import checkpoint as _ckpt

                    def _attn_layer(_x, _m, _q, _nf, _w, _layer=layer, _l=_li):
                        _w2 = self._route_attention(_l, _x, _q, _layer, data, _w)
                        return _layer(_m, _q, _nf, data.edge_index, data.edge_type, size, _w2)

                    hidden = _ckpt(_attn_layer, layer_input, _msg, query, node_features,
                                   edge_weight, use_reentrant=False)
                else:
                    if getattr(self, "attn_node", None) is not None:
                        _ew = self._route_attention(_li, layer_input, query, layer, data, edge_weight)
                    # Bellman-Ford iteration, we send the original boundary condition in addition to the updated node states
                    hidden = layer(
                        _msg,
                        query,
                        node_features,
                        data.edge_index,
                        data.edge_type,
                        size,
                        _ew,
                    )
                if self.short_cut and hidden.shape == layer_input.shape:
                    # residual connection here. UNGATED `layer_input`, not `_msg`: the
                    # gate belongs on the messages this node sends, not on the state it
                    # keeps.
                    hidden = hidden + layer_input
                hiddens.append(hidden)
                edge_weights.append(edge_weight)
                layer_input = hidden
            # Cached for the loss. A list of [B, N] per layer, or empty when CCMP is off,
            # so a control run pays nothing and the attribute always exists.
            self._resp_pred = _rp
            self._resp_gstat = _rp_stat

            # original query (relation type) embeddings
            node_query = query.unsqueeze(1).expand(
                -1, data.num_nodes, -1
            )  # (batch_size, num_nodes, input_dim)
            if self.concat_hidden:
                output = torch.cat(hiddens + [node_query], dim=-1)
            else:
                output = torch.cat([hiddens[-1], node_query], dim=-1)

        return {
            "node_feature": output,
            "edge_weights": edge_weights,
        }

    def forward(self, data, node_features, relation_representations, query):
        # initialize relations in each NBFNet layer (with uinque projection internally)
        for layer in self.layers:
            layer.relation = relation_representations

        # we already did traversal_dropout in the outer loop of UltraQuery
        # if self.training:
        #     # Edge dropout in the training mode
        #     # here we want to remove immediate edges (head, relation, tail) from the edge_index and edge_types
        #     # to make NBFNet iteration learn non-trivial paths
        #     data = self.remove_easy_edges(data, h_index, t_index, r_index)

        # node features arrive in shape (bs, num_nodes, dim)
        # NBFNet needs batch size on the first place
        output = self.bellmanford(
            data, node_features, query
        )  # (num_nodes, batch_size, feature_dim）
        if self.return_hidden:
            return output["node_feature"]
        else:
            score = self.mlp(output["node_feature"]).squeeze(-1)  # (bs, num_nodes)
            # return only the score
            return score

    def visualize(self, data, sample, node_features, relation_representations, query):
        for layer in self.layers:
            layer.relation = relation_representations

        output = self.bellmanford(
            data, node_features, query, separate_grad=True
        )  # (num_nodes, batch_size, feature_dim）
        node_feature = output["node_feature"]
        edge_weights = output["edge_weights"]
        question_entities_mask = sample["start_nodes_mask"]
        target_entities_mask = sample["target_nodes_mask"]
        query_entities_index = question_entities_mask.nonzero(as_tuple=True)[1]
        target_entities_index = target_entities_mask.nonzero(as_tuple=True)[1]

        paths_results = {}
        for t_index in target_entities_index:
            index = (
                t_index.unsqueeze(0)
                .unsqueeze(0)
                .unsqueeze(-1)
                .expand(-1, -1, node_feature.shape[-1])
            )
            feature = node_feature.gather(1, index).squeeze(0)
            score = self.mlp(feature).squeeze(-1)

            edge_grads = autograd.grad(score, edge_weights, retain_graph=True)
            distances, back_edges = self.beam_search_distance(
                data, edge_grads, query_entities_index, t_index, self.num_beam
            )
            paths, weights = self.topk_average_length(
                distances, back_edges, t_index, self.path_topk
            )
            paths_results[t_index.item()] = (paths, weights)
        return paths_results


In [ ]:
%%writefile /content/gfm-rag/gfmrag/trainers/base_trainer.py
import logging
import os
from abc import ABC, abstractmethod
from dataclasses import dataclass
from itertools import islice
from typing import Any

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from gfmrag import utils
from gfmrag.graph_index_datasets.graph_dataset_loader import (
    GraphDataset,
    GraphDatasetLoader,
)
from gfmrag.utils.wandb_utils import (
    log_metrics,
    log_model_checkpoint,
)

from .training_args import TrainingArguments

logger = logging.getLogger(__name__)
# Disable on non-master processes
if utils.get_rank() != 0:
    logger.setLevel(logging.CRITICAL + 1)


@dataclass
class TaskDataset:
    """Task-specific dataset for GFM-RAG models."""

    name: str
    graph: Any
    data_loader: DataLoader


class BaseTrainer(ABC):
    """
    Base trainer class for GFM-RAG models, similar to HuggingFace Trainer.
    """

    separator = ">" * 30
    line = "-" * 30

    def __init__(
        self,
        output_dir: str,
        args: TrainingArguments,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        train_graph_dataset_loader: GraphDatasetLoader | None = None,
        eval_graph_dataset_loader: GraphDatasetLoader | None = None,
        **kwargs: Any,
    ) -> None:
        self.output_dir = output_dir
        self.model = model
        self.optimizer = optimizer
        self.args = args
        self.train_graph_dataset_loader = train_graph_dataset_loader
        self.eval_graph_dataset_loader = eval_graph_dataset_loader

        # Evaluation strategy
        self.eval_strategy = args.eval_strategy
        self.eval_steps = args.eval_steps

        # Set up distributed training
        self.device = utils.get_device()
        self.world_size = utils.get_world_size()
        self.rank = utils.get_rank()

        # Training state
        self.state: dict[str, Any] = {
            "epoch": 0,
            "global_step": 0,
            "best_metric": float("-inf") if args.greater_is_better else float("inf"),
            "best_epoch": -1,
        }

        # Create output directory
        os.makedirs(self.output_dir, exist_ok=True)

        # Set up model for training
        self._setup_model()

    def _setup_model(self) -> None:
        """Set up the model for training."""

        self.model = self.model.to(self.device)
        # Configure model precision based on config
        self.model, self.dtype = utils.configure_model_precision(
            self.model, self.device, self.args.dtype
        )

        self.use_amp = self.dtype != torch.float32
        self.enable_grad_scaler = self.dtype not in [torch.float32, torch.bfloat16]
        self.scaler = torch.amp.GradScaler(
            self.device.type, enabled=self.enable_grad_scaler
        )
        # Resume AFTER the precision cast, not before. Loading optimizer state
        # while the params are still fp32 makes load_state_dict cast exp_avg to
        # fp32; after the bf16 cast the first Adam step then mixes fp32 state
        # with bf16 grads and dies in _foreach_lerp_ (expected float, got
        # BFloat16). Loading here matches the dtype layout a fresh run creates.
        if self.args.resume_from_checkpoint:
            self._load_checkpoint(self.args.resume_from_checkpoint)

        if self.world_size > 1 and not self.args.split_graph_training:
            self.parallel_model = nn.parallel.DistributedDataParallel(
                self.model, device_ids=[self.device]
            )
        else:
            self.parallel_model = self.model

    def _load_checkpoint(self, checkpoint_path: str) -> None:
        """Load a checkpoint."""
        if os.path.exists(checkpoint_path):
            logger.info(f"Loading checkpoint from {checkpoint_path}")
            state = torch.load(
                checkpoint_path, map_location=self.device, weights_only=False
            )

            # Load model state
            if "model" in state:
                self.model.load_state_dict(state["model"], strict=False)

            # Load optimizer state
            if "optimizer" in state and hasattr(self, "optimizer"):
                try:
                    self.optimizer.load_state_dict(state["optimizer"])
                    logger.info("Loaded optimizer state from checkpoint")
                except Exception as e:
                    logger.warning(f"Could not load optimizer state: {e}")

            # Load training state
            if "epoch" in state:
                self.state["epoch"] = state["epoch"]
            if "global_step" in state:
                self.state["global_step"] = state["global_step"]
            if "best_metric" in state:
                self.state["best_metric"] = state["best_metric"]
            if "best_epoch" in state:
                self.state["best_epoch"] = state["best_epoch"]
            # hasattr, because _load_checkpoint is reached from _setup_model(), which
            # __init__ calls BEFORE it builds the AMP scaler. On the resume path
            # self.scaler therefore does not exist yet. The optimizer load above
            # already guards for the same reason; this line was missed, so every
            # resume died with a bare AttributeError.
            if "scaler" in state and hasattr(self, "scaler"):
                self.scaler.load_state_dict(state["scaler"])

            logger.info(
                f"Resumed from epoch {self.state['epoch']}, global step {self.state['global_step']}"
            )
        else:
            logger.warning(f"Checkpoint {checkpoint_path} does not exist")

    def _save_checkpoint(self, output_dir: str, is_best: bool = False) -> None:
        """Save a checkpoint."""
        if utils.get_rank() == 0:
            state = {
                "model": self.model.state_dict(),
                "optimizer": self.optimizer.state_dict(),
                "scaler": self.scaler.state_dict(),
                "epoch": self.state["epoch"],
                "global_step": self.state["global_step"],
                "best_metric": self.state["best_metric"],
                "best_epoch": self.state["best_epoch"],
            }

            # Save best checkpoint
            if is_best:
                best_path = os.path.join(output_dir, "model_best.pth")
                torch.save(state, best_path)
                logger.info(f"Saved best model to {best_path}")

                # Log model checkpoint to wandb
                log_model_checkpoint(
                    best_path,
                    f"best-epoch-{self.state['epoch']}-step-{self.state['global_step']}",
                    metadata={
                        "epoch": self.state["epoch"],
                        "best_metric": self.state["best_metric"],
                        "best": True,
                    },
                )
            # Save regular checkpoint
            elif not self.args.save_best_only:
                checkpoint_path = os.path.join(
                    output_dir,
                    f"checkpoint-epoch-{self.state['epoch']}-step-{self.state['global_step']}.pth",
                )
                torch.save(state, checkpoint_path)
                logger.info(f"Saved checkpoint to {checkpoint_path}")
                log_model_checkpoint(
                    checkpoint_path,
                    f"checkpoint-epoch-{self.state['epoch']}-step-{self.state['global_step']}",
                    metadata={
                        "epoch": self.state["epoch"],
                        "global_step": self.state["global_step"],
                    },
                )

    def _log_metrics(
        self, logs: dict[str, Any], step: int | None = None, prefix: str = ""
    ) -> None:
        """Log metrics."""
        if utils.get_rank() == 0:
            if step is None:
                step = self.state["global_step"]  # type: ignore

            # Log to console
            order = sorted(list(logs.keys()))
            for key in order:
                logger.info(f"{key}: {logs[key]:.4f}")

            # Add Prefix to the logs
            if prefix:
                log_with_prefix = {f"{prefix}/{k}": v for k, v in logs.items()}
            else:
                log_with_prefix = logs

            # Add step to logs
            logs_with_step = {**log_with_prefix, "step": step}

            # Log to wandb
            log_metrics(logs_with_step)

    @abstractmethod
    def _create_task_dataset(
        self, graph_dataset: GraphDataset, is_train: bool = True
    ) -> TaskDataset:
        """
        Create a task-specific dataset from the graph dataset

        Args:
            graph_dataset (GraphDataset): The graph dataset to use.
            is_train (bool): Whether this is for training.

        Returns:
            TaskDataset
        """
        pass

    @abstractmethod
    def train_step(
        self, batch: Any, task_dataset: TaskDataset
    ) -> dict[str, float | torch.Tensor]:
        """
        Perform a single training step.

        Args:
            batch: Training batch from dataloader
            task_dataset: Information about the current dataset

        Returns:
            Dictionary containing loss and other metrics
        """
        pass

    @abstractmethod
    def evaluate(self) -> dict[str, float]:
        """
        Perform the evaluation.

        Returns:
            Dictionary containing evaluation metrics
        """
        pass

    def train(self) -> None:
        """Main training loop."""
        if self.args.do_train:
            logger.info("***** Running training *****")
            logger.info(f"  Num epochs = {self.args.num_epoch}")
            logger.info(
                f"  Instantaneous batch size per device = {self.args.train_batch_size}"
            )
            logger.info(
                f"  Total train batch size (w. parallel & distributed) = {self.args.train_batch_size * self.world_size}"
            )

            start_epoch = self.state["epoch"]

            for epoch in range(start_epoch, self.args.num_epoch):  # type: ignore
                self.state["epoch"] = epoch + 1

                if utils.get_rank() == 0:
                    logger.info(f"{'=' * 50}")
                    logger.info(f"Epoch {self.state['epoch']} / {self.args.num_epoch}")
                    logger.info(f"{'=' * 50}")

                # Training
                self._train_epoch()
                utils.synchronize()

                # Evaluation by epoch
                if (
                    self.eval_strategy == "epoch"
                    and self.eval_graph_dataset_loader is not None
                ):
                    eval_metrics = self.evaluate()
                    self._log_metrics(eval_metrics, prefix="eval")
                    self._maybe_save_best_model(eval_metrics)

                # Save checkpoint
                if not self.args.save_best_only:
                    self._save_checkpoint(self.output_dir)

            utils.synchronize()
            # Load best model at the end
            if self.args.load_best_model_at_end:
                best_model_path = os.path.join(self.output_dir, "model_best.pth")
                if os.path.exists(best_model_path):
                    logger.info("Loading best model for final evaluation")
                    self._load_checkpoint(best_model_path)

            logger.info("Training completed!")

        # Final evaluation
        if self.eval_graph_dataset_loader is not None and self.args.do_eval:
            logger.info("***** Running final evaluation *****")
            final_metrics = self.evaluate()
            self._log_metrics(final_metrics, prefix="final")

    def _train_epoch(self) -> None:
        """Train for one epoch."""
        if self.train_graph_dataset_loader is None:
            logger.warning("No training dataset loader provided")
            return

        self.parallel_model.train()

        epoch_losses = []
        epoch_metrics: dict[str, list[float]] = {}

        # Set epoch for data loader
        if hasattr(self.train_graph_dataset_loader, "set_epoch"):
            self.train_graph_dataset_loader.set_epoch(self.state["epoch"])  # type: ignore

        for graph_dataset in self.train_graph_dataset_loader:
            dataset_name = graph_dataset.name
            task_dataset = self._create_task_dataset(graph_dataset, is_train=True)
            data_loader = task_dataset.data_loader

            # Set epoch for sampler
            if hasattr(data_loader, "sampler") and hasattr(
                data_loader.sampler, "set_epoch"
            ):
                data_loader.sampler.set_epoch(self.state["epoch"])  # type: ignore

            # Limit steps per epoch if specified
            if self.args.max_steps_per_epoch:
                data_iterator = islice(data_loader, self.args.max_steps_per_epoch)
                total_steps = self.args.max_steps_per_epoch
            else:
                data_iterator = data_loader
                total_steps = len(data_loader)

            progress_bar = tqdm(
                data_iterator,
                desc=f"Training {dataset_name} - Epoch {self.state['epoch']}",
                total=total_steps,
                disable=not utils.is_main_process(),
            )

            for batch in progress_bar:
                # Training step
                with torch.amp.autocast(
                    device_type=self.device.type, dtype=self.dtype, enabled=self.use_amp
                ):
                    step_metrics = self.train_step(batch, task_dataset)

                    assert "loss" in step_metrics, (
                        "Training step must return 'loss' in metrics"
                    )

                    # Backward pass
                    loss = step_metrics["loss"]
                    self.scaler.scale(loss).backward()

                    # Split-graph training: manual gradient sync (no DDP wrapper)
                    if self.args.split_graph_training and self.world_size > 1:
                        self.scaler.unscale_(self.optimizer)
                        for param in self.model.parameters():
                            if param.grad is not None:
                                dist.all_reduce(param.grad, op=dist.ReduceOp.AVG)

                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()

                epoch_losses.append(loss.item())  # type: ignore

                # Convert step metrics to float for logging
                step_metrics = {
                    k: v.item() if isinstance(v, torch.Tensor) else v
                    for k, v in step_metrics.items()
                }

                # Accumulate metrics
                for key, value in step_metrics.items():
                    if key not in epoch_metrics:
                        epoch_metrics[key] = []
                    epoch_metrics[key].append(value)

                self.state["global_step"] += 1

                # Log step metrics
                if self.state["global_step"] % self.args.logging_steps == 0:
                    self._log_metrics(step_metrics, prefix="train")

                # Evaluation by step
                if (
                    self.eval_strategy == "step"
                    and self.eval_graph_dataset_loader is not None
                    and self.eval_steps is not None
                    and self.state["global_step"] % self.eval_steps == 0
                ):
                    eval_metrics = self.evaluate()
                    self._log_metrics(eval_metrics, prefix="eval")
                    self._maybe_save_best_model(eval_metrics)

                    # Save checkpoint
                    if not self.args.save_best_only:
                        self._save_checkpoint(self.output_dir)

                # Update progress bar. EVERY term, not just the total: the total is a
                # weighted sum of three or four objectives and a change in it says nothing
                # about which one moved. `ccmp_pos` in particular is the collapse detector
                # -- if the gold-side share of supervised nodes goes to zero the target has
                # degenerated and the run is already dead, which is worth seeing at step 50
                # rather than after four epochs.
                progress_bar.set_postfix(
                    **{k: round(float(v), 4)
                       for k, v in step_metrics.items()
                       if k in ("loss", "hn_fused", "hn_graph", "ccmp",
                                "ccmp_conf", "ccmp_pos", "pop_anchor")})

        # Log epoch averages
        if utils.get_rank() == 0:
            epoch_avg_metrics = {
                f"epoch_{k}": np.mean(v) for k, v in epoch_metrics.items()
            }
            epoch_avg_metrics["epoch"] = self.state["epoch"]
            self._log_metrics(epoch_avg_metrics, prefix="train")
            # ONE LINE PER EPOCH, EVERY COMPONENT. _log_metrics goes to wandb, which the
            # runs disable, so without this the per-term averages are computed and thrown
            # away and only the total reaches stdout.
            _ord = ["loss", "hn_fused", "hn_graph", "ccmp", "aux_graph", "pop_anchor",
                    "ccmp_conf", "ccmp_pos", "ccmp_nodes", "resid_cover", "resid_negs"]
            _have = [k for k in _ord if k in epoch_metrics] + \
                    [k for k in sorted(epoch_metrics) if k not in _ord]
            print("[loss] epoch %d  " % self.state["epoch"]
                  + "  ".join("%s %.4f" % (k, np.mean(epoch_metrics[k])) for k in _have),
                  flush=True)

            logger.info(
                f"Epoch {self.state['epoch']} completed - Average loss: {np.mean(epoch_losses):.4f}"
            )

    def _maybe_save_best_model(self, eval_metrics: dict[str, float]) -> None:
        """Save model if it's the best so far."""
        if self.args.metric_for_best_model is None:
            return

        metric_value = eval_metrics.get(self.args.metric_for_best_model)
        if metric_value is None:
            logger.warning(
                f"Metric {self.args.metric_for_best_model} not found in eval metrics"
            )
            return

        is_best = (
            self.args.greater_is_better and metric_value > self.state["best_metric"]
        ) or (
            not self.args.greater_is_better and metric_value < self.state["best_metric"]
        )

        if is_best:
            self.state["best_metric"] = metric_value
            self.state["best_epoch"] = self.state["epoch"]
            if utils.get_rank() == 0:
                logger.info(
                    f"New best model! {self.args.metric_for_best_model}: {metric_value:.4f} at epoch {self.state['epoch']}"
                )
            self._save_checkpoint(self.output_dir, is_best=True)

        else:
            if utils.get_rank() == 0:
                logger.info(
                    f"Current best {self.args.metric_for_best_model}: {self.state['best_metric']:.4f} at epoch {self.state['best_epoch']}, not updated"
                )


In [ ]:
%%writefile /content/gfm-rag/gfmrag/trainers/fusion_trainer.py
"""
fusion_trainer.py — SFTTrainer for the CARGO fusion (FusionGraphReasoner / RoutedFusionReasoner).

Two objectives, selected by env FUSION_OBJECTIVE:

- "bce_pcr" (default, legacy): the config losses (bce + pcr) act on the FUSED document scores, plus a
  small graph-alone ListCE aux (weight AUX_W). This is the run whose graph stayed inert (aux frozen at
  ~log N): the operator already satisfies bce/pcr on the easy queries, so the GNN never gets a gradient.

- "hardneg" (the report's Eq 3.9 objective): a softmax cross-entropy over a per-query lineup whose
  negatives are the OPERATOR'S OWN top-ranked hubs — {gold} u operator-top-`HARDNEG_HUB` u `HARDNEG_RAND`
  random docs. Ranking the gold above the docs the operator already loves can only be done with the graph,
  so this is what actually teaches the graph to fix the operator's cross-domain misses. It is applied to
  BOTH the fused score (trains gate/router + operator scalars + graph jointly) AND the graph-alone score
  (weight AUX_W — trains the GNN DIRECTLY, so it still learns even when the fusion is initialised
  near-operator and the fused-path gradient into the graph is tiny).

LOSS V2 — three flag-gated, dissim-targeted edits to the hardneg objective.
All default OFF, giving bit-identical legacy behavior:

1. PER_GOLD=1 — per-gold contrastive. Legacy pools golds in one logsumexp, which is a
   soft-max: one easy gold satisfies the query and a buried dissim gold free-rides with
   ~zero gradient. Per-gold, EVERY gold must individually beat the lineup:
       L = sum_g w_g * [ logsumexp(negs u {g}) - s_g ] / sum_g w_g
2. MISS_W_AUX=1 / MISS_W_FUSED=1 — miss-weighting: w_g = log1p(operator rank of gold g),
   capped at MISS_W_CAP (default 8), normalised by the weight sum so the loss scale is
   stable. Concentrates gradient on the golds the operator buries (68% of dissim golds
   sit past rank 100). Recommended always-on for the graph-alone aux (no gate/router in
   that path); on the FUSED term only under convex routing — with the per-query additive
   gate, amplifying the gradient of graph-noisy queries is what taught the gate backwards.
   With PER_GOLD=0 the pooled query term is weighted by its WORST-ranked gold.
3. HARDNEG_GRAPH=K — graph-mined negatives: the graph-alone top-K (detached, golds
   removed) join the lineup, so the contrastive also pushes DOWN docs the graph
   over-scores (PPR domain-hub flooding) instead of only pushing golds up past operator
   hubs. Early in training the GNN is ~random so these are just extra random negatives;
   the term becomes self-adversarial as the graph learns.

Only train_step is overridden; evaluate()/predict() are inherited unchanged and already consume the
fused score, because the fusion lives inside the model's forward().
"""
import os
import json

import torch

from gfmrag.losses import ListCELoss
from gfmrag.models import cqig as cqig_mod
from gfmrag.models.ultra import query_utils

from .sft_trainer import SFTTrainer


def _emb(batch):
    """A single [1, D] question vector for a batch of any size."""
    e = batch["question_embeddings"].float()
    return e.reshape(-1, e.shape[-1]).mean(0, keepdim=True)


class FusionSFTTrainer(SFTTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._aux_loss_fn = ListCELoss()
        self._aux_w = float(os.environ.get("AUX_W", "0.1"))
        self._objective = os.environ.get("FUSION_OBJECTIVE", "bce_pcr")
        self._hn_hub = int(os.environ.get("HARDNEG_HUB", "50"))    # operator-top-K hubs per query
        self._hn_rand = int(os.environ.get("HARDNEG_RAND", "50"))  # random negatives per query
        # Anchor weight for the learned semantic scorer's popularity predictor. Matches
        # --pop_lambda in semantic_scorer.py so the fusion continues training the model
        # under the objective it was selected under, not a different one.
        self._sem_pop_lambda = float(os.environ.get("SEM_POP_LAMBDA", "1.0"))
        # --- Cross-Query Informativeness Gating (inert unless the model has a gate) ---
        self._cqig_m = int(os.environ.get("CQIG_M", "16"))            # reference queries
        self._cqig_pool_size = int(os.environ.get("CQIG_POOL", "64"))  # pool to choose from
        self._cqig_recal = int(os.environ.get("CQIG_RECAL", "1000"))   # steps; 0 = calibrate once
        # Calibration rounds. 1 is exact for a single gated layer; 2 re-runs the whole
        # calibration with the first round's gates active, which is what removes the stale
        # centering when several layers are gated. Costs one more pair of reference passes.
        self._cqig_rounds = int(os.environ.get("CQIG_ROUNDS", "1"))
        # Queries per reference forward. 1 is the safe default: the calibration state is
        # [B, N, D] and the graphs here have ~60k nodes.
        self._cqig_ref_batch = int(os.environ.get("CQIG_REF_BATCH", "1"))
        # Hard negatives per query for the gold-vs-negative informativeness AUC. Matches
        # HARDNEG_HUB so the diagnostic asks about the same documents the loss does.
        self._cqig_hn_k = int(os.environ.get("CQIG_HN_K", os.environ.get("HARDNEG_HUB", "50")))
        # How the frozen source references are re-linked to a target graph.
        #   exact     string equality on entity names, i.e. what the indexer does. Correct
        #             on the source graph and near-useless off it: physics resolved 501/501
        #             on its train graph and 25/501 on its own test graph.
        #   semantic  exact first, then the K nearest target nodes to the seed's frozen
        #             source embedding, above a source-selected cosine floor, weighted by
        #             softmax(cos / T) and normalised so the seed's total weight is
        #             unchanged. Still zero-shot: frozen encoder, source-chosen floor, no
        #             target label or query touched.
        self._cqig_link = os.environ.get("CQIG_LINK", "exact").strip().lower()
        assert self._cqig_link in ("exact", "semantic"), (
            f"CQIG_LINK={self._cqig_link!r}; expected 'exact' or 'semantic'")
        self._cqig_link_k = int(os.environ.get("CQIG_LINK_K", "3"))
        # Softmax temperature over cosines. At 0.05 a 0.1 gap in cosine is a ~7x weight
        # ratio: sharp enough that the best match dominates, soft enough that a genuine
        # near-tie splits rather than being decided by encoder noise.
        self._cqig_link_temp = float(os.environ.get("CQIG_LINK_T", "0.05"))
        # The rejection floor. "auto" measures it on the SOURCE graph (see _cqig_link_floor);
        # a number pins it. Either way it is fixed before any target graph is seen.
        self._cqig_link_min = os.environ.get("CQIG_LINK_MIN", "auto").strip().lower()
        self._cqig_link_q = float(os.environ.get("CQIG_LINK_Q", "0.5"))
        # --- loss v2 flags (all default to exact legacy behavior) ---
        self._per_gold = os.environ.get("PER_GOLD", "0") == "1"
        self._hn_graph = int(os.environ.get("HARDNEG_GRAPH", "0"))       # graph-top-K negatives
        self._miss_w_fused = os.environ.get("MISS_W_FUSED", "0") == "1"
        self._miss_w_aux = os.environ.get("MISS_W_AUX", "0") == "1"
        self._miss_w_cap = float(os.environ.get("MISS_W_CAP", "8.0"))
        # --- RESID_PRIOR: residual supervision against the graph's own structural prior ---
        #
        # MEASURED PROBLEM. A zero-parameter random walk on these graphs scores 0.245/0.273
        # nDCG@5; the trained 6-layer GNN scores 0.213-0.273. The graph-alone contrastive
        # asks the GNN to rank golds above operator hubs, which personalised PageRank from
        # the same seeds largely does already, so the term is near-satisfied at init and the
        # surviving gradient points back at the structure the walk exploits. The learned
        # component is close to free-lunch topology.
        #
        # THE EDIT. Restrict each gold's negatives to the documents the PRIOR already ranks
        # ABOVE it. Documents the prior ordered correctly contribute exactly zero gradient,
        # so the loss can only be reduced by capacity the walk does not have. This is
        # orthogonal to MISS_W_*, which reweights by the SEMANTIC channel's rank and
        # therefore never told the graph anything about what the graph already knew.
        #
        # Costs one cached PPR per query: T sparse mat-muls, computed once and reused for
        # every later epoch, against 6 dense GNN layers per step.
        self._resid_prior = os.environ.get("RESID_PRIOR", "0") == "1"
        self._resid_k = int(os.environ.get("RESID_K", "200"))       # cap on |negatives|
        self._resid_T = int(os.environ.get("RESID_T", "3"))         # walk length
        self._resid_alpha = float(os.environ.get("RESID_ALPHA", "0.15"))   # restart prob
        # Golds the prior already ranks first have an EMPTY negative set and would vanish
        # from the loss entirely, leaving the GNN unconstrained on everything it gets right.
        # They keep a small term against the ordinary lineup instead.
        self._resid_anchor = float(os.environ.get("RESID_ANCHOR", "0.1"))
        self._ppr_A = None          # row-normalised transition, one per graph
        self._ppr_A_key = None
        self._prior_cache = {}      # (graph key, query id) -> document prior, float16 CPU

        # --- CCMP: contrastive continuation message passing ------------------------
        self._ccmp = os.environ.get("CCMP", "0") == "1"
        self._ccmp_w = float(os.environ.get("CCMP_W", "0.1"))     # lambda_r
        self._ccmp_neg = int(os.environ.get("CCMP_NEG", "64"))    # |H^sem_q|
        # Over-fetch semantic errors before applying the graph-reachability filter.
        # CCMP_POOL=256 with CCMP_NEG=64 is the paper setting: it preserves semantic
        # hardness while giving the graph four times as many candidates from which to
        # find errors it can actually reach within the GNN horizon.
        self._ccmp_pool = max(
            self._ccmp_neg, int(os.environ.get("CCMP_POOL", "256"))
        )
        # Select the two sides independently. CCMP_M remains a backwards-compatible
        # total cap for old runs; when supplied alone it is split evenly. New runs use
        # 512 gold-favouring + 512 error-favouring nodes per layer.
        _legacy_m = os.environ.get("CCMP_M")
        _legacy_side = max(1, int(_legacy_m) // 2) if _legacy_m else 512
        self._ccmp_m_pos = int(os.environ.get("CCMP_M_POS", str(_legacy_side)))
        self._ccmp_m_neg = int(os.environ.get("CCMP_M_NEG", str(_legacy_side)))
        self._ccmp_cmin = float(os.environ.get("CCMP_CMIN", "0.2"))
        # RESIDUAL CCMP. Default OFF, so every run made before this is byte-identical.
        # ON: supervision is restricted to the golds the semantic scorer has NOT already
        # resolved, U_q = {g in G_q : exists d not in G_q with s_sem(q,d) >= s_sem(q,g)},
        # and the negatives to the reachable errors that outrank them. Queries with
        # U_q empty are supervised to leave propagation unchanged (g_qv = 1) instead of
        # being asked to improve a ranking that is already correct. Nothing changes at
        # inference: the head still predicts from node states alone.
        self._ccmp_resid = os.environ.get("CCMP_RESIDUAL", "0") == "1"
        # Weight of the identity (neutral) bucket in the class-balanced mean. 0 drops the
        # identity supervision and keeps only the restriction to unresolved golds, which
        # separates the two halves of the change.
        self._ccmp_id_w = float(os.environ.get("CCMP_IDENTITY_W", "1.0"))
        self._ccmp_P = None
        self._ccmp_key = None
        # The selected endpoint set and its continuation targets are fixed on first use,
        # then cached. This keeps CCMP's supervision stationary while the semantic and
        # graph scorers continue to train.
        self._ccmp_cache = {}        # (graph key, query id) -> (idx, y, c, meta) CPU
        if self._ccmp:
            print(
                f"[ccmp targets] reachable semantic errors: pool={self._ccmp_pool} "
                f"keep={self._ccmp_neg}; graph fallback on shortfall; "
                f"nodes/layer=+{self._ccmp_m_pos}/-{self._ccmp_m_neg}"
                + (f"; RESIDUAL on (identity_w={self._ccmp_id_w:g})"
                   if self._ccmp_resid else "; residual off"),
                flush=True,
            )

    def build_lineups(self, target_doc, s_op, g_mine=None):
        """One candidate lineup per query, built ONCE and reused by both losses.

        The fused and graph-alone terms are meant to face the SAME lineup, so that the
        only difference between them is which score is being trained. Building the
        negatives inside each call gave them the same hard negatives (top-k is
        deterministic) but freshly drawn randoms, which is not the design and adds
        variance for nothing.
        """
        B, n_doc = target_doc.shape
        dev = target_doc.device
        out = []
        for b in range(B):
            pos = target_doc[b].nonzero(as_tuple=True)[0]
            if pos.numel() == 0:
                out.append(None)
                continue
            # A UNION, NOT A CONCATENATION. The semantic and graph top-K overlap heavily
            # and torch.randint samples WITH replacement, so cat() put the same paper in
            # the denominator two or three times and silently gave it two or three times
            # the negative weight -- worst for exactly the papers both rankers over-score.
            taken = torch.zeros(n_doc, dtype=torch.bool, device=dev)
            taken[pos] = True
            # Over-fetch by the most that could already be claimed, so a fresh top-k is
            # always available without a .item() sync inside the training loop.
            budget = pos.numel() + self._hn_hub + max(self._hn_graph, 0)

            def _top_fresh(scores, k):
                """Top-k of `scores` not already claimed; marks what it returns."""
                if scores is None or k <= 0:
                    return None
                idx = scores.topk(min(k + budget, n_doc)).indices
                idx = idx[~taken[idx]][:k]
                taken[idx] = True
                return idx

            negs = [x for x in (
                _top_fresh(s_op[b], self._hn_hub),          # the semantic scorer's own hubs
                _top_fresh(g_mine[b] if g_mine is not None else None, self._hn_graph),
            ) if x is not None and x.numel() > 0]
            # EXACTLY `_hn_rand` DISTINCT random non-golds, rather than "50 draws minus
            # whatever collided", which quietly delivered fewer. A permutation is exact;
            # at 1k-20k documents it costs far less than the forward pass it rides on.
            if self._hn_rand > 0:
                pool = torch.randperm(n_doc, device=dev)
                pool = pool[~taken[pool]][: self._hn_rand]
                if pool.numel():
                    negs.append(pool)
            out.append(torch.cat(negs) if negs else pos.new_empty(0))
        return out

    def _contrastive_hardneg(self, doc_scores, target_doc, s_op, g_mine=None, miss_w=False,
                             lineups=None):
        """Contrastive over {gold(s)} u operator-top-K hubs [u graph-top-K] u random negatives.

        doc_scores : [B, n_doc] scores to train (fused or graph-alone), require grad.
        target_doc : [B, n_doc] {0,1} gold mask over document nodes.
        s_op       : [B, n_doc] operator scores (detached) — mines hard-negative hubs + miss weights.
        g_mine     : [B, n_doc] graph-alone scores (DETACHED) — mines HARDNEG_GRAPH extra negatives.
        miss_w     : weight each gold by log1p(its operator rank), capped at MISS_W_CAP.
        """
        B, n_doc = doc_scores.shape
        dev = doc_scores.device
        loss = doc_scores.new_zeros(())
        wsum = 0.0
        for b in range(B):
            pos = target_doc[b].nonzero(as_tuple=True)[0]
            if pos.numel() == 0:
                continue
            # Built once per step and shared by both losses. Falls back to building its
            # own only when called without one, which keeps older callers working.
            neg = (lineups[b] if lineups is not None
                   else self.build_lineups(target_doc, s_op, g_mine)[b])
            neg_logits = doc_scores[b, neg]
            # per-gold miss weights: how badly does the operator rank each gold?
            if miss_w:
                ranks = (s_op[b].unsqueeze(0) > s_op[b, pos].unsqueeze(1)).sum(1).float() + 1.0
                w = torch.log1p(ranks).clamp(max=self._miss_w_cap)
            else:
                w = torch.ones(pos.numel(), device=dev)
            if self._per_gold:
                # every gold must individually beat the lineup (no free-riding behind a sibling)
                for j in range(pos.numel()):
                    lg = torch.cat([neg_logits, doc_scores[b, pos[j : j + 1]]])
                    loss = loss + w[j] * (torch.logsumexp(lg, 0) - doc_scores[b, pos[j]])
                    wsum += float(w[j])
            else:
                # legacy pooled multi-positive; miss_w weights the query by its worst-ranked gold
                logits = torch.cat([doc_scores[b, pos], neg_logits])
                log_z = torch.logsumexp(logits, 0)
                log_pos = torch.logsumexp(logits[: pos.numel()], 0)
                wq = float(w.max())
                loss = loss + wq * (log_z - log_pos)
                wsum += wq
        return loss / max(wsum, 1e-6)

    # ------------------------------------------------- RESID_PRIOR: the structural prior
    def _prior_doc(self, graph, batch, doc_ids):
        """Personalised PageRank over the KG from this query's seeds, restricted to documents.

        Parameter-free and detached: this is the baseline the GNN has to beat, so nothing
        about it may depend on anything the GNN learns. Cached per (graph, seed set), which
        makes it a first-epoch cost only. Cached on CPU in float16 because the train graph
        holds 1304 queries x 4676 documents and that is 12 MB rather than 24.
        """
        dev = batch["start_nodes_mask"].device
        N = graph.num_nodes
        gkey = f"{id(graph)}:{N}"
        if self._ppr_A_key != gkey:
            ei = graph.edge_index
            # SYMMETRISED. The KG is directed, and a walk that can only travel head->tail
            # cannot reach a document from an entity the document mentions, which is the
            # dominant path in this corpus. Both directions, then row-normalise.
            r = torch.cat([ei[0], ei[1]])
            c = torch.cat([ei[1], ei[0]])
            deg = torch.zeros(N, device=dev).index_add_(
                0, r, torch.ones(r.numel(), device=dev))
            w = 1.0 / deg.clamp(min=1.0)[r]
            self._ppr_A = torch.sparse_coo_tensor(
                torch.stack([c, r]), w, (N, N), device=dev).coalesce()
            self._ppr_A_key = gkey
            self._prior_cache.clear()

        seeds = batch["start_nodes_mask"].float()                    # [B, N]
        out = seeds.new_zeros(seeds.shape[0], doc_ids.numel())
        for b in range(seeds.shape[0]):
            idx = seeds[b].nonzero(as_tuple=True)[0]
            key = (gkey, tuple(idx.tolist()))
            hit = self._prior_cache.get(key)
            if hit is None:
                x0 = seeds[b : b + 1]
                s = x0.sum()
                if float(s) <= 0:
                    # no seed resolved: a uniform prior, which makes EVERY document a
                    # negative for every gold. Skipped by the loss instead.
                    hit = torch.zeros(doc_ids.numel(), dtype=torch.float16)
                else:
                    # AUTOCAST OFF and no grad. train_step runs under bfloat16 AMP and
                    # sparse.mm has no bf16 kernel on every build; a crash here would cost
                    # a whole run. The prior is data, not a learned quantity, so neither
                    # autocast nor autograd has any business touching it.
                    with torch.autocast(device_type=x0.device.type, enabled=False), \
                            torch.no_grad():
                        x0 = (x0 / s).float()
                        x = x0
                        for _ in range(self._resid_T):
                            x = ((1.0 - self._resid_alpha)
                                 * torch.sparse.mm(self._ppr_A, x.t()).t()
                                 + self._resid_alpha * x0)
                    hit = x[0, doc_ids].detach().half().cpu()
                self._prior_cache[key] = hit
            out[b] = hit.to(dev).float()
        return out

    def _contrastive_resid(self, doc_scores, target_doc, prior_doc, lineups):
        """Negatives = the documents the PRIOR already ranks above this gold.

        A gold the walk already places above every non-gold has nothing to fix and drops
        out (except for the anchor); a gold the walk buries under 300 documents is asked to
        climb past the worst of them. The loss therefore measures only what the GNN adds
        to the topology, which is the quantity the run is trying to move.
        """
        B = doc_scores.shape[0]
        loss = doc_scores.new_zeros(())
        wsum, n_gold, n_touch, n_neg = 0.0, 0, 0, 0
        for b in range(B):
            pos = target_doc[b].nonzero(as_tuple=True)[0]
            if pos.numel() == 0:
                continue
            pr = prior_doc[b]
            if float(pr.max()) <= 0:
                continue                        # unseeded query: the prior says nothing
            for j in range(pos.numel()):
                g = pos[j]
                n_gold += 1
                above = (pr > pr[g]).nonzero(as_tuple=True)[0]
                if above.numel():
                    # sibling golds are not negatives, whatever the prior thinks of them
                    above = above[~torch.isin(above, pos)]
                if above.numel() == 0:
                    if self._resid_anchor > 0 and lineups is not None and lineups[b] is not None:
                        lg = torch.cat([doc_scores[b, lineups[b]], doc_scores[b, g : g + 1]])
                        loss = loss + self._resid_anchor * (
                            torch.logsumexp(lg, 0) - doc_scores[b, g])
                        wsum += self._resid_anchor
                    continue
                if above.numel() > self._resid_k:
                    # keep the prior's HIGHEST-scored offenders. Those are the documents
                    # actually occupying the top of the ranking the gold has to enter;
                    # the tail of a 3000-long set is noise and would dominate by count.
                    above = above[pr[above].topk(self._resid_k).indices]
                n_touch += 1
                n_neg += int(above.numel())
                lg = torch.cat([doc_scores[b, above], doc_scores[b, g : g + 1]])
                loss = loss + (torch.logsumexp(lg, 0) - doc_scores[b, g])
                wsum += 1.0
        stats = {
            # what fraction of golds the prior actually gets wrong, i.e. how much of the
            # data this loss can even see. If it is near zero the walk is already right
            # and there is nothing to learn; if it is near one the prior is useless here.
            "resid_cover": n_touch / max(n_gold, 1),
            "resid_negs": n_neg / max(n_touch, 1),
        }
        return loss / max(wsum, 1e-6), stats

    # ------------------------------------------------------------------- CCMP targets
    def _ccmp_hit(self, onehot, h):
        """Pr[a walk from v reaches this column's target within h hops], in [0, 1].

        Absorbing recurrence x <- 1_T + (1 - 1_T) P x. NOT sum_t P^t 1_T, which is the
        expected number of visits: unbounded, and it over-counts nodes sitting on short
        cycles, so it is not the probability the ratio is supposed to be a ratio of.
        """
        # SELF-GUARDING. Correct today only because the one caller wraps it, and there is
        # no bf16 addmm_sparse_cuda kernel: under autocast this raises rather than falling
        # back, so a second caller would be a crash, not a slow path.
        with torch.no_grad(), torch.autocast(
                device_type=onehot.device.type, enabled=False):
            x = onehot.clone().float()
            keep = 1.0 - x
            for _ in range(h):
                x = onehot.float() + keep * torch.sparse.mm(self._ccmp_P.float(), x)
        return x

    def _ccmp_targets(self, graph, batch, doc_ids, s_op, g_mine=None):
        """(idx, y, c, meta) per query, with idx/y/c each [L, M]. The endpoint
        set is selected on the query's first occurrence and then cached, making CCMP's
        intermediate supervision stationary while the two retrieval channels train.

        PER-TARGET COLUMNS, REDUCED PER SIDE. Seeding B+ from |G| nodes and B- from |H|
        nodes -- the form as originally written -- makes y ~ |G|/(|G|+|H|) ~ 0.01 and
        c = |2y-1| ~ 1 at essentially EVERY node, so the objective collapses to "predict
        0, confidently", the gate shuts globally and delta is the only thing propagating.
        Reducing each side separately is what makes a hub land at y ~ 0.5, c ~ 0, which
        is the ambiguity the confidence weight exists to express.

        ONE TARGET PER (q, l, v), over the gold UNION. yhat carries no gold index, so a
        per-gold target would only ever be fit to its c-weighted mean over golds -- i.e.
        exactly the "one easy gold satisfies the supervision" failure that per-gold
        computation is supposed to prevent.
        """
        dev = doc_ids.device
        N = graph.num_nodes
        gkey = f"{id(graph)}:{N}"
        if self._ccmp_key != gkey:
            ei = graph.edge_index
            r = torch.cat([ei[0], ei[1]])
            c = torch.cat([ei[1], ei[0]])
            deg = torch.zeros(N, device=dev).index_add_(
                0, r, torch.ones(r.numel(), device=dev))
            self._ccmp_P = torch.sparse_coo_tensor(
                torch.stack([r, c]), 1.0 / deg.clamp(min=1.0)[r],
                (N, N), device=dev).coalesce()
            self._ccmp_key = gkey
            self._ccmp_cache.clear()
        L = len(getattr(self.model.base.entity_model, "resp_proj", []) or [])
        tgt = batch["target_nodes_mask"]
        seeds = batch["start_nodes_mask"]
        out = []
        for b in range(tgt.shape[0]):
            gp = tgt[b, doc_ids].nonzero(as_tuple=True)[0]
            if gp.numel() == 0 or float(seeds[b].sum()) <= 0:
                out.append(None)
                continue
            # KEYED ON THE QUERY ID. Golds + seed COUNT is not a query identity: two
            # queries sharing golds and seed count would collide, and the negatives are
            # the SEMANTIC scorer's top-K, which differ per query, so the second query
            # would silently train against the first one's distractors.
            key = (gkey, str(batch["id"][b]))
            hit = self._ccmp_cache.get(key)
            if hit is None:
                with torch.autocast(device_type=dev.type, enabled=False), torch.no_grad():
                    # STRUCTURAL REACH. A^(0) = the query's seeds, A^(l+1) = A^(l) u
                    # N(A^(l)). Besides restricting layer-l supervision to A^(l), the
                    # final `reach` identifies documents the graph can reach within the
                    # complete L-layer horizon. An unreachable negative has B-=0 from
                    # every query-conditioned state and therefore cannot teach CCMP which
                    # route to suppress.
                    reach = (seeds[b] > 0).float()
                    reaches = []
                    for _ in range(L):
                        reaches.append(reach.clone())
                        nxt = torch.sparse.mm(
                            self._ccmp_P, reach.unsqueeze(1)
                        ).squeeze(1)
                        reach = ((nxt > 0) | (reach > 0)).float()
                    reachable_doc = reach[doc_ids] > 0

                    # Start from the scorer's most convincing errors, but retain only
                    # errors the graph could propagate to in L layers. We over-fetch
                    # CCMP_POOL (256 by default) and keep the first CCMP_NEG (64).
                    gold_doc = torch.zeros(
                        doc_ids.numel(), dtype=torch.bool, device=dev
                    )
                    gold_doc[gp] = True
                    s1 = s_op[b].detach().float().clone()
                    s1[gold_doc] = -torch.inf

                    # WHICH GOLDS THE SEMANTIC SCORER HAS NOT RESOLVED. A gold is
                    # unresolved when at least one non-gold scores at least as highly,
                    # i.e. s_sem(q,g) <= max_{d not in G_q} s_sem(q,d). With CCMP_RESIDUAL
                    # off, gp_use is every gold and no threshold is applied, which is the
                    # historical behaviour exactly.
                    gp_use, neutral = gp, False
                    if self._ccmp_resid:
                        max_neg = s1.max()
                        unres = gp[s_op[b].detach().float()[gp] <= max_neg]
                        if unres.numel() == 0:
                            # Every gold already outranks every non-gold. There is no
                            # error for the graph to correct, so CCMP is supervised to be
                            # a no-op here rather than perturbing a correct ranking. Node
                            # SELECTION is left untouched (below) so the represented set
                            # is the same one the contrastive queries use; only the target
                            # becomes neutral.
                            neutral = True
                        else:
                            gp_use = unres
                            # Negatives become the errors that stand between the query and
                            # its missed golds: non-golds scoring at least as highly as the
                            # lowest unresolved gold. Documents ranked below every missed
                            # gold are not obstructing anything and carry no signal about
                            # which route to suppress.
                            thr = s_op[b].detach().float()[gp_use].min()
                            s1[s1 < thr] = -torch.inf
                    pool_k = min(self._ccmp_pool, s1.numel())
                    sem_pool = s1.topk(pool_k).indices
                    # torch.isfinite drops the -inf padding topk returns once the
                    # residual threshold has masked most of the corpus.
                    sem_ok = (reachable_doc[sem_pool] & ~gold_doc[sem_pool]
                              & torch.isfinite(s1[sem_pool]))
                    sem_reachable = sem_pool[sem_ok]
                    neg_idx = sem_reachable[: self._ccmp_neg]
                    n_sem = int(neg_idx.numel())

                    # Some queries have fewer than 64 reachable documents among the
                    # semantic top 256. Fill only the missing slots with the graph's
                    # highest-ranked reachable errors. Never re-add a gold or a semantic
                    # negative already selected. If the reachable subgraph itself holds
                    # fewer than 64 non-golds, use the smaller honest endpoint set rather
                    # than reintroducing unreachable papers.
                    n_graph = 0
                    missing = self._ccmp_neg - n_sem
                    if missing > 0 and g_mine is not None:
                        eligible = reachable_doc & ~gold_doc
                        if self._ccmp_resid and not neutral:
                            # The graph-score fallback respects the same threshold, or the
                            # "errors that outrank the missed golds" definition would leak.
                            eligible = eligible & torch.isfinite(s1)
                        if neg_idx.numel() > 0:
                            eligible[neg_idx] = False
                        fill_k = min(missing, int(eligible.sum()))
                        if fill_k > 0:
                            g1 = g_mine[b].detach().float().clone()
                            g1[~eligible] = -torch.inf
                            graph_fill = g1.topk(fill_k).indices
                            neg_idx = torch.cat([neg_idx, graph_fill])
                            n_graph = int(graph_fill.numel())

                    # A seed component can contain only gold documents. In that rare
                    # case no valid negative continuation exists, so omitting CCMP for
                    # this query is preferable to taking a mean over an empty B- side.
                    if neg_idx.numel() == 0:
                        out.append(None)
                        continue

                    # B+ is seeded from the UNRESOLVED golds under CCMP_RESIDUAL, and
                    # from every gold otherwise.
                    gi, ni = doc_ids[gp_use], doc_ids[neg_idx]
                    oh = torch.zeros(N, gi.numel() + ni.numel(), device=dev)
                    oh[gi, torch.arange(gi.numel(), device=dev)] = 1.0
                    oh[ni, torch.arange(ni.numel(), device=dev) + gi.numel()] = 1.0
                    # Supervision at layer l is restricted to A^(l), because
                    # h^(l) at an unreached node carries no query information at all: it
                    # is still the static entity embedding that early-late fusion put
                    # there. Asking the head to predict a query-specific target from a
                    # query-independent state is asking it to fit noise, and on this graph
                    # it is nearly all of the sampled supervision.
                    I, Y, C = [], [], []
                    cand_pos = cand_neg = sel_pos = sel_neg = 0
                    for l in range(L):
                        x = self._ccmp_hit(oh, L - l)
                        bp = x[:, :gi.numel()].mean(1)
                        bn = x[:, gi.numel():].mean(1)
                        y = bp / (bp + bn + 1e-6)
                        # c = |B+ - B-| / (B+ + B-), NOT |2y - 1|.
                        #
                        # The two agree exactly wherever the node reaches something. They
                        # differ on the case that dominates this graph: a node reaching
                        # NEITHER set has B+ = B- = 0, so y = 0/eps = 0 and |2y-1| = 1.
                        # The old form therefore labelled every unreachable node
                        # "confidently negative" with full weight, and since the M nodes
                        # are chosen by top-c, the supervision was almost entirely those
                        # nodes: measured at 90% of all nodes passing at layer 1 rising to
                        # 100% by layer 6, with ~0% gold-side. The loss reduced to "predict
                        # zero everywhere", after which a mean-normalised gate returns ~1
                        # and CCMP is a no-op. This form sends those nodes to c = 0.
                        cc = (bp - bn).abs() / (bp + bn + 1e-6)
                        cc = cc * reaches[l]
                        # BALANCED NODE SELECTION. A single top-M over confidence was
                        # dominated by error-favouring nodes (~93% on this graph). The
                        # BCE was class-balanced afterwards, but the representation set
                        # itself contained very few positive routes. Rank the two sides
                        # independently so scarce gold-favouring states cannot be crowded
                        # out before the loss sees them. y=0.5 is genuinely ambiguous and
                        # occupies neither quota.
                        pos_all = ((y > 0.5) & (cc > 0)).nonzero(
                            as_tuple=True
                        )[0]
                        neg_all = ((y < 0.5) & (cc > 0)).nonzero(
                            as_tuple=True
                        )[0]
                        kp = min(self._ccmp_m_pos, int(pos_all.numel()))
                        kn = min(self._ccmp_m_neg, int(neg_all.numel()))
                        cand_pos += int(pos_all.numel())
                        cand_neg += int(neg_all.numel())
                        sel_pos += kp
                        sel_neg += kn
                        pos_idx = (
                            pos_all[cc[pos_all].topk(kp).indices]
                            if kp > 0 else pos_all
                        )
                        neg_idx_nodes = (
                            neg_all[cc[neg_all].topk(kn).indices]
                            if kn > 0 else neg_all
                        )
                        idx = torch.cat([pos_idx, neg_idx_nodes])
                        yy, csel = y[idx], cc[idx]
                        if neutral:
                            # IDENTITY SUPERVISION. A constant prediction across the
                            # reached nodes makes the mean-normalised gate
                            # ybar = (eps+yhat)/mean(eps+yhat) equal 1, hence
                            # g = (1-eta) + eta*1 = 1 and propagation is unchanged. 0.5 is
                            # the constant that also minimises the BCE at p = 0.5, and it
                            # is the sentinel _ccmp_loss routes to the neutral bucket.
                            # Full confidence, because "do not move" is not an ambiguous
                            # instruction; padding stays at c = 0 and is still dropped.
                            yy = torch.full_like(yy, 0.5)
                            csel = torch.ones_like(csel)

                        # Keep fixed [L, M_pos+M_neg] tensors for the CPU cache. Padding
                        # has c=0 and is removed by CCMP_CMIN before the BCE, so it cannot
                        # affect the loss or the reported supervised-node counts.
                        cap = self._ccmp_m_pos + self._ccmp_m_neg
                        pad = cap - idx.numel()
                        if pad > 0:
                            idx = torch.cat([
                                idx,
                                torch.zeros(pad, dtype=torch.long, device=dev),
                            ])
                            yy = torch.cat([yy, torch.full(
                                (pad,), 0.5, dtype=y.dtype, device=dev
                            )])
                            csel = torch.cat([csel, torch.zeros(
                                pad, dtype=cc.dtype, device=dev
                            )])
                        I.append(idx); Y.append(yy); C.append(csel)
                hit = (torch.stack(I).cpu(), torch.stack(Y).half().cpu(),
                       torch.stack(C).half().cpu(),
                       # Counts make the reachability filter auditable in the existing
                       # step and epoch logs without storing any document identifiers.
                       (n_sem, n_graph, int(sem_reachable.numel()), pool_k,
                        cand_pos, cand_neg, sel_pos, sel_neg,
                        # residual diagnostics: is this query neutral, and how many of
                        # its golds the semantic scorer left unresolved.
                        bool(neutral), int(gp_use.numel()), int(gp.numel())))
                self._ccmp_cache[key] = hit
            out.append(hit)
        return out

    def _ccmp_loss(self, preds, targets):
        """Confidence-weighted BCE between yhat^(l)_v and the continuation target."""
        if not preds:
            return None, {}
        dev = preds[0].device
        pn = preds[0].new_zeros(())      # positive-side numerator (y > 1/2)
        nn_ = preds[0].new_zeros(())     # negative-side numerator
        un_ = preds[0].new_zeros(())    # neutral / identity numerator (y == 1/2)
        pd = nd = ud = 0.0
        nsup, csum, npos = 0, 0.0, 0
        nquery = n_sem = n_graph = n_neg = n_pool_reach = n_pool = 0
        n_cand_pos = n_cand_neg = n_sel_pos = n_sel_neg = 0
        n_neutral = n_unres_gold = n_gold = 0
        for b, hit in enumerate(targets):
            if hit is None:
                continue
            idx, y, c, meta = hit
            # 11-tuple under CCMP_RESIDUAL, 8-tuple on a cache written before it. Reading
            # both keeps a warm _ccmp_cache from an earlier cell usable.
            (sem_count, graph_count, pool_reach, pool_count,
             cand_pos, cand_neg, sel_pos, sel_neg) = meta[:8]
            is_neutral, n_used, n_all = (meta[8:] if len(meta) >= 11
                                         else (False, 0, 0))
            n_neutral += int(bool(is_neutral))
            n_unres_gold += (0 if is_neutral else n_used)
            n_gold += n_all
            nquery += 1
            n_sem += sem_count
            n_graph += graph_count
            n_neg += sem_count + graph_count
            n_pool_reach += pool_reach
            n_pool += pool_count
            n_cand_pos += cand_pos
            n_cand_neg += cand_neg
            n_sel_pos += sel_pos
            n_sel_neg += sel_neg
            for l in range(min(len(preds), idx.shape[0])):
                ii = idx[l].to(dev)
                yy = y[l].to(dev).float()
                cc = c[l].to(dev).float()
                keep = cc > self._ccmp_cmin
                if not bool(keep.any()):
                    continue
                ii, yy, cc = ii[keep], yy[keep], cc[keep]
                p = preds[l][b, ii].float().clamp(1e-6, 1 - 1e-6)
                bce = -(yy * p.log() + (1 - yy) * (1 - p).log())
                # CLASS-BALANCED. Even after the confidence fix the reached targets run
                # ~93% negative on this graph, so an unbalanced mean is minimised by
                # "predict 0 everywhere" -- the same degenerate solution by a slower route.
                # Each side is normalised by its own confidence mass and the two are
                # averaged, so the head cannot buy the loss down by collapsing.
                # THREE BUCKETS, NOT TWO. y == 0.5 is the identity sentinel written by
                # _ccmp_targets for queries the semantic scorer already resolved. Left in
                # the negative bucket it would read as "predict 0", which is the collapse
                # the class balance exists to prevent, and it would drag the gate down on
                # exactly the same-field queries this change is meant to protect.
                # Ordinary targets never land on 0.5: pos_all/neg_all are strict.
                u = yy == 0.5
                m = yy > 0.5
                if bool(m.any()):
                    pn = pn + (cc[m] * bce[m]).sum(); pd += float(cc[m].sum())
                    npos += int(m.sum())
                neg = (~m) & (~u)
                if bool(neg.any()):
                    nn_ = nn_ + (cc[neg] * bce[neg]).sum(); nd += float(cc[neg].sum())
                if bool(u.any()):
                    un_ = un_ + (cc[u] * bce[u]).sum(); ud += float(cc[u].sum())
                nsup += int(ii.numel())
                csum += float(cc.sum())
        # Mean over whichever buckets carry mass, each normalised by its own confidence.
        # With no neutral targets this is 0.5*(pn/pd) + 0.5*(nn_/nd), identical to before.
        # CCMP_IDENTITY_W=0 drops the identity term, isolating the restriction to
        # unresolved golds from the instruction to stand still.
        terms, weights = [], []
        if pd > 0:
            terms.append(pn / pd); weights.append(1.0)
        if nd > 0:
            terms.append(nn_ / nd); weights.append(1.0)
        if ud > 0 and self._ccmp_id_w > 0:
            terms.append(un_ / ud); weights.append(self._ccmp_id_w)
        if not terms:
            return None, {}
        wsum = sum(weights)
        loss = sum(w * t for w, t in zip(weights, terms)) / wsum
        return loss, {"ccmp_nodes": nsup / max(len(targets), 1),
                      "ccmp_conf": csum / max(nsup, 1),
                      # share of supervised nodes on the gold side. If this is ~0 the
                      # target has collapsed and nothing downstream can work.
                      "ccmp_pos": npos / max(nsup, 1),
                      "ccmp_nodes_pos": npos / max(nquery, 1),
                      "ccmp_nodes_neg": (nsup - npos) / max(nquery, 1),
                      # Pre-selection prevalence remains the collapse diagnostic now
                      # that the selected representation set is deliberately balanced.
                      "ccmp_cand_pos": n_cand_pos / max(nquery, 1),
                      "ccmp_cand_neg": n_cand_neg / max(nquery, 1),
                      "ccmp_cand_pos_frac": n_cand_pos / max(
                          n_cand_pos + n_cand_neg, 1
                      ),
                      "ccmp_sel_pos": n_sel_pos / max(nquery, 1),
                      "ccmp_sel_neg": n_sel_neg / max(nquery, 1),
                      "ccmp_neg_sem": n_sem / max(nquery, 1),
                      "ccmp_neg_graph": n_graph / max(nquery, 1),
                      "ccmp_neg_count": n_neg / max(nquery, 1),
                      "ccmp_pool_reach": n_pool_reach / max(n_pool, 1),
                      # RESIDUAL DIAGNOSTICS. Read these first on a residual run.
                      # ccmp_resolved is the share of queries the semantic scorer already
                      # ranks perfectly, i.e. the share on which CCMP is now a supervised
                      # no-op. If it is ~0 the change cannot do anything and the arm is
                      # the old CCMP; if it is ~1 nothing is being corrected.
                      # ccmp_unres_frac is the share of golds still contested on the
                      # remaining queries, i.e. how much narrower B+ has become.
                      "ccmp_resolved": n_neutral / max(nquery, 1),
                      "ccmp_unres_gold": n_unres_gold / max(nquery - n_neutral, 1),
                      "ccmp_unres_frac": n_unres_gold / max(n_gold, 1)}

    # ------------------------------------------------------------------ CQIG plumbing
    #
    # PROTOCOL. One bank of M source problems is chosen ONCE, from the training queries, and
    # stored on the model in a graph-independent form (frozen question embedding + start-node
    # NAMES). Every graph, including graphs the model has never seen, is calibrated by
    # re-linking those SAME problems against its own vocabulary. Nothing is ever read from
    # the target corpus's query set, so the evaluation is not transductive.
    @staticmethod
    def _cqig_key(task_dataset):
        """Identify a graph. NOT the node count: two corpora could share one."""
        return getattr(task_dataset, "name", None) or f"graph@{id(task_dataset)}"

    def _cqig_gate(self):
        m = getattr(self, "model", None)
        return getattr(m, "cqig", None) if m is not None else None

    @staticmethod
    def _cqig_vocab(ds, which):
        v = getattr(ds, f"cqig_{which}", None)
        assert v, (
            f"task dataset {getattr(ds, 'name', ds)!r} carries no {which}; CQIG re-links its "
            f"reference problems by entity NAME, so it needs the graph's vocabulary. "
            f"_create_task_dataset attaches it.")
        return v

    @staticmethod
    def _cqig_node_emb(ds):
        """This graph's frozen node-text embeddings, or None if it was indexed without them.

        `graph.x` is what the indexer wrote by encoding every node name with the same frozen
        text encoder the questions went through, so a seed name and a target node name are
        already in one space and no encoder has to be carried to inference time.
        """
        graph = getattr(ds, "graph", None)
        return getattr(graph, "x", None) if graph is not None else None

    @staticmethod
    def _cqig_refs_from_batch(batch, id2node, node_emb=None):
        """Split a loader batch into PORTABLE per-query references.

        Portable means nothing in it is sized to, or indexed by, this graph: the question
        embedding is frozen text, and the seeds are the entity names the indexer looked up
        in node2id, recovered here from the mask through id2node together with their
        attachment weights. One entry per QUERY, never per batch.

        `node_emb` is the SOURCE graph's node features. When given, each seed also carries
        its frozen embedding, which is what the semantic linker matches against a target
        vocabulary. It is a property of the seed's text under a frozen encoder, not of the
        source graph's topology, so it travels as cleanly as the name does.
        """
        e = batch["question_embeddings"].detach().float().cpu()
        m = batch["start_nodes_mask"].detach().float().cpu()
        ids = batch.get("id")
        out = []
        for k in range(e.shape[0]):
            nz = torch.nonzero(m[k], as_tuple=False).flatten().tolist()
            rows = [j for j in nz if j in id2node]
            seeds = [(id2node[j], float(m[k, j])) for j in rows]
            se = None
            if node_emb is not None and rows:
                # Same order as `seeds`; set_reference_bank asserts on that pairing.
                se = node_emb[torch.tensor(rows, dtype=torch.long)].detach().float().cpu()
            out.append({"qemb": e[k:k + 1].clone(), "seeds": seeds, "seed_emb": se,
                        "id": (str(ids[k]) if ids is not None else None)})
        return out

    def _cqig_link_floor(self, refs, node_emb):
        """Choose the semantic linker's rejection threshold from SOURCE data only.

        The question the floor has to answer is "how similar are two node names that mean
        the same thing, in THIS encoder's geometry?", and that is answerable without any
        target corpus. For every reference seed, take its nearest OTHER node in the source
        graph: those are the encoder's own near-synonyms, at the scale this graph's
        vocabulary actually produces. The floor is a quantile of that distribution.

        Deliberately not a round number like 0.7. A hand-set constant is a hyperparameter
        tuned on whatever corpus it was first tried on, and it would not survive a change of
        encoder; this rescales with the encoder because it is measured in it.
        """
        emb = torch.cat([r["seed_emb"] for r in refs if r.get("seed_emb") is not None], 0)
        if emb.numel() == 0:
            return 0.0, 0
        # top-2: the first is the seed's own node at cosine 1, the second is its neighbour.
        cos, _ = cqig_mod.topk_cosine(emb.to(node_emb.device), node_emb, 2)
        nn_cos = cos[:, 1].float().cpu()
        q = min(max(self._cqig_link_q, 0.0), 1.0)
        return float(torch.quantile(nn_cos, q)), int(nn_cos.numel())

    @staticmethod
    def _cqig_farthest_point(refs, m):
        """Pick m maximally spread references by farthest-point on question embeddings."""
        embs = torch.cat([r["qemb"] for r in refs], 0)
        embs = embs / embs.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        picked = [0]
        while len(picked) < min(m, len(refs)):
            d = (1.0 - embs @ embs[picked].T).min(dim=1).values
            d[torch.tensor(picked)] = -1.0
            picked.append(int(d.argmax()))
        return [refs[i] for i in picked]

    def _cqig_select_reference(self, batch, ds):
        """Fill a pool of individual TRAINING queries, then freeze M of them as the bank.

        The bank is handed to the MODEL, not kept on the trainer, so it rides in the
        checkpoint and a reloaded model can calibrate a corpus the trainer never saw.
        """
        g = self._cqig_gate()
        if g is None:
            return False
        if g.ref_bank:
            return True
        id2node = self._cqig_vocab(ds, "id2node")
        # SOURCE node features. Only needed for CQIG_LINK=semantic, but stored whenever the
        # graph has them: the bank rides in the checkpoint, and a bank frozen without seed
        # embeddings cannot be linked semantically later without retraining.
        src_emb = self._cqig_node_emb(ds)
        self._cqig_pool = getattr(self, "_cqig_pool", [])
        self._cqig_pool.extend(self._cqig_refs_from_batch(batch, id2node, src_emb))
        if len(self._cqig_pool) < self._cqig_pool_size:
            return False
        pool = self._cqig_pool[: self._cqig_pool_size]
        refs = self._cqig_farthest_point(pool, self._cqig_m)
        # EXACTLY M, OR SAY SO. Every graph is calibrated from this one bank, so M is the
        # same on all of them by construction -- but only if the bank really holds M.
        assert len(refs) == self._cqig_m, (
            f"asked for M={self._cqig_m} reference queries but the pool of "
            f"{len(pool)} yielded {len(refs)}; raise CQIG_POOL above CQIG_M")
        # THE FLOOR IS FROZEN WITH THE BANK, on the source graph, before any target graph
        # exists. Measuring it later against a target would make the threshold a function
        # of the corpus being transferred to, which is exactly what zero-shot forbids.
        floor, n_meas = 0.0, 0
        if self._cqig_link == "semantic":
            assert src_emb is not None, (
                "CQIG_LINK=semantic needs the source graph's node embeddings (graph.x), "
                "which are absent; this dataset was indexed without node features")
            if self._cqig_link_min == "auto":
                floor, n_meas = self._cqig_link_floor(refs, src_emb.to(self.device))
            else:
                floor = float(self._cqig_link_min)
        g.set_reference_bank(refs, meta={
            "m": len(refs), "pool": self._cqig_pool_size,
            "source": self._cqig_key(ds),
            "selection": "farthest-point on question embeddings, per query",
            "link": self._cqig_link, "link_k": self._cqig_link_k,
            "link_temp": self._cqig_link_temp, "link_floor": floor})
        self._cqig_pool = []
        seeds = [len(r["seeds"]) for r in refs]
        print(f"[cqig] source bank frozen: M={len(refs)} individual training queries from "
              f"'{self._cqig_key(ds)}' (pool {self._cqig_pool_size}); "
              f"seeds/query min={min(seeds)} median={sorted(seeds)[len(seeds) // 2]} "
              f"max={max(seeds)}; stored on the model", flush=True)
        if self._cqig_link == "semantic":
            how = (f"source nearest-neighbour cosine, q={self._cqig_link_q:g} over "
                   f"{n_meas} seeds" if self._cqig_link_min == "auto"
                   else f"pinned by CQIG_LINK_MIN={self._cqig_link_min}")
            print(f"[cqig] link=semantic: K={self._cqig_link_k} T={self._cqig_link_temp:g} "
                  f"floor={floor:.4f} [{how}]; seed weight is split by softmax(cos/T) over "
                  f"the survivors, so each seed's total attachment weight is unchanged",
                  flush=True)
        return True

    def _cqig_calibrate(self, graph, ds, why):
        """Re-link the source bank to THIS graph, calibrate, then measure what the gate does."""
        g = self._cqig_gate()
        key = self._cqig_key(ds)
        node2id = self._cqig_vocab(ds, "node2id")
        # The floor travels with the BANK, not with the trainer: a checkpoint calibrated on
        # a new corpus must reject at the same threshold the bank was frozen under, even in
        # a process where CQIG_LINK_MIN was never set.
        meta = g.ref_meta or {}
        batches, cov = g.materialise(
            node2id, int(graph.num_nodes), device=self.device,
            batch_size=self._cqig_ref_batch,
            node_emb=(graph.x if self._cqig_link == "semantic" else None),
            link=self._cqig_link, link_k=self._cqig_link_k,
            link_temp=meta.get("link_temp", self._cqig_link_temp),
            link_floor=meta.get("link_floor", 0.0))
        # SAME PRECISION CONTEXT AS EVERY OTHER FORWARD IN THIS TRAINER. Training, eval and
        # predict all wrap their forwards in autocast. The train-side calibration inherited
        # that by accident, being called from inside the training loop; the inference-side
        # call sits outside it, so float32 activations met bfloat16 weights.
        with torch.amp.autocast(device_type=self.device.type, dtype=self.dtype,
                                enabled=self.use_amp):
            rep = self.model.cqig_calibrate(graph, key, ref_batches=batches,
                                            device=self.device,
                                            rounds=self._cqig_rounds)
        print(f"[cqig] {why}: graph='{key}' M={cov['queries']} queries "
              f"({cov['batches']} forwards x {self._cqig_rounds} round(s) x 2 passes) "
              f"seed-coverage={cov['seed_coverage']:.1%} "
              f"({cov['seeds_found']}/{cov['seeds_total']}) "
              f"queries_with_no_seed={cov['queries_with_no_seed']} "
              f"stats={g.stats_bytes() / 2**20:.0f} MB", flush=True)
        # THE SPLIT, ALWAYS. `exact` is directly comparable to every run made before the
        # semantic linker existed, so the two arms can be read against each other rather
        # than against an aggregate that moved for an unstated reason.
        if cov["link"] == "semantic":
            how = (f"K={cov['link_k']} floor={cov['link_floor']:.4f}, median best cos "
                   f"{cov['sem_cos_median']:.3f}, {cov['sem_nodes_per_seed']:.2f} nodes/seed"
                   if cov["seeds_semantic"] else
                   f"K={cov['link_k']} floor={cov['link_floor']:.4f}, none needed")
            print(f"[cqig]   linking: exact {cov['seeds_exact']} "
                  f"({cov['exact_coverage']:.1%})  + semantic {cov['seeds_semantic']} "
                  f"({how})  = {cov['seeds_found']}/{cov['seeds_total']}; "
                  f"unmatched {cov['seeds_unmatched']}", flush=True)
        # EVERY GATED LAYER, not just the first. With a multi-layer arm the interesting
        # failure is one layer doing the work and the rest sitting at a constant.
        for j in sorted(rep):
            r = rep[j]
            v, iq, gq = r["var_q"], r["I"], r["gate"]
            print(f"[cqig]   layer {j + 1}/{g.n_layers}: nodes={r['nodes']} "
                  f"responding={r['live']} ({r['responding_frac']:.2%})  "
                  f"V p10/p50/p90 = {v['p10']:.4e} {v['p50']:.4e} {v['p90']:.4e}  "
                  f"scale={r['scale']:.4e}  tau_ref={r['tau_ref']:.4e} "
                  f"[{r['tau_src']}]", flush=True)
            print(f"[cqig]     I    min/p10/p50/p90/max = {iq['min']:.4e} {iq['p10']:.4e} "
                  f"{iq['p50']:.4e} {iq['p90']:.4e} {iq['max']:.4e}  "
                  f"std={iq['std']:.4e}  responded={r['resp_frac']:.2%} over "
                  f"{r['sample_queries']} reference queries", flush=True)
            print(f"[cqig]     g    min/p10/p50/p90/max = {gq['min']:.4e} {gq['p10']:.4e} "
                  f"{gq['p50']:.4e} {gq['p90']:.4e} {gq['max']:.4e}  "
                  f"std={gq['std']:.4e}  span={gq['max'] - gq['min']:.4e}", flush=True)
            # THE COEFFICIENT THE ARM ACTUALLY APPLIES. Identical to g under op="gate", so
            # the line is only printed when it would say something different -- and there it
            # is the one that matters, because g is not what multiplies anything.
            if r.get("mu_source", "ref") == "zero":
                # The ablation must be visible in the calibration block itself, not only in
                # the startup banner, because that block is what gets pasted around.
                print(f"[cqig]     mu=ZERO ABLATION: mu is 0 on all {r['mu_zero']} nodes, "
                      f"so I = ||h||^2/scale and the reference bank contributes nothing to "
                      f"this arm. Its score is the magnitude-gate floor, not the method.",
                      flush=True)
            if r.get("op", "gate") != "gate":
                cq = r["coef"]
                print(f"[cqig]     op={r['op']}: h - c*mu, c min/p10/p50/p90/max = "
                      f"{cq['min']:.4e} {cq['p10']:.4e} {cq['p50']:.4e} {cq['p90']:.4e} "
                      f"{cq['max']:.4e}   mu=0 on {r['mu_zero']} nodes "
                      f"({r['mu_zero_frac']:.2%}), which this op cannot touch at all",
                      flush=True)
            print(f"[cqig]     lam={r['lam']:.6e} alpha={r['alpha']:.6e} "
                  f"tau={r['tau']:.6e} gate_at_I=0 {r['gate_unreached']:.4e}", flush=True)
            if r["degenerate"]:
                print(f"[cqig]     WARNING: no node responded to any reference query at "
                      f"layer {j + 1}; the scale fell back to 1.0 and I is meaningless.",
                      flush=True)
            elif gq["max"] - gq["min"] < 1e-3:
                print(f"[cqig]     WARNING: the gate spans {gq['max'] - gq['min']:.2e} "
                      f"across nodes. That is a near-constant rescaling, not a gate, and "
                      f"this arm will read as a no-op whatever the final metrics say.",
                      flush=True)
        if cov["seed_coverage"] < 0.2:
            fix = ("" if cov["link"] == "semantic" else
                   " Exact name lookup is the likely cause, not the corpus: set "
                   "CQIG_LINK=semantic to link the remainder by embedding.")
            print(f"[cqig]   WARNING: only {cov['seed_coverage']:.1%} of the source seeds "
                  f"resolve in '{key}'. The reference problems barely touch this graph, so "
                  f"its statistics are not measuring a response to them.{fix}", flush=True)
        elif cov["link"] == "semantic" and cov["exact_coverage"] < 0.2:
            print(f"[cqig]   note: exact lookup alone would have reached "
                  f"{cov['exact_coverage']:.1%} on '{key}'; the semantic linker carried it "
                  f"to {cov['seed_coverage']:.1%}. That gap is what this arm is testing.",
                  flush=True)
        gb = g.stats_bytes() / 2**30
        if gb > 4.0:
            print(f"[cqig]   WARNING: the reference statistics now hold {gb:.1f} GiB of "
                  f"GPU memory ({len(g.gate_layers)} gated layers x "
                  f"{len(g.known_graphs())} graphs). mu is one float32 row per node per "
                  f"gated layer, so gating fewer layers is the lever if this run runs out "
                  f"of memory.", flush=True)
        return key

    def _cqig_report_live(self, why):
        """What the gate did on the queries that were actually scored, plus the AUC that
        decides whether informativeness has anything to do with relevance."""
        g = self._cqig_gate()
        if g is None or not g.recording():
            return
        rep = g.live_report()
        if not rep:
            g.stop_live()
            return
        for j in sorted(rep):
            r = rep[j]
            what = "gate" if r.get("op", "gate") == "gate" else f"{r['op']} coef"
            print(f"[cqig] {why} {what}, layer {j + 1}: n={r['n']:.4e}  "
                  f"min/p10/p50/p90/max = {r['min']:.4e} {r['p10']:.4e} {r['p50']:.4e} "
                  f"{r['p90']:.4e} {r['max']:.4e}  mean={r['mean']:.4e} "
                  f"std={r['std']:.4e}  span={r['max'] - r['min']:.4e}  "
                  f"lam={float(g.lam(j).detach()):.6e} "
                  f"alpha={float(g.alpha(j).detach()):.6e} "
                  f"tau={g.tau(j):.6e}", flush=True)
            # HOW BIG THE EDIT WAS, measured on the states. A coefficient far from 1 that
            # moves the state by 1e-3 is a no-op wearing a large number, and with
            # `layer_norm: yes` downstream that is the failure worth being able to see.
            print(f"[cqig]   edit ||h~-h||/||h||: mean={r['edit_rel_mean']:.4e} "
                  f"max={r['edit_rel_max']:.4e} on {r['edit_frac']:.2%} of live nodes",
                  flush=True)
            if r["auc_n"]:
                print(f"[cqig]   informativeness AUC, gold vs semantic-top-"
                      f"{self._cqig_hn_k}: {r['auc']:.4f} over {r['auc_n']} queries  "
                      f"(mean I gold {r['I_gold']:.4e} vs negative {r['I_neg']:.4e})",
                      flush=True)
                if abs(r["auc"] - 0.5) < 0.02:
                    print(f"[cqig]   NOTE: an AUC of {r['auc']:.4f} says a high I is no "
                          f"likelier on a gold paper than on a hard negative. The gate's "
                          f"premise, not its calibration, is what is failing.", flush=True)
        g.stop_live()

    def evaluate(self) -> dict:
        """Measure the gate on the real evaluation queries, around the inherited eval.

        Reset before and report after, so the numbers belong to one evaluation rather than
        accumulating across epochs. Purely observational: `reset_live` switches on a
        histogram and a label-side AUC, neither of which is read by any calibration.
        """
        g = self._cqig_gate()
        if g is not None and g.ref_bank:
            g.reset_live()
        m = super().evaluate()
        self._cqig_report_live("eval")
        return m

    def predict(self) -> dict:
        """Same measurement around the pass that writes the predictions file.

        The AUC stays empty here, because prediction batches carry no gold mask; the gate
        distribution is the part that matters, since these are the scores the benchmark
        numbers are computed from.
        """
        g = self._cqig_gate()
        if g is not None and g.ref_bank:
            g.reset_live()
        out = super().predict()
        self._cqig_report_live("predict")
        return out

    # ------------------------------------------------------------ path interpretation
    @torch.no_grad()
    def _channel_scores(self, graph, batch):
        """One forward; returns doc node ids and the four per-document score vectors
        (fused, graph-alone raw, multi-view scorer, Qwen3 cosine), all in nodes.csv doc order."""
        with torch.amp.autocast(device_type=self.device.type, dtype=self.dtype, enabled=self.use_amp):
            pred = self.model(graph, batch)
        did = self.model._doc_ids
        fused = pred[0, did].float()
        graph_raw = self.model._raw_doc[0].float()
        sem = self.model._s_op[0].float()
        if getattr(self.model, "semantic", "mlp") == "operator":
            # operator-scorer checkpoint (the July v1 fusion, e.g. TOMATO on the OpenIE graph): there is
            # no multi-view table, the 'scorer' channel is S_op and the Qwen3 cosine comes from the
            # operator components; callers get (None, None) instead of a views table.
            tag, row = self.model._row[str(batch["id"][0])]
            dense = self.model._dense[tag][row].float().to(fused.device)
            return did, {"fused": fused, "graph": graph_raw, "scorer": sem, "dense": dense}, (None, None)
        tag, row = self.model._sem_row[str(batch["id"][0])]
        tab = self.model._sem_tab[tag]
        dense = tab["dense"][row].float().to(fused.device)
        return did, {"fused": fused, "graph": graph_raw, "scorer": sem, "dense": dense}, (tab, row)

    def _rank_without(self, graph, batch, triples, node, n_random=0):
        """Rank of `node` under the graph and fused channels after dropping the given (h, t, r)
        edges from the graph (or `n_random` random edges when triples is None). Edits the graph
        in place for one forward pass and restores it."""
        ei, et = graph.edge_index, graph.edge_type
        keep = torch.ones(ei.shape[1], dtype=torch.bool, device=ei.device)
        if triples is None:
            if n_random > 0:
                drop = torch.randperm(ei.shape[1], device=ei.device)[:n_random]
                keep[drop] = False
        else:
            for h, t, r in set(tuple(int(x) for x in tr) for tr in triples):
                m = (ei[0] == h) & (ei[1] == t) & (et == r)
                if m.sum() == 0:
                    m = (ei[0] == h) & (ei[1] == t)
                keep &= ~m
        removed = int((~keep).sum().item())
        em = self.model.base.entity_model
        try:
            graph.edge_index, graph.edge_type = ei[:, keep], et[keep]
            em._ccmp_adj_key = None
            with torch.no_grad():
                did, ch, _ = self._channel_scores(graph, batch)
            j = (did == node).nonzero(as_tuple=True)[0]
            out = {"removed": removed}
            if j.numel():
                j = int(j[0])
                for k in ("graph", "fused"):
                    out[k] = int((ch[k] > ch[k][j]).sum().item()) + 1
        finally:
            graph.edge_index, graph.edge_type = ei, et
            em._ccmp_adj_key = None
        return out

    @staticmethod
    def _valid_paths(paths, weights, seeds):
        """Drop decoded paths that do not start at a seed or whose hops do not chain. The beam
        backtrack can emit a placeholder first hop (node 0 -> node 0) when a node's layer-0 beam
        slot was never filled; those are artefacts, not routes. Returns (paths, weights, n_dropped)."""
        keep_p, keep_w = [], []
        for path, w in zip(paths, weights):
            ok = len(path) > 0 and int(path[0][0]) in seeds and all(int(a[1]) == int(b[0]) for a, b in zip(path, path[1:]))
            if ok:
                keep_p.append(path); keep_w.append(w)
        return keep_p, keep_w, len(paths) - len(keep_p)

    def _interpret_target(self, graph, batch, node, em, eta, id2node, id2rel):
        """Gradient beam search from the query's seeds to one node; returns (paths with the CCMP
        gate per hop, frontier-mean responsibility per layer, raw (h, t, r) paths, n_dropped)."""
        seeds = set(batch["start_nodes_mask"][0].nonzero(as_tuple=True)[0].tolist())
        sample = dict(batch); tm = torch.zeros_like(batch["target_nodes_mask"]); tm[0, node] = 1.0
        sample["target_nodes_mask"] = tm
        if getattr(em, "resp_proj", None) is not None:
            em._ccmp_seeds = sample["start_nodes_mask"]
        em._keep_reach = True
        with torch.enable_grad(), torch.amp.autocast(device_type=self.device.type, dtype=self.dtype, enabled=self.use_amp):
            pr = self.model.base.visualize(graph, sample)
        em._keep_reach = False
        rp = [x[0].detach().float() for x in getattr(em, "_resp_pred", [])]
        reach = getattr(em, "_reach_layers", [])
        fm = []
        for l, y in enumerate(rp):
            r_ = reach[l] if l < len(reach) and reach[l] is not None else None
            fm.append(float(y[r_[0].bool()].mean()) if r_ is not None and r_.sum() > 0 else float(y.mean()))
        paths, weights = pr.get(int(node), ([], []))
        paths, weights, dropped = self._valid_paths(paths, weights, seeds)
        paths_out = []
        for path, w in zip(paths, weights):
            hops = []
            for l, (h, t, r) in enumerate(path):
                hop = {"layer": l, "head": id2node[h], "rel": id2rel.get(r, str(r)), "tail": id2node[t]}
                if l < len(rp):
                    y = float(rp[l][h]); hop["resp"] = y; hop["frontier_mean"] = fm[l]
                    hop["gate"] = (1 - eta) + eta * (y / max(fm[l], 1e-6))
                    if l < len(reach) and reach[l] is not None:
                        hop["reached"] = bool(reach[l][0, h] > 0)
                hops.append(hop)
            paths_out.append({"weight": float(w), "hops": hops})
        return paths_out, fm, paths, dropped

    def _edge_grads_for(self, graph, batch, node, em):
        """Per-layer d(graph score of `node`)/d(edge weight), [E] per layer, captured from the same
        forward the beam search uses (base.visualize), so the numbers are exactly the beam's."""
        sample = dict(batch); tm = torch.zeros_like(batch["target_nodes_mask"]); tm[0, node] = 1.0
        sample["target_nodes_mask"] = tm
        if getattr(em, "resp_proj", None) is not None:
            em._ccmp_seeds = sample["start_nodes_mask"]
        cap = {}
        orig = em.beam_search_distance
        def spy(data, edge_grads, h_index, t_index, num_beam=10):
            cap["eg"] = [g.detach().float() for g in edge_grads]
            return orig(data, edge_grads, h_index, t_index, num_beam)
        em.beam_search_distance = spy
        try:
            with torch.enable_grad(), torch.amp.autocast(device_type=self.device.type, dtype=self.dtype, enabled=self.use_amp):
                pr = self.model.base.visualize(graph, sample)
        finally:
            em.beam_search_distance = orig
        return cap["eg"], pr

    def _direct_path_weights(self, graph, edge_grads, paths):
        """Weight of each given path [(h, t, r), ...] under the given per-layer edge gradients: the
        beam's own definition (hop i taken at layer i, mean over hops), evaluated on a fixed route so
        the same route can be compared across gating conditions. None when an edge is missing."""
        key = getattr(self, "_edge_key_cache", None)
        if key is None or key[0] != id(graph):
            ei = graph.edge_index.cpu(); et = graph.edge_type.cpu()
            lut = {}
            for e, (h, t, r) in enumerate(zip(ei[0].tolist(), ei[1].tolist(), et.tolist())):
                lut.setdefault((h, t, r), []).append(e)
            key = (id(graph), lut); self._edge_key_cache = key
        lut = key[1]; out = []
        for path in paths:
            vals = []
            for i, (h, t, r) in enumerate(path):
                es = lut.get((int(h), int(t), int(r)))
                if es is None or i >= len(edge_grads):
                    vals = None; break
                vals.append(max(float(edge_grads[i][e]) for e in es))
            out.append(None if vals is None else sum(vals) / len(vals))
        return out

    def _gate_attribution(self, graph, batch, node, em, id2node, raw_paths, topn=15):
        """Path interpretation through the gate: d(graph score of `node`)/d g_u^(l) for every node u and
        layer l, and the first-order CCMP contribution c_u^(l) = d s/d g * (g - 1). Summed over layers
        this says how much each node's gating moved the gold's score; along a route it says where on the
        route CCMP acted, at every layer, not only the one the hop was attributed to."""
        em.resp_gate = True; em._gate_layers = None; em._gate_nodes = None; em._gate_grad = True
        try:
            with torch.enable_grad(), torch.amp.autocast(device_type=self.device.type, dtype=self.dtype, enabled=self.use_amp):
                self.model(graph, batch)
                did = self.model._doc_ids
                j = (did == node).nonzero(as_tuple=True)[0]
                score = self.model._raw_doc[0, j].float().sum()
                gates = list(em._gate_tensors)
                grads = torch.autograd.grad(score, gates, allow_unused=True)
        finally:
            em._gate_grad = False; em._gate_tensors = []
        G = [g[0].detach().float().cpu() for g in gates]
        D = [(torch.zeros_like(G[l]) if grads[l] is None else grads[l][0].detach().float().cpu()) for l in range(len(G))]
        C = [D[l] * (G[l] - 1.0) for l in range(len(G))]
        total = torch.stack(C).sum(0)
        senders = sorted({int(h) for p_ in raw_paths for (h, t, r) in p_})
        per_sender = {id2node[u]: {"gate": [float(G[l][u]) for l in range(len(G))],
                                   "dscore_dg": [float(D[l][u]) for l in range(len(G))],
                                   "contrib": [float(C[l][u]) for l in range(len(G))],
                                   "contrib_total": float(total[u])} for u in senders}
        routes = [{"senders": [id2node[int(h)] for (h, t, r) in p_], "contrib_total": float(sum(total[int(h)] for (h, t, r) in p_))} for p_ in raw_paths]
        top = torch.topk(total.abs(), min(topn, total.numel())).indices.tolist()
        top_nodes = [{"node": id2node[u], "contrib_total": float(total[u]), "on_route": u in set(senders),
                      "gate": [round(float(G[l][u]), 3) for l in range(len(G))]} for u in top]
        return {"score_gate_on": float(score), "sum_contrib_all_nodes": float(total.sum()),
                "sum_contrib_route_senders": float(sum(total[u] for u in senders)),
                "per_layer_sum_all": [float(c.sum()) for c in C], "route_senders": per_sender, "routes": routes, "top_nodes": top_nodes}

    def gate_decomposition(self, qids, out_path, golds=None, num_beam=10, path_topk=5, max_golds=2):
        """Which gates move a route's weight? For each (query, gold): the top routes under the full
        CCMP gate, then the SAME routes' weights, the gold's graph score and its graph / fused ranks
        under: gate off; gate only at the layers the route is attributed to (0..L-1); gate only at
        the later layers; gate only on the route's own sender nodes; gate on every node except them.
        Same trained weights throughout; only the inference-time gate mask changes."""
        import numpy as np
        self.model.eval()
        em = self.model.base.entity_model
        assert getattr(em, "resp_proj", None) is not None, "gate_decomposition needs a CCMP checkpoint"
        em.num_beam, em.path_topk = int(num_beam), int(path_topk)
        eta = float(getattr(em, "resp_eta", 0.5))
        want = [str(q) for q in qids]
        results = []
        for test_dataset in self.eval_graph_dataset_loader:
            src = test_dataset.data
            graph = src.graph.to(self.device); data = src.test_data
            id2node = src.id2node; id2rel = {v: k for k, v in src.rel2id.items()}
            raw = {str(x["id"]): x for x in src.raw_test_data}
            pos = {str(data[i]["id"]): i for i in range(len(data)) if str(data[i]["id"]) in set(want)}
            n_layers = len(em.layers)
            for sid in want:
                if sid not in pos: continue
                item = data[pos[sid]]
                batch = {"question_embeddings": item["question_embeddings"].unsqueeze(0).to(self.device),
                         "start_nodes_mask": item["start_nodes_mask"].unsqueeze(0).to(self.device),
                         "target_nodes_mask": item["target_nodes_mask"].unsqueeze(0).to(self.device), "id": [sid]}
                did, ch, _ = self._channel_scores(graph, batch)
                gold_pos = (batch["target_nodes_mask"][0, did] > 0).nonzero(as_tuple=True)[0]
                pinned = set((golds or {}).get(sid, []))
                forced = [j for j in gold_pos.tolist() if id2node[int(did[j])] in pinned]
                best = forced if forced else sorted(gold_pos.tolist(), key=lambda j: int((ch["fused"] > ch["fused"][j]).sum()))[:max_golds]
                rec = {"id": sid, "question": raw.get(sid, {}).get("question", ""), "stratum": raw.get(sid, {}).get("stratum"),
                       "seeds": [id2node[x] for x in batch["start_nodes_mask"][0].nonzero(as_tuple=True)[0].tolist()], "targets": []}
                for j in best:
                    node = int(did[j]); gname = id2node[node]
                    # baseline: full gate -> the routes we follow through every condition
                    em.resp_gate = True; em._gate_layers = None; em._gate_nodes = None
                    paths_out, fm, raw_paths, dropped = self._interpret_target(graph, batch, node, em, eta, id2node, id2rel)
                    seeds = set(batch["start_nodes_mask"][0].nonzero(as_tuple=True)[0].tolist())
                    route_senders = sorted({int(h) for p_ in raw_paths for (h, t, r) in p_})
                    L = max((len(p_) for p_ in raw_paths), default=1)
                    mask = torch.zeros(graph.num_nodes, dtype=torch.bool); mask[route_senders] = True
                    conds = [("gate_on", dict(resp_gate=True, _gate_layers=None, _gate_nodes=None)),
                             ("gate_off", dict(resp_gate=False, _gate_layers=None, _gate_nodes=None)),
                             ("gate_layers_attributed", dict(resp_gate=True, _gate_layers=set(range(L)), _gate_nodes=None)),
                             ("gate_layers_later", dict(resp_gate=True, _gate_layers=set(range(L, n_layers)), _gate_nodes=None)),
                             ("gate_route_senders_only", dict(resp_gate=True, _gate_layers=None, _gate_nodes=mask)),
                             ("gate_all_but_route_senders", dict(resp_gate=True, _gate_layers=None, _gate_nodes=~mask))]
                    tgt = {"doc": gname, "attributed_layers": L, "n_route_senders": len(route_senders),
                           "frontier_mean_resp": fm, "routes": [{"hops": p["hops"], "weight_beam": p["weight"]} for p in paths_out],
                           "conditions": {}}
                    try:
                        tgt["gate_attribution"] = self._gate_attribution(graph, batch, node, em, id2node, raw_paths)
                    except Exception as _e:   # attribution is additive to the decomposition; never lose the rest
                        tgt["gate_attribution"] = {"error": str(_e)[:300]}
                        print(f"[gate-decomp] attribution failed for {sid} -> {gname}: {_e}", flush=True)
                    for cname, cfg in conds:
                        for k_, v_ in cfg.items(): setattr(em, k_, v_)
                        did_c, ch_c, _ = self._channel_scores(graph, batch)
                        ranks = {k: int((v > v[j]).sum().item()) + 1 for k, v in ch_c.items()}
                        eg, pr = self._edge_grads_for(graph, batch, node, em)
                        w_direct = self._direct_path_weights(graph, eg, raw_paths)
                        top_here, top_w = pr.get(node, ([], []))
                        top_here, top_w, _ = self._valid_paths(top_here, top_w, seeds)
                        tgt["conditions"][cname] = {"rank": ranks, "graph_score": float(ch_c["graph"][j]),
                                                    "graph_score_gap_to_top": float(ch_c["graph"].max() - ch_c["graph"][j]),
                                                    "route_weights_direct": w_direct,
                                                    "top_route_here": ([{"head": id2node[h], "rel": id2rel.get(r, str(r)), "tail": id2node[t]} for (h, t, r) in top_here[0]] if top_here else None),
                                                    "top_route_here_weight": (float(top_w[0]) if top_here else None)}
                    em.resp_gate = True; em._gate_layers = None; em._gate_nodes = None
                    ga = tgt.get("gate_attribution", {})
                    if "score_gate_on" in ga:
                        ga["delta_score_on_minus_off"] = float(tgt["conditions"]["gate_on"]["graph_score"] - tgt["conditions"]["gate_off"]["graph_score"])
                    rec["targets"].append(tgt)
                    c = tgt["conditions"]
                    print(f"[gate-decomp] {sid} -> {gname}: graph rank on {c['gate_on']['rank']['graph']} off {c['gate_off']['rank']['graph']} "
                          f"| route-1 weight on {c['gate_on']['route_weights_direct'][:1]} off {c['gate_off']['route_weights_direct'][:1]} "
                          f"L01 {c['gate_layers_attributed']['route_weights_direct'][:1]} later {c['gate_layers_later']['route_weights_direct'][:1]} "
                          f"own {c['gate_route_senders_only']['route_weights_direct'][:1]} others {c['gate_all_but_route_senders']['route_weights_direct'][:1]}", flush=True)
                results.append(rec)
        os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
        json.dump(results, open(out_path, "w"), indent=1)
        print(f"[gate-decomp] wrote {len(results)} queries -> {out_path}")
        return results

    def _min_hops(self, graph, seeds, targets, max_hops):
        """Fewest hops from any seed to each target node (undirected BFS, sparse matmul frontier),
        None when the target is not reached within max_hops."""
        if not seeds or not targets:
            return {}
        adj = getattr(self, "_bfs_adj", None)
        if adj is None or getattr(self, "_bfs_adj_key", None) != id(graph):
            ei = graph.edge_index
            r2 = torch.cat([ei[0], ei[1]]); c2 = torch.cat([ei[1], ei[0]])
            adj = torch.sparse_coo_tensor(torch.stack([r2, c2]), torch.ones(r2.numel(), device=ei.device),
                                          (graph.num_nodes, graph.num_nodes)).coalesce()
            self._bfs_adj, self._bfs_adj_key = adj, id(graph)
        n = graph.num_nodes
        visited = torch.zeros(n, device=adj.device); visited[seeds] = 1.0
        frontier = visited.clone()
        out = {int(t): (0 if visited[t] > 0 else None) for t in targets}
        for h in range(1, int(max_hops) + 1):
            if all(v is not None for v in out.values()):
                break
            frontier = ((torch.sparse.mm(adj, frontier.unsqueeze(1)).squeeze(1) > 0).float() * (1 - visited))
            if frontier.sum() == 0:
                break
            visited = visited + frontier
            for t in out:
                if out[t] is None and frontier[t] > 0:
                    out[t] = h
        return out

    def interpret(self, qids, out_path, probes_path=None, num_beam=10, path_topk=5,
                  max_golds=2, top_views=3, do_paths=True, golds=None, necessity=False, distractor=False, dump_k=0):
        """Path interpretations, NBFNet-style, for the graph channel of the fusion model.

        For each requested query: the rank of every gold under each channel (fused, graph
        alone, multi-view scorer, raw Qwen3 cosine), the top-k paths from the query's seed
        frames to its best-ranked gold (beam search over the gradient of the graph score
        w.r.t. each layer's edge weights, exactly the GFM-RAG / NBFNet recipe), and along
        every path the CCMP responsibility of each hop's sender node at the layer it was
        used, normalised by that layer's frontier mean (i.e. the gate the model applied).
        Also the scorer's views that matched the gold best, with their probe text.
        """
        import numpy as np
        self.model.eval()
        em = self.model.base.entity_model
        em.num_beam, em.path_topk = int(num_beam), int(path_topk)
        eta = float(getattr(em, "resp_eta", 0.5))
        probes = {}
        if probes_path and os.path.exists(probes_path):
            for line in open(probes_path):
                if line.strip():
                    r = json.loads(line); probes[str(r["id"])] = r.get("probes", [])
        want = [str(q) for q in qids]
        results = []
        for test_dataset in self.eval_graph_dataset_loader:
            src = test_dataset.data
            graph = src.graph.to(self.device)
            data = src.test_data
            id2node = src.id2node
            id2rel = {v: k for k, v in src.rel2id.items()}
            raw = {str(x["id"]): x for x in src.raw_test_data}
            pos = {}
            for i in range(len(data)):
                sid = str(data[i]["id"])
                if sid in want: pos[sid] = i
            missing = [q for q in want if q not in pos]
            if missing:
                print(f"[interpret] {len(missing)} requested ids not in {test_dataset.name}: {missing[:5]}")
            for sid in want:
                if sid not in pos: continue
                item = data[pos[sid]]
                batch = {"question_embeddings": item["question_embeddings"].unsqueeze(0).to(self.device),
                         "start_nodes_mask": item["start_nodes_mask"].unsqueeze(0).to(self.device),
                         "target_nodes_mask": item["target_nodes_mask"].unsqueeze(0).to(self.device),
                         "id": [sid]}
                did, ch, (tab, row) = self._channel_scores(graph, batch)
                n_doc = did.numel()
                gold_pos = (batch["target_nodes_mask"][0, did] > 0).nonzero(as_tuple=True)[0]
                doc_name = [id2node[int(did[j])] for j in gold_pos.tolist()]
                ranks = {}
                for k, v in ch.items():
                    ranks[k] = {doc_name[a]: int((v > v[j]).sum().item()) + 1 for a, j in enumerate(gold_pos.tolist())}
                # `golds` pins the documents to interpret (e.g. the one gold the picker chose); otherwise
                # the query's best-ranked golds under the fused score.
                pinned = set((golds or {}).get(sid, []))
                forced = [j for j in gold_pos.tolist() if id2node[int(did[j])] in pinned]
                best = forced if forced else sorted(gold_pos.tolist(), key=lambda j: ranks["fused"][id2node[int(did[j])]])[:max_golds]
                # scorer views for each gold
                if tab is not None:
                    Hq = np.asarray(tab["H"][row])[:, tab["col"]].astype(np.float32)   # [J, n_doc]
                    vmask = tab["mask"][row].numpy() > 0
                else:                   # operator-scorer checkpoint: no hypothetical-answer views
                    Hq, vmask = None, None
                seeds = batch["start_nodes_mask"][0].nonzero(as_tuple=True)[0].tolist()
                # Structural floor for the hop-count figure: the fewest hops from ANY seed node to each
                # gold, undirected BFS on the graph, capped at the reasoner's depth. The reasoner's top
                # path can only be this long or longer; the gap between the two is what the figure shows.
                min_hops = self._min_hops(graph, seeds, [int(did[j]) for j in gold_pos.tolist()], len(em.layers))
                rec = {"id": sid, "question": raw.get(sid, {}).get("question", ""),
                       "stratum": raw.get(sid, {}).get("stratum"), "golds": doc_name, "ranks": ranks,
                       "n_doc": int(n_doc), "seeds": [id2node[x] for x in seeds], "targets": []}
                if dump_k:
                    # per-channel top-K (doc, score) lists, for post-hoc fusion experiments
                    rec["channels"] = {k: [[id2node[int(did[jj])], float(v[jj])] for jj in torch.topk(v, min(int(dump_k), v.numel())).indices.tolist()] for k, v in ch.items()}
                    rec["gold_scores"] = {k: {doc_name[a]: float(v[j]) for a, j in enumerate(gold_pos.tolist())} for k, v in ch.items()}
                    _g = getattr(self.model, "_last_gamma", None); rec["gamma_q"] = float(_g[0]) if _g is not None else None
                for j in best:
                    gname = id2node[int(did[j])]
                    tv = [(int(a), float(Hq[a, j])) for a in np.argsort(-Hq[:, j]) if vmask[a]][:top_views] if Hq is not None else []
                    views = [{"view": a, "match": m, "text": (probes.get(sid, [None] * (a + 1))[a] if a < len(probes.get(sid, [])) else None)}
                             for a, m in tv]
                    if not do_paths:        # scan mode: ranks and views only, no gradient beam search
                        rec["targets"].append({"doc": gname, "rank": {k: ranks[k][gname] for k in ranks},
                                               "dense_cos": float(ch["dense"][j]), "views": views, "min_hops": min_hops.get(int(did[j])),
                                               "frontier_mean_resp": [], "paths": []})
                        continue
                    # ---- paths: gradient beam search on the GRAPH channel's score for this gold
                    paths_out, fm, paths, dropped = self._interpret_target(graph, batch, int(did[j]), em, eta, id2node, id2rel)
                    nec = None
                    if necessity and paths:
                        nec = {}
                        for kk in (1, 3):
                            nec[f"top{kk}"] = self._rank_without(graph, batch, [x for p_ in paths[:kk] for x in p_], int(did[j]))
                        nec["random"] = self._rank_without(graph, batch, None, int(did[j]), n_random=nec["top3"]["removed"])
                    rec["targets"].append({"doc": gname, "rank": {k: ranks[k][gname] for k in ranks},
                                           "dense_cos": float(ch["dense"][j]), "views": views, "min_hops": min_hops.get(int(did[j])),
                                           "frontier_mean_resp": fm, "paths": paths_out, "dropped_paths": dropped, "necessity": nec})
                    print(f"[interpret] {sid} -> {gname}: ranks {rec['targets'][-1]['rank']} | {len(paths_out)} paths ({dropped} artefacts dropped)")
                if do_paths and distractor:
                    order = torch.argsort(ch["graph"], descending=True).tolist(); gset = set(gold_pos.tolist())
                    jd = next((o for o in order if o not in gset), None)
                    if jd is not None:
                        dp, dfm, draw, ddrop = self._interpret_target(graph, batch, int(did[jd]), em, eta, id2node, id2rel)
                        rec["distractor"] = {"doc": id2node[int(did[jd])], "rank": {k: int((v > v[jd]).sum().item()) + 1 for k, v in ch.items()},
                                             "frontier_mean_resp": dfm, "paths": dp, "dropped_paths": ddrop}
                        if necessity and draw:
                            rec["distractor"]["necessity"] = {"top1": self._rank_without(graph, batch, [x for p_ in draw[:1] for x in p_], int(did[jd]))}
                if not do_paths and len(results) % 25 == 0:
                    print(f"[interpret] scanned {len(results) + 1}/{len(want)}", flush=True)
                results.append(rec)
        os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
        json.dump(results, open(out_path, "w"), indent=1)
        print(f"[interpret] wrote {len(results)} queries -> {out_path}")
        return results

    def _cqig_maybe_calibrate(self, graph, batch, task_dataset):
        """Called from train_step. Keeps the TRAIN graph's statistics current; mu drifts as
        the model trains, so it is recomputed every CQIG_RECAL steps."""
        g = self._cqig_gate()
        if g is None:
            return
        self._cqig_step = getattr(self, "_cqig_step", 0) + 1
        if not self._cqig_select_reference(batch, task_dataset):
            g.mode = "off"                          # no bank yet: run ungated, not wrongly
            return
        key = self._cqig_key(task_dataset)
        due = (not g.calibrated(key)) or (
            self._cqig_recal > 0 and self._cqig_step % self._cqig_recal == 0)
        if due:
            self._cqig_train_key = self._cqig_calibrate(graph, task_dataset, "train")
        else:
            self.model.cqig_use_graph(key)          # eval may have left another graph active

    def _cqig_for_inference(self, task_dataset):
        """Calibrate the graph about to be scored, from the SOURCE bank re-linked to it.

        This is the zero-shot path: the target corpus supplies its vocabulary and its graph,
        and nothing else. Its own queries are never inspected.
        """
        g = self._cqig_gate()
        if g is None:
            return
        if not g.ref_bank:
            # The bank is frozen a few hundred training steps in. An evaluation before that
            # must run UNGATED: the only calibrated graph is the train graph, and its mu has
            # the wrong number of rows for this one.
            g.mode = "off"
            return
        graph = task_dataset.graph.to(self.device)
        key = self._cqig_key(task_dataset)
        due = (not g.calibrated(key)) or (
            self._cqig_recal > 0
            and getattr(self, "_cqig_infer_at", -1) != getattr(self, "_cqig_step", 0))
        if due:
            self._cqig_calibrate(graph, task_dataset, "inference (source refs re-linked)")
            self._cqig_infer_at = getattr(self, "_cqig_step", 0)
        else:
            self.model.cqig_use_graph(key)

    def _create_task_dataset(self, dataset, is_train=True, **kw):
        """The one seam that train, evaluate AND predict all pass through.

        Evaluation and prediction run on a different graph from training, and mu has one
        row per node, so scoring the test graph against the train graph's statistics is not
        merely wrong but shape-invalid. Calibrating here means every consumer gets the
        right graph's statistics without each one needing its own hook.
        """
        ds = super()._create_task_dataset(dataset, is_train=is_train, **kw)
        g = self._cqig_gate()
        if g is not None:
            # The vocabulary rides along so CQIG can re-link its source problems by NAME.
            src = getattr(dataset, "data", dataset)
            ds.cqig_node2id = getattr(src, "node2id", None)
            ds.cqig_id2node = getattr(src, "id2node", None)
            if is_train:
                self._cqig_train_key = self._cqig_key(ds)
            else:
                self._cqig_for_inference(ds)
        return ds

    def train_step(self, batch, task_dataset):
        graph = task_dataset.graph.to(self.device)
        batch = query_utils.cuda(batch, device=self.device)
        self._cqig_maybe_calibrate(graph, batch, task_dataset)

        pred = self.parallel_model(graph, batch)          # FUSED [B, N]
        target = batch["target_nodes_mask"]

        total = torch.tensor(0.0, device=self.device, requires_grad=True)
        step_metrics = {}

        if self._objective == "hardneg":
            did = self.model._doc_ids
            # Whichever scorer the model's `semantic` setting selected: handcrafted
            # operator or the learned sorted-MLP. Both contrastive terms below mine
            # their negatives from it, so the graph is always trained to fix the misses
            # of the scorer actually in use.
            s_op = self.model._s_op                        # [B, n_doc] detached
            tgt_doc = target[:, did]
            raw = getattr(self.model, "_raw_doc", None)
            # THE AUXILIARY LOSS MUST TRAIN WHAT THE FUSION RANKS ON. The fused score
            # uses z(s_graph); z is scale-invariant, so a contrastive loss on the RAW
            # graph score can be driven down simply by scaling every score up, a
            # direction that leaves the fused ranking untouched. That is a free way for
            # hn_graph to fall while hn_fused and validation do not move. Mining still
            # uses the raw score, because top-k is scale-free either way.
            raw_l = getattr(self.model, "_raw_doc_z", None)
            if raw_l is None:
                raw_l = raw                     # older reasoners that cache only the raw score
            g_mine = raw.detach() if (raw is not None and self._hn_graph > 0) else None
            # ONE LINEUP FOR BOTH TERMS. Same golds, same hubs, same randoms, so the only
            # thing that differs between the fused and graph-alone losses is which score
            # is being trained -- which is the whole point of having both.
            lineups = self.build_lineups(tgt_doc, s_op, g_mine)
            # (1) fused-score contrastive: trains gate/router + operator scalars + graph jointly
            l_fused = self._contrastive_hardneg(
                pred[:, did], tgt_doc, s_op, g_mine=g_mine, miss_w=self._miss_w_fused,
                lineups=lineups
            )
            step_metrics["hn_fused"] = l_fused.item()
            total = total + l_fused
            # (2) graph-alone contrastive: trains the GNN DIRECTLY (survives a near-zero gate/alpha)
            # `_aux_ok` is False for the semantic-prior reasoner, which has no standalone
            # graph ranking: its `_raw_doc` is the belief CORRECTION, kept for diagnostics.
            # Training a contrastive on it would demand that the correction be a ranker in
            # its own right, which is not what it is.
            if self._aux_w > 0 and raw_l is not None and getattr(self.model, "_aux_ok", True):
                if self._resid_prior:
                    # REPLACES the standard graph-alone term rather than adding to it, so
                    # the arm differs from its control in exactly one thing: which
                    # documents count as negatives for the graph channel.
                    prior_doc = self._prior_doc(graph, batch, did)
                    l_graph, rstats = self._contrastive_resid(
                        raw_l, tgt_doc, prior_doc, lineups)
                    step_metrics.update(rstats)
                else:
                    l_graph = self._contrastive_hardneg(
                        raw_l, tgt_doc, s_op, g_mine=g_mine, miss_w=self._miss_w_aux,
                        lineups=lineups
                    )
                step_metrics["hn_graph"] = l_graph.item()
                total = total + self._aux_w * l_graph
            # (3) CCMP: intermediate responsibility. ADDED to the endpoint terms, not
            # substituted for them -- the graph must still be trained against the
            # scorer's own mistakes, which is its job. This term only says WHICH
            # intermediate nodes should be carrying the query while it does that.
            if self._ccmp and self._ccmp_w > 0:
                preds = getattr(self.model.base.entity_model, "_resp_pred", None)
                if preds:
                    tg = self._ccmp_targets(graph, batch, did, s_op, g_mine=g_mine)
                    l_crp, cstats = self._ccmp_loss(preds, tg)
                    if l_crp is not None:
                        step_metrics["ccmp"] = l_crp.item()
                        step_metrics.update(cstats)
                        _gs = getattr(self.model.base.entity_model,
                                      "_resp_gstat", None)
                        if _gs:
                            step_metrics["ccmp_gmax"] = max(x[1] for x in _gs)
                            step_metrics["ccmp_gp95"] = max(x[2] for x in _gs)
                        total = total + self._ccmp_w * l_crp
        else:
            for sft_loss in self.loss_functions:
                tids = graph.nodes_by_type[sft_loss.target_node_type]
                loss = sft_loss.loss_fn(pred[:, tids], target[:, tids])
                step_metrics[sft_loss.name] = loss.item()
                total = total + sft_loss.weight * loss

            # graph-alone auxiliary term (teach the GNN independently of the gate)
            if self._aux_w > 0 and getattr(self.model, "_raw_doc", None) is not None:
                did = self.model._doc_ids
                aux = self._aux_loss_fn(self.model._raw_doc, target[:, did])
                step_metrics["aux_graph"] = aux.item()
                total = total + self._aux_w * aux

        # POPULARITY ANCHOR, only under semantic='mlp'. The learned scorer's popularity
        # term is a network, and the fusion's ranking gradient reaches it exactly as the
        # standalone run's did -- so it needs the same anchor, or p_hat stops meaning "how
        # generally matchable is this paper" and starts meaning "whatever lowers this loss".
        # Zero for the operator channel, which has no predictor.
        if self._sem_pop_lambda > 0 and hasattr(self.model, "semantic_aux_loss"):
            l_pop = self.model.semantic_aux_loss()
            if torch.is_tensor(l_pop):
                step_metrics["pop_anchor"] = l_pop.item()
                total = total + self._sem_pop_lambda * l_pop

        step_metrics["loss"] = total
        return step_metrics


In [ ]:
%%writefile /content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training_fusion.yaml
# G-Reasoner SFT — CARGO fusion (v16sc graph + operator, learned jointly in ONE run).
# = base sft_training.yaml, but: (1) model is FusionGraphReasoner (wraps GraphReasoner, fuses the
# operator score into the document logits with a learned per-query gate); (2) losses are bce + pcr on
# the FUSED score (NO mse distillation — the operator is a fused signal, not a teacher); (3) the
# trainer adds a small graph-alone aux loss (weight AUX_W) so the from-scratch GNN learns alongside.
# Operator scores come from env OPERATOR_SCORES (train) / OPERATOR_SCORES_TEST (test).
hydra:
  run:
    dir: outputs/qa_finetune/${now:%Y-%m-%d}/${now:%H-%M-%S}
  searchpath:
    - pkg://gfmrag.workflow.config

defaults:
  - _self_
  - text_emb_model: qwen3
  - wandb: default

seed: 1024
timeout: 60
save_pretrained: no
load_model_from_pretrained: null

datasets:
  _target_: gfmrag.graph_index_datasets.GraphIndexDataset
  cfgs:
    root: ./data
    force_reload: False
    text_emb_model_cfgs: ${text_emb_model}
  train_names:
    - tomato_train_v16sc
  valid_names:
    - tomato_test_v16sc
  init_datasets: True
  feat_dim: 1024
  max_datasets_in_memory: 10
  data_loading_workers: 4

# Fusion model: wraps the v16sc GraphReasoner; entity_model is instantiated and passed through.
# The operator is recomputed live from cached ingredients (OPERATOR_COMPONENTS[_TEST]) so its
# scalars w=[w0,w1,w2] and beta are LEARNED jointly (warm-started at fitted values, base LR).
model:
  _target_: gfmrag.models.fusion_reasoner.FusionGraphReasoner
  gamma_init: 0.01         # gate near zero: on step 1 the model IS the operator (opt-in graph, protects same-domain)
  gate_hidden: 8
  op_lr_scale: 1.0         # operator scalars learn at the full base LR (4 params, warm-started, fast to fit)
  # Which scorer supplies the semantic channel. "operator" is the handcrafted sum+max
  # scorer with the leave-one-out anti-hub term; "mlp" is the learned sorted-input MLP
  # with a predicted popularity discount, warm-started from SEMANTIC_CKPT/SEMANTIC_POPNET.
  # Everything else about the fusion is identical, so the pair is a controlled comparison.
  semantic: operator
  semantic_train: true     # keep the learned scorer training under the fusion's ranking loss
  # Cross-Query Informativeness Gating. cqig=false attaches no hooks at all, so the
  # ungated arm is bit-identical to a build without it.
  cqig: false
  cqig_lam: 0.1            # initial max damping; NOT 0, which would be a dead gradient
  # null on both means "take it from the environment" (CQIG_NORM / CQIG_LAYERS), which is
  # how the notebook varies them per arm without pushing a string like "3-6" through
  # Hydra's override quoting. Set them here to pin an arm from the config instead.
  #   cqig_norm:   layer (default) | node | energy   -- see the header of cqig.py
  #   cqig_layers: last (default) | all | "6" | "3-6" | "3,5,6"   -- 1-INDEXED
  #   cqig_op:     gate (default) | centre | centre-fixed        -- CQIG_OP
  #     Which OPERATOR the same statistic drives. gate scales the state by g; centre
  #     subtracts (1-g)*mu, so an uninformative node sends its residual instead of a
  #     quieter copy of itself; centre-fixed subtracts a constant lam*mu, which is the
  #     query-independent rung of the ladder. lam=0 recovers the ungated model under all
  #     three, so the ablation is still one scalar.
  #   cqig_mu:     ref (default) | zero                          -- CQIG_MU
  #     zero forces mu=0, which degenerates I to ||h||^2/s: an activation-magnitude gate
  #     with no reference bank in it. THE ABLATION. If it reproduces the method's score,
  #     the bank was never load-bearing. Illegal with a centring op, which would be a
  #     literal no-op at mu=0.
  cqig_norm: null
  cqig_layers: null
  cqig_op: null
  cqig_mu: null
  use_ent_emb: early-late-fusion
  dtype: bfloat16
  entity_model:
    _target_: gfmrag.models.ultra.models.QueryNBFNet
    input_dim: 1024
    hidden_dims: [1024, 1024, 1024, 1024, 1024, 1024]
    message_func: distmult
    aggregate_func: sum
    short_cut: yes
    layer_norm: yes
    return_hidden: True

# Loss: gold supervision on the FUSED document score (no distillation term).
losses:
  - name: bce_loss
    loss:
      _target_: gfmrag.losses.BCELoss
      adversarial_temperature: 0.2
    weight: 0.3
    target_node_type: document
  - name: pcr_loss
    loss:
      _target_: gfmrag.losses.ListCELoss
    weight: 0.7
    target_node_type: document

optimizer:
  _target_: torch.optim.AdamW
  lr: 5.0e-4

trainer:
  _target_: gfmrag.trainers.fusion_trainer.FusionSFTTrainer   # adds graph-alone aux loss (AUX_W)
  args:
    _target_: gfmrag.trainers.TrainingArguments
    train_batch_size: 4
    num_epoch: 10
    logging_steps: 100
    max_steps_per_epoch: null
    resume_from_checkpoint: null
    do_train: true
    do_eval: true
    save_best_only: yes
    metric_for_best_model: document_mrr
    dtype: ${model.dtype}
    split_graph_inference: false
    split_graph_training: false
    split_graph_partition: contiguous
  metrics: [mrr, hits@1, hits@2, hits@3, hits@5, hits@10, hits@20, recall@2, recall@3, recall@5, recall@10, recall@20]
  target_types: [document]


In [ ]:
%%writefile /content/gfm-rag/gfmrag/workflow/ccmp_checkpoint_selection.py
"""Prefer SIR-4 merged-graph CCMP checkpoints and their matching test graph/scorer.

Run names and scorer locations follow prep/build_sir4_hyb_notebook.py and
prep/build_sir4_openie_notebook.py. Availability is checked on mounted Drive.
"""
import json
from pathlib import Path
import re
import shutil
import zipfile


def merged_candidates(dataset, drive):
    if dataset not in {"sir4_cs", "sir4_biology", "sir4_physics", "sir4_matsci"}:
        return []
    root = Path(drive) / "outputs/sir4_hyb"
    default_batch = 1 if dataset == "sir4_cs" else 2
    preferred = root / f"{dataset}_hyb_ccmp_e10_b{default_batch}" / "model_best.pth"
    pattern = re.compile(rf"{re.escape(dataset)}_hyb_ccmp_e(\d+)_b(\d+)")
    candidates = []
    for path in root.glob(f"{dataset}_hyb_ccmp_e*_b*/model_best.pth"):
        match = pattern.fullmatch(path.parent.name)
        if match and path.is_file() and path.stat().st_size:
            candidates.append((path, int(match[1]), int(match[2])))
    # Prefer the standard e10/default-batch recipe; then other available runs by
    # epoch and batch, deterministically. Never select by test performance or mtime.
    candidates.sort(key=lambda x: (x[0] != preferred, -x[1], x[2] != default_batch, x[2], str(x[0])))
    return [p for p, _, _ in candidates]


def ensure_graph(dataset, graph, family, drive, data_root):
    root = Path(data_root) / graph
    needed = [f"processed/stage1/{f}" for f in ("nodes.csv", "edges.csv", "relations.csv", "test.json")]
    if not all((root / p).is_file() for p in needed):
        bundle = Path(drive) / "sir4_hyb_bundle.zip"
        if family != "merged" or not bundle.is_file():
            raise FileNotFoundError(f"Selected {family} checkpoint requires {graph}. "
                                    f"Its graph files are missing; merged graphs are supplied by {bundle}. "
                                    "Checkpoint selection was not silently changed.")
        prefix = f"retriever/data/{graph}/"
        with zipfile.ZipFile(bundle) as z:
            if not all(prefix + p in z.namelist() for p in needed):
                raise ValueError(f"{bundle} does not contain the complete test graph {graph}")
            # Extract just this graph, without overwriting other datasets or caches.
            for info in z.infolist():
                if not info.filename.startswith(prefix) or info.is_dir():
                    continue
                relative = Path(info.filename[len(prefix):])
                if relative.is_absolute() or ".." in relative.parts:
                    raise ValueError(f"Invalid bundle member: {info.filename}")
                dest = root / relative
                dest.parent.mkdir(parents=True, exist_ok=True)
                with z.open(info) as source, dest.open("wb") as target:
                    shutil.copyfileobj(source, target)
    docs = root / "raw/documents.json"
    if not docs.is_file():
        docs.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(Path(data_root) / f"{dataset}_test/raw/documents.json", docs)


def select_checkpoint(dataset, spec, drive, data_root, prefer_merged=True):
    """Select once per dataset; every intervention/precision shares this selection.

    A missing merged checkpoint permits fallback. A present checkpoint with
    missing/inconsistent assets is an error, not a silent change of experiment.
    Strict tensor loading and exact graph/path validation happen in the workflow.
    """
    available = merged_candidates(dataset, drive)
    if prefer_merged and available:
        ckpt, graph, family, name, skey = available[0], f"{dataset}_test_hyb", "merged", "merged_ccmp", "field"
        reason = "Merged-graph CCMP checkpoint available; preferred over frame-only checkpoint."
        meta_path = ckpt.parent / "arm.json"
        metadata = json.loads(meta_path.read_text()) if meta_path.is_file() else {}
        expected = {"field": dataset.removeprefix("sir4_"), "graph": "hyb", "valid": graph,
                    "train": f"{dataset}_train_hyb", "ccmp": True, "semantic": "mlp"}
        for key, value in expected.items():
            if key in metadata and metadata[key] != value:
                raise ValueError(f"{meta_path}: {key}={metadata[key]!r}, expected {value!r}")
        scorer_dirs = [f"outputs/sir4_hyb/semantic_{dataset}", spec["sem"]["field"][0],
                       f"outputs/{dataset}_ablations_v1/{dataset}/semantic"]
        scorer = next((rel for rel in scorer_dirs
                       if all((Path(drive) / rel / f"{stem}_semantic_mlp_fixedloss_{dataset}.{ext}").is_file()
                              for stem, ext in (("params", "json"), ("popnet", "pt")))), None)
        if scorer is None:
            raise FileNotFoundError(f"Merged checkpoint found, but its field scorer files are missing: {scorer_dirs}")
        semantic_specs = {"field": (scorer, dataset)}
    else:
        arms = [a for a in spec["arms"] if a[0] == "frame_ccmp"]
        if len(arms) != 1:
            raise ValueError(f"{dataset}: expected one frame CCMP fallback specification")
        name, relative, _, skey, _ = arms[0]
        relative = relative if isinstance(relative, list) else [relative]
        ckpt = next((Path(drive) / r / "model_best.pth" for r in relative
                     if (Path(drive) / r / "model_best.pth").is_file()
                     and (Path(drive) / r / "model_best.pth").stat().st_size), None)
        if ckpt is None:
            raise FileNotFoundError(f"{dataset}: neither a merged nor fallback CCMP checkpoint is available")
        graph, family, metadata = spec["frame"], "frame", {}
        semantic_specs = spec["sem"]
        reason = ("FALLBACK: no merged-graph CCMP checkpoint found under outputs/sir4_hyb; using the previous frame checkpoint."
                  if prefer_merged else "Frame checkpoint explicitly requested (PREFER_MERGED=False).")
    ensure_graph(dataset, graph, family, drive, data_root)
    return json.loads(json.dumps({"dataset": dataset, "family": family, "arm": name, "checkpoint": str(ckpt),
            "graph": graph, "scorer_key": skey, "semantic_specs": semantic_specs,
            "reason": reason, "prefer_merged": prefer_merged,
            "available_merged_checkpoints": [str(p) for p in available], "arm_metadata": metadata}))


In [ ]:
%%writefile /content/gfm-rag/gfmrag/workflow/ccmp_mechanism_core.py
"""Matched, fixed-path CCMP interventions. Embedded in colab_table4_openie.ipynb.

The intervention is applied AFTER native gate normalization and BEFORE the dtype
cast / outgoing-message multiplication. Counterfactual gates are frozen from the
native-on pass. No counterfactual is renormalized or recomputed from its new states.
"""
import csv
import hashlib
import json
import math
from pathlib import Path

import torch

VERSION = "ccmp-mechanism-v1"
CONDITIONS = (
    "off", "native", "frozen", "path_only", "outside_full",
    "outside_suppress", "outside_amplify",
)


def patch_engine(path):
    """Install an opt-in hook in the single-process branch; ordinary runs unchanged."""
    path = Path(path)
    text = path.read_text()
    marker = "# CCMP_MECHANISM_HOOK_V1"
    if marker in text:
        return
    anchor = """                else:
                    _msg = layer_input
                if separate_grad:
                    edge_weight = edge_weight.clone().requires_grad_()
"""
    replacement = """                else:
                    _msg = layer_input
                # CCMP_MECHANISM_HOOK_V1: analysis-only, after normalization.
                _hook = getattr(self, "_ccmp_intervention", None)
                if _hook is not None:
                    _native_gate = (_gt if _heads is not None and self.resp_gate
                                    else torch.ones_like(layer_input[..., 0]).float())
                    _chosen_gate = _hook(_li, _native_gate, layer_input,
                                         self._reach_layers[_li])
                    _msg = layer_input * _chosen_gate.unsqueeze(-1).to(layer_input.dtype)
                if separate_grad:
                    edge_weight = edge_weight.clone().requires_grad_()
"""
    if text.count(anchor) != 1:
        raise RuntimeError("Engine changed: expected exactly one CCMP intervention anchor")
    text = text.replace(anchor, replacement)
    compile(text, str(path), "exec")
    path.write_text(text)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    tmp.replace(path)


def choose_gate(condition, baseline, protected):
    """Pure counterfactual policy. No normalization after intervention."""
    base = baseline.detach()
    one = torch.ones_like(base)
    if condition == "off":
        return one
    if condition == "frozen":
        return base
    if condition == "path_only":
        return torch.where(protected, base, one)
    if condition == "outside_full":
        return torch.where(protected, one, base)
    if condition == "outside_suppress":
        return torch.where(protected, one, base.clamp(max=1))
    if condition == "outside_amplify":
        return torch.where(protected, one, base.clamp(min=1))
    raise ValueError(condition)


def resolve_paths(case, src, graph, batch, layers, protection):
    """Exact strings and edge IDs; fail on missing/ambiguous edges or wrong targets."""
    node2id, rel2id = src.node2id, src.rel2id
    target = node2id[case["gold_id"]]
    if not bool(batch["target_nodes_mask"][0, target] > 0):
        raise ValueError(f"Pinned document is not a gold: {case['gold_id']}")
    paths = []
    protected = torch.zeros((layers, 1, graph.num_nodes), dtype=torch.bool,
                            device=graph.edge_index.device)
    for p in case["paths"]:
        hops, nodes = [], []
        for i, hop in enumerate(p["hops"]):
            if hop["layer"] != i or i >= layers:
                raise ValueError("Fixed paths must start at layer 0 and fit the model depth")
            h, t, r = node2id[hop["head"]], node2id[hop["tail"]], rel2id[hop["rel"]]
            if i == 0:
                if not bool(batch["start_nodes_mask"][0, h] > 0):
                    raise ValueError(f"Path does not start at a seed: {hop['head']}")
                nodes.append(h)
            elif nodes[-1] != h:
                raise ValueError("Disconnected fixed path")
            nodes.append(t)
            match = ((graph.edge_index[0] == h) & (graph.edge_index[1] == t)
                     & (graph.edge_type == r)).nonzero(as_tuple=True)[0]
            if match.numel() != 1:
                raise ValueError(f"Expected one exact edge, found {match.numel()}: {hop}")
            hops.append({**hop, "head_id": h, "tail_id": t, "rel_id": r,
                         "edge_id": int(match[0])})
            protected[i, 0, h] = True
        if not hops or nodes[-1] != target or len(nodes) != len(set(nodes)):
            raise ValueError("Path must be simple and end at the exact pinned gold")
        if protection == "nodes_all_layers":
            protected[:, 0, nodes] = True
        elif protection != "sender_layer":
            raise ValueError(protection)
        paths.append({"name": p["name"], "hops": hops})
    if not paths:
        raise ValueError("At least one fixed path is required")
    return target, paths, protected


def gate_stats(gate, reached):
    v = gate.detach().float()[reached.bool()]
    if not v.numel():
        return {"n": 0}
    return {"n": v.numel(), "mean": float(v.mean()), "min": float(v.min()),
            "max": float(v.max()), "suppressed": int((v < 1).sum()),
            "amplified": int((v > 1).sum()), "exactly_one": int((v == 1).sum()),
            "suppression_mass": float((1 - v).clamp(min=0).sum()),
            "amplification_mass": float((v - 1).clamp(min=0).sum())}


class GateRecorder:
    def __init__(self, condition, protected, baseline=None):
        self.condition, self.protected, self.baseline = condition, protected, baseline
        self.gates, self.stats = [], []

    def __call__(self, layer, native_gate, layer_input, reached):
        if reached is None or layer != len(self.gates):
            raise RuntimeError("Expected ordered single-query gates with structural reach")
        if self.condition == "native":
            gate = native_gate                 # retain derivatives through the gate
        else:
            base = native_gate if self.baseline is None else self.baseline[layer]
            gate = choose_gate(self.condition, base, self.protected[layer])
        if not torch.isfinite(gate).all():
            raise RuntimeError("Non-finite gate")
        applied = gate.to(layer_input.dtype).float()
        if self.condition.startswith("outside_"):
            if not torch.all(applied[self.protected[layer]] == 1):
                raise RuntimeError("Protected path gate changed")
        self.gates.append(gate.detach().clone())
        self.stats.append({"layer": layer, "message_dtype": str(layer_input.dtype),
                           "raw": gate_stats(gate, reached),
                           "applied": gate_stats(applied, reached),
                           "outside_applied": gate_stats(applied, reached.bool() & ~self.protected[layer]),
                           "protected_count": int(self.protected[layer].sum())})
        return gate


def score_metrics(scores, j, gold_mask):
    v = scores.detach().float()
    if not torch.isfinite(v).all():
        raise RuntimeError("Non-finite document scores")
    nongold = v[~gold_mask]
    return {"score": float(v[j]), "rank": int((v > v[j]).sum()) + 1,
            "ties": int((v == v[j]).sum()),
            "margin_to_best_nongold": float(v[j] - nongold.max()) if nongold.numel() else None}


def run_condition(model, graph, batch, target, paths, protected, condition,
                  dtype, baseline=None, competitors=None, diagnostics=None):
    """Use the true fusion forward, with the legacy visualize edge-gradient convention."""
    em = model.base.entity_model
    recorder = GateRecorder(condition, protected, baseline)
    original_bf = em.bellmanford
    old_gate, old_keep = em.resp_gate, getattr(em, "_keep_reach", False)
    if getattr(em, "_ccmp_intervention", None) is not None:
        raise RuntimeError("Another intervention is already installed")
    captured = {}

    def capture(*args, **kwargs):
        kwargs["separate_grad"] = True
        out = original_bf(*args, **kwargs)
        captured["edge_weights"] = out["edge_weights"]
        return out

    em.bellmanford = capture
    em._ccmp_intervention, em._keep_reach = recorder, True
    em.resp_gate = condition != "off"
    try:
        with torch.enable_grad(), torch.autocast(device_type=batch["question_embeddings"].device.type,
                                               dtype=dtype, enabled=dtype != torch.float32):
            pred = model(graph, batch)
            did = model._doc_ids
            js = (did == target).nonzero(as_tuple=True)[0]
            if js.numel() != 1:
                raise RuntimeError("Pinned gold is not exactly one document node")
            j = int(js[0])
            scores = {"graph": model._raw_doc[0].float(), "fused": pred[0, did].float()}
            edge_grads = torch.autograd.grad(scores["graph"][j], captured["edge_weights"])
        if diagnostics is not None:
            diagnostics.update(edge_grads=tuple(g.detach() for g in edge_grads),
                               doc_ids=did.detach(), target_position=j)
        if len(recorder.gates) != len(em.layers):
            raise RuntimeError("Hook did not record every layer; check engine patch")
        scores = {k: v.detach() for k, v in scores.items()}
        gm = batch["target_nodes_mask"][0, did] > 0
        result = {"condition": condition, "channels": {k: score_metrics(v, j, gm) for k, v in scores.items()},
                  "gate_stats": recorder.stats, "paths": []}
        for p in paths:
            hops = []
            for hop in p["hops"]:
                l, h, e = hop["layer"], hop["head_id"], hop["edge_id"]
                raw = recorder.gates[l][0, h]
                message_dtype = getattr(torch, recorder.stats[l]["message_dtype"].split(".")[-1])
                hops.append({**hop, "edge_gradient": float(edge_grads[l][e]),
                             "g_raw": float(raw), "g_applied": float(raw.to(message_dtype)),
                             "protected": bool(protected[l, 0, h])})
            result["paths"].append({"name": p["name"], "weight": sum(h["edge_gradient"] for h in hops) / len(hops),
                                    "hops": hops})
        if competitors is None:
            competitors = {}
            for k, v in scores.items():
                ix = (~gm).nonzero(as_tuple=True)[0]
                competitors[k] = ix[v[ix].topk(min(5, ix.numel())).indices].tolist()
        result["competitors"] = {
            k: [{"node_id": int(did[ix]), "score": float(v[ix]), "gold_margin": float(v[j] - v[ix])}
                for ix in competitors[k]] for k, v in scores.items()}
        detached = {k: v.detach().float().cpu().clone() for k, v in scores.items()}
        detached["scorer"] = model._s_op[0].detach().float().cpu().clone()
        return result, recorder.gates, detached, competitors
    finally:
        em.bellmanford = original_bf
        em.resp_gate, em._keep_reach = old_gate, old_keep
        del em._ccmp_intervention
        em._resp_pred, em._reach_layers = [], []


def score_deltas(a, b):
    return {k: float((a[k] - b[k]).abs().max()) for k in a}


def numerical_allowance(reference, dtype, repeat_error=None):
    """Absolute allowance for repeated GPU reductions at the requested dtype.

    The scorer is a separate float32 channel. Graph/fused scores inherit the GNN
    arithmetic dtype; bf16 sparse reductions need an allowance proportional to
    their scale even when the applied gates are bit-identical.
    """
    repeat_error = repeat_error or {}
    allowance = {}
    for k, value in reference.items():
        scale = max(1.0, float(value.detach().abs().max()))
        roundoff = 0.0 if dtype == torch.float32 or k == "scorer" else 2 * torch.finfo(dtype).eps * scale
        allowance[k] = max(5 * float(repeat_error.get(k, 0.0)), roundoff)
    return allowance


def compare_scores(a, b, atol, rtol, label, extra_atol=None):
    checks = {}
    extra_atol = extra_atol or {}
    for k in a:
        delta = float((a[k] - b[k]).abs().max())
        checks[k] = delta
        allowance = float(extra_atol.get(k, 0.0))
        if not torch.allclose(a[k], b[k], atol=atol + allowance, rtol=rtol):
            raise RuntimeError(f"{label}: {k} score mismatch, max absolute error {delta}; "
                               f"measured/dtype allowance {allowance}")
    return checks


def experiment(model, graph, batch, src, case, dtype, protection="nodes_all_layers",
               atol=1e-5, rtol=1e-4):
    em = model.base.entity_model
    if getattr(em, "resp_proj", None) is None or not em.resp_gate_norm:
        raise RuntimeError("Requires a checkpoint with mean-normalized CCMP heads")
    if getattr(em, "route_mode", "") or getattr(em, "attn_node", None) is not None:
        raise RuntimeError("Disable alternate routing for a CCMP-only experiment")
    if getattr(graph, "dist_context", None) is not None:
        raise RuntimeError("Run on one GPU without graph partitioning")
    model.eval()
    target, paths, protected = resolve_paths(case, src, graph, batch, len(em.layers), protection)
    results, tensors, gate_runs, gates, comps = {}, {}, {}, None, None
    # Native off remains the actual ungated code path (plus identity hook).
    for condition in CONDITIONS:
        rec, gs, sc, comp = run_condition(model, graph, batch, target, paths, protected,
                                         condition, dtype, gates, comps)
        results[condition], tensors[condition], gate_runs[condition] = rec, sc, gs
        if condition == "off":
            comps = comp  # same five off-run non-gold competitors in all conditions
        if condition == "native":
            gates = gs
        compare_scores({"scorer": tensors["off"]["scorer"]}, {"scorer": sc["scorer"]},
                       atol, rtol, "Semantic scorer must stay fixed")
        print(f"  {case['name']} / {condition}: graph rank {rec['channels']['graph']['rank']}, "
              f"w={[round(p['weight'], 6) for p in rec['paths']]}", flush=True)
    # The causal replay is defined by its applied gates. Check these exactly before
    # allowing for nondeterministic low-precision sparse reductions in its scores.
    gate_replay = {
        "raw_bit_identical": all(torch.equal(a, b) for a, b in zip(gate_runs["native"], gate_runs["frozen"])),
        "applied_bit_identical": all(torch.equal(a.to(dtype), b.to(dtype))
                                     for a, b in zip(gate_runs["native"], gate_runs["frozen"])),
        "target_ranks_identical": all(results["native"]["channels"][k]["rank"]
                                      == results["frozen"]["channels"][k]["rank"]
                                      for k in ("graph", "fused")),
    }
    if not all(gate_replay.values()):
        raise RuntimeError("Frozen replay did not apply the exact saved native gates")
    repeats, repeat_tensors, repeat_deltas = {}, {}, {}
    for condition in ("off", "native", "frozen"):
        rec, _, sc, _ = run_condition(model, graph, batch, target, paths, protected, condition,
                                      dtype, gates, comps)
        repeats[condition], repeat_tensors[condition] = rec, sc
        repeat_deltas[condition] = score_deltas(tensors[condition], sc)
    native_allowance = numerical_allowance(tensors["native"], dtype, repeat_deltas["native"])
    frozen_allowance = numerical_allowance(tensors["frozen"], dtype, repeat_deltas["frozen"])
    replay_allowance = {k: max(native_allowance[k], frozen_allowance[k]) for k in native_allowance}
    checks = {"gate_replay": gate_replay, "repeat_variation": repeat_deltas,
              "frozen_replay_allowance": replay_allowance,
              "frozen_replay": compare_scores(tensors["native"], tensors["frozen"], atol, rtol,
                                               "Frozen replay", replay_allowance)}
    # Verify that instrumentation / separate_grad did not change deployed forward scores.
    old_gate = em.resp_gate
    try:
        for condition in ("off", "native"):
            em.resp_gate = condition == "native"
            with torch.no_grad(), torch.autocast(device_type=batch["question_embeddings"].device.type,
                                                dtype=dtype, enabled=dtype != torch.float32):
                pred = model(graph, batch)
                plain = {"graph": model._raw_doc[0].float().cpu(),
                         "fused": pred[0, model._doc_ids].float().cpu(),
                         "scorer": model._s_op[0].float().cpu()}
            allowance = numerical_allowance(tensors[condition], dtype, repeat_deltas[condition])
            checks[condition + "_uninstrumented_allowance"] = allowance
            checks[condition + "_uninstrumented"] = compare_scores(
                tensors[condition], plain, atol, rtol, "Uninstrumented forward", allowance)
    finally:
        em.resp_gate = old_gate
    effects = []
    for i, p in enumerate(paths):
        w = {k: r["paths"][i]["weight"] for k, r in results.items()}
        if not all(math.isfinite(v) for v in w.values()):
            raise RuntimeError(f"Non-finite path attribution: {p['name']}")
        repeat_error = max(abs(repeats[k]["paths"][i]["weight"] - w[k]) for k in repeats)
        # Numerical screening only; this is NOT a confidence interval or hypothesis test.
        tol = max(atol, rtol * max(abs(x) for x in w.values()), 5 * repeat_error)
        suppression = w["outside_suppress"] - w["off"]
        n_suppressed = sum(s["outside_applied"].get("suppressed", 0)
                           for s in results["outside_suppress"]["gate_stats"])
        verdict = ("no_effective_suppression" if n_suppressed == 0 else
                   "increases" if suppression > tol else
                   "decreases" if suppression < -tol else "numerically_unresolved")
        effects.append({"path": p["name"], "weights": w, "numerical_tolerance": tol,
                        "repeat_attribution_error": repeat_error,
                        "native_minus_off": w["native"] - w["off"],
                        "frozen_minus_off": w["frozen"] - w["off"],
                        "gate_derivative_effect": w["native"] - w["frozen"],
                        "path_only_effect": w["path_only"] - w["off"],
                        "outside_full_effect": w["outside_full"] - w["off"],
                        "outside_suppression_effect": suppression,
                        "outside_amplification_effect": w["outside_amplify"] - w["off"],
                        "outside_suppression_verdict": verdict,
                        "path_outside_interaction": w["frozen"] - w["path_only"] - w["outside_full"] + w["off"],
                        "suppression_amplification_interaction": w["outside_full"] - w["outside_suppress"] - w["outside_amplify"] + w["off"]})
    for rec in results.values():
        for values in rec["competitors"].values():
            for v in values:
                v["document"] = src.id2node[v["node_id"]]
    historical = {}
    refs = case.get("historical_reference", {}) if case.get("historical_reference_applicable", True) else {}
    for condition, ref in refs.items():
        rec = results[condition]
        delta = {p["name"]: p["weight"] - ref["path_weights"][p["name"]] for p in rec["paths"]}
        historical[condition] = {
            "path_weight_deltas": delta,
            "weights_match_tolerance": all(math.isclose(p["weight"], ref["path_weights"][p["name"]],
                                                        abs_tol=atol, rel_tol=rtol) for p in rec["paths"]),
            "rank_deltas": {k: rec["channels"][k]["rank"] - ref["ranks"][k] for k in ("graph", "fused")}}
    # CPU gate tensors are returned separately for reproducibility; never used as old-run cache inputs.
    gate_archive = {"native_raw": torch.stack(gates).cpu(), "protected": protected.cpu(),
                    "message_dtypes": [s["message_dtype"] for s in results["native"]["gate_stats"]],
                    "paths": paths, "precision": str(dtype)}
    return {"case": case, "protection": protection, "conditions": results,
            "checks": checks, "repeats": repeats, "effects": effects,
            "historical_comparison": historical}, gate_archive


def render_report(payload, out_dir):
    """Only current, hash-validated measurements enter the table and TikZ data files."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    lines = ["# CCMP fixed-path intervention results", "",
             "All conditions use the same checkpoint, graph, query, gold and fixed paths. "
             "Positive Δw means increased gradient attribution, not necessarily improved ranking.", "",
             "Outside means outside the protected node set (all layers by default). Gates are frozen "
             "from native CCMP, then clamped without renormalization. These selected cases do not estimate "
             "a population effect. Frozen replay requires bit-identical applied gates. Score checks include "
             "measured repeat variation and a recorded bfloat16 scale allowance; these numerical tolerances "
             "are not statistical confidence intervals.", ""]
    selection = payload["manifest"].get("selection") or {}
    if selection:
        lines += [f"Graph: `{selection['graph']}`. Checkpoint family: **{selection['family']}**.", "",
                  f"Checkpoint: `{selection['checkpoint']}`.", "", selection["reason"], ""]
    rows, hops, effects = [], [], []
    for record in payload["results"]:
        case = record["case"]
        lines += [f"## {case['name']} ({payload['manifest']['precision']})", "",
                  f"Protection: `{record['protection']}`. Gold: `{case['gold_id']}`.", "",
                  "| Condition | Graph rank | Fused rank | Graph score | " + " | ".join(p["name"] + " w" for p in case["paths"]) + " |",
                  "|---|---:|---:|---:|" + "---:|" * len(case["paths"])]
        for condition, r in record["conditions"].items():
            ch = r["channels"]
            lines.append(f"| {condition} | {ch['graph']['rank']} | {ch['fused']['rank']} | {ch['graph']['score']:.6g} | "
                         + " | ".join(f"{p['weight']:.6g}" for p in r["paths"]) + " |")
            for p in r["paths"]:
                common = {"case": case["name"], "query_id": case["query_id"], "gold_id": case["gold_id"],
                          "graph": selection.get("graph", ""), "checkpoint_family": selection.get("family", ""),
                          "precision": payload["manifest"]["precision"],
                          "condition": condition, "path": p["name"], "weight": p["weight"]}
                rows.append({**common, **{f"{k}_{m}": v for k, ms in ch.items() for m, v in ms.items()}})
                hops.extend({**common, **h} for h in p["hops"])
        lines += ["", "| Path | Native Δw | Gate derivative Δw | Outside suppression Δw | Suppression result |",
                  "|---|---:|---:|---:|---|"]
        for e in record["effects"]:
            lines.append(f"| {e['path']} | {e['native_minus_off']:.6g} | {e['gate_derivative_effect']:.6g} | "
                         f"{e['outside_suppression_effect']:.6g} | {e['outside_suppression_verdict']} |")
            effects.append({"case": case["name"], **{k: v for k, v in e.items() if k != "weights"}})
        lines += ["", "An increase under outside_suppress supports suppression being sufficient to raise this path's "
                  "attribution under this intervention. It does not show that all other messages were irrelevant, "
                  "or that suppression fully explains native CCMP. Native minus frozen isolates differentiation "
                  "through gates at a matched forward pass. Inspect ranks and gold margins separately.", ""]
        if record.get("historical_comparison"):
            history = record["historical_comparison"]
            matched = all(v["weights_match_tolerance"] and all(d == 0 for d in v["rank_deltas"].values())
                          for v in history.values())
            lines += ["Historical figure check: " + ("off/on weights and graph/fused ranks reproduce at the configured tolerance."
                       if matched else "the old figure's weights or ranks do not reproduce exactly at this precision. "
                       "Use these new matched measurements; inspect historical_comparison in results.json before reusing old percentages."), ""]
        elif case.get("historical_reference_applicable") is False:
            lines += ["The old figure used a frame-only checkpoint/graph. Its weights are retained as provenance, "
                      "not used as a numerical reproduction target for this merged-graph experiment.", ""]
    (out / "report.md").write_text("\n".join(lines))
    for name, data in (("paths.csv", rows), ("hops.csv", hops), ("effects.csv", effects)):
        if data:
            with (out / name).open("w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=list(data[0]))
                writer.writeheader()
                writer.writerows(data)
    # Simple numeric TSV that pgfplotstable / TikZ can consume without escaped document strings.
    with (out / "tikz_values.tsv").open("w") as f:
        f.write("case\tcondition\tpath\thop\tw\tgraw\tgapplied\n")
        for ci, record in enumerate(payload["results"]):
            for vi, condition in enumerate(CONDITIONS):
                for pi, p in enumerate(record["conditions"][condition]["paths"]):
                    for hi, h in enumerate(p["hops"]):
                        f.write(f"{ci}\t{vi}\t{pi}\t{hi}\t{p['weight']:.9g}\t{h['g_raw']:.9g}\t{h['g_applied']:.9g}\n")
    atomic_json(out / "tikz_index.json", {"cases": [r["case"]["name"] for r in payload["results"]],
                                         "conditions": CONDITIONS,
                                         "paths": [[p["name"] for p in r["case"]["paths"]] for r in payload["results"]]})


In [ ]:
%%writefile /content/gfm-rag/gfmrag/workflow/ccmp_mechanism.py
"""Installed as gfmrag.workflow.ccmp_mechanism by the Colab notebook. No training."""
try:
    import torchvision.io as _tvio
    if not hasattr(_tvio, "VideoReader"):
        class _NoVideoReader:
            def __init__(self, *a, **k):
                raise RuntimeError("torchvision video API removed")
        _tvio.VideoReader = _NoVideoReader
except Exception:
    pass

import json
import os
import random
from pathlib import Path

import hydra
import numpy as np
import torch
from hydra.core.hydra_config import HydraConfig
from hydra.utils import instantiate
from omegaconf import DictConfig, OmegaConf

from gfmrag import utils
from gfmrag.graph_index_datasets import GraphDatasetLoader
from gfmrag.trainers.sft_trainer import SFTLoss
from gfmrag.workflow.ccmp_mechanism_core import (
    VERSION, atomic_json, digest, experiment, render_report, sha256_file,
)


def fingerprint_inputs(cfg, src, cases):
    root = Path(__file__).resolve().parents[1]
    source = {str(p.relative_to(root)): sha256_file(p)
              for p in sorted(root.rglob("*.py"))}
    files = list(src.processed_graph) + [str(Path(src.processed_dir) / "test.pt")]
    mode = getattr(cfg.mechanism, "mode", "interventions")
    if mode == "paths":
        files += [str(Path(src.raw_dir) / "test.json"),
                  str(Path(cfg.datasets.cfgs.root) / cfg.datasets.valid_names[0] / "raw/documents.json")]
    selection = None
    if getattr(cfg.mechanism, "selection", None):
        selection = json.loads(Path(cfg.mechanism.selection).read_text())
        if (selection["checkpoint"] != str(cfg.mechanism.ckpt)
                or selection["graph"] != str(cfg.datasets.valid_names[0])):
            raise ValueError("Selected checkpoint/graph differs from workflow configuration")
        files.append(str(cfg.mechanism.selection))
    for key in ("OPERATOR_COMPONENTS_TEST", "SEMANTIC_COMPONENTS_TEST", "SEMANTIC_CKPT", "SEMANTIC_POPNET"):
        p = os.environ.get(key)
        if p:
            files.append(p)
            if key == "SEMANTIC_COMPONENTS_TEST":
                with np.load(p, allow_pickle=True) as z:
                    files.append(str(z["h_path"]))
    inputs = {p: sha256_file(p) for p in sorted(set(files))}
    manifest = {
        "version": VERSION, "cases": cases, "checkpoint": str(cfg.mechanism.ckpt),
        "analysis_mode": mode,
        "checkpoint_sha256": sha256_file(cfg.mechanism.ckpt),
        "selection": selection,
        "engine_sha256": digest(source), "source_files": source, "inputs": inputs,
        "config": OmegaConf.to_container(cfg, resolve=True), "precision": str(cfg.model.dtype),
        "protection": str(cfg.mechanism.protection), "seed": int(cfg.seed),
        "torch": torch.__version__, "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu",
        "environment": {k: v for k, v in sorted(os.environ.items())
                        if k.startswith(("CCMP", "ROUTE", "FUSION_", "SEM_", "SEED_", "CQIG"))},
        "attribution": ("finite-beam path discovery" if mode == "paths" else "fixed-path measurement")
                       + "; mean edge gradients from graph gold score; legacy linked layer clones, same as visualize; not a product of gates",
        "gate_policy": ("native differentiable CCMP gates, normalized by the model and cast to message dtype"
                        if mode == "paths" else
                        "freeze native-on gates; intervene after normalization; cast to message dtype; no renormalization"),
    }
    # RESUME only changes execution, not the experiment identity.
    manifest["config"]["mechanism"].pop("resume", None)
    for case in cases:
        source = case.get("selection_source")
        if source is not None:
            if (source["checkpoint"] != str(cfg.mechanism.ckpt)
                    or source["graph"] != str(cfg.datasets.valid_names[0])
                    or (source.get("checkpoint_sha256") is not None
                        and source["checkpoint_sha256"] != manifest["checkpoint_sha256"])):
                raise ValueError("Discovered paths were selected using a different checkpoint or graph")
    return manifest


@hydra.main(config_path="config/gfm_reasoner", config_name="sft_training_fusion", version_base=None)
def main(cfg: DictConfig):
    mode = getattr(cfg.mechanism, "mode", "interventions")
    if mode not in {"interventions", "paths"}:
        raise ValueError(f"Unknown analysis mode: {mode}")
    if mode == "paths":
        from gfmrag.workflow.scientific_paths import interpret_scientific_paths, render_scientific_table
    if int(os.environ.get("WORLD_SIZE", 1)) != 1:
        raise RuntimeError("Use a single GPU process")
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    if str(cfg.model.dtype) == "bfloat16" and not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):
        raise RuntimeError("bfloat16 requested: use an Ampere-or-newer GPU or explicitly choose float32")
    if cfg.trainer.args.resume_from_checkpoint is not None:
        raise RuntimeError("Training resume must be disabled for this experiment")
    feat_dim = set(utils.init_multi_dataset(cfg, 1, 0))
    if len(feat_dim) != 1 or len(cfg.datasets.valid_names) != 1:
        raise RuntimeError("Expected one test graph with one feature width")
    model = instantiate(cfg.model, feat_dim=feat_dim.pop())
    state = torch.load(cfg.mechanism.ckpt, map_location="cpu", weights_only=False)["model"]
    # No dropping mismatched tensors, no newly initialized missing CCMP/scorer weights.
    model.load_state_dict(state, strict=True)
    if not any("resp_proj" in k for k in state):
        raise RuntimeError("Checkpoint has no CCMP responsibility head")
    del state
    loader = GraphDatasetLoader(cfg.datasets, cfg.datasets.valid_names, shuffle=False,
                               max_datasets_in_memory=1, data_loading_workers=0)
    losses = [SFTLoss(name=lc.name, loss_fn=instantiate(lc.loss), weight=lc.weight,
                      target_node_type=lc.target_node_type,
                      is_distillation_loss=lc.get("is_distillation_loss", False)) for lc in cfg.losses]
    trainer = instantiate(cfg.trainer, output_dir=HydraConfig.get().runtime.output_dir,
                          model=model, optimizer=instantiate(cfg.optimizer, model.parameters()),
                          loss_functions=losses, train_graph_dataset_loader=loader,
                          eval_graph_dataset_loader=loader)
    if trainer.dtype != getattr(torch, str(cfg.model.dtype)):
        raise RuntimeError("Actual precision differs from requested precision")
    cases = json.loads(Path(cfg.mechanism.cases).read_text())
    if not cases or len({c["name"] for c in cases}) != len(cases):
        raise RuntimeError("Cases must be nonempty with unique names")
    completed = []
    try:
        for dataset in loader:
            src = dataset.data
            manifest = fingerprint_inputs(cfg, src, cases)
            run_id = digest(manifest)
            out = Path(cfg.mechanism.out) / run_id[:20]
            out.mkdir(parents=True, exist_ok=True)
            atomic_json(out / "manifest.json", manifest)
            graph = src.graph.to(trainer.device)
            if mode == "paths":
                documents = json.loads((Path(cfg.datasets.cfgs.root) / dataset.name / "raw/documents.json").read_text())
            pos = {str(src.test_data[i]["id"]): i for i in range(len(src.test_data))}
            for case in cases:
                if case["query_id"] not in pos:
                    raise RuntimeError(f"Pinned query missing: {case['query_id']}")
                token = digest(case)[:20]
                result_file, gate_file = out / f"case_{token}.json", out / f"gates_{token}.pt"
                cached = json.loads(result_file.read_text()) if result_file.exists() else None
                if (bool(cfg.mechanism.resume) and cached and cached.get("run_id") == run_id
                        and gate_file.exists() and cached.get("gate_sha256") == sha256_file(gate_file)):
                    print(f"[verified cache] {case['name']} {run_id[:20]}", flush=True)
                    completed.append(cached["result"])
                    continue
                item = src.test_data[pos[case["query_id"]]]
                batch = {k: item[k].unsqueeze(0).to(trainer.device)
                         for k in ("question_embeddings", "start_nodes_mask", "target_nodes_mask")}
                batch["id"] = [case["query_id"]]
                if mode == "paths":
                    result, gates = interpret_scientific_paths(
                        trainer.model, graph, batch, src, case, trainer.dtype, documents,
                        top_k=int(cfg.mechanism.top_k), beam_size=int(cfg.mechanism.beam_size))
                else:
                    result, gates = experiment(trainer.model, graph, batch, src, case, trainer.dtype,
                                               protection=str(cfg.mechanism.protection),
                                               atol=float(cfg.mechanism.atol), rtol=float(cfg.mechanism.rtol))
                tmp = gate_file.with_suffix(".tmp")
                torch.save(gates, tmp)
                tmp.replace(gate_file)
                atomic_json(result_file, {"run_id": run_id, "gate_sha256": sha256_file(gate_file), "result": result})
                completed.append(result)
            payload = {"manifest": manifest, "run_id": run_id, "status": "complete", "results": completed}
            atomic_json(out / "results.json", payload)
            if mode == "paths":
                render_scientific_table(payload, out)
            else:
                render_report(payload, out)
            atomic_json(Path(cfg.mechanism.out) / "latest.json",
                        {"version": VERSION, "run_id": run_id, "status": "complete", "results": str(out / "results.json")})
            print(f"[complete] {out / ('table4.md' if mode == 'paths' else 'report.md')}", flush=True)
        if len(completed) != len(cases):
            raise RuntimeError("Not all requested cases completed")
    finally:
        loader.shutdown()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/gfm-rag/gfmrag/workflow/scientific_paths.py
"""Table-4-style scientific paths from the CURRENT model, using NBFNet beam search.

The target is a pinned benchmark-relevant PAPER. Rankings are reported even when
it is retrieved poorly. Paths are post-hoc gradient attributions, not generated
answers, proofs, probabilities, or experiments validating scientific transfer.
"""
import csv
import json
import math
from pathlib import Path

import torch

if __package__ == "gfmrag.workflow":
    from .ccmp_mechanism_core import (run_condition, resolve_paths, score_metrics, atomic_json,
                                      compare_scores, numerical_allowance)
else:
    from ccmp_mechanism import (run_condition, resolve_paths, score_metrics, atomic_json,
                                compare_scores, numerical_allowance)


def document_title(name, documents):
    value = documents.get(name, name)
    if isinstance(value, dict):
        return str(value.get("title") or value.get("text") or name).split("\n")[0]
    return str(value).split("\n")[0].split(". ")[0].rstrip(".")


def graph_domains(node, graph, src):
    if "in_field" not in src.rel2id:
        return []
    mask = (graph.edge_index[0] == node) & (graph.edge_type == src.rel2id["in_field"])
    return sorted({src.id2node[int(i)].removeprefix("[domain] ") for i in graph.edge_index[1, mask].tolist()})


def decode_candidates(raw_paths, raw_weights, edge_grads, graph, batch, src,
                      gold_id, gates, gate_stats, documents, top_k):
    """Keep genuine simple paths; never silently replace a weak path with a curated one."""
    id2rel = {v: k for k, v in src.rel2id.items()}
    valid, rejected, seen = [], [], set()
    for raw, beam_weight in zip(raw_paths, raw_weights):
        raw = tuple(tuple(int(v) for v in hop) for hop in raw)
        if raw in seen:
            continue
        seen.add(raw)
        if not math.isfinite(float(beam_weight)):
            rejected.append("nonfinite beam score")
            continue
        hops = [{"layer": l, "head": src.id2node[h], "rel": id2rel[r], "tail": src.id2node[t]}
                for l, (h, t, r) in enumerate(raw)]
        try:
            _, resolved, _ = resolve_paths({"gold_id": gold_id, "paths": [{"name": "candidate", "hops": hops}]},
                                           src, graph, batch, len(edge_grads), "sender_layer")
        except ValueError as e:
            rejected.append(str(e))
            continue
        detailed = resolved[0]["hops"]
        for hop in detailed:
            l, h = hop["layer"], hop["head_id"]
            gate = gates[l][0, h]
            dtype = getattr(torch, gate_stats[l]["message_dtype"].split(".")[-1])
            hop.update(edge_gradient=float(edge_grads[l][hop["edge_id"]]),
                       g_raw=float(gate), g_applied=float(gate.to(dtype)))
        weight = sum(h["edge_gradient"] for h in detailed) / len(detailed)
        if not math.isfinite(weight):
            raise ValueError("Nonfinite path attribution")
        nodes = [detailed[0]["head"]] + [h["tail"] for h in detailed]
        labels = [h["rel"].removeprefix("inverse_") for h in detailed]
        flags = []
        if len(detailed) == 1:
            flags.append("single hop")
        if any(n.startswith("[domain]") for n in nodes):
            flags.append("contains domain hub")
        if any(r in {"equivalent", "same_as"} for r in labels):
            flags.append("contains alias link")
        mechanism = {"achieves", "overcomes", "works_via", "limited_by", "concerns", "explains",
                     "paper_achieves", "paper_overcomes", "paper_works_via", "paper_limited_by",
                     "paper_concerns", "paper_explains"}
        if not mechanism.intersection(labels):
            flags.append("no mechanism/function/limitation relation")
        valid.append({"weight": weight, "beam_weight": float(beam_weight), "hops": detailed,
                      "flags": flags, "papers_on_path": [n for n in dict.fromkeys(nodes) if n in documents]})
    valid.sort(key=lambda p: (-p["weight"], tuple((h["head"], h["rel"], h["tail"]) for h in p["hops"])))
    for rank, p in enumerate(valid, 1):
        p.update(name=f"P{rank}", rank_in_candidates=rank)
    return {"top_paths": valid[:top_k],
            "top_multihop_paths": [p for p in valid if len(p["hops"]) >= 2][:top_k],
            "valid_candidate_count": len(valid), "rejected_count": len(rejected),
            "rejection_reasons": sorted(set(rejected))}


def interpret_scientific_paths(model, graph, batch, src, case, dtype, documents,
                               top_k=5, beam_size=10):
    """Find the highest-attribution paths returned by the engine's finite beam.

    No historical paths/weights enter selection. Beam search ranks cumulative
    gradient scores, not hop count; candidate weights are divided by path length.
    """
    if top_k < 1 or beam_size < top_k:
        raise ValueError("Require beam_size >= top_k >= 1")
    model.eval()
    em = model.base.entity_model
    if (getattr(em, "resp_proj", None) is None or not em.resp_gate_norm or getattr(em, "route_mode", "")
            or getattr(em, "attn_node", None) is not None or getattr(graph, "dist_context", None) is not None):
        raise ValueError("This notebook requires single-process CCMP, without alternate routing")
    target = src.node2id[case["gold_id"]]
    if not bool(batch["target_nodes_mask"][0, target] > 0):
        raise ValueError("Target must be a benchmark gold paper")
    protected = torch.zeros((len(em.layers), 1, graph.num_nodes), device=graph.edge_index.device, dtype=torch.bool)
    diagnostics = {}
    # Same graph-score forward and gradients as the matched CCMP workflow, but no
    # preselected paths. This is the actual native-CCMP forward pass.
    record, gates, scores, _ = run_condition(model, graph, batch, target, [], protected,
                                            "native", dtype, diagnostics=diagnostics)
    old_gate = em.resp_gate
    try:
        em.resp_gate = True
        with torch.no_grad(), torch.autocast(device_type=graph.edge_index.device.type,
                                            dtype=dtype, enabled=dtype != torch.float32):
            plain = model(graph, batch)
            plain_scores = {"graph": model._raw_doc[0].detach().float().cpu(),
                            "fused": plain[0, model._doc_ids].detach().float().cpu(),
                            "scorer": model._s_op[0].detach().float().cpu()}
        forward_allowance = numerical_allowance(scores, dtype)
        forward_check = compare_scores(scores, plain_scores, 1e-5, 1e-4,
                                       "Path attribution vs uninstrumented inference", forward_allowance)
    finally:
        em.resp_gate = old_gate
    seed_ids = batch["start_nodes_mask"][0].nonzero(as_tuple=True)[0]
    with torch.no_grad():
        distances, back_edges = em.beam_search_distance(
            graph, diagnostics["edge_grads"], seed_ids, target, num_beam=beam_size)
        # Inspect every target candidate across depths before removing cyclic /
        # placeholder paths. Asking for only k initially can leave too few valid paths.
        raw_paths, raw_weights = em.topk_average_length(
            distances, back_edges, target, k=beam_size * len(em.layers))
    found = decode_candidates(raw_paths, raw_weights, diagnostics["edge_grads"], graph, batch, src,
                              case["gold_id"], gates, record["gate_stats"], documents, top_k)
    did, j = diagnostics["doc_ids"].cpu(), diagnostics["target_position"]
    gm = batch["target_nodes_mask"][0, diagnostics["doc_ids"]].cpu() > 0
    ranks = dict(record["channels"])
    ranks["scorer"] = score_metrics(scores["scorer"], j, gm)
    tag_row = getattr(model, "_sem_row", {}).get(str(batch["id"][0]))
    if tag_row is not None:
        tag, row = tag_row
        dense = model._sem_tab[tag]["dense"][row].detach().float().cpu()
        if dense.numel() != did.numel():
            raise ValueError("Dense reference does not match the model's document order")
        ranks["dense"] = score_metrics(dense, j, gm)
    else:
        ranks["dense"] = None
    raw_query = next((q for q in src.raw_test_data if str(q["id"]) == case["query_id"]), None)
    if raw_query is None:
        raise ValueError("Question text / stratum missing")
    top_documents = {k: [{"id": src.id2node[int(did[i])], "title": document_title(src.id2node[int(did[i])], documents),
                          "score": float(v[i])} for i in v.topk(min(top_k, v.numel())).indices.tolist()]
                     for k, v in scores.items() if k in {"graph", "fused"}}
    result = {"case": case, "question": raw_query["question"], "query_field": case["dataset"].removeprefix("sir4_"),
              "benchmark_stratum": raw_query.get("stratum"), "target_id": case["gold_id"],
              "target_title": document_title(case["gold_id"], documents),
              "target_domains": graph_domains(target, graph, src),
              "gold_papers": [{"id": d, "title": document_title(d, documents)} for d in raw_query["supporting_documents"]],
              "channels": ranks, "top_documents": top_documents, "gate_stats": record["gate_stats"], **found,
              "forward_score_check": forward_check, "forward_score_allowance": forward_allowance,
              "beam_size": beam_size, "requested_top_k": top_k,
              "search_note": "Approximate top simple paths among finite-beam candidates; no paths are invented if none qualify.",
              "scope": "Attribution to the graph score of a preselected relevant paper; not an answer-generation trace."}
    paper_ids = {d for p in found["top_paths"] + found["top_multihop_paths"] for d in p["papers_on_path"]}
    result["paper_titles"] = {d: document_title(d, documents) for d in sorted(paper_ids)}
    print(f"[paths] {case['name']}: {found['valid_candidate_count']} valid candidates; "
          f"graph rank {ranks['graph']['rank']}, fused rank {ranks['fused']['rank']}", flush=True)
    for p in result["top_paths"]:
        print(f"  {p['name']} w={p['weight']:.6g}: " + " -> ".join(
            f"({h['head']}, {h['rel']}, {h['tail']})" for h in p["hops"]), flush=True)
    return result, {"native_raw": torch.stack(gates).cpu(),
                    "message_dtypes": [s["message_dtype"] for s in record["gate_stats"]]}


def tex_escape(text):
    replacements = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
                    "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
    return "".join(replacements.get(c, c) for c in str(text))


def relation_text(rel, tex=False):
    inverse = rel.startswith("inverse_")
    value = rel.removeprefix("inverse_").replace("_", " ")
    return (tex_escape(value) + (r"$^{-1}$" if inverse else "")) if tex else value + ("⁻¹" if inverse else "")


def path_text(path, titles, tex=False):
    def name(n):
        s = titles.get(n, n)
        return tex_escape(s) if tex else s.replace("|", "\\|")
    triples = [f"({name(h['head'])}, {relation_text(h['rel'], tex)}, {name(h['tail'])})" for h in path["hops"]]
    return (r" $\rightarrow$ " if tex else " → ").join(triples)


def render_scientific_table(payload, out_dir):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    selection = payload["manifest"].get("selection") or {}
    md = ["# Scientific path interpretations", "",
          f"Graph: `{selection.get('graph', '')}`. Checkpoint: `{selection.get('checkpoint', '')}`.", "",
          "Paths are discovered from the current checkpoint. Weights are gradient attributions, not probabilities. "
          "The target is a preselected benchmark-relevant paper; its actual retrieval rank is reported. "
          "The cross stratum is a dataset annotation, and target domains are graph labels.", ""]
    tex = [r"% Include with \usepackage{booktabs,tabularx}. Inverse relations are traversed backwards.",
           r"\begin{table*}[t]", r"\centering\small", r"\renewcommand{\arraystretch}{1.15}",
           r"\begin{tabularx}{\textwidth}{@{}p{0.13\textwidth}X@{}}", r"\toprule"]
    csv_rows, intervention_cases = [], []
    for record in payload["results"]:
        title_map = record["paper_titles"]
        name = record["case"]["name"]
        info = [("Question", record["question"]), ("Query field", record["query_field"]),
                ("Relevant paper", record["target_title"]), ("Paper domain", ", ".join(record["target_domains"]) or "unknown"),
                ("Benchmark stratum", record["benchmark_stratum"] or "unknown")]
        md += [f"## {name}", ""] + [f"**{label}:** {text}\n" for label, text in info]
        ranks = " | ".join(f"{k}: {v['rank']}" for k, v in record["channels"].items() if v is not None)
        md += [f"**Target ranks (1 is best):** {ranks}", "",
               "**Top fused retrieved papers:** " + "; ".join(f"{i}. {d['title']}" for i, d in enumerate(record["top_documents"]["fused"], 1)), "",
               "### Top paths, including single-hop paths", "",
               "| Path | Weight | Hops | Applied gates, in hop order | Graph route | Notes |", "|---|---:|---:|---|---|---|"]
        for p in record["top_paths"]:
            gate_text = ", ".join(f"{h['g_applied']:.6g}" for h in p["hops"])
            md.append(f"| {p['name']} | {p['weight']:.6g} | {len(p['hops'])} | {gate_text} | {path_text(p, title_map)} | {', '.join(p['flags'])} |")
        if not record["top_paths"]:
            md += ["No valid simple path was returned by the configured beam search."]
        md += ["", "### Highest-ranked multi-hop subset", "",
               "This subset requires at least two hops; P-numbers retain their rank among all valid beam candidates.", ""]
        for p in record["top_multihop_paths"]:
            md += [f"**{p['name']} — w={p['weight']:.6g}:** {path_text(p, title_map)}", "",
                   "Applied sender gates: " + ", ".join(f"hop {h['layer']+1}: {h['g_applied']:.6g}" for h in p["hops"]) + ".", ""]
        if not record["top_multihop_paths"]:
            md += ["No multi-hop path was returned. This example does not currently provide a multi-hop path illustration.", ""]
        md += ["**Papers appearing on displayed paths:** " + "; ".join(title_map.values()), "",
               "**Interpretation:** inspect whether the route connects a scientific problem/function/limitation to a relevant "
               "method in another domain. A domain hub or alias alone is weak evidence. These paths do not show that the "
               "retrieved method has experimentally solved the query's problem.", ""]
        for label, text in info:
            tex.append(tex_escape(label) + " & " + tex_escape(text) + r" \\")
        tex.append("Ranks & " + tex_escape(ranks) + r" \\")
        for index, p in enumerate(record["top_paths"][:2]):
            tex.append(("Top paths" if index == 0 else "") + f" & {p['weight']:.4f}: " + path_text(p, title_map, tex=True) + r" \\")
        tex.append(r"\midrule")
        for p in record["top_paths"] + record["top_multihop_paths"]:
            for h in p["hops"]:
                row = {"case": name, "path": p["name"], "weight": p["weight"], "graph": selection.get("graph", ""), **h}
                if row not in csv_rows:
                    csv_rows.append(row)
        if record["top_paths"]:
            c = {k: v for k, v in record["case"].items() if not k.startswith("historical_") and k not in {"paths", "source"}}
            c["paths"] = [{"name": p["name"], "hops": [{k: h[k] for k in ("layer", "head", "rel", "tail")} for h in p["hops"]]}
                          for p in record["top_paths"][:2]]
            c["selection_source"] = {"run_id": payload["run_id"], "graph": selection.get("graph"),
                                     "checkpoint": selection.get("checkpoint"),
                                     "checkpoint_sha256": payload["manifest"].get("checkpoint_sha256"),
                                     "precision": payload["manifest"].get("precision"),
                                     "rule": "top two native simple paths; no multi-hop filtering"}
            c["historical_reference_applicable"] = False
            intervention_cases.append(c)
    tex += [r"\bottomrule\end{tabularx}",
            r"\caption{Gradient-attributed scientific retrieval paths from the current checkpoint. Targets are selected benchmark-relevant papers; ranks report actual retrieval. "
            r"Relation $r^{-1}$ traverses an original edge in reverse. Weights are not probabilities. Beam search is approximate.}",
            r"\label{tab:scientific-paths}", r"\end{table*}"]
    (out / "table4.md").write_text("\n".join(md) + "\n")
    (out / "table4.tex").write_text("\n".join(tex) + "\n")
    if csv_rows:
        with (out / "path_hops.csv").open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(csv_rows[0]))
            writer.writeheader()
            writer.writerows(csv_rows)
    atomic_json(out / "discovered_cases_for_interventions.json", intervention_cases)
    # Editable TikZ path strips; use \input inside a document with the listed packages.
    tikz = [r"% Preamble: \usepackage{tikz}\usetikzlibrary{arrows.meta}",
            r"% One strip per discovered top path. Coordinates and boxes are editable."]
    for rec in payload["results"]:
        for p in rec["top_paths"][:2]:
            hops = p["hops"]
            nodes = [hops[0]["head"]] + [h["tail"] for h in hops]
            spacing = 15.0 / max(1, len(hops))
            width = min(2.6, spacing - 0.30)
            tikz += [r"\begin{tikzpicture}[>=Stealth, every node/.style={font=\scriptsize}]",
                     r"\node[anchor=west,align=left,text width=16cm] at (-1.3,2.4) {" + tex_escape(rec["case"]["name"])
                     + f"; {p['name']}; $w={p['weight']:.4f}$" + "};"]
            for i, n in enumerate(nodes):
                label = rec["paper_titles"].get(n, n)
                tikz.append(r"\node[draw,rounded corners=2pt,align=center,text width=" + f"{width:.2f}cm,minimum height=1.0cm"
                            + f"] (n{i}) at ({i*spacing:.2f},0) " + "{" + tex_escape(label) + "};")
            for i, h in enumerate(hops):
                label = relation_text(h["rel"], tex=True) + rf"\\$g={h['g_applied']:.4f}$"
                tikz.append(rf"\draw[->] (n{i}) -- node[above,align=center,font=\tiny,text width={spacing:.2f}cm] "
                            + "{" + label + "}" + f" (n{i+1});")
            tikz += [r"\end{tikzpicture}", r"\par\medskip"]
    (out / "top_paths.tikz.tex").write_text("\n".join(tikz) + "\n")


## 5. Restore the embedding model

In [ ]:
# Restore Qwen3-Embedding from Drive, or download and cache it once.
if NEED_ENGINE:
    import os, shutil
    QDIR = "/content/qwen3"
    DRIVE_QWEN = f"{DRIVE}/qwen3-embedding-0.6b"
    BASE = "https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main"
    TOK = os.environ.get("HF_TOKEN", "")
    AUTH = f'-H "Authorization: Bearer {TOK}"' if TOK else ""
    NEED = ["model.safetensors","config.json","config_sentence_transformers.json","modules.json",
            "tokenizer.json","tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]

    def ready(d):
        return (all(os.path.exists(f"{d}/{f}") for f in NEED)
                and os.path.getsize(f"{d}/model.safetensors") > 1_000_000_000
                and os.path.getsize(f"{d}/tokenizer.json") > 11_000_000)

    if not ready(QDIR) and ready(DRIVE_QWEN):
        print("restoring Qwen3 from Drive cache ..."); shutil.copytree(DRIVE_QWEN, QDIR, dirs_exist_ok=True)

    if not ready(QDIR):
        os.makedirs(f"{QDIR}/1_Pooling", exist_ok=True)
        # 1) big weight via aria2c (run ONCE — a repeat can delete the finished file)
        if not (os.path.exists(f"{QDIR}/model.safetensors") and os.path.getsize(f"{QDIR}/model.safetensors") > 1_000_000_000):
            os.system("apt-get -qq install -y aria2")
            os.system(f'aria2c -x16 -s16 -k1M --max-tries=5 --retry-wait=2 --file-allocation=none '
                      f'{AUTH} -d {QDIR} -o model.safetensors "{BASE}/model.safetensors"')
        # 2) small files bypass Xet — plain curl
        for f in ["config.json","config_sentence_transformers.json","modules.json",
                  "tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]:
            os.system(f'curl -sSL -f {AUTH} "{BASE}/{f}" -o "{QDIR}/{f}"')
        # 3) tokenizer.json (11 MB, also Xet) — retry until a request lands on the good CDN
        for i in range(20):
            os.system(f"rm -f {QDIR}/tokenizer.json")
            os.system(f'curl -sSL -f {AUTH} "{BASE}/tokenizer.json" -o {QDIR}/tokenizer.json')
            if os.path.exists(f"{QDIR}/tokenizer.json") and os.path.getsize(f"{QDIR}/tokenizer.json") > 11_000_000:
                print(f"tokenizer.json ok on try {i+1}"); break
        assert ready(QDIR), "Qwen3 incomplete — re-run this cell (aria2c may need another pass)"
        os.makedirs(os.path.dirname(DRIVE_QWEN), exist_ok=True)
        shutil.copytree(QDIR, DRIVE_QWEN, dirs_exist_ok=True); print("cached Qwen3 to Drive")

    # load by LOCAL PATH — never by hub name again
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(QDIR)
    print("Qwen3 loaded offline:", _m.encode(["test"], normalize_embeddings=True).shape)  # (1, 1024)
    del _m
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 6. Select checkpoints and prepare graph-aligned inputs

In [ ]:
# Import the visible modules written above and patch the analysis hook.
import importlib.util

core_path = Path("/content/gfm-rag/gfmrag/workflow/ccmp_mechanism_core.py")
core_spec = importlib.util.spec_from_file_location("ccmp_mechanism_core", core_path)
_core = importlib.util.module_from_spec(core_spec)
core_spec.loader.exec_module(_core)
_core.patch_engine("/content/gfm-rag/gfmrag/models/ultra/models.py")

selection_path = Path("/content/gfm-rag/gfmrag/workflow/ccmp_checkpoint_selection.py")
selection_spec = importlib.util.spec_from_file_location("ccmp_checkpoint_selection", selection_path)
_selection = importlib.util.module_from_spec(selection_spec)
selection_spec.loader.exec_module(_selection)

CASES = [{'name': 'biology_optimal_transport',
  'dataset': 'sir4_biology',
  'query_id': '10.1021_acsomega.5c12723',
  'gold_id': '10.1007/s11263-023-01831-9',
  'source': 'results/qualitative/drive_scan_sir4_biology/hops_frame_ccmp.json',
  'paths': [{'name': 'P1',
             'hops': [{'layer': 0,
                       'head': '[limitation] sensitivity to class imbalance',
                       'rel': 'inverse_overcomes',
                       'tail': '[method] optimal transport-based method'},
                      {'layer': 1,
                       'head': '[method] optimal transport-based method',
                       'rel': 'inverse_contributes',
                       'tail': '10.1007/s11263-023-01831-9'}]},
            {'name': 'P2',
             'hops': [{'layer': 0,
                       'head': '[function] improve classification performance on imbalanced datasets',
                       'rel': 'inverse_achieves',
                       'tail': '[method] optimal transport-based method'},
                      {'layer': 1,
                       'head': '[method] optimal transport-based method',
                       'rel': 'inverse_contributes',
                       'tail': '10.1007/s11263-023-01831-9'}]}],
  'historical_reference': {'native': {'ranks': {'fused': 8, 'graph': 3, 'scorer': 13, 'dense': 113},
                                      'path_weights': {'P1': 12.4375, 'P2': 11.3125}},
                           'off': {'ranks': {'fused': 10, 'graph': 436, 'scorer': 13, 'dense': 113},
                                   'path_weights': {'P1': 9.28125, 'P2': 8.0625}}},
  'historical_reference_note': 'Old qualitative dumps only; never used as measurements or gate replay '
                               'inputs. Check numerical reproduction before reusing the old figure '
                               'percentages.'},
 {'name': 'creativity_fixation',
  'dataset': 'sir4_cs',
  'query_id': '10.48550_arxiv.2602.20408',
  'gold_id': '10.3758/bf03202751',
  'source': 'results/qualitative/drive_scan_sir4_cs/hops_frame_ccmp.json',
  'paths': [{'name': 'P1',
             'hops': [{'layer': 0,
                       'head': '[function] examine incubation effects on problem-solving improvement',
                       'rel': 'inverse_achieves',
                       'tail': '[method] fixation induction methodology'},
                      {'layer': 1,
                       'head': '[method] fixation induction methodology',
                       'rel': 'inverse_contributes',
                       'tail': '10.2307/1422851'},
                      {'layer': 2,
                       'head': '10.2307/1422851',
                       'rel': 'in_field',
                       'tail': '[domain] cognitive psychology'},
                      {'layer': 3,
                       'head': '[domain] cognitive psychology',
                       'rel': 'inverse_in_field',
                       'tail': '10.3758/bf03202751'}]},
            {'name': 'P2',
             'hops': [{'layer': 0,
                       'head': '[function] creative generation process',
                       'rel': 'inverse_concerns',
                       'tail': '[finding] enhanced focus and relevance in creative outputs'},
                      {'layer': 1,
                       'head': '[finding] enhanced focus and relevance in creative outputs',
                       'rel': 'inverse_reports',
                       'tail': '10.3758/bf03202751'}]}],
  'historical_reference': {'native': {'ranks': {'fused': 18, 'graph': 2, 'scorer': 49, 'dense': 169},
                                      'path_weights': {'P1': 11.075366973876953, 'P2': 3.734375}},
                           'off': {'ranks': {'fused': 25, 'graph': 4, 'scorer': 49, 'dense': 169},
                                   'path_weights': {'P1': 15.080484390258789, 'P2': 5.59375}}},
  'historical_reference_note': 'Old qualitative dumps only; never used as measurements or gate replay '
                               'inputs. Check numerical reproduction before reusing the old figure '
                               'percentages.'},
 {'name': 'memory_reconsolidation',
  'dataset': 'sir4_cs',
  'query_id': '10.48550_arxiv.2603.03985',
  'gold_id': '10.1111/j.1749-6632.2010.05443.x',
  'source': 'results/qualitative/drive_scan_sir4_cs/hops_frame_ccmp.json',
  'paths': [{'name': 'P1',
             'hops': [{'layer': 0,
                       'head': '[function] memory reconsolidation',
                       'rel': 'inverse_concerns',
                       'tail': '[finding] return to a transient unstable state'},
                      {'layer': 1,
                       'head': '[finding] return to a transient unstable state',
                       'rel': 'inverse_reports',
                       'tail': '10.1111/j.1749-6632.2010.05443.x'}]},
            {'name': 'P2',
             'hops': [{'layer': 0,
                       'head': '[function] memory recall dynamics',
                       'rel': 'inverse_concerns',
                       'tail': '[finding] predictable patterns in recall length and summarization'},
                      {'layer': 1,
                       'head': '[finding] predictable patterns in recall length and summarization',
                       'rel': 'inverse_reports',
                       'tail': '10.1103/g1cz-wk1l'},
                      {'layer': 2,
                       'head': '10.1103/g1cz-wk1l',
                       'rel': 'in_field',
                       'tail': '[domain] cognitive psychology'},
                      {'layer': 3,
                       'head': '[domain] cognitive psychology',
                       'rel': 'inverse_in_field',
                       'tail': '10.1111/j.1749-6632.2010.05443.x'}]}],
  'historical_reference': {'native': {'ranks': {'fused': 28, 'graph': 2, 'scorer': 93, 'dense': 380},
                                      'path_weights': {'P1': 12.515625, 'P2': 6.358192443847656}},
                           'off': {'ranks': {'fused': 37, 'graph': 3, 'scorer': 93, 'dense': 380},
                                   'path_weights': {'P1': 15.8125, 'P2': 14.0867919921875}}},
  'historical_reference_note': 'Old qualitative dumps only; never used as measurements or gate replay '
                               'inputs. Check numerical reproduction before reusing the old figure '
                               'percentages.'}]
assert all(any(case["dataset"] == dataset for case in CASES) for dataset in DATASETS)
print("Loaded", len(CASES), "question/target examples. The path notebook ignores their old paths.")


In [ ]:
"""Readable Colab setup helpers for the scientific-path and CCMP notebooks.

This file is inserted directly into a notebook code cell. It is deliberately
ordinary Python: no encoded source, exec, eval, or generated code strings.
"""
import copy
import fnmatch
from pathlib import Path

import numpy as np
import torch

import scigraphir_paths as cp


def copy_new(source, destination, pattern="*"):
    """Copy files that are missing or have a different size."""
    if not os.path.isdir(source):
        return 0
    copied = 0
    for root, _, files in os.walk(source):
        relative_root = os.path.relpath(root, source)
        for filename in files:
            if not fnmatch.fnmatch(filename, pattern):
                continue
            source_file = f"{root}/{filename}"
            destination_file = f"{destination}/{relative_root}/{filename}"
            if (os.path.exists(destination_file)
                    and os.path.getsize(destination_file) == os.path.getsize(source_file)):
                continue
            os.makedirs(os.path.dirname(destination_file), exist_ok=True)
            shutil.copy(source_file, destination_file)
            copied += 1
    return copied


def restore_index(graph_name):
    source = f"{CACHE}/index/{graph_name}"
    if not os.path.isdir(source):
        print(f"  {graph_name}: no cached index on Drive (built on first use)")
        return
    copied = sum(copy_new(f"{source}/{directory}",
                          f"{DATA_ROOT}/{graph_name}/processed/{directory}")
                 for directory in os.listdir(source))
    print(f"  {graph_name}: index restored ({copied} files)")


def save_index(graph_name):
    processed = f"{DATA_ROOT}/{graph_name}/processed"
    for directory in os.listdir(processed):
        if directory != "stage1":
            copy_new(f"{processed}/{directory}", f"{CACHE}/index/{graph_name}/{directory}")


def operator_components(graph_name):
    return f"{DATA_ROOT}/{graph_name}/operator_components{OP_SLUG}.npz"


def semantic_components(graph_name):
    return f"{DATA_ROOT}/{graph_name}/semantic_components{OP_SLUG}.npz"


def semantic_components_are_current(path):
    if not os.path.exists(path):
        return False
    with np.load(path, allow_pickle=True) as saved:
        return os.path.exists(str(saved["h_path"])) and "qwen" in str(saved["encoder"]).lower()


def set_dataset(dataset_name):
    """Switch every scoped cache/path to one SIR-4 field."""
    global DATASET, S, CACHE, QUERIES, env
    DATASET = dataset_name
    S = copy.deepcopy(SPEC[dataset_name])
    CACHE = f"{DRIVE}/outputs/{dataset_name}/cache"
    QUERIES = f"{DATA_ROOT}/{dataset_name}_test/raw/test.json"
    os.makedirs(CACHE, exist_ok=True)
    for key in list(os.environ):
        if key.startswith(("CCMP", "ROUTE", "STRAT_", "CQIG", "RESID_", "MISS_W",
                           "FUSION_", "SEED_")):
            os.environ.pop(key)
    os.environ["SCIGRAPHIR_DATASET"] = dataset_name
    env = dict(os.environ, SCIGRAPHIR_ROOT=SCIGRAPHIR_ROOT, SCIGRAPHIR_DATASET=dataset_name,
               PYTHONUNBUFFERED="1")
    cp.set_dataset(dataset_name)
    print(cp.banner())


def prepare_components(graph_names):
    """Restore/build graph-aligned operator and semantic tables."""
    global EMB_LOCAL, SEMF
    started = time.time()
    EMB_LOCAL = f"{SCIGRAPHIR_ROOT}/outputs/caches/op_emb"
    restored = copy_new(f"{CACHE}/op_emb", EMB_LOCAL, "test_*")
    print(f"embedding cache: {restored} test-split files restored ({time.time()-started:.0f}s)")

    SEMF = {}
    for key, (relative, scorer_dataset) in S["sem"].items():
        source = f"{DRIVE}/{relative}"
        if key == "field":
            location = f"{S4}/results/semantic_{scorer_dataset}"
            os.makedirs(location, exist_ok=True)
            copy_new(source, location)
        else:
            location = source
        SEMF[key] = (
            f"{location}/params_semantic_mlp_fixedloss_{scorer_dataset}.json",
            f"{location}/popnet_semantic_mlp_fixedloss_{scorer_dataset}.pt",
        )
        for filename in SEMF[key]:
            assert os.path.exists(filename), f"scorer file missing: {filename}"
        with open(SEMF[key][0]) as stream:
            jmax = json.load(stream)["jmax"]
        print(f"  scorer '{key}': jmax {jmax} ({relative})")

    for graph_name in graph_names:
        restore_index(graph_name)
        operator_file = operator_components(graph_name)
        if not os.path.exists(operator_file):
            cached = f"{CACHE}/{graph_name}_operator_components{OP_SLUG}.npz"
            if os.path.exists(cached):
                shutil.copy(cached, operator_file)
            else:
                command = [
                    sys.executable, "-u",
                    "precompute/precompute_operator_components.py",
                    "--dataset", DATASET, "--graph", graph_name,
                    "--split", "test", "--model", OP_MODEL,
                ]
                sh(command, KGDIR)
                shutil.copy(operator_file, cached)

        semantic_file = semantic_components(graph_name)
        if not semantic_components_are_current(semantic_file):
            command = [
                sys.executable, "-u",
                "precompute/precompute_semantic_components.py",
                "--dataset", DATASET, "--model", OP_MODEL,
                "--graph", graph_name, "--split", "test",
            ]
            sh(command, KGDIR)
        with np.load(semantic_file, allow_pickle=True) as saved:
            shape = tuple(int(value) for value in saved["h_shape"])
            jmax = int(saved["Jmax"])
        print(f"  {graph_name}: H {shape}, Jmax={jmax}")

    written = copy_new(EMB_LOCAL, f"{CACHE}/op_emb", "test_*")
    print(f"{written} new embedding files written back to Drive")


def inspect_checkpoint(checkpoint):
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)["model"]
    responsibility = [key for key in state if "resp_" in key]
    semantic = [key for key in state if key.startswith("sem_")]
    hidden = next((int(state[key].shape[0]) for key in responsibility
                   if key.endswith("resp_proj.0.weight")), None)
    jmax = next((int(state[key].shape[1]) - 2 for key in semantic
                 if key.endswith("sem_net.0.weight")), None)
    return {"tensors": len(state), "resp_keys": len(responsibility),
            "sem_keys": len(semantic), "ccmp_hid": hidden, "jmax": jmax}


def model_environment(checkpoint, graph_name, scorer_key, gate=True):
    """Construct the exact environment expected by FusionGraphReasoner."""
    info = inspect_checkpoint(checkpoint)
    semantic_checkpoint, semantic_popnet = SEMF[scorer_key]
    with open(semantic_checkpoint) as stream:
        scorer = json.load(stream)
    assert int(scorer["jmax"]) == info["jmax"], (
        f"scorer width mismatch: checkpoint jmax={info['jmax']} vs "
        f"scorer '{scorer_key}' jmax={scorer['jmax']}")
    values = {
        "WANDB_MODE": "disabled",
        "HYDRA_FULL_ERROR": "1",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "OPERATOR_COMPONENTS": operator_components(graph_name),
        "OPERATOR_COMPONENTS_TEST": operator_components(graph_name),
        "SEMANTIC_COMPONENTS": semantic_components(graph_name),
        "SEMANTIC_COMPONENTS_TEST": semantic_components(graph_name),
        "SEMANTIC_CKPT": semantic_checkpoint,
        "SEMANTIC_POPNET": semantic_popnet,
        "SEM_POP_LAMBDA": "1.0",
        "FUSION_OBJECTIVE": "hardneg",
        "HARDNEG_HUB": "50",
        "HARDNEG_RAND": "50",
        "AUX_W": "1.0",
        "PER_GOLD": "1",
        "HARDNEG_GRAPH": "50",
        "STRAT_TEST": QUERIES,
    }
    assert info["resp_keys"], "selected checkpoint has no CCMP responsibility head"
    values.update(CCMP="1", CCMP_HID=str(info["ccmp_hid"]),
                  CCMP_GATE="1" if gate else "0", CCMP_GATE_NORM="1", CCMP_ETA="0.5")
    arm_file = f"{os.path.dirname(checkpoint)}/arm.json"
    if os.path.exists(arm_file):
        with open(arm_file) as stream:
            arm = json.load(stream)
        if arm.get("ccmp_eta") is not None:
            values["CCMP_ETA"] = str(arm["ccmp_eta"])
    return values, info


def hydra_common(graph_name):
    return [
        "--config-path", "config/gfm_reasoner",
        "--config-name", "sft_training_fusion",
        "text_emb_model=qwen3_st",
        f"datasets.cfgs.root={DATA_ROOT}",
        "datasets.cfgs.force_reload=False",
        f"datasets.train_names=[{graph_name}]",
        f"datasets.valid_names=[{graph_name}]",
        "model.semantic=mlp",
        "model.cqig=false",
    ]


## 7. Run fixed-path interventions

In [ ]:
# Run all seven conditions plus repeat/replay checks, per fixed case and precision.
import copy
from pathlib import Path
from IPython.display import display, Markdown


RESULTS = []
for d in DATASETS:
    set_dataset(d)
    selection = _selection.select_checkpoint(d, S, DRIVE, DATA_ROOT, prefer_merged=PREFER_MERGED)
    print(f"[checkpoint] {selection['reason']}")
    print(f"[checkpoint] {selection['checkpoint']}\n[graph] {selection['graph']}\n[scorer] {selection['scorer_key']}")
    ckpt, graph, skey = selection["checkpoint"], selection["graph"], selection["scorer_key"]
    GRAPHS = [graph]
    S["sem"] = selection["semantic_specs"]
    selection_file = f"{RUNS}/ccmp_mechanism_selection_{d}.json"
    with open(selection_file, "w") as f:
        json.dump(selection, f, indent=2)
    prepare_components(GRAPHS)
    model_environment_values, info = model_environment(ckpt, graph, skey, gate=True)
    assert info["resp_keys"] and model_environment_values["CCMP_GATE"] == "1"
    selected_cases = copy.deepcopy([c for c in CASES if c["dataset"] == d])
    for case in selected_cases:
        # Original figure measurements came from the frame checkpoint/graph.
        # Keep them as provenance, but do not call merged results a reproduction failure.
        case["historical_reference_applicable"] = selection["family"] == "frame" and not case.get("selection_source")
    case_file = f"{RUNS}/ccmp_mechanism_cases_{d}.json"
    with open(case_file, "w") as f:
        json.dump(selected_cases, f, indent=2)
    for precision in PRECISIONS:
        output_root = f"{DRIVE}/outputs/ccmp_mechanism/{d}/{selection['family']}/{precision}/{PROTECTION}"
        run_dir = f"{RUNS}/ccmp_mechanism/{d}/{selection['family']}/{precision}/{PROTECTION}"
        os.makedirs(run_dir, exist_ok=True)
        command = [sys.executable, "-u", "-m", "gfmrag.workflow.ccmp_mechanism"]
        command += hydra_common(graph)
        command += [f"model.dtype={precision}", "datasets.data_loading_workers=0",
                    f"+mechanism.ckpt={ckpt}", f"+mechanism.cases={case_file}",
                    f"+mechanism.selection={selection_file}",
                    f"+mechanism.out={output_root}", f"+mechanism.protection={PROTECTION}",
                    f"+mechanism.resume={str(RESUME).lower()}",
                    f"+mechanism.atol={ATOL}", f"+mechanism.rtol={RTOL}",
                    f"hydra.run.dir={run_dir}"]
        sh(command, "/content/gfm-rag", extra=model_environment_values, log=f"{run_dir}/console.log")
        latest = json.load(open(f"{output_root}/latest.json"))
        assert latest["status"] == "complete" and latest["version"] == _core.VERSION
        result = json.load(open(latest["results"]))
        assert result["run_id"] == latest["run_id"] and result["manifest"]["cases"] == selected_cases
        assert result["manifest"]["precision"] == precision and result["status"] == "complete"
        assert result["manifest"]["selection"] == selection
        assert len(result["results"]) == len(selected_cases)
        RESULTS.append((d, precision, latest["results"], result))
        display(Markdown(Path(latest["results"]).with_name("report.md").read_text()))
        print("Results, CSV, full gate tensors and TikZ values:", str(Path(latest["results"]).parent))
    save_index(graph)



## 8. Compare precision and inspect evidence

In [ ]:
# Combined precision comparison
import pandas as pd

effect_rows = []
for dataset, precision, path, payload in RESULTS:
    for result in payload["results"]:
        for effect in result["effects"]:
            effect_rows.append({"dataset": dataset, "precision": precision, "case": result["case"]["name"],
                                "graph": payload["manifest"]["selection"]["graph"],
                                "checkpoint_family": payload["manifest"]["selection"]["family"],
                                "path": effect["path"], "native_delta_w": effect["native_minus_off"],
                                "gate_derivative_delta_w": effect["gate_derivative_effect"],
                                "outside_suppression_delta_w": effect["outside_suppression_effect"],
                                "suppression_result": effect["outside_suppression_verdict"],
                                "results_file": path})
effects = pd.DataFrame(effect_rows)
display(effects.drop(columns="results_file"))
for (case, path), group in effects.groupby(["case", "path"]):
    if len(group) > 1 and group.suppression_result.nunique() > 1:
        print(f"PRECISION-SENSITIVE: {case} / {path}; inspect both runs before making a figure claim.")
print("paths.csv contains weights; hops.csv has g_raw, g_applied and every edge gradient.")
print("The numerical screen is not a statistical test; these cases cannot establish a dataset-wide effect.")


### Reading the result

- `outside_suppress − off > 0` supports the competing-message suppression explanation for that path and example.
- `native − frozen` measures the contribution of differentiating through the gates while replaying identical applied gates.
- `path_only − off` measures the effect of retaining native gates only on the selected path nodes.
- A larger path weight does not imply a better retrieval rank.

Frozen replay requires bit-identical raw/applied gates and identical target ranks. CUDA sparse reductions can vary between equivalent bfloat16 passes, so score checks record repeat variation and an explicit dtype-and-score-scale allowance. Float32 retains the tight configured tolerance. These are numerical checks, not confidence intervals.
